In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import random
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Final, Mapping

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
from IPython.display import display


# ============================================================
# Runtime contract
# ============================================================

MINIMUM_PYTHON_VERSION: Final[tuple[int, int]] = (3, 10)

if sys.version_info < MINIMUM_PYTHON_VERSION:
    required_version = ".".join(map(str, MINIMUM_PYTHON_VERSION))
    observed_version = platform.python_version()

    raise RuntimeError(
        f"Python {required_version} or newer is required; "
        f"observed Python {observed_version}."
    )

warnings.filterwarnings(
    "error",
    category=pd.errors.SettingWithCopyWarning,
)

pd.options.display.max_columns = 200
pd.options.display.width = 180
pd.options.display.float_format = "{:,.10g}".format


# ============================================================
# Frozen project identity
# ============================================================

NOTEBOOK_NAME: Final[str] = "08_HAWKES_DIAGNOSTICS.ipynb"
NOTEBOOK_STAGE: Final[str] = "HAWKES_DIAGNOSTICS"

SOURCE_RUN_PREFIX: Final[str] = (
    "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
)
V01_RUN_ID: Final[str] = "v0_1_20260714T090616Z_e82325081a81"
COMBINED_OUTPUT_PREFIX: Final[str] = (
    f"{SOURCE_RUN_PREFIX}__{V01_RUN_ID}"
)

SOURCE_SET_SHA256: Final[str] = (
    "132c83531eec615d279408b5c06f402973114ba3058dfadd2fe58e2e67184c4b"
)
RUN_CONFIG_SHA256: Final[str] = (
    "14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a770bee995f688d59617"
)
RUN_IDENTITY_SHA256: Final[str] = (
    "5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9df19204c42e37fe4198"
)


# ============================================================
# Frozen event-stream authority
# ============================================================

PRIMARY_EVENT_REPRESENTATION: Final[str] = (
    "SAME_MS_SAME_SIDE_BURSTS"
)
PRIMARY_EVENT_TIME_COLUMN: Final[str] = "event_time_ns"
PRIMARY_EVENT_PARTITION_COLUMN: Final[str] = "event_partition"
PRIMARY_TIMESTAMP_INTERFACE: Final[str] = (
    "SIMULTANEOUS_EVENT_BATCH_REQUIRED"
)
PRIMARY_SCORING_RULE: Final[str] = "STRICT_PRE_BATCH"

EVENT_SIDES: Final[tuple[str, str]] = ("BUY", "SELL")

DIAGNOSTIC_PARTITIONS: Final[tuple[str, str]] = (
    "DEVELOPMENT",
    "CALIBRATION",
)
LOCKED_EVALUATION_PARTITION: Final[str] = "CALIBRATION"
PROTECTED_PARTITIONS: Final[tuple[str, str]] = (
    "VALIDATION",
    "ENGINEERING_HOLDOUT",
)

CALIBRATION_PARAMETER_UPDATES: Final[int] = 0
WITHIN_BATCH_ZERO_LAG_EXCITATION_ALLOWED: Final[bool] = False
TIMESTAMP_JITTER_ALLOWED: Final[bool] = False
EVENT_REORDERING_ALLOWED: Final[bool] = False
FABRICATED_PREHISTORY_ALLOWED: Final[bool] = False

EXPECTED_PRIMARY_EVENT_ROWS: Final[int] = 13_887
EXPECTED_PRIMARY_SCORING_BATCHES: Final[int] = 13_564

EXPECTED_DEVELOPMENT_EVENT_ROWS: Final[int] = 7_004
EXPECTED_DEVELOPMENT_BATCH_ROWS: Final[int] = 6_859

EXPECTED_CALIBRATION_EVENT_ROWS: Final[int] = 2_493
EXPECTED_CALIBRATION_BATCH_ROWS: Final[int] = 2_400


# ============================================================
# Frozen Notebook 07 model authority
# ============================================================

EXPECTED_NOTEBOOK07_TERMINAL_STATUS: Final[str] = (
    "PASS_HAWKES_ESTIMATION_RESTRICTED_FIRST"
)
EXPECTED_SELECTED_MODEL_ID: Final[str] = (
    "H1_DIAGONAL_SHARED_DECAY"
)
EXPECTED_SELECTED_MODEL_FAMILY: Final[str] = (
    "DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES"
)
EXPECTED_ESTIMATOR_LABEL: Final[str] = (
    "STRICT_PRE_BATCH_COARSENED_TIME_QUASI_MLE"
)
EXPECTED_KERNEL_FAMILY: Final[str] = (
    "SINGLE_EXPONENTIAL_PER_AUTHORIZED_CHANNEL"
)
EXPECTED_SELECTED_SPECTRAL_RADIUS: Final[float] = 0.1892556591

AUTHORIZED_CHANNELS: Final[tuple[str, str]] = (
    "BUY_TO_BUY",
    "SELL_TO_SELL",
)
FIXED_ZERO_CHANNELS: Final[tuple[str, str]] = (
    "BUY_TO_SELL",
    "SELL_TO_BUY",
)

MODEL_REPLACEMENT_ALLOWED: Final[bool] = False
HAWKES_REFIT_ALLOWED: Final[bool] = False
CROSS_EXCITATION_FIT_ALLOWED: Final[bool] = False
STATE_DEPENDENT_HAWKES_ALLOWED: Final[bool] = False


# ============================================================
# Frozen Notebook 06 comparison authority
# ============================================================

FORMAL_COUNT_COMPARATOR: Final[str] = (
    "D0_SIDE_CONSTANT_POISSON"
)
SELECTED_ROLLING_BASELINE: Final[str] = (
    "R_SIDE_ROLLING_250MS_POISSON"
)
PRIMARY_SIMPLE_COUNT_BASELINE: Final[str] = (
    "E_SIDE_EWMA_250MS_POISSON"
)
SELECTED_RENEWAL_FAMILY: Final[str] = "GAMMA"


# ============================================================
# Project paths
# ============================================================

PROJECT_ROOT: Final[Path] = Path(r"D:\Clown Project")
V00_ROOT: Final[Path] = PROJECT_ROOT / "V0.0"
V01_ROOT: Final[Path] = PROJECT_ROOT / "V0.1"

V01_DATA_ROOT: Final[Path] = V01_ROOT / "data"
V01_PROCESSED_ROOT: Final[Path] = V01_DATA_ROOT / "processed"
V01_EVENT_ROOT: Final[Path] = V01_PROCESSED_ROOT / "events"
V01_POINT_PROCESS_ROOT: Final[Path] = (
    V01_PROCESSED_ROOT / "point_process"
)

V01_ARTIFACT_ROOT: Final[Path] = V01_ROOT / "artifacts"
V01_MANIFEST_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "manifests"
V01_AUDIT_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "audit_tables"
V01_MODEL_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "models"
V01_DIAGNOSTIC_ROOT: Final[Path] = (
    V01_ARTIFACT_ROOT / "diagnostics"
)
V01_RECONCILIATION_ROOT: Final[Path] = (
    V01_ARTIFACT_ROOT / "reconciliation"
)
V01_HANDOFF_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "handoff"
V01_FIGURE_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "figures"

NOTEBOOK08_AUDIT_ROOT: Final[Path] = (
    V01_AUDIT_ROOT / "08_hawkes_diagnostics"
)
NOTEBOOK08_DIAGNOSTIC_ROOT: Final[Path] = (
    V01_DIAGNOSTIC_ROOT / "08_hawkes_diagnostics"
)
NOTEBOOK08_RECONCILIATION_ROOT: Final[Path] = (
    V01_RECONCILIATION_ROOT / "08_hawkes_diagnostics"
)
NOTEBOOK08_FIGURE_ROOT: Final[Path] = (
    V01_FIGURE_ROOT / "08_hawkes_diagnostics"
)

NOTEBOOK07_TERMINAL_DECISION_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / f"{COMBINED_OUTPUT_PREFIX}__07_terminal_decision.json"
)
NOTEBOOK07_PERSISTENCE_INDEX_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / f"{COMBINED_OUTPUT_PREFIX}__07_hawkes_persistence_index.json"
)
NOTEBOOK07_READBACK_AUDIT_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / f"{COMBINED_OUTPUT_PREFIX}__07_semantic_readback_audit.json"
)
NOTEBOOK07_FINAL_ACCEPTANCE_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / f"{COMBINED_OUTPUT_PREFIX}__07_hawkes_estimation_final_acceptance.json"
)
NOTEBOOK07_OUTPUT_MANIFEST_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / f"{COMBINED_OUTPUT_PREFIX}__07_hawkes_estimation_output_manifest.json"
)
NOTEBOOK07_TO_08_HANDOFF_PATH: Final[Path] = (
    V01_HANDOFF_ROOT
    / f"{COMBINED_OUTPUT_PREFIX}__07_to_08_hawkes_diagnostics_handoff.json"
)

NOTEBOOK06_TERMINAL_DECISION_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_terminal_decision.json"
)
NOTEBOOK06_FINAL_ACCEPTANCE_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_final_acceptance.json"
)
NOTEBOOK06_OUTPUT_MANIFEST_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_output_manifest.json"
)
NOTEBOOK06_READBACK_AUDIT_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_readback_audit.json"
)

# This cell performs no filesystem reads or writes.
# Artifact loading and output-directory creation remain blocked until
# the upstream identity, checksum, and protected-partition gates pass.


# ============================================================
# Time and diagnostic contracts
# ============================================================

NANOSECONDS_PER_MILLISECOND: Final[int] = 1_000_000
NANOSECONDS_PER_SECOND: Final[int] = 1_000_000_000

EVENT_TIME_STORAGE_UNIT: Final[str] = "NANOSECONDS"
MODEL_TIME_UNIT: Final[str] = "SECONDS"

COUNT_GRID_WIDTH_NS: Final[int] = NANOSECONDS_PER_MILLISECOND
COUNT_GRID_WIDTH_SECONDS: Final[float] = (
    COUNT_GRID_WIDTH_NS / NANOSECONDS_PER_SECOND
)

COMPENSATOR_ENGINE: Final[str] = (
    "EXACT_PIECEWISE_EXPONENTIAL_INTEGRATION"
)
PRIMARY_ADEQUACY_REFERENCE: Final[str] = (
    "COARSENING_AWARE_PARAMETRIC_SIMULATION_ENVELOPES"
)
RAW_P_VALUE_DECISION_ROLE: Final[str] = "REPORT_ONLY"

RESIDUAL_ACF_MAX_LAG: Final[int] = 50
LJUNG_BOX_LAGS: Final[tuple[int, int, int]] = (10, 20, 50)
CROSS_CORRELATION_MAX_LAG_MS: Final[int] = 30

BLOCK_BOOTSTRAP_SECONDS: Final[tuple[int, int, int]] = (
    5,
    10,
    30,
)
N_BLOCK_BOOTSTRAP_REPLICATES: Final[int] = 2_000
N_MODEL_ENVELOPE_REPLICATES: Final[int] = 1_000
BOOTSTRAP_CONFIDENCE_LEVEL: Final[float] = 0.95

FLOAT_ABSOLUTE_TOLERANCE: Final[float] = 1e-10
FLOAT_RELATIVE_TOLERANCE: Final[float] = 1e-8
SPECTRAL_RADIUS_ABSOLUTE_TOLERANCE: Final[float] = 1e-10
INTENSITY_FLOOR_PER_SECOND: Final[float] = 1e-12


# ============================================================
# Deterministic execution
# ============================================================

RANDOM_SEED: Final[int] = 20260720

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
RNG = np.random.default_rng(RANDOM_SEED)


# ============================================================
# Permitted terminal statuses
# ============================================================

PERMITTED_TERMINAL_STATUSES: Final[tuple[str, ...]] = (
    "PASS_HAWKES_DIAGNOSTICS_RESTRICTED_H1",
    "CONDITIONAL_PASS_H1_RESIDUAL_MISSPECIFICATION",
    "CONDITIONAL_PASS_CROSS_SIDE_EXTENSION_RECOMMENDED",
    "FAIL_HAWKES_NO_INCREMENTAL_VALUE",
    "FAIL_DIAGNOSTIC_PIPELINE_INVALID",
)


# ============================================================
# Immutable notebook configuration
# ============================================================

@dataclass(frozen=True, slots=True)
class Notebook08Config:
    """Immutable configuration for the frozen Hawkes diagnostic run."""

    notebook_name: str
    notebook_stage: str
    source_run_prefix: str
    v01_run_id: str
    random_seed: int
    event_representation: str
    event_time_column: str
    event_partition_column: str
    timestamp_interface: str
    scoring_rule: str
    diagnostic_partitions: tuple[str, ...]
    locked_evaluation_partition: str
    protected_partitions: tuple[str, ...]
    selected_model_id: str
    selected_model_family: str
    estimator_label: str
    authorized_channels: tuple[str, ...]
    fixed_zero_channels: tuple[str, ...]
    primary_count_baseline: str
    compensator_engine: str
    primary_adequacy_reference: str
    residual_acf_max_lag: int
    ljung_box_lags: tuple[int, ...]
    cross_correlation_max_lag_ms: int
    block_bootstrap_seconds: tuple[int, ...]
    block_bootstrap_replicates: int
    model_envelope_replicates: int
    bootstrap_confidence_level: float
    calibration_parameter_updates: int


NOTEBOOK_CONFIG = Notebook08Config(
    notebook_name=NOTEBOOK_NAME,
    notebook_stage=NOTEBOOK_STAGE,
    source_run_prefix=SOURCE_RUN_PREFIX,
    v01_run_id=V01_RUN_ID,
    random_seed=RANDOM_SEED,
    event_representation=PRIMARY_EVENT_REPRESENTATION,
    event_time_column=PRIMARY_EVENT_TIME_COLUMN,
    event_partition_column=PRIMARY_EVENT_PARTITION_COLUMN,
    timestamp_interface=PRIMARY_TIMESTAMP_INTERFACE,
    scoring_rule=PRIMARY_SCORING_RULE,
    diagnostic_partitions=DIAGNOSTIC_PARTITIONS,
    locked_evaluation_partition=LOCKED_EVALUATION_PARTITION,
    protected_partitions=PROTECTED_PARTITIONS,
    selected_model_id=EXPECTED_SELECTED_MODEL_ID,
    selected_model_family=EXPECTED_SELECTED_MODEL_FAMILY,
    estimator_label=EXPECTED_ESTIMATOR_LABEL,
    authorized_channels=AUTHORIZED_CHANNELS,
    fixed_zero_channels=FIXED_ZERO_CHANNELS,
    primary_count_baseline=PRIMARY_SIMPLE_COUNT_BASELINE,
    compensator_engine=COMPENSATOR_ENGINE,
    primary_adequacy_reference=PRIMARY_ADEQUACY_REFERENCE,
    residual_acf_max_lag=RESIDUAL_ACF_MAX_LAG,
    ljung_box_lags=LJUNG_BOX_LAGS,
    cross_correlation_max_lag_ms=CROSS_CORRELATION_MAX_LAG_MS,
    block_bootstrap_seconds=BLOCK_BOOTSTRAP_SECONDS,
    block_bootstrap_replicates=N_BLOCK_BOOTSTRAP_REPLICATES,
    model_envelope_replicates=N_MODEL_ENVELOPE_REPLICATES,
    bootstrap_confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    calibration_parameter_updates=CALIBRATION_PARAMETER_UPDATES,
)


def canonical_json_sha256(
    payload: Mapping[str, Any],
) -> str:
    """Return the SHA-256 digest of a canonical JSON mapping."""
    canonical_payload = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(canonical_payload).hexdigest()


NOTEBOOK_CONFIG_SHA256: Final[str] = canonical_json_sha256(
    asdict(NOTEBOOK_CONFIG)
)


# ============================================================
# Clean-kernel state sentinels
# ============================================================

UPSTREAM_AUTHORITY_VERIFIED = False
FROZEN_MODEL_LOADED = False
FROZEN_REPLAY_RECONCILED = False
PROTECTED_PARTITION_CONTENT_LOADED = False
FILESYSTEM_WRITES_PERFORMED = False


# ============================================================
# Runtime provenance and initialization summary
# ============================================================

runtime_provenance = pd.DataFrame(
    [
        {
            "component": "python",
            "version": platform.python_version(),
        },
        {
            "component": "numpy",
            "version": np.__version__,
        },
        {
            "component": "pandas",
            "version": pd.__version__,
        },
        {
            "component": "scipy",
            "version": scipy.__version__,
        },
        {
            "component": "matplotlib",
            "version": matplotlib.__version__,
        },
        {
            "component": "platform",
            "version": platform.platform(),
        },
    ]
)

setup_summary = pd.DataFrame(
    [
        {
            "field": "notebook",
            "value": NOTEBOOK_NAME,
        },
        {
            "field": "source_run_prefix",
            "value": SOURCE_RUN_PREFIX,
        },
        {
            "field": "v01_run_id",
            "value": V01_RUN_ID,
        },
        {
            "field": "selected_model_id",
            "value": EXPECTED_SELECTED_MODEL_ID,
        },
        {
            "field": "locked_evaluation_partition",
            "value": LOCKED_EVALUATION_PARTITION,
        },
        {
            "field": "primary_simple_count_baseline",
            "value": PRIMARY_SIMPLE_COUNT_BASELINE,
        },
        {
            "field": "compensator_engine",
            "value": COMPENSATOR_ENGINE,
        },
        {
            "field": "primary_adequacy_reference",
            "value": PRIMARY_ADEQUACY_REFERENCE,
        },
        {
            "field": "calibration_parameter_updates",
            "value": CALIBRATION_PARAMETER_UPDATES,
        },
        {
            "field": "protected_partitions",
            "value": ", ".join(PROTECTED_PARTITIONS),
        },
        {
            "field": "random_seed",
            "value": RANDOM_SEED,
        },
        {
            "field": "notebook_config_sha256",
            "value": NOTEBOOK_CONFIG_SHA256,
        },
        {
            "field": "upstream_authority_verified",
            "value": UPSTREAM_AUTHORITY_VERIFIED,
        },
        {
            "field": "protected_partition_content_loaded",
            "value": PROTECTED_PARTITION_CONTENT_LOADED,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(setup_summary)
display(runtime_provenance)

print(
    "Notebook 08 runtime and immutable diagnostic configuration initialized. "
    "No upstream artifacts, analytical event data, protected partition content, "
    "or V0.0 references have been loaded. No filesystem writes were performed."
)

,field,value
0,notebook,08_HAWKES_DIAGNOSTICS.ipynb
1,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
2,v01_run_id,v0_1_20260714T090616Z_e82325081a81
3,selected_model_id,H1_DIAGONAL_SHARED_DECAY
4,locked_evaluation_partition,CALIBRATION
5,primary_simple_count_baseline,E_SIDE_EWMA_250MS_POISSON
6,compensator_engine,EXACT_PIECEWISE_EXPONENTIAL_INTEGRATION
7,primary_adequacy_reference,COARSENING_AWARE_PARAMETRIC_SIMULATION_ENVELOPES
8,calibration_parameter_updates,0
9,protected_partitions,"VALIDATION, ENGINEERING_HOLDOUT"


,component,version
0,python,3.11.9
1,numpy,2.2.0
2,pandas,2.3.2
3,scipy,1.16.1
4,matplotlib,3.10.6
5,platform,Windows-10-10.0.26200-SP0


Notebook 08 runtime and immutable diagnostic configuration initialized. No upstream artifacts, analytical event data, protected partition content, or V0.0 references have been loaded. No filesystem writes were performed.


In [4]:
# ============================================================
# Verify Notebook 06 authority and Notebook 07-to-08 handoff
# ============================================================

import os
import re
from pathlib import Path
from typing import Any, Final, Mapping


SHA256_RE: Final[re.Pattern[str]] = re.compile(r"^[0-9a-f]{64}$")

NB06_HANDOFF_PATH: Final[Path] = (
    V01_HANDOFF_ROOT
    / "06_point_process_baselines_to_07_hawkes_estimation_handoff.json"
)

NB06_PATHS: Final[dict[str, Path]] = {
    "terminal_decision": NOTEBOOK06_TERMINAL_DECISION_PATH,
    "final_acceptance": NOTEBOOK06_FINAL_ACCEPTANCE_PATH,
    "output_manifest": NOTEBOOK06_OUTPUT_MANIFEST_PATH,
    "readback_audit": NOTEBOOK06_READBACK_AUDIT_PATH,
}

NB06_SCHEMAS: Final[dict[str, tuple[str, str]]] = {
    "terminal_decision": (
        "NOTEBOOK_06_TERMINAL_DECISION",
        "NOTEBOOK_06_TERMINAL_DECISION_V1",
    ),
    "final_acceptance": (
        "NOTEBOOK_06_FINAL_ACCEPTANCE",
        "NOTEBOOK_06_FINAL_ACCEPTANCE_V1",
    ),
    "output_manifest": (
        "NOTEBOOK_06_OUTPUT_MANIFEST",
        "NOTEBOOK_06_OUTPUT_MANIFEST_V1",
    ),
    "readback_audit": (
        "NOTEBOOK_06_READBACK_AUDIT",
        "NOTEBOOK_06_READBACK_AUDIT_V1",
    ),
}

NB07_PATHS: Final[dict[str, Path]] = {
    "terminal_decision": NOTEBOOK07_TERMINAL_DECISION_PATH,
    "final_acceptance": NOTEBOOK07_FINAL_ACCEPTANCE_PATH,
    "output_manifest": NOTEBOOK07_OUTPUT_MANIFEST_PATH,
    "persistence_index": NOTEBOOK07_PERSISTENCE_INDEX_PATH,
    "readback_audit": NOTEBOOK07_READBACK_AUDIT_PATH,
    "handoff": NOTEBOOK07_TO_08_HANDOFF_PATH,
}

NB07_SCHEMAS: Final[dict[str, tuple[str, str]]] = {
    "terminal_decision": (
        "NOTEBOOK_07_TERMINAL_DECISION",
        "NOTEBOOK_07_TERMINAL_DECISION_V1",
    ),
    "final_acceptance": (
        "NOTEBOOK_07_HAWKES_ESTIMATION_FINAL_ACCEPTANCE",
        "NOTEBOOK_07_HAWKES_ESTIMATION_FINAL_ACCEPTANCE_V1",
    ),
    "output_manifest": (
        "NOTEBOOK_07_HAWKES_ESTIMATION_OUTPUT_MANIFEST",
        "NOTEBOOK_07_HAWKES_ESTIMATION_OUTPUT_MANIFEST_V1",
    ),
    "persistence_index": (
        "NOTEBOOK_07_HAWKES_PERSISTENCE_INDEX",
        "NOTEBOOK_07_HAWKES_PERSISTENCE_INDEX_V1",
    ),
    "readback_audit": (
        "NOTEBOOK_07_HAWKES_SEMANTIC_READBACK_AUDIT",
        "NOTEBOOK_07_HAWKES_SEMANTIC_READBACK_AUDIT_V1",
    ),
    "handoff": (
        "NOTEBOOK_07_TO_NOTEBOOK_08_HAWKES_HANDOFF",
        "NOTEBOOK_07_TO_NOTEBOOK_08_HAWKES_HANDOFF_V1",
    ),
}


def require(condition: bool, message: str) -> None:
    """Raise when a blocking contract condition fails."""
    if not condition:
        raise RuntimeError(message)


def file_sha256(path: Path) -> str:
    """Return a file SHA-256 without loading the whole file."""
    require(path.is_file(), f"Missing file: {path}")

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    """Load a JSON artifact and require a top-level object."""
    require(
        path.is_file(),
        f"Missing JSON artifact: {path}",
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        value = json.load(handle)

    require(
        isinstance(value, dict),
        f"Top-level JSON must be an object: {path}",
    )

    return value


def verify_self_hash(
    path: Path,
) -> tuple[dict[str, Any], str, str]:
    """Verify a flat JSON document containing payload_sha256."""
    document = load_json(path)
    registered_hash = document.get("payload_sha256")

    require(
        isinstance(registered_hash, str)
        and bool(SHA256_RE.fullmatch(registered_hash)),
        f"Invalid payload_sha256: {path}",
    )

    payload_without_hash = {
        key: value
        for key, value in document.items()
        if key != "payload_sha256"
    }

    observed_semantic_hash = canonical_json_sha256(
        payload_without_hash
    )

    require(
        observed_semantic_hash == registered_hash,
        f"Self-hash mismatch: {path}",
    )

    return (
        document,
        observed_semantic_hash,
        file_sha256(path),
    )


def expect(
    document: Mapping[str, Any],
    label: str,
    **fields: Any,
) -> None:
    """Require exact values for selected document fields."""
    mismatches = {
        field_name: {
            "expected": expected_value,
            "observed": document.get(field_name),
        }
        for field_name, expected_value in fields.items()
        if document.get(field_name) != expected_value
    }

    require(
        not mismatches,
        f"{label} contract mismatch: {mismatches}",
    )


def normalized_absolute(path: Path) -> str:
    """Return a normalized absolute path."""
    return os.path.normcase(
        os.path.abspath(
            os.fspath(path)
        )
    )


def inside(
    path: Path,
    root: Path,
) -> bool:
    """Return whether a path is contained by a root."""
    try:
        return os.path.commonpath(
            [
                normalized_absolute(path),
                normalized_absolute(root),
            ]
        ) == normalized_absolute(root)
    except ValueError:
        return False


def relative_key(
    path: str | Path,
) -> str:
    """Normalize a relative path for manifest lookup."""
    return (
        str(path)
        .replace("\\", "/")
        .lstrip("./")
        .casefold()
    )


def resolve_v01(
    path_text: str | Path,
) -> Path:
    """Resolve an absolute or V0.1-relative path."""
    path = Path(path_text)

    if path.is_absolute():
        return path

    return V01_ROOT / path


# ------------------------------------------------------------
# Guards and declared-path verification
# ------------------------------------------------------------

require(
    not UPSTREAM_AUTHORITY_VERIFIED,
    "Upstream authority is already verified.",
)

require(
    not PROTECTED_PARTITION_CONTENT_LOADED,
    "Protected partition content is already open.",
)

require(
    not FILESYSTEM_WRITES_PERFORMED,
    "A filesystem write occurred before verification.",
)

for artifact_path in (
    NB06_HANDOFF_PATH,
    *NB06_PATHS.values(),
    *NB07_PATHS.values(),
):
    require(
        inside(
            artifact_path,
            V01_ROOT,
        ),
        f"Artifact is outside V0.1: {artifact_path}",
    )

    require(
        not inside(
            artifact_path,
            V00_ROOT,
        ),
        f"Artifact resolves inside V0.0: {artifact_path}",
    )

    require(
        artifact_path.is_file(),
        f"Required artifact is missing: {artifact_path}",
    )


# ------------------------------------------------------------
# Notebook 06 handoff and self-hashed control artifacts
# ------------------------------------------------------------

(
    nb06_handoff,
    nb06_handoff_semantic_sha256,
    nb06_handoff_file_sha256,
) = verify_self_hash(
    NB06_HANDOFF_PATH
)

expect(
    nb06_handoff,
    "Notebook 06-to-07 handoff",
    artifact_type=(
        "NOTEBOOK_06_TO_NOTEBOOK_07_HANDOFF"
    ),
    schema_version=(
        "NOTEBOOK_06_TO_NOTEBOOK_07_HANDOFF_V1"
    ),
    producer="06_POINT_PROCESS_BASELINES.ipynb",
    consumer="07_HAWKES_ESTIMATION.ipynb",
    source_run_prefix=SOURCE_RUN_PREFIX,
    source_set_sha256=SOURCE_SET_SHA256,
    v0_1_run_id=V01_RUN_ID,
    run_config_sha256=RUN_CONFIG_SHA256,
    run_identity_sha256=RUN_IDENTITY_SHA256,
    status="PASS_HAWKES_RESTRICTED_FIRST",
)

nb06_authorization = nb06_handoff.get(
    "authorization"
)

require(
    isinstance(
        nb06_authorization,
        dict,
    ),
    "Notebook 06 handoff lacks authorization.",
)

expect(
    nb06_authorization,
    "Notebook 06 authorization",
    authorization_state=(
        "AUTHORIZED_RESTRICTED_FIRST"
    ),
    terminal_status=(
        "PASS_HAWKES_RESTRICTED_FIRST"
    ),
    notebook_07_authorized=True,
    restricted_first_required=True,
    state_dependent_hawkes_authorized=False,
    hawkes_superiority_claim_authorized=False,
    strategy_or_quoting_use_authorized=False,
)

nb06_control_registry = nb06_handoff.get(
    "control_artifacts"
)

require(
    isinstance(
        nb06_control_registry,
        dict,
    ),
    "Notebook 06 handoff lacks control_artifacts.",
)

artifact_rows: list[dict[str, Any]] = [
    {
        "artifact_role": "nb06_to_07_handoff",
        "path": str(
            NB06_HANDOFF_PATH
        ),
        "file_sha256": (
            nb06_handoff_file_sha256
        ),
        "semantic_sha256": (
            nb06_handoff_semantic_sha256
        ),
        "passed": True,
    }
]

NB06_DOCUMENTS: dict[
    str,
    dict[str, Any],
] = {}

for role, artifact_path in NB06_PATHS.items():
    reference = nb06_control_registry.get(
        role
    )

    require(
        isinstance(
            reference,
            dict,
        ),
        f"Notebook 06 handoff omits {role}.",
    )

    referenced_path = Path(
        str(
            reference.get(
                "path",
                "",
            )
        )
    )

    require(
        normalized_absolute(
            referenced_path
        )
        == normalized_absolute(
            artifact_path
        ),
        f"Notebook 06 registered path mismatch: {role}",
    )

    (
        document,
        semantic_sha256,
        raw_file_sha256,
    ) = verify_self_hash(
        artifact_path
    )

    (
        expected_artifact_type,
        expected_schema_version,
    ) = NB06_SCHEMAS[
        role
    ]

    expect(
        document,
        f"Notebook 06 {role}",
        artifact_type=(
            expected_artifact_type
        ),
        schema_version=(
            expected_schema_version
        ),
        source_run_prefix=(
            SOURCE_RUN_PREFIX
        ),
        v0_1_run_id=(
            V01_RUN_ID
        ),
    )

    # This schema intentionally does not repeat the three
    # run-hash fields. Its own payload hash and handoff binding
    # provide its integrity and identity chain.
    if role != "readback_audit":
        expect(
            document,
            f"Notebook 06 {role} identity",
            source_set_sha256=(
                SOURCE_SET_SHA256
            ),
            run_config_sha256=(
                RUN_CONFIG_SHA256
            ),
            run_identity_sha256=(
                RUN_IDENTITY_SHA256
            ),
        )

    require(
        semantic_sha256
        == reference.get(
            "payload_sha256"
        ),
        f"Notebook 06 semantic hash mismatch: {role}",
    )

    require(
        raw_file_sha256
        == reference.get(
            "sha256"
        ),
        f"Notebook 06 file hash mismatch: {role}",
    )

    NB06_DOCUMENTS[
        role
    ] = document

    artifact_rows.append(
        {
            "artifact_role": (
                f"nb06_{role}"
            ),
            "path": str(
                artifact_path
            ),
            "file_sha256": (
                raw_file_sha256
            ),
            "semantic_sha256": (
                semantic_sha256
            ),
            "passed": True,
        }
    )


nb06_terminal = NB06_DOCUMENTS[
    "terminal_decision"
]

expect(
    nb06_terminal,
    "Notebook 06 terminal decision",
    terminal_status=(
        "PASS_HAWKES_RESTRICTED_FIRST"
    ),
    authorization_state=(
        "AUTHORIZED_RESTRICTED_FIRST"
    ),
    formal_count_comparator=(
        FORMAL_COUNT_COMPARATOR
    ),
    primary_simple_count_baseline=(
        PRIMARY_SIMPLE_COUNT_BASELINE
    ),
    notebook_07_authorized=True,
    restricted_first_required=True,
    status="PASS",
)

expect(
    NB06_DOCUMENTS[
        "final_acceptance"
    ],
    "Notebook 06 final acceptance",
    terminal_status=(
        "PASS_HAWKES_RESTRICTED_FIRST"
    ),
    authorization_state=(
        "AUTHORIZED_RESTRICTED_FIRST"
    ),
    blocking_failure_count=0,
    notebook_07_authorized=True,
    restricted_first_required=True,
    status=(
        "PASS_HAWKES_RESTRICTED_FIRST"
    ),
)

expect(
    NB06_DOCUMENTS[
        "readback_audit"
    ],
    "Notebook 06 readback audit",
    failed_artifact_count=0,
    all_paths_inside_v0_1=True,
    any_path_inside_v0_0=False,
    v0_0_reference_hash_stable=True,
    status="PASS",
)


# ------------------------------------------------------------
# Notebook 07 identities and semantic hash chain
# ------------------------------------------------------------

NB07_DOCUMENTS: dict[
    str,
    dict[str, Any],
] = {
    role: load_json(
        artifact_path
    )
    for role, artifact_path in (
        NB07_PATHS.items()
    )
}

for role, document in NB07_DOCUMENTS.items():
    (
        expected_artifact_type,
        expected_schema_version,
    ) = NB07_SCHEMAS[
        role
    ]

    expect(
        document,
        f"Notebook 07 {role}",
        artifact_type=(
            expected_artifact_type
        ),
        schema_version=(
            expected_schema_version
        ),
        source_run_prefix=(
            SOURCE_RUN_PREFIX
        ),
        source_set_sha256=(
            SOURCE_SET_SHA256
        ),
        v0_1_run_id=(
            V01_RUN_ID
        ),
        run_config_sha256=(
            RUN_CONFIG_SHA256
        ),
        run_identity_sha256=(
            RUN_IDENTITY_SHA256
        ),
    )


nb07_terminal_package = (
    NB07_DOCUMENTS[
        "terminal_decision"
    ]
)

terminal_components = (
    nb07_terminal_package.get(
        "components"
    )
)

terminal_semantic_hashes = (
    nb07_terminal_package.get(
        "semantic_hashes"
    )
)

require(
    isinstance(
        terminal_components,
        dict,
    ),
    "Terminal role package lacks components.",
)

require(
    isinstance(
        terminal_semantic_hashes,
        dict,
    ),
    "Terminal role package lacks semantic_hashes.",
)

terminal_payload = (
    terminal_components.get(
        "terminal_decision"
    )
)

require(
    isinstance(
        terminal_payload,
        dict,
    ),
    "Terminal payload is missing.",
)

terminal_semantic_sha256 = (
    canonical_json_sha256(
        terminal_payload
    )
)

require(
    terminal_semantic_sha256
    == terminal_components.get(
        "terminal_decision_sha256"
    )
    == terminal_semantic_hashes.get(
        "terminal_decision_sha256"
    ),
    (
        "Notebook 07 terminal semantic hash "
        "verification failed."
    ),
)

expect(
    nb07_terminal_package,
    "Notebook 07 terminal role package",
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    artifact_role=(
        "terminal_decision"
    ),
    status="PASS",
)


nb07_final = NB07_DOCUMENTS[
    "final_acceptance"
]

nb07_manifest = NB07_DOCUMENTS[
    "output_manifest"
]

nb07_persistence = NB07_DOCUMENTS[
    "persistence_index"
]

nb07_readback = NB07_DOCUMENTS[
    "readback_audit"
]

nb07_handoff = NB07_DOCUMENTS[
    "handoff"
]


expect(
    nb07_handoff,
    "Notebook 07-to-08 handoff",
    producer=(
        "07_HAWKES_ESTIMATION.ipynb"
    ),
    consumer=(
        NOTEBOOK_NAME
    ),
    notebook07_terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    authorization_scope=(
        "NOTEBOOK_08_DIAGNOSTICS_ONLY"
    ),
    notebook08_authorized_after_manifest_verification=True,
    status=(
        "PASS_HANDOFF_FOR_HAWKES_DIAGNOSTICS"
    ),
)

expect(
    nb07_final,
    "Notebook 07 final acceptance",
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    blocking_failure_count=0,
    semantic_readback_completed=True,
    semantic_readback_passed=True,
    calibration_parameter_updates=0,
    notebook08_handoff_completed=True,
    notebook08_authorized_scope=(
        "HAWKES_DIAGNOSTICS_ONLY"
    ),
    next_operation=(
        "BEGIN_NOTEBOOK08_HAWKES_DIAGNOSTICS"
    ),
)

expect(
    nb07_manifest,
    "Notebook 07 output manifest",
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    calibration_parameter_updates=0,
    notebook08_authorization_scope=(
        "HAWKES_DIAGNOSTICS_ONLY"
    ),
    status=(
        "PASS_COMPLETE_OUTPUT_MANIFEST"
    ),
)

expect(
    nb07_readback,
    "Notebook 07 readback audit",
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    all_files_exist=True,
    all_hashes_match=True,
    all_parse_checks_passed=True,
    all_semantic_checks_passed=True,
    calibration_parameter_updates=0,
    status=(
        "PASS_SEMANTIC_READBACK"
    ),
)

expect(
    nb07_persistence,
    "Notebook 07 persistence index",
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    immediate_byte_verification_passed=True,
    v0_0_files_unchanged=True,
    v0_1_selected_model_unchanged=True,
    status=(
        "PASS_CORE_PERSISTENCE_PENDING_FINAL_READBACK"
    ),
)


readback_semantic_sha256 = (
    canonical_json_sha256(
        nb07_readback
    )
)

handoff_semantic_sha256 = (
    canonical_json_sha256(
        nb07_handoff
    )
)

final_semantic_sha256 = (
    canonical_json_sha256(
        nb07_final
    )
)

persistence_file_sha256 = (
    file_sha256(
        NB07_PATHS[
            "persistence_index"
        ]
    )
)


for document in (
    nb07_handoff,
    nb07_final,
    nb07_manifest,
    nb07_persistence,
    nb07_readback,
):
    require(
        document.get(
            "terminal_decision_sha256"
        )
        == terminal_semantic_sha256,
        (
            "Notebook 07 terminal-decision "
            "hash chain is broken."
        ),
    )


for document in (
    nb07_handoff,
    nb07_final,
    nb07_manifest,
):
    require(
        document.get(
            "semantic_readback_sha256"
        )
        == readback_semantic_sha256,
        (
            "Notebook 07 semantic-readback "
            "hash chain is broken."
        ),
    )


for document in (
    nb07_final,
    nb07_manifest,
):
    require(
        document.get(
            "notebook07_to_notebook08_handoff_sha256"
        )
        == handoff_semantic_sha256,
        (
            "Notebook 07-to-08 handoff "
            "hash chain is broken."
        ),
    )


require(
    nb07_manifest.get(
        "final_acceptance_sha256"
    )
    == final_semantic_sha256,
    (
        "Notebook 07 final-acceptance "
        "hash chain is broken."
    ),
)

require(
    nb07_final.get(
        "persistence_index_file_sha256"
    )
    == persistence_file_sha256,
    (
        "Notebook 07 persistence-index "
        "file hash mismatch."
    ),
)

require(
    nb07_readback.get(
        "persistence_index_file_sha256"
    )
    == persistence_file_sha256,
    (
        "Notebook 07 readback persistence "
        "hash mismatch."
    ),
)


# ------------------------------------------------------------
# Frozen model and authorization contract
# ------------------------------------------------------------

selected_model = nb07_handoff.get(
    "selected_model"
)

frozen_parameters = nb07_handoff.get(
    "frozen_parameters"
)

history_contract = nb07_handoff.get(
    "history_contract"
)

claim_limits = nb07_handoff.get(
    "claim_limits"
)

protected_access = nb07_handoff.get(
    "protected_partition_content_loaded"
)

for label, value in (
    (
        "selected_model",
        selected_model,
    ),
    (
        "frozen_parameters",
        frozen_parameters,
    ),
    (
        "history_contract",
        history_contract,
    ),
    (
        "claim_limits",
        claim_limits,
    ),
    (
        "protected_partition_content_loaded",
        protected_access,
    ),
):
    require(
        isinstance(
            value,
            dict,
        ),
        f"Notebook 07 handoff lacks {label}.",
    )


expect(
    selected_model,
    "Frozen selected model",
    model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    model_family=(
        EXPECTED_SELECTED_MODEL_FAMILY
    ),
    kernel_family=(
        EXPECTED_KERNEL_FAMILY
    ),
    estimator_label=(
        EXPECTED_ESTIMATOR_LABEL
    ),
    primary_event_representation=(
        PRIMARY_EVENT_REPRESENTATION
    ),
    primary_timestamp_column=(
        PRIMARY_EVENT_TIME_COLUMN
    ),
    shared_decay=True,
    cross_excitation_fixed_zero=True,
)

expect(
    history_contract,
    "Frozen history contract",
    development_left_censored=True,
    fabricated_prehistory=False,
    calibration_initial_state_source=(
        "TERMINAL_DEVELOPMENT_STATE"
    ),
    calibration_reset_authorized=False,
    strict_pre_batch_scoring=True,
    within_batch_zero_lag_excitation=False,
)

expect(
    protected_access,
    "Protected partition contract",
    **{
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
)

require(
    all(
        value is False
        for value in claim_limits.values()
    ),
    (
        "Notebook 07 handoff authorizes "
        "a prohibited claim."
    ),
)


spectral_radius = float(
    frozen_parameters[
        "spectral_radius"
    ]
)

require(
    np.isclose(
        spectral_radius,
        EXPECTED_SELECTED_SPECTRAL_RADIUS,
        rtol=(
            FLOAT_RELATIVE_TOLERANCE
        ),
        atol=(
            SPECTRAL_RADIUS_ABSOLUTE_TOLERANCE
        ),
    ),
    (
        "Frozen spectral radius differs "
        "from the Notebook 07 contract."
    ),
)

require(
    0.0
    <= spectral_radius
    < 1.0,
    "Frozen Hawkes model is non-stationary.",
)

require(
    float(
        frozen_parameters[
            "kappa_buy_to_sell"
        ]
    )
    == 0.0
    and float(
        frozen_parameters[
            "kappa_sell_to_buy"
        ]
    )
    == 0.0,
    (
        "Frozen cross-excitation is not "
        "exactly zero."
    ),
)

require(
    np.isclose(
        float(
            frozen_parameters[
                "beta_buy_per_second"
            ]
        ),
        float(
            frozen_parameters[
                "beta_sell_per_second"
            ]
        ),
        rtol=(
            FLOAT_RELATIVE_TOLERANCE
        ),
        atol=(
            FLOAT_ABSOLUTE_TOLERANCE
        ),
    ),
    "The shared-decay contract is violated.",
)


selected_model_semantic_sha256 = str(
    selected_model[
        "selected_model_package_sha256"
    ]
)

for document in (
    nb07_final,
    nb07_manifest,
    nb07_persistence,
):
    require(
        document.get(
            "selected_model_package_sha256"
        )
        == selected_model_semantic_sha256,
        (
            "Notebook 07 selected-model "
            "hash chain is broken."
        ),
    )


# ------------------------------------------------------------
# Manifest and required role-package verification
# ------------------------------------------------------------

manifest_records = nb07_manifest.get(
    "files"
)

require(
    isinstance(
        manifest_records,
        list,
    ),
    (
        "Notebook 07 output manifest "
        "lacks files."
    ),
)

manifest_index: dict[
    str,
    dict[str, Any],
] = {}

for record in manifest_records:
    require(
        isinstance(
            record,
            dict,
        ),
        "Malformed Notebook 07 manifest record.",
    )

    record_path = record.get(
        "relative_path"
    )

    require(
        isinstance(
            record_path,
            str,
        ),
        (
            "Notebook 07 manifest record "
            "lacks relative_path."
        ),
    )

    key = relative_key(
        record_path
    )

    require(
        key not in manifest_index,
        (
            "Duplicate Notebook 07 manifest "
            f"path: {record_path}"
        ),
    )

    manifest_index[
        key
    ] = record


# The output manifest does not list itself by design.
for role, artifact_path in NB07_PATHS.items():
    if role == "output_manifest":
        continue

    key = relative_key(
        artifact_path.relative_to(
            V01_ROOT
        )
    )

    require(
        key in manifest_index,
        (
            "Notebook 07 manifest omits "
            f"{role}."
        ),
    )

    require(
        file_sha256(
            artifact_path
        )
        == manifest_index[
            key
        ].get(
            "sha256"
        ),
        (
            "Notebook 07 file hash "
            f"mismatch: {role}"
        ),
    )


required_artifact_paths = (
    nb07_handoff.get(
        "required_artifact_paths"
    )
)

require(
    isinstance(
        required_artifact_paths,
        dict,
    )
    and bool(
        required_artifact_paths
    ),
    (
        "Notebook 07 handoff lacks "
        "required_artifact_paths."
    ),
)

NOTEBOOK07_REQUIRED_ARTIFACT_PATHS: dict[
    str,
    Path,
] = {}

required_artifact_rows: list[
    dict[str, Any]
] = []

for role, path_text in (
    required_artifact_paths.items()
):
    require(
        isinstance(
            path_text,
            str,
        )
        and bool(
            path_text.strip()
        ),
        f"Invalid required path: {role}",
    )

    artifact_path = resolve_v01(
        path_text
    )

    require(
        inside(
            artifact_path,
            V01_ROOT,
        ),
        (
            "Required artifact is outside "
            f"V0.1: {artifact_path}"
        ),
    )

    require(
        not inside(
            artifact_path,
            V00_ROOT,
        ),
        (
            "Required artifact resolves "
            f"inside V0.0: {artifact_path}"
        ),
    )

    require(
        artifact_path.is_file(),
        (
            "Required Notebook 07 artifact "
            f"is missing: {artifact_path}"
        ),
    )

    key = relative_key(
        artifact_path.relative_to(
            V01_ROOT
        )
    )

    require(
        key in manifest_index,
        (
            "Notebook 07 manifest omits "
            f"required role: {role}"
        ),
    )

    observed_sha256 = file_sha256(
        artifact_path
    )

    require(
        observed_sha256
        == manifest_index[
            key
        ].get(
            "sha256"
        ),
        (
            "Required artifact hash "
            f"mismatch: {role}"
        ),
    )

    NOTEBOOK07_REQUIRED_ARTIFACT_PATHS[
        str(role)
    ] = artifact_path

    required_artifact_rows.append(
        {
            "artifact_role": str(
                role
            ),
            "path": str(
                artifact_path
            ),
            "size_bytes": int(
                artifact_path.stat().st_size
            ),
            "file_sha256": (
                observed_sha256
            ),
            "passed": True,
        }
    )


# ------------------------------------------------------------
# Verify the selected-model semantic payload itself
# ------------------------------------------------------------

selected_model_role_path = (
    NOTEBOOK07_REQUIRED_ARTIFACT_PATHS.get(
        "selected_model_package"
    )
)

require(
    selected_model_role_path is not None,
    (
        "Selected-model role package "
        "is not registered."
    ),
)

selected_model_role = load_json(
    selected_model_role_path
)

expect(
    selected_model_role,
    "Selected-model role package",
    artifact_type=(
        "NOTEBOOK_07_SELECTED_MODEL_PACKAGE"
    ),
    schema_version=(
        "NOTEBOOK_07_SELECTED_MODEL_PACKAGE_V1"
    ),
    source_run_prefix=(
        SOURCE_RUN_PREFIX
    ),
    source_set_sha256=(
        SOURCE_SET_SHA256
    ),
    v0_1_run_id=(
        V01_RUN_ID
    ),
    run_config_sha256=(
        RUN_CONFIG_SHA256
    ),
    run_identity_sha256=(
        RUN_IDENTITY_SHA256
    ),
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    artifact_role=(
        "selected_model_package"
    ),
    status="PASS",
)

selected_components = (
    selected_model_role.get(
        "components"
    )
)

selected_semantic_hashes = (
    selected_model_role.get(
        "semantic_hashes"
    )
)

require(
    isinstance(
        selected_components,
        dict,
    ),
    (
        "Selected-model package "
        "lacks components."
    ),
)

require(
    isinstance(
        selected_semantic_hashes,
        dict,
    ),
    (
        "Selected-model package lacks "
        "semantic_hashes."
    ),
)

selected_model_payload = (
    selected_components.get(
        "selected_model_package"
    )
)

require(
    isinstance(
        selected_model_payload,
        dict,
    ),
    "Selected-model payload is missing.",
)

observed_selected_model_semantic_sha256 = (
    canonical_json_sha256(
        selected_model_payload
    )
)

require(
    observed_selected_model_semantic_sha256
    == selected_model_semantic_sha256
    == selected_components.get(
        "selected_model_package_sha256"
    )
    == selected_semantic_hashes.get(
        "selected_model_package_sha256"
    ),
    (
        "Selected-model semantic hash "
        "verification failed."
    ),
)


# ------------------------------------------------------------
# Publish verified authority for the next cell
# ------------------------------------------------------------

for role, artifact_path in NB07_PATHS.items():
    document = NB07_DOCUMENTS[
        role
    ]

    artifact_rows.append(
        {
            "artifact_role": (
                f"nb07_{role}"
            ),
            "path": str(
                artifact_path
            ),
            "file_sha256": (
                file_sha256(
                    artifact_path
                )
            ),
            "semantic_sha256": (
                terminal_semantic_sha256
                if role
                == "terminal_decision"
                else canonical_json_sha256(
                    document
                )
            ),
            "passed": True,
        }
    )


UPSTREAM_CONTROL_ARTIFACT_AUDIT = (
    pd.DataFrame(
        artifact_rows
    )
)

NOTEBOOK07_REQUIRED_ARTIFACT_AUDIT = (
    pd.DataFrame(
        required_artifact_rows
    )
)

BASELINE_SELECTION_AUTHORITY = pd.DataFrame(
    [
        {
            "selection_role": (
                "FORMAL_COUNT_COMPARATOR"
            ),
            "process_name": (
                "BUY_SELL_COUNT_GRID"
            ),
            "model_id": (
                nb06_terminal[
                    "formal_count_comparator"
                ]
            ),
            "source": (
                "NB06_TERMINAL_DECISION"
            ),
        },
        {
            "selection_role": (
                "PRIMARY_SIMPLE_COUNT_BASELINE"
            ),
            "process_name": (
                "BUY_SELL_COUNT_GRID"
            ),
            "model_id": (
                nb06_terminal[
                    "primary_simple_count_baseline"
                ]
            ),
            "source": (
                "NB06_TERMINAL_DECISION"
            ),
        },
    ]
)

FROZEN_HAWKES_HANDOFF = pd.DataFrame(
    [
        {
            "field": "model_id",
            "value": (
                selected_model[
                    "model_id"
                ]
            ),
        },
        {
            "field": "model_family",
            "value": (
                selected_model[
                    "model_family"
                ]
            ),
        },
        {
            "field": "estimator_label",
            "value": (
                selected_model[
                    "estimator_label"
                ]
            ),
        },
        {
            "field": "mu_buy_per_second",
            "value": (
                frozen_parameters[
                    "mu_buy_per_second"
                ]
            ),
        },
        {
            "field": "mu_sell_per_second",
            "value": (
                frozen_parameters[
                    "mu_sell_per_second"
                ]
            ),
        },
        {
            "field": "kappa_buy_to_buy",
            "value": (
                frozen_parameters[
                    "kappa_buy_to_buy"
                ]
            ),
        },
        {
            "field": "kappa_sell_to_sell",
            "value": (
                frozen_parameters[
                    "kappa_sell_to_sell"
                ]
            ),
        },
        {
            "field": "beta_shared_per_second",
            "value": (
                frozen_parameters[
                    "beta_buy_per_second"
                ]
            ),
        },
        {
            "field": "spectral_radius",
            "value": (
                spectral_radius
            ),
        },
        {
            "field": (
                "calibration_parameter_updates"
            ),
            "value": 0,
        },
        {
            "field": (
                "authorization_scope"
            ),
            "value": (
                nb07_handoff[
                    "authorization_scope"
                ]
            ),
        },
    ]
)


require(
    UPSTREAM_CONTROL_ARTIFACT_AUDIT[
        "passed"
    ].all(),
    "A control artifact failed verification.",
)

require(
    NOTEBOOK07_REQUIRED_ARTIFACT_AUDIT[
        "passed"
    ].all(),
    "A required artifact failed verification.",
)

require(
    not PROTECTED_PARTITION_CONTENT_LOADED,
    "Protected partition content was opened.",
)

require(
    not FILESYSTEM_WRITES_PERFORMED,
    "This cell performed a filesystem write.",
)

UPSTREAM_AUTHORITY_VERIFIED = True


display(
    UPSTREAM_CONTROL_ARTIFACT_AUDIT
)

display(
    BASELINE_SELECTION_AUTHORITY
)

display(
    FROZEN_HAWKES_HANDOFF
)

display(
    NOTEBOOK07_REQUIRED_ARTIFACT_AUDIT
)

print(
    "Notebook 06 baseline authority and Notebook 07 Hawkes "
    "handoff verified. "
    f"Control artifacts: {len(UPSTREAM_CONTROL_ARTIFACT_AUDIT)}. "
    f"Required Notebook 07 role packages: "
    f"{len(NOTEBOOK07_REQUIRED_ARTIFACT_AUDIT)}. "
    "Protected partitions remain unopened. "
    "No filesystem writes were performed."
)

,artifact_role,path,file_sha256,semantic_sha256,passed
0,nb06_to_07_handoff,D:\Clown Project\V0.1\artifacts\handoff\06_poi...,c7c57532241580395a4aa29121f20fcb40385fb2aeecf0...,81b9a8463b10bcd48e25de82bb39e378906ce0950c8c34...,True
1,nb06_terminal_decision,D:\Clown Project\V0.1\artifacts\manifests\06_p...,6c53c509fe1122883e217d741f195dd76cfeda975f8203...,5c802aa59fa79feb43c4172a46ddd8a23541c9ed3b76e6...,True
2,nb06_final_acceptance,D:\Clown Project\V0.1\artifacts\manifests\06_p...,7ee4127508d49809f39e99b799458681a83133901038ba...,8df9376da776056408531eef3ca528b53f680407e266ed...,True
3,nb06_output_manifest,D:\Clown Project\V0.1\artifacts\manifests\06_p...,56dfc09bc96b0ce8456f4801e6c553d63694e58a620f0b...,e708a18ea7fd988c7870d8f8121385dcaf6ca3eca3a6a5...,True
4,nb06_readback_audit,D:\Clown Project\V0.1\artifacts\manifests\06_p...,27945455c732c3ae81a9ff92c0d3ee9b11029d85fd94b0...,26001d4bf989a3d05f5bfbc9338094e48a087d63cbbc79...,True
5,nb07_terminal_decision,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,d1a6a55b97ce3f4e2cf8bd4958b42124d9b609f56c39dd...,f8377079cde7eb5f6a58852d023ea0622690ec20230dbc...,True
6,nb07_final_acceptance,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,32f462888a36d37d95e7e8b0e69a9297c7b074203c2827...,32f462888a36d37d95e7e8b0e69a9297c7b074203c2827...,True
7,nb07_output_manifest,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,abc93a03ba93ab41ec6fccede348391f38c96661599041...,abc93a03ba93ab41ec6fccede348391f38c96661599041...,True
8,nb07_persistence_index,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,c3d7adb1b9aa6af3e9297d2d925f084c8b3bc2216aaf1d...,c3d7adb1b9aa6af3e9297d2d925f084c8b3bc2216aaf1d...,True
9,nb07_readback_audit,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,5895b20ca4fb416db962820adefc2a01c73290caa616b8...,5895b20ca4fb416db962820adefc2a01c73290caa616b8...,True


,selection_role,process_name,model_id,source
0,FORMAL_COUNT_COMPARATOR,BUY_SELL_COUNT_GRID,D0_SIDE_CONSTANT_POISSON,NB06_TERMINAL_DECISION
1,PRIMARY_SIMPLE_COUNT_BASELINE,BUY_SELL_COUNT_GRID,E_SIDE_EWMA_250MS_POISSON,NB06_TERMINAL_DECISION


,field,value
0,model_id,H1_DIAGONAL_SHARED_DECAY
1,model_family,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES
2,estimator_label,STRICT_PRE_BATCH_COARSENED_TIME_QUASI_MLE
3,mu_buy_per_second,1.536124799
4,mu_sell_per_second,1.767916532
5,kappa_buy_to_buy,0.1892556591
6,kappa_sell_to_sell,0.1126637105
7,beta_shared_per_second,70.67941658
8,spectral_radius,0.1892556591
9,calibration_parameter_updates,0


,artifact_role,path,size_bytes,file_sha256,passed
0,candidate_fit_summary,D:\Clown Project\V0.1\artifacts\audit_tables\B...,7431,7810ed7e1e15fb35ffb3cc366c504982936056ede18c35...,True
1,candidate_registry,D:\Clown Project\V0.1\artifacts\models\BTCUSDT...,2887,a2c45f85ac54438f8df1eb79032e1230dab81dc368db59...,True
2,candidate_selection,D:\Clown Project\V0.1\artifacts\models\BTCUSDT...,5257,1f65ab0bf3925c38d85e07464c1eb294fdc26c53c9d6a4...,True
3,chronological_fold_geometry,D:\Clown Project\V0.1\artifacts\audit_tables\B...,5266,0853351ddffec720c0bcd470fcc7137e6a5658981aec44...,True
4,chronological_fold_results,D:\Clown Project\V0.1\artifacts\diagnostics\BT...,19191,5beee543ad4bc736fb569755d5bdad79749fd416f29724...,True
5,chronological_stability_audit,D:\Clown Project\V0.1\artifacts\diagnostics\BT...,15442,c6943fb9150f0b1fd9de0880550693384e5158da88da7f...,True
6,left_edge_and_boundary_contract,D:\Clown Project\V0.1\artifacts\diagnostics\BT...,4982,19b3323fb3bbb6d17b38ff689afa66c3a146e5c81f4ac7...,True
7,likelihood_engine_validation,D:\Clown Project\V0.1\artifacts\diagnostics\BT...,3700,dc73b5e1e35bc742fa2eb31ae1dadf555e7ef855f0c506...,True
8,locked_calibration_package,D:\Clown Project\V0.1\artifacts\models\BTCUSDT...,3150,8b2c688af53e3178c7c15739af5d5a0a6d0faf86fb9b75...,True
9,locked_calibration_replay,D:\Clown Project\V0.1\artifacts\diagnostics\BT...,3178805,2f5d9cbb895c5c9d38c4606b895a1f93ce60d0aaa0d6a0...,True


Notebook 06 baseline authority and Notebook 07 Hawkes handoff verified. Control artifacts: 11. Required Notebook 07 role packages: 18. Protected partitions remain unopened. No filesystem writes were performed.


In [5]:
# ============================================================
# Protected analytical load and event/batch reconciliation
# ============================================================

import pyarrow.dataset as ds
import pyarrow.parquet as pq


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    UPSTREAM_AUTHORITY_VERIFIED,
    "Upstream authority has not been verified.",
)

require(
    not bool(
        globals().get(
            "ANALYTICAL_CONTENT_LOADED",
            False,
        )
    ),
    "Analytical content is already loaded.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded before authorization.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted during analytical loading.",
)


# ------------------------------------------------------------
# Analytical partition contract
# ------------------------------------------------------------

FIT_PARTITION: Final[str] = "DEVELOPMENT"

ANALYTICAL_PARTITIONS: Final[tuple[str, str]] = (
    FIT_PARTITION,
    LOCKED_EVALUATION_PARTITION,
)

PARTITION_ORDER_MAP: Final[Mapping[str, int]] = {
    "DEVELOPMENT": 1,
    "CALIBRATION": 2,
    "VALIDATION": 3,
    "ENGINEERING_HOLDOUT": 4,
}

EXPECTED_ANALYTICAL_EVENT_ROWS: Final[int] = (
    EXPECTED_DEVELOPMENT_EVENT_ROWS
    + EXPECTED_CALIBRATION_EVENT_ROWS
)

EXPECTED_ANALYTICAL_BATCH_ROWS: Final[int] = (
    EXPECTED_DEVELOPMENT_BATCH_ROWS
    + EXPECTED_CALIBRATION_BATCH_ROWS
)

EXPECTED_ANALYTICAL_COUNTS_BY_PARTITION: Final[
    Mapping[str, Mapping[str, int]]
] = {
    "DEVELOPMENT": {
        "event_count": EXPECTED_DEVELOPMENT_EVENT_ROWS,
        "batch_count": EXPECTED_DEVELOPMENT_BATCH_ROWS,
    },
    "CALIBRATION": {
        "event_count": EXPECTED_CALIBRATION_EVENT_ROWS,
        "batch_count": EXPECTED_CALIBRATION_BATCH_ROWS,
    },
}


# ------------------------------------------------------------
# Verified upstream-data registry
# ------------------------------------------------------------

upstream_data_records = nb06_handoff.get(
    "authoritative_upstream_data_artifacts",
)

require(
    isinstance(
        upstream_data_records,
        list,
    ),
    "Notebook 06 handoff has no valid upstream-data registry.",
)

require(
    len(upstream_data_records) == 6,
    (
        "Notebook 06 handoff must register exactly six "
        "authoritative upstream data artifacts."
    ),
)

NOTEBOOK08_UPSTREAM_DATA_REGISTRY = pd.DataFrame(
    upstream_data_records
)

required_registry_columns: Final[set[str]] = {
    "label",
    "artifact_type",
    "full_row_count",
    "full_column_count",
    "raw_file_sha256",
    "table_payload_sha256",
    "contract_payload_sha256",
    "resolved_path",
    "verified",
}

missing_registry_columns = (
    required_registry_columns
    - set(
        NOTEBOOK08_UPSTREAM_DATA_REGISTRY.columns
    )
)

require(
    not missing_registry_columns,
    (
        "The upstream-data registry is missing columns: "
        f"{sorted(missing_registry_columns)}"
    ),
)

expected_upstream_artifacts: Final[
    Mapping[str, tuple[str, int, int]]
] = {
    "primary_estimation_events": (
        "NOTEBOOK_04_PRIMARY_ESTIMATION_EVENTS",
        EXPECTED_PRIMARY_EVENT_ROWS,
        88,
    ),
    "primary_exact_time_batches": (
        "NOTEBOOK_04_PRIMARY_EXACT_TIME_BATCHES",
        EXPECTED_PRIMARY_SCORING_BATCHES,
        27,
    ),
    "primary_scoring_batches": (
        "NOTEBOOK_04_PRIMARY_SCORING_BATCHES",
        EXPECTED_PRIMARY_SCORING_BATCHES,
        25,
    ),
    "primary_event_to_batch_membership": (
        "NOTEBOOK_04_PRIMARY_EVENT_TO_BATCH_MEMBERSHIP",
        EXPECTED_PRIMARY_EVENT_ROWS,
        10,
    ),
    "observation_window_contract": (
        "NOTEBOOK_04_OBSERVATION_WINDOW_CONTRACT",
        4,
        19,
    ),
    "primary_batch_market_state_features": (
        "PRIMARY_BATCH_MARKET_STATE_FEATURES",
        EXPECTED_PRIMARY_SCORING_BATCHES,
        577,
    ),
}

require(
    set(
        NOTEBOOK08_UPSTREAM_DATA_REGISTRY[
            "label"
        ]
    )
    == set(expected_upstream_artifacts),
    (
        "Notebook 06 registers an unexpected set of "
        "authoritative upstream data artifacts."
    ),
)

upstream_file_audit_rows: list[
    dict[str, Any]
] = []

for registry_row in (
    NOTEBOOK08_UPSTREAM_DATA_REGISTRY
    .sort_values(
        "label",
        kind="stable",
    )
    .to_dict(
        orient="records",
    )
):
    label = str(
        registry_row[
            "label"
        ]
    )

    artifact_type = str(
        registry_row[
            "artifact_type"
        ]
    )

    source_path = Path(
        str(
            registry_row[
                "resolved_path"
            ]
        )
    )

    registered_sha256 = str(
        registry_row[
            "raw_file_sha256"
        ]
    )

    (
        expected_artifact_type,
        expected_row_count,
        expected_column_count,
    ) = expected_upstream_artifacts[
        label
    ]

    require(
        artifact_type
        == expected_artifact_type,
        (
            f"Unexpected artifact type for {label}: "
            f"{artifact_type}"
        ),
    )

    require(
        int(
            registry_row[
                "full_row_count"
            ]
        )
        == expected_row_count,
        (
            f"Unexpected full row count for {label}: "
            f"{registry_row['full_row_count']}"
        ),
    )

    require(
        int(
            registry_row[
                "full_column_count"
            ]
        )
        == expected_column_count,
        (
            f"Unexpected full column count for {label}: "
            f"{registry_row['full_column_count']}"
        ),
    )

    require(
        bool(
            registry_row[
                "verified"
            ]
        ),
        f"Notebook 06 did not verify {label}.",
    )

    require(
        source_path.is_file(),
        f"Registered upstream file is missing: {source_path}",
    )

    require(
        inside(
            source_path,
            V01_ROOT,
        ),
        f"Registered upstream file is outside V0.1: {source_path}",
    )

    require(
        not inside(
            source_path,
            V00_ROOT,
        ),
        f"Registered upstream file resolves inside V0.0: {source_path}",
    )

    require(
        source_path.suffix.casefold()
        == ".parquet",
        (
            f"Expected Parquet for {label}; "
            f"observed {source_path.suffix!r}."
        ),
    )

    require(
        bool(
            SHA256_RE.fullmatch(
                registered_sha256
            )
        ),
        f"Invalid registered SHA-256 for {label}.",
    )

    observed_sha256 = file_sha256(
        source_path
    )

    require(
        observed_sha256
        == registered_sha256,
        f"Upstream file hash mismatch for {label}.",
    )

    upstream_file_audit_rows.append(
        {
            "label": label,
            "artifact_type": artifact_type,
            "resolved_path": str(
                source_path
            ),
            "full_row_count": expected_row_count,
            "full_column_count": expected_column_count,
            "registered_sha256": registered_sha256,
            "observed_sha256": observed_sha256,
            "hash_matches": True,
            "inside_v0_1": True,
            "inside_v0_0": False,
            "analytical_content_loaded": False,
            "status": "PASS",
        }
    )

NOTEBOOK08_UPSTREAM_FILE_AUDIT = pd.DataFrame(
    upstream_file_audit_rows
)


# ------------------------------------------------------------
# Minimal authoritative column interfaces
# ------------------------------------------------------------

EVENT_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_id",
    "primary_event_number",
    "partition_event_index",
    "primary_event_batch_id",
    "primary_event_batch_number",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_event_time_ns",
    "relative_event_time_seconds",
    "event_side",
    "event_side_code",
    "event_print_count",
)

EXACT_BATCH_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_batch_time_ns",
    "relative_batch_time_seconds",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "unique_side_count",
    "batch_print_count",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "contract_start_ns",
    "contract_end_exclusive_ns",
    "event_clock_origin_ns",
    "observation_window_duration_ns",
)

SCORING_BATCH_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_batch_time_ns",
    "relative_batch_time_seconds",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "unique_side_count",
    "batch_print_count",
    "batch_quantity",
    "batch_notional",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "event_clock_origin_ns",
    "observation_window_duration_ns",
    "score_with_history_strictly_before_batch_time",
    "zero_lag_within_batch_excitation_allowed",
    "apply_batch_excitation_after_all_members_scored",
    "collector_sequence_is_trace_order_only",
    "timestamp_jitter_allowed",
    "event_removal_allowed",
)

MEMBERSHIP_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_id",
    "primary_event_batch_number",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "event_side",
    "event_side_code",
    "primary_event_batch_id",
    "primary_event_number",
    "partition_event_index",
)


# ------------------------------------------------------------
# Loading and normalization helpers
# ------------------------------------------------------------

def registered_upstream_path(
    label: str,
) -> Path:
    """Return one verified upstream path."""
    matching_rows = (
        NOTEBOOK08_UPSTREAM_DATA_REGISTRY.loc[
            NOTEBOOK08_UPSTREAM_DATA_REGISTRY[
                "label"
            ].eq(label)
        ]
    )

    require(
        len(matching_rows) == 1,
        (
            "Expected exactly one upstream artifact for "
            f"{label!r}; observed {len(matching_rows)}."
        ),
    )

    source_path = Path(
        str(
            matching_rows.iloc[0][
                "resolved_path"
            ]
        )
    )

    require(
        source_path.is_file(),
        f"Registered upstream path is missing: {source_path}",
    )

    require(
        source_path.suffix.casefold()
        == ".parquet",
        f"Registered upstream artifact is not Parquet: {source_path}",
    )

    return source_path


def parquet_schema_columns(
    source_path: Path,
) -> tuple[str, ...]:
    """Read a physical Parquet schema without loading rows."""
    return tuple(
        pq.ParquetFile(
            source_path
        ).schema_arrow.names
    )


def require_parquet_columns(
    source_path: Path,
    required_columns: tuple[str, ...],
    *,
    label: str,
) -> None:
    """Require a complete Parquet column interface."""
    available_columns = set(
        parquet_schema_columns(
            source_path
        )
    )

    missing_columns = sorted(
        set(required_columns)
        - available_columns
    )

    require(
        not missing_columns,
        (
            f"{label} is missing required columns: "
            f"{missing_columns}"
        ),
    )


def load_filtered_parquet(
    source_path: Path,
    *,
    columns: tuple[str, ...],
    partitions: tuple[str, ...],
    label: str,
) -> pd.DataFrame:
    """
    Materialize only DEVELOPMENT and CALIBRATION through an
    Arrow predicate. Protected rows are never converted to pandas.
    """
    require_parquet_columns(
        source_path,
        columns,
        label=label,
    )

    dataset = ds.dataset(
        source_path,
        format="parquet",
    )

    partition_filter = ds.field(
        PRIMARY_EVENT_PARTITION_COLUMN
    ).isin(
        list(partitions)
    )

    arrow_table = dataset.to_table(
        columns=list(columns),
        filter=partition_filter,
    )

    frame = arrow_table.to_pandas(
        types_mapper=None,
        self_destruct=True,
        split_blocks=True,
    )

    require(
        PRIMARY_EVENT_PARTITION_COLUMN
        in frame.columns,
        (
            f"{label} lost partition column "
            f"{PRIMARY_EVENT_PARTITION_COLUMN!r}."
        ),
    )

    frame[
        PRIMARY_EVENT_PARTITION_COLUMN
    ] = (
        frame[
            PRIMARY_EVENT_PARTITION_COLUMN
        ]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    observed_partitions = set(
        frame[
            PRIMARY_EVENT_PARTITION_COLUMN
        ]
        .dropna()
        .tolist()
    )

    require(
        observed_partitions
        == set(partitions),
        (
            f"{label} partition mismatch: "
            f"{sorted(observed_partitions)}"
        ),
    )

    require(
        not observed_partitions.intersection(
            PROTECTED_PARTITIONS
        ),
        f"{label} materialized protected partition content.",
    )

    return frame


def parse_exact_int64(
    values: pd.Series,
    *,
    label: str,
) -> pd.Series:
    """Convert integer-like values to exact non-null int64."""
    require(
        not values.isna().any(),
        f"{label} contains missing values.",
    )

    require(
        not pd.api.types.is_bool_dtype(
            values.dtype
        ),
        f"{label} must not be Boolean.",
    )

    if pd.api.types.is_integer_dtype(
        values.dtype
    ):
        return values.astype(
            "int64"
        )

    if pd.api.types.is_float_dtype(
        values.dtype
    ):
        numeric_values = values.to_numpy(
            dtype=np.float64
        )

        require(
            np.isfinite(
                numeric_values
            ).all(),
            f"{label} contains non-finite values.",
        )

        require(
            np.equal(
                numeric_values,
                np.rint(
                    numeric_values
                ),
            ).all(),
            f"{label} contains non-integral values.",
        )

        return pd.Series(
            np.rint(
                numeric_values
            ).astype(
                np.int64
            ),
            index=values.index,
            name=values.name,
        )

    numeric_values = pd.to_numeric(
        values.astype(
            "string"
        ),
        errors="raise",
    )

    require(
        not numeric_values.isna().any(),
        f"{label} could not be parsed exactly.",
    )

    return numeric_values.astype(
        "int64"
    )


def normalize_boolean(
    values: pd.Series,
    *,
    label: str,
) -> pd.Series:
    """Return a strict non-null Boolean series."""
    require(
        not values.isna().any(),
        f"{label} contains missing Boolean values.",
    )

    if pd.api.types.is_bool_dtype(
        values.dtype
    ):
        return values.astype(
            bool
        )

    normalized_text = (
        values.astype(
            "string"
        )
        .str.strip()
        .str.lower()
    )

    require(
        normalized_text.isin(
            (
                "true",
                "false",
                "1",
                "0",
            )
        ).all(),
        f"{label} contains invalid Boolean encodings.",
    )

    return normalized_text.map(
        {
            "true": True,
            "1": True,
            "false": False,
            "0": False,
        }
    ).astype(
        bool
    )


def deterministic_sort(
    frame: pd.DataFrame,
    columns: tuple[str, ...],
) -> pd.DataFrame:
    """Return a stable deterministically ordered copy."""
    require(
        set(columns).issubset(
            frame.columns
        ),
        (
            "Deterministic sort columns are unavailable: "
            f"{columns}"
        ),
    )

    return (
        frame.sort_values(
            list(columns),
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# Resolve registered sources
# ------------------------------------------------------------

PRIMARY_EVENTS_PATH = registered_upstream_path(
    "primary_estimation_events"
)

EXACT_BATCHES_PATH = registered_upstream_path(
    "primary_exact_time_batches"
)

SCORING_BATCHES_PATH = registered_upstream_path(
    "primary_scoring_batches"
)

EVENT_BATCH_MEMBERSHIP_PATH = registered_upstream_path(
    "primary_event_to_batch_membership"
)

OBSERVATION_WINDOW_CONTRACT_PATH = registered_upstream_path(
    "observation_window_contract"
)


# ------------------------------------------------------------
# Load observation-window metadata
#
# These are contract metadata rows, not analytical event rows.
# ------------------------------------------------------------

OBSERVATION_WINDOW_CONTRACT = pd.read_parquet(
    OBSERVATION_WINDOW_CONTRACT_PATH,
    engine="pyarrow",
)

required_window_columns: Final[
    tuple[str, ...]
] = (
    "partition_order",
    "event_partition",
    "contract_start_ns",
    "contract_end_exclusive_ns",
    "contract_duration_ns",
    "event_clock_origin_ns",
    "left_censoring_duration_ns",
    "observation_window_duration_ns",
    "primary_event_count",
    "primary_batch_count",
    "simultaneous_batch_count",
    "mixed_side_batch_count",
    "contract_interval_convention",
    "left_censoring_preserved",
)

missing_window_columns = sorted(
    set(
        required_window_columns
    )
    - set(
        OBSERVATION_WINDOW_CONTRACT.columns
    )
)

require(
    not missing_window_columns,
    (
        "Observation-window contract is missing columns: "
        f"{missing_window_columns}"
    ),
)

OBSERVATION_WINDOW_CONTRACT = (
    OBSERVATION_WINDOW_CONTRACT.loc[
        :,
        list(
            required_window_columns
        ),
    ]
    .copy()
)

OBSERVATION_WINDOW_CONTRACT[
    "event_partition"
] = (
    OBSERVATION_WINDOW_CONTRACT[
        "event_partition"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)

for integer_column in (
    "partition_order",
    "contract_start_ns",
    "contract_end_exclusive_ns",
    "contract_duration_ns",
    "event_clock_origin_ns",
    "left_censoring_duration_ns",
    "observation_window_duration_ns",
    "primary_event_count",
    "primary_batch_count",
    "simultaneous_batch_count",
    "mixed_side_batch_count",
):
    OBSERVATION_WINDOW_CONTRACT[
        integer_column
    ] = parse_exact_int64(
        OBSERVATION_WINDOW_CONTRACT[
            integer_column
        ],
        label=(
            "OBSERVATION_WINDOW_CONTRACT."
            f"{integer_column}"
        ),
    )

OBSERVATION_WINDOW_CONTRACT[
    "left_censoring_preserved"
] = normalize_boolean(
    OBSERVATION_WINDOW_CONTRACT[
        "left_censoring_preserved"
    ],
    label=(
        "OBSERVATION_WINDOW_CONTRACT."
        "left_censoring_preserved"
    ),
)

OBSERVATION_WINDOW_CONTRACT = deterministic_sort(
    OBSERVATION_WINDOW_CONTRACT,
    (
        "partition_order",
    ),
)

require(
    len(
        OBSERVATION_WINDOW_CONTRACT
    )
    == len(
        PARTITION_ORDER_MAP
    ),
    "Observation-window contract must contain four rows.",
)

require(
    OBSERVATION_WINDOW_CONTRACT[
        "event_partition"
    ].is_unique,
    "Observation-window contract contains duplicate partitions.",
)

require(
    set(
        OBSERVATION_WINDOW_CONTRACT[
            "event_partition"
        ]
    )
    == set(
        PARTITION_ORDER_MAP
    ),
    "Observation-window contract has an unexpected partition set.",
)

require(
    OBSERVATION_WINDOW_CONTRACT[
        "contract_interval_convention"
    ].eq(
        "[start, end)"
    ).all(),
    "A partition violates the half-open interval contract.",
)

require(
    OBSERVATION_WINDOW_CONTRACT[
        "left_censoring_preserved"
    ].all(),
    "A partition does not preserve left censoring.",
)

contract_starts_ns = (
    OBSERVATION_WINDOW_CONTRACT[
        "contract_start_ns"
    ].to_numpy(
        dtype=np.int64
    )
)

contract_ends_ns = (
    OBSERVATION_WINDOW_CONTRACT[
        "contract_end_exclusive_ns"
    ].to_numpy(
        dtype=np.int64
    )
)

contract_durations_ns = (
    OBSERVATION_WINDOW_CONTRACT[
        "contract_duration_ns"
    ].to_numpy(
        dtype=np.int64
    )
)

require(
    np.all(
        contract_ends_ns
        > contract_starts_ns
    ),
    "A partition contract has nonpositive duration.",
)

require(
    np.array_equal(
        contract_ends_ns
        - contract_starts_ns,
        contract_durations_ns,
    ),
    "A partition contract duration is inconsistent.",
)

require(
    np.all(
        contract_starts_ns[1:]
        >= contract_ends_ns[:-1]
    ),
    "Partition contract windows overlap.",
)


# ------------------------------------------------------------
# Filtered DEVELOPMENT and CALIBRATION analytical loads
# ------------------------------------------------------------

PRIMARY_ESTIMATION_EVENTS_ANALYTICAL = (
    load_filtered_parquet(
        PRIMARY_EVENTS_PATH,
        columns=EVENT_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_estimation_events",
    )
)

PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL = (
    load_filtered_parquet(
        EXACT_BATCHES_PATH,
        columns=EXACT_BATCH_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_exact_time_batches",
    )
)

PRIMARY_SCORING_BATCHES_ANALYTICAL = (
    load_filtered_parquet(
        SCORING_BATCHES_PATH,
        columns=SCORING_BATCH_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_scoring_batches",
    )
)

PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL = (
    load_filtered_parquet(
        EVENT_BATCH_MEMBERSHIP_PATH,
        columns=MEMBERSHIP_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_event_to_batch_membership",
    )
)


# ------------------------------------------------------------
# Exact dtype normalization
# ------------------------------------------------------------

for (
    frame_label,
    frame,
    integer_columns,
) in (
    (
        "PRIMARY_ESTIMATION_EVENTS_ANALYTICAL",
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
        (
            "primary_event_number",
            "partition_event_index",
            "primary_event_batch_number",
            "event_partition_order",
            "event_time_ns",
            "relative_event_time_ns",
            "event_side_code",
            "event_print_count",
        ),
    ),
    (
        "PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL",
        PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL,
        (
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_time_ns",
            "relative_batch_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "unique_side_count",
            "batch_print_count",
            "contract_start_ns",
            "contract_end_exclusive_ns",
            "event_clock_origin_ns",
            "observation_window_duration_ns",
        ),
    ),
    (
        "PRIMARY_SCORING_BATCHES_ANALYTICAL",
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
        (
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_time_ns",
            "relative_batch_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "unique_side_count",
            "batch_print_count",
            "event_clock_origin_ns",
            "observation_window_duration_ns",
        ),
    ),
    (
        "PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL",
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
        (
            "primary_event_batch_number",
            "event_partition_order",
            "event_time_ns",
            "event_side_code",
            "primary_event_number",
            "partition_event_index",
        ),
    ),
):
    for integer_column in integer_columns:
        frame[
            integer_column
        ] = parse_exact_int64(
            frame[
                integer_column
            ],
            label=(
                f"{frame_label}."
                f"{integer_column}"
            ),
        )

for frame in (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
    PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
):
    frame[
        "event_side"
    ] = (
        frame[
            "event_side"
        ]
        .astype("string")
        .str.strip()
        .str.upper()
    )

for boolean_column in (
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
):
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
        boolean_column
    ] = normalize_boolean(
        PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
            boolean_column
        ],
        label=(
            "PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL."
            f"{boolean_column}"
        ),
    )

for boolean_column in (
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "score_with_history_strictly_before_batch_time",
    "zero_lag_within_batch_excitation_allowed",
    "apply_batch_excitation_after_all_members_scored",
    "collector_sequence_is_trace_order_only",
    "timestamp_jitter_allowed",
    "event_removal_allowed",
):
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        boolean_column
    ] = normalize_boolean(
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            boolean_column
        ],
        label=(
            "PRIMARY_SCORING_BATCHES_ANALYTICAL."
            f"{boolean_column}"
        ),
    )


# ------------------------------------------------------------
# Stable chronological ordering
# ------------------------------------------------------------

PRIMARY_ESTIMATION_EVENTS_ANALYTICAL = deterministic_sort(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
    (
        "event_partition_order",
        "partition_event_index",
        "primary_event_number",
    ),
)

PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL = deterministic_sort(
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL,
    (
        "event_partition_order",
        "partition_batch_index",
        "primary_event_batch_number",
    ),
)

PRIMARY_SCORING_BATCHES_ANALYTICAL = deterministic_sort(
    PRIMARY_SCORING_BATCHES_ANALYTICAL,
    (
        "event_partition_order",
        "partition_batch_index",
        "primary_event_batch_number",
    ),
)

PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL = (
    deterministic_sort(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
        (
            "event_partition_order",
            "partition_event_index",
            "primary_event_number",
        ),
    )
)


# ------------------------------------------------------------
# Protected-partition firewall
# ------------------------------------------------------------

for frame_label, frame in (
    (
        "PRIMARY_ESTIMATION_EVENTS_ANALYTICAL",
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
    ),
    (
        "PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL",
        PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL,
    ),
    (
        "PRIMARY_SCORING_BATCHES_ANALYTICAL",
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
    ),
    (
        "PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL",
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
    ),
):
    observed_partitions = set(
        frame[
            "event_partition"
        ]
        .dropna()
        .tolist()
    )

    require(
        observed_partitions
        == set(
            ANALYTICAL_PARTITIONS
        ),
        (
            f"{frame_label} contains an unexpected "
            f"partition set: {sorted(observed_partitions)}"
        ),
    )

    require(
        not observed_partitions.intersection(
            PROTECTED_PARTITIONS
        ),
        f"{frame_label} contains protected partition content.",
    )


# ------------------------------------------------------------
# Partition and row-count reconciliation
# ------------------------------------------------------------

partition_reconciliation_rows: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    partition_events = (
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
            PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    partition_batches = (
        PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
            PRIMARY_SCORING_BATCHES_ANALYTICAL[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    expected_counts = (
        EXPECTED_ANALYTICAL_COUNTS_BY_PARTITION[
            partition_name
        ]
    )

    observed_event_count = len(
        partition_events
    )

    observed_batch_count = len(
        partition_batches
    )

    require(
        observed_event_count
        == expected_counts[
            "event_count"
        ],
        (
            f"{partition_name} event-count mismatch: "
            f"expected={expected_counts['event_count']:,}; "
            f"observed={observed_event_count:,}."
        ),
    )

    require(
        observed_batch_count
        == expected_counts[
            "batch_count"
        ],
        (
            f"{partition_name} batch-count mismatch: "
            f"expected={expected_counts['batch_count']:,}; "
            f"observed={observed_batch_count:,}."
        ),
    )

    partition_reconciliation_rows.append(
        {
            "event_partition": partition_name,
            "expected_event_count": expected_counts[
                "event_count"
            ],
            "observed_event_count": observed_event_count,
            "expected_batch_count": expected_counts[
                "batch_count"
            ],
            "observed_batch_count": observed_batch_count,
            "buy_event_count": int(
                partition_events[
                    "event_side"
                ].eq(
                    "BUY"
                ).sum()
            ),
            "sell_event_count": int(
                partition_events[
                    "event_side"
                ].eq(
                    "SELL"
                ).sum()
            ),
            "first_event_time_ns": int(
                partition_events[
                    "event_time_ns"
                ].iloc[0]
            ),
            "last_event_time_ns": int(
                partition_events[
                    "event_time_ns"
                ].iloc[-1]
            ),
            "simultaneous_batch_count": int(
                partition_batches[
                    "simultaneous_batch_required_flag"
                ].sum()
            ),
            "mixed_side_batch_count": int(
                partition_batches[
                    "mixed_side_batch_flag"
                ].sum()
            ),
            "status": "PASS",
        }
    )

NOTEBOOK08_ANALYTICAL_PARTITION_RECONCILIATION = (
    pd.DataFrame(
        partition_reconciliation_rows
    )
)

require(
    len(
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL
    )
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "Total analytical event count is incorrect.",
)

require(
    len(
        PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL
    )
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "Total analytical exact-time batch count is incorrect.",
)

require(
    len(
        PRIMARY_SCORING_BATCHES_ANALYTICAL
    )
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "Total analytical scoring-batch count is incorrect.",
)

require(
    len(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL
    )
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "Total event-to-batch membership count is incorrect.",
)


# ------------------------------------------------------------
# Event identity, side, and chronology gates
# ------------------------------------------------------------

require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "primary_event_id"
    ].notna().all(),
    "At least one analytical event lacks an event ID.",
)

require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "primary_event_id"
    ].is_unique,
    "Analytical event IDs are not unique.",
)

require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "primary_event_number"
    ].is_unique,
    "Analytical event numbers are not unique.",
)

require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_side"
    ].isin(
        EVENT_SIDES
    ).all(),
    "Analytical event sides are invalid.",
)

expected_side_codes = (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_side"
    ].map(
        {
            "BUY": 0,
            "SELL": 1,
        }
    )
)

require(
    np.array_equal(
        expected_side_codes.to_numpy(
            dtype=np.int64
        ),
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "event_side_code"
        ].to_numpy(
            dtype=np.int64
        ),
    ),
    "Event-side codes violate BUY=0, SELL=1.",
)

require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_time_ns"
    ].is_monotonic_increasing,
    "Analytical event times are not globally nondecreasing.",
)

require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_print_count"
    ].ge(
        1
    ).all(),
    "An analytical event has a nonpositive print count.",
)


# ------------------------------------------------------------
# Event-to-batch membership reconciliation
# ------------------------------------------------------------

require(
    PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL[
        "primary_event_id"
    ].is_unique,
    "An analytical event appears more than once in membership.",
)

require(
    set(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL[
            "primary_event_id"
        ]
    )
    == set(
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "primary_event_id"
        ]
    ),
    "Event and membership ID sets do not reconcile.",
)

event_membership_comparison = (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        [
            "primary_event_id",
            "primary_event_number",
            "partition_event_index",
            "primary_event_batch_id",
            "primary_event_batch_number",
            "event_partition_order",
            "event_partition",
            "event_time_ns",
            "event_side",
            "event_side_code",
        ]
    ]
    .merge(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
        on="primary_event_id",
        how="outer",
        validate="one_to_one",
        suffixes=(
            "_event",
            "_membership",
        ),
        indicator=True,
    )
)

require(
    event_membership_comparison[
        "_merge"
    ].eq(
        "both"
    ).all(),
    "Event-to-batch membership merge is incomplete.",
)

for comparison_field in (
    "primary_event_number",
    "partition_event_index",
    "primary_event_batch_id",
    "primary_event_batch_number",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "event_side",
    "event_side_code",
):
    require(
        event_membership_comparison[
            f"{comparison_field}_event"
        ].eq(
            event_membership_comparison[
                f"{comparison_field}_membership"
            ]
        ).all(),
        (
            "Event-to-batch membership mismatch in "
            f"{comparison_field!r}."
        ),
    )


# ------------------------------------------------------------
# Exact-time and scoring-batch reconciliation
# ------------------------------------------------------------

require(
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
        "primary_event_batch_id"
    ].is_unique,
    "Exact-time batch IDs are not unique.",
)

require(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "primary_event_batch_id"
    ].is_unique,
    "Scoring-batch IDs are not unique.",
)

require(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "event_time_ns"
    ].is_monotonic_increasing,
    "Analytical batch times are not globally nondecreasing.",
)

batch_comparison_columns: Final[
    tuple[str, ...]
] = (
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_batch_time_ns",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "unique_side_count",
    "batch_print_count",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "event_clock_origin_ns",
    "observation_window_duration_ns",
)

batch_identity_comparison = (
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
        [
            "primary_event_batch_id",
            *batch_comparison_columns,
        ]
    ]
    .merge(
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            [
                "primary_event_batch_id",
                *batch_comparison_columns,
            ]
        ],
        on="primary_event_batch_id",
        how="outer",
        validate="one_to_one",
        suffixes=(
            "_exact",
            "_scoring",
        ),
        indicator=True,
    )
)

require(
    batch_identity_comparison[
        "_merge"
    ].eq(
        "both"
    ).all(),
    "Exact-time and scoring-batch ID sets do not reconcile.",
)

for comparison_field in batch_comparison_columns:
    require(
        batch_identity_comparison[
            f"{comparison_field}_exact"
        ].eq(
            batch_identity_comparison[
                f"{comparison_field}_scoring"
            ]
        ).all(),
        (
            "Exact-time and scoring-batch mismatch in "
            f"{comparison_field!r}."
        ),
    )


# ------------------------------------------------------------
# Batch conservation and strict scoring semantics
# ------------------------------------------------------------

scoring_batches = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL
)

require(
    scoring_batches[
        "batch_event_count"
    ].ge(
        1
    ).all(),
    "A scoring batch contains no events.",
)

require(
    scoring_batches[
        "buy_event_count"
    ].ge(
        0
    ).all(),
    "A scoring batch has a negative BUY count.",
)

require(
    scoring_batches[
        "sell_event_count"
    ].ge(
        0
    ).all(),
    "A scoring batch has a negative SELL count.",
)

require(
    (
        scoring_batches[
            "buy_event_count"
        ]
        + scoring_batches[
            "sell_event_count"
        ]
    ).eq(
        scoring_batches[
            "batch_event_count"
        ]
    ).all(),
    "BUY and SELL counts do not conserve batch events.",
)

require(
    int(
        scoring_batches[
            "batch_event_count"
        ].sum()
    )
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "Batch event counts do not conserve analytical events.",
)

expected_unique_side_count = (
    scoring_batches[
        "buy_event_count"
    ].gt(
        0
    ).astype(
        "int8"
    )
    + scoring_batches[
        "sell_event_count"
    ].gt(
        0
    ).astype(
        "int8"
    )
)

require(
    expected_unique_side_count.eq(
        scoring_batches[
            "unique_side_count"
        ]
    ).all(),
    "Batch unique-side counts are incorrect.",
)

require(
    scoring_batches[
        "mixed_side_batch_flag"
    ].eq(
        scoring_batches[
            "unique_side_count"
        ].gt(
            1
        )
    ).all(),
    "Mixed-side batch flags are incorrect.",
)

require(
    scoring_batches[
        "simultaneous_batch_required_flag"
    ].eq(
        scoring_batches[
            "batch_event_count"
        ].gt(
            1
        )
    ).all(),
    "Simultaneous-batch flags are incorrect.",
)

require(
    scoring_batches[
        "score_with_history_strictly_before_batch_time"
    ].all(),
    "A batch violates strict pre-batch scoring.",
)

require(
    not scoring_batches[
        "zero_lag_within_batch_excitation_allowed"
    ].any(),
    "A batch permits zero-lag within-batch excitation.",
)

require(
    scoring_batches[
        "apply_batch_excitation_after_all_members_scored"
    ].all(),
    "A batch applies excitation before complete scoring.",
)

require(
    scoring_batches[
        "collector_sequence_is_trace_order_only"
    ].all(),
    "Collector sequence is incorrectly treated as elapsed time.",
)

require(
    not scoring_batches[
        "timestamp_jitter_allowed"
    ].any(),
    "A batch permits timestamp jitter.",
)

require(
    not scoring_batches[
        "event_removal_allowed"
    ].any(),
    "A batch permits event removal.",
)


# ------------------------------------------------------------
# Batch-membership count reconciliation
# ------------------------------------------------------------

membership_batch_summary = (
    PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL.groupby(
        [
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
        ],
        observed=True,
        sort=False,
        dropna=False,
    )
    .agg(
        membership_event_count=(
            "primary_event_id",
            "size",
        ),
        membership_buy_event_count=(
            "event_side",
            lambda values: int(
                values.eq(
                    "BUY"
                ).sum()
            ),
        ),
        membership_sell_event_count=(
            "event_side",
            lambda values: int(
                values.eq(
                    "SELL"
                ).sum()
            ),
        ),
    )
    .reset_index()
)

membership_batch_reconciliation = (
    scoring_batches[
        [
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
        ]
    ]
    .merge(
        membership_batch_summary,
        on=[
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
        ],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

require(
    membership_batch_reconciliation[
        "_merge"
    ].eq(
        "both"
    ).all(),
    "Batch membership does not reconcile with scoring batches.",
)

require(
    membership_batch_reconciliation[
        "batch_event_count"
    ].eq(
        membership_batch_reconciliation[
            "membership_event_count"
        ]
    ).all(),
    "Membership event counts differ from scoring-batch counts.",
)

require(
    membership_batch_reconciliation[
        "buy_event_count"
    ].eq(
        membership_batch_reconciliation[
            "membership_buy_event_count"
        ]
    ).all(),
    "Membership BUY counts differ from scoring-batch counts.",
)

require(
    membership_batch_reconciliation[
        "sell_event_count"
    ].eq(
        membership_batch_reconciliation[
            "membership_sell_event_count"
        ]
    ).all(),
    "Membership SELL counts differ from scoring-batch counts.",
)


# ------------------------------------------------------------
# Observation-window containment and boundary continuity
# ------------------------------------------------------------

window_lookup = (
    OBSERVATION_WINDOW_CONTRACT.set_index(
        "event_partition"
    )
)

for partition_name in ANALYTICAL_PARTITIONS:
    partition_events = (
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
            PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    contract_start_ns = int(
        window_lookup.loc[
            partition_name,
            "contract_start_ns",
        ]
    )

    contract_end_exclusive_ns = int(
        window_lookup.loc[
            partition_name,
            "contract_end_exclusive_ns",
        ]
    )

    require(
        partition_events[
            "event_time_ns"
        ].ge(
            contract_start_ns
        ).all(),
        f"{partition_name} contains events before contract start.",
    )

    require(
        partition_events[
            "event_time_ns"
        ].lt(
            contract_end_exclusive_ns
        ).all(),
        f"{partition_name} contains events outside its contract.",
    )

DEVELOPMENT_START_NS: Final[int] = int(
    window_lookup.loc[
        FIT_PARTITION,
        "contract_start_ns",
    ]
)

DEVELOPMENT_END_EXCLUSIVE_NS: Final[int] = int(
    window_lookup.loc[
        FIT_PARTITION,
        "contract_end_exclusive_ns",
    ]
)

CALIBRATION_START_NS: Final[int] = int(
    window_lookup.loc[
        LOCKED_EVALUATION_PARTITION,
        "contract_start_ns",
    ]
)

CALIBRATION_END_EXCLUSIVE_NS: Final[int] = int(
    window_lookup.loc[
        LOCKED_EVALUATION_PARTITION,
        "contract_end_exclusive_ns",
    ]
)

require(
    CALIBRATION_START_NS
    == DEVELOPMENT_END_EXCLUSIVE_NS,
    (
        "DEVELOPMENT and CALIBRATION windows are not "
        "exactly contiguous."
    ),
)

development_last_event_time_ns = int(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "event_partition"
        ].eq(
            FIT_PARTITION
        ),
        "event_time_ns",
    ].iloc[-1]
)

calibration_first_event_time_ns = int(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "event_partition"
        ].eq(
            LOCKED_EVALUATION_PARTITION
        ),
        "event_time_ns",
    ].iloc[0]
)

require(
    calibration_first_event_time_ns
    > development_last_event_time_ns,
    (
        "The first CALIBRATION event does not occur strictly "
        "after the final DEVELOPMENT event."
    ),
)

DEVELOPMENT_TO_CALIBRATION_EVENT_GAP_NS: Final[int] = (
    calibration_first_event_time_ns
    - development_last_event_time_ns
)


# ------------------------------------------------------------
# Access-state ledger
# ------------------------------------------------------------

PARTITION_CONTENT_LOADED: dict[
    str,
    bool,
] = {
    "DEVELOPMENT": True,
    "CALIBRATION": True,
    "VALIDATION": False,
    "ENGINEERING_HOLDOUT": False,
}

ANALYTICAL_CONTENT_LOADED = True
PROTECTED_PARTITION_METADATA_LOADED = True
FUTURE_LABEL_TABLE_LOADED = False
MARKET_STATE_FEATURE_VALUES_LOADED = False
PROTECTED_PARTITION_CONTENT_LOADED = False
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)


# ------------------------------------------------------------
# Load and reconciliation summaries
# ------------------------------------------------------------

exact_time_batch_summary = pd.DataFrame(
    [
        {
            "metric": "analytical_event_rows",
            "value": len(
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL
            ),
        },
        {
            "metric": "analytical_batch_rows",
            "value": len(
                PRIMARY_SCORING_BATCHES_ANALYTICAL
            ),
        },
        {
            "metric": "buy_event_rows",
            "value": int(
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                    "event_side"
                ].eq(
                    "BUY"
                ).sum()
            ),
        },
        {
            "metric": "sell_event_rows",
            "value": int(
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                    "event_side"
                ].eq(
                    "SELL"
                ).sum()
            ),
        },
        {
            "metric": "simultaneous_batches",
            "value": int(
                scoring_batches[
                    "simultaneous_batch_required_flag"
                ].sum()
            ),
        },
        {
            "metric": "mixed_side_batches",
            "value": int(
                scoring_batches[
                    "mixed_side_batch_flag"
                ].sum()
            ),
        },
        {
            "metric": "maximum_events_in_one_batch",
            "value": int(
                scoring_batches[
                    "batch_event_count"
                ].max()
            ),
        },
        {
            "metric": "development_to_calibration_event_gap_ns",
            "value": DEVELOPMENT_TO_CALIBRATION_EVENT_GAP_NS,
        },
        {
            "metric": "validation_content_loaded",
            "value": False,
        },
        {
            "metric": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "metric": "future_label_table_loaded",
            "value": False,
        },
        {
            "metric": "market_state_feature_values_loaded",
            "value": False,
        },
        {
            "metric": "filesystem_writes_performed",
            "value": False,
        },
    ]
)

display(
    NOTEBOOK08_UPSTREAM_FILE_AUDIT[
        [
            "label",
            "artifact_type",
            "full_row_count",
            "full_column_count",
            "hash_matches",
            "status",
        ]
    ]
)

display(
    NOTEBOOK08_ANALYTICAL_PARTITION_RECONCILIATION
)

display(
    exact_time_batch_summary
)

print(
    "DEVELOPMENT and CALIBRATION analytical content loaded "
    "through partition-filtered Arrow scans. Event identities, "
    "batch memberships, counts, timestamps, contract windows, "
    "and strict pre-batch scoring semantics reconciled. "
    "VALIDATION and ENGINEERING_HOLDOUT content remain unopened. "
    "Future labels and market-state feature values remain unopened. "
    "No filesystem writes were performed."
)

,label,artifact_type,full_row_count,full_column_count,hash_matches,status
0,observation_window_contract,NOTEBOOK_04_OBSERVATION_WINDOW_CONTRACT,4,19,True,PASS
1,primary_batch_market_state_features,PRIMARY_BATCH_MARKET_STATE_FEATURES,13564,577,True,PASS
2,primary_estimation_events,NOTEBOOK_04_PRIMARY_ESTIMATION_EVENTS,13887,88,True,PASS
3,primary_event_to_batch_membership,NOTEBOOK_04_PRIMARY_EVENT_TO_BATCH_MEMBERSHIP,13887,10,True,PASS
4,primary_exact_time_batches,NOTEBOOK_04_PRIMARY_EXACT_TIME_BATCHES,13564,27,True,PASS
5,primary_scoring_batches,NOTEBOOK_04_PRIMARY_SCORING_BATCHES,13564,25,True,PASS


,event_partition,expected_event_count,observed_event_count,expected_batch_count,observed_batch_count,buy_event_count,sell_event_count,first_event_time_ns,last_event_time_ns,simultaneous_batch_count,mixed_side_batch_count,status
0,DEVELOPMENT,7004,7004,6859,6859,3414,3590,1783665468766951600,1783667269232205000,111,26,PASS
1,CALIBRATION,2493,2493,2400,2400,1230,1263,1783667270546982000,1783667989349685300,68,5,PASS


,metric,value
0,analytical_event_rows,9497
1,analytical_batch_rows,9259
2,buy_event_rows,4644
3,sell_event_rows,4853
4,simultaneous_batches,179
5,mixed_side_batches,31
6,maximum_events_in_one_batch,6
7,development_to_calibration_event_gap_ns,1314777000
8,validation_content_loaded,False
9,engineering_holdout_content_loaded,False


DEVELOPMENT and CALIBRATION analytical content loaded through partition-filtered Arrow scans. Event identities, batch memberships, counts, timestamps, contract windows, and strict pre-batch scoring semantics reconciled. VALIDATION and ENGINEERING_HOLDOUT content remain unopened. Future labels and market-state feature values remain unopened. No filesystem writes were performed.


In [6]:
# ============================================================
# Load the frozen H1 model and independently replay
# DEVELOPMENT through locked CALIBRATION
# ============================================================

import math
from dataclasses import dataclass


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    UPSTREAM_AUTHORITY_VERIFIED,
    "Upstream authority has not been verified.",
)

require(
    bool(
        globals().get(
            "ANALYTICAL_CONTENT_LOADED",
            False,
        )
    ),
    "Analytical DEVELOPMENT and CALIBRATION content is unavailable.",
)

require(
    not FROZEN_MODEL_LOADED,
    "The frozen Hawkes model is already loaded.",
)

require(
    not FROZEN_REPLAY_RECONCILED,
    "The frozen Hawkes replay is already reconciled.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Portable role-package helpers
# ------------------------------------------------------------

def require_mapping(
    value: Any,
    *,
    label: str,
) -> Mapping[str, Any]:
    """Return a mapping or raise a blocking contract error."""
    require(
        isinstance(
            value,
            Mapping,
        ),
        f"{label} must be a mapping.",
    )

    return value


def dataframe_from_role_component(
    component: Any,
    *,
    label: str,
) -> pd.DataFrame:
    """Reconstruct one persisted DataFrame role component."""
    component_mapping = require_mapping(
        component,
        label=label,
    )

    expect(
        component_mapping,
        label,
        object_type="DATAFRAME",
    )

    columns = component_mapping.get(
        "columns"
    )

    records = component_mapping.get(
        "records"
    )

    require(
        isinstance(
            columns,
            list,
        )
        and all(
            isinstance(
                column,
                str,
            )
            for column in columns
        ),
        f"{label} has an invalid column registry.",
    )

    require(
        isinstance(
            records,
            list,
        )
        and all(
            isinstance(
                record,
                Mapping,
            )
            for record in records
        ),
        f"{label} has an invalid record registry.",
    )

    frame = pd.DataFrame.from_records(
        records,
        columns=columns,
    )

    require(
        len(frame)
        == int(
            component_mapping[
                "row_count"
            ]
        ),
        f"{label} row count does not reconcile.",
    )

    require(
        len(frame.columns)
        == int(
            component_mapping[
                "column_count"
            ]
        ),
        f"{label} column count does not reconcile.",
    )

    require(
        list(
            frame.columns
        )
        == columns,
        f"{label} column order does not reconcile.",
    )

    return frame


def require_close(
    observed: float,
    expected: float,
    *,
    label: str,
    relative_tolerance: float = FLOAT_RELATIVE_TOLERANCE,
    absolute_tolerance: float = 1e-8,
) -> None:
    """Require two finite floating-point values to agree."""
    observed_float = float(
        observed
    )

    expected_float = float(
        expected
    )

    require(
        math.isfinite(
            observed_float
        ),
        f"{label}: observed value is non-finite.",
    )

    require(
        math.isfinite(
            expected_float
        ),
        f"{label}: expected value is non-finite.",
    )

    require(
        math.isclose(
            observed_float,
            expected_float,
            rel_tol=relative_tolerance,
            abs_tol=absolute_tolerance,
        ),
        (
            f"{label} mismatch: "
            f"observed={observed_float!r}, "
            f"expected={expected_float!r}."
        ),
    )


# ------------------------------------------------------------
# Read the selected-model role package from disk
# ------------------------------------------------------------

SELECTED_MODEL_ROLE_PATH: Final[Path] = (
    NOTEBOOK07_REQUIRED_ARTIFACT_PATHS[
        "selected_model_package"
    ]
)

SELECTED_MODEL_ROLE_PACKAGE = load_json(
    SELECTED_MODEL_ROLE_PATH
)

expect(
    SELECTED_MODEL_ROLE_PACKAGE,
    "Notebook 07 selected-model role package",
    artifact_type=(
        "NOTEBOOK_07_SELECTED_MODEL_PACKAGE"
    ),
    schema_version=(
        "NOTEBOOK_07_SELECTED_MODEL_PACKAGE_V1"
    ),
    source_run_prefix=SOURCE_RUN_PREFIX,
    source_set_sha256=SOURCE_SET_SHA256,
    v0_1_run_id=V01_RUN_ID,
    run_config_sha256=RUN_CONFIG_SHA256,
    run_identity_sha256=RUN_IDENTITY_SHA256,
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    artifact_role=(
        "selected_model_package"
    ),
    status="PASS",
)

selected_model_components = require_mapping(
    SELECTED_MODEL_ROLE_PACKAGE.get(
        "components"
    ),
    label="selected_model_role.components",
)

FROZEN_SELECTED_MODEL_PACKAGE = require_mapping(
    selected_model_components.get(
        "selected_model_package"
    ),
    label=(
        "selected_model_role.components."
        "selected_model_package"
    ),
)

FROZEN_SELECTED_MODEL_PACKAGE_SHA256: Final[str] = (
    canonical_json_sha256(
        FROZEN_SELECTED_MODEL_PACKAGE
    )
)

require(
    FROZEN_SELECTED_MODEL_PACKAGE_SHA256
    == selected_model_components.get(
        "selected_model_package_sha256"
    )
    == SELECTED_MODEL_ROLE_PACKAGE.get(
        "semantic_hashes",
        {},
    ).get(
        "selected_model_package_sha256"
    )
    == selected_model_semantic_sha256,
    "The selected-model semantic hash does not reconcile.",
)

expect(
    FROZEN_SELECTED_MODEL_PACKAGE,
    "Frozen selected Hawkes model",
    artifact_type=(
        "NOTEBOOK_07_SELECTED_HAWKES_MODEL"
    ),
    schema_version=(
        "NOTEBOOK_07_SELECTED_HAWKES_MODEL_V1"
    ),
    producer="07_HAWKES_ESTIMATION.ipynb",
    source_run_prefix=SOURCE_RUN_PREFIX,
    v0_1_run_id=V01_RUN_ID,
    model_id=EXPECTED_SELECTED_MODEL_ID,
    model_family=EXPECTED_SELECTED_MODEL_FAMILY,
    estimator_label=EXPECTED_ESTIMATOR_LABEL,
    fit_partition="DEVELOPMENT",
    fit_event_count=EXPECTED_DEVELOPMENT_EVENT_ROWS,
    fit_batch_count=EXPECTED_DEVELOPMENT_BATCH_ROWS,
    fit_start_ns=DEVELOPMENT_START_NS,
    fit_end_exclusive_ns=DEVELOPMENT_END_EXCLUSIVE_NS,
    calibration_used_for_fitting=False,
    calibration_used_for_selection=False,
    calibration_parameter_updates=0,
    kernel_family=EXPECTED_KERNEL_FAMILY,
    shared_decay=True,
    future_labels_used=False,
    market_state_features_used=False,
    status="FROZEN_DEVELOPMENT_SELECTED_MODEL",
)

require(
    tuple(
        FROZEN_SELECTED_MODEL_PACKAGE[
            "authorized_channels"
        ]
    )
    == AUTHORIZED_CHANNELS,
    "The frozen authorized-channel set changed.",
)

require(
    tuple(
        FROZEN_SELECTED_MODEL_PACKAGE[
            "fixed_zero_channels"
        ]
    )
    == FIXED_ZERO_CHANNELS,
    "The frozen zero-channel set changed.",
)


# ------------------------------------------------------------
# Frozen parameter containers
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class FrozenDiagonalHawkesParameters:
    """Immutable natural parameters for the selected H1 model."""

    model_id: str
    mu_buy: float
    mu_sell: float
    kappa_buy: float
    kappa_sell: float
    beta_buy: float
    beta_sell: float


@dataclass(frozen=True, slots=True)
class FrozenHawkesState:
    """Excitation state expressed at one exact nanosecond."""

    state_time_ns: int
    buy_excitation: float
    sell_excitation: float


@dataclass(frozen=True, slots=True)
class FrozenHawkesReplayResult:
    """Exact strict-pre-batch replay result."""

    model_id: str
    observation_start_ns: int
    observation_end_exclusive_ns: int
    event_count: int
    batch_count: int
    log_event_term: float
    compensator_buy: float
    compensator_sell: float
    total_compensator: float
    log_likelihood: float
    negative_log_likelihood: float
    final_state: FrozenHawkesState
    replay: pd.DataFrame


frozen_parameter_payload = require_mapping(
    FROZEN_SELECTED_MODEL_PACKAGE.get(
        "parameters"
    ),
    label="frozen_model.parameters",
)

FROZEN_H1_PARAMETERS = FrozenDiagonalHawkesParameters(
    model_id=EXPECTED_SELECTED_MODEL_ID,
    mu_buy=float(
        frozen_parameter_payload[
            "mu_buy_per_second"
        ]
    ),
    mu_sell=float(
        frozen_parameter_payload[
            "mu_sell_per_second"
        ]
    ),
    kappa_buy=float(
        frozen_parameter_payload[
            "kappa_buy"
        ]
    ),
    kappa_sell=float(
        frozen_parameter_payload[
            "kappa_sell"
        ]
    ),
    beta_buy=float(
        frozen_parameter_payload[
            "beta_buy_per_second"
        ]
    ),
    beta_sell=float(
        frozen_parameter_payload[
            "beta_sell_per_second"
        ]
    ),
)


# ------------------------------------------------------------
# Parameter and stationarity verification
# ------------------------------------------------------------

frozen_parameter_vector_before = np.asarray(
    [
        FROZEN_H1_PARAMETERS.mu_buy,
        FROZEN_H1_PARAMETERS.mu_sell,
        FROZEN_H1_PARAMETERS.kappa_buy,
        FROZEN_H1_PARAMETERS.kappa_sell,
        FROZEN_H1_PARAMETERS.beta_buy,
        FROZEN_H1_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

require(
    np.isfinite(
        frozen_parameter_vector_before
    ).all(),
    "The frozen Hawkes parameters are non-finite.",
)

require(
    FROZEN_H1_PARAMETERS.mu_buy
    > INTENSITY_FLOOR_PER_SECOND,
    "The frozen BUY base intensity is not positive.",
)

require(
    FROZEN_H1_PARAMETERS.mu_sell
    > INTENSITY_FLOOR_PER_SECOND,
    "The frozen SELL base intensity is not positive.",
)

require(
    0.0
    <= FROZEN_H1_PARAMETERS.kappa_buy
    < 1.0,
    "The frozen BUY excitation mass is inadmissible.",
)

require(
    0.0
    <= FROZEN_H1_PARAMETERS.kappa_sell
    < 1.0,
    "The frozen SELL excitation mass is inadmissible.",
)

require(
    FROZEN_H1_PARAMETERS.beta_buy
    > 0.0,
    "The frozen BUY decay is not positive.",
)

require(
    FROZEN_H1_PARAMETERS.beta_sell
    > 0.0,
    "The frozen SELL decay is not positive.",
)

require_close(
    FROZEN_H1_PARAMETERS.beta_buy,
    FROZEN_H1_PARAMETERS.beta_sell,
    label="Shared BUY/SELL decay",
    absolute_tolerance=FLOAT_ABSOLUTE_TOLERANCE,
)

FROZEN_H1_SPECTRAL_RADIUS: Final[float] = max(
    FROZEN_H1_PARAMETERS.kappa_buy,
    FROZEN_H1_PARAMETERS.kappa_sell,
)

require(
    FROZEN_H1_SPECTRAL_RADIUS
    < 1.0,
    "The frozen H1 model is non-stationary.",
)

require_close(
    FROZEN_H1_SPECTRAL_RADIUS,
    EXPECTED_SELECTED_SPECTRAL_RADIUS,
    label="Frozen H1 spectral radius",
    absolute_tolerance=(
        SPECTRAL_RADIUS_ABSOLUTE_TOLERANCE
    ),
)

require_close(
    FROZEN_H1_PARAMETERS.mu_buy,
    frozen_parameters[
        "mu_buy_per_second"
    ],
    label="Handoff BUY base intensity",
)

require_close(
    FROZEN_H1_PARAMETERS.mu_sell,
    frozen_parameters[
        "mu_sell_per_second"
    ],
    label="Handoff SELL base intensity",
)

require_close(
    FROZEN_H1_PARAMETERS.kappa_buy,
    frozen_parameters[
        "kappa_buy_to_buy"
    ],
    label="Handoff BUY excitation mass",
)

require_close(
    FROZEN_H1_PARAMETERS.kappa_sell,
    frozen_parameters[
        "kappa_sell_to_sell"
    ],
    label="Handoff SELL excitation mass",
)

require_close(
    FROZEN_H1_PARAMETERS.beta_buy,
    frozen_parameters[
        "beta_buy_per_second"
    ],
    label="Handoff BUY decay",
)

require_close(
    FROZEN_H1_PARAMETERS.beta_sell,
    frozen_parameters[
        "beta_sell_per_second"
    ],
    label="Handoff SELL decay",
)

require(
    float(
        frozen_parameters[
            "kappa_buy_to_sell"
        ]
    )
    == 0.0,
    "BUY-to-SELL excitation is not frozen at zero.",
)

require(
    float(
        frozen_parameters[
            "kappa_sell_to_buy"
        ]
    )
    == 0.0,
    "SELL-to-BUY excitation is not frozen at zero.",
)


# ------------------------------------------------------------
# Independent exact replay engine
# ------------------------------------------------------------

def exact_excitation_integral(
    excitation_at_interval_start: float,
    decay_rate: float,
    elapsed_seconds: float,
) -> float:
    """Integrate one exponentially decaying excitation state."""
    require(
        math.isfinite(
            excitation_at_interval_start
        )
        and excitation_at_interval_start
        >= 0.0,
        "Excitation at interval start must be finite and nonnegative.",
    )

    require(
        math.isfinite(
            decay_rate
        )
        and decay_rate
        > 0.0,
        "Decay rate must be finite and positive.",
    )

    require(
        math.isfinite(
            elapsed_seconds
        )
        and elapsed_seconds
        >= 0.0,
        "Elapsed time must be finite and nonnegative.",
    )

    if elapsed_seconds == 0.0:
        return 0.0

    return (
        excitation_at_interval_start
        * (
            -math.expm1(
                -decay_rate
                * elapsed_seconds
            )
        )
        / decay_rate
    )


def validate_frozen_batch_frame(
    batch_frame: pd.DataFrame,
    *,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
) -> pd.DataFrame:
    """Return a validated exact-time BUY/SELL batch table."""
    required_columns = (
        "event_time_ns",
        "buy_event_count",
        "sell_event_count",
    )

    missing_columns = sorted(
        set(
            required_columns
        )
        - set(
            batch_frame.columns
        )
    )

    require(
        not missing_columns,
        (
            "Frozen replay batch input is missing columns: "
            f"{missing_columns}"
        ),
    )

    validated = (
        batch_frame.loc[
            :,
            list(
                required_columns
            ),
        ]
        .copy()
    )

    for integer_column in required_columns:
        validated[
            integer_column
        ] = parse_exact_int64(
            validated[
                integer_column
            ],
            label=(
                "FROZEN_REPLAY."
                f"{integer_column}"
            ),
        )

    validated = deterministic_sort(
        validated,
        (
            "event_time_ns",
        ),
    )

    require(
        validated[
            "event_time_ns"
        ].is_unique,
        "Frozen replay input contains duplicate batch timestamps.",
    )

    require(
        validated[
            "event_time_ns"
        ].is_monotonic_increasing,
        "Frozen replay input is not chronologically ordered.",
    )

    require(
        validated[
            "buy_event_count"
        ].ge(
            0
        ).all(),
        "Frozen replay input contains a negative BUY count.",
    )

    require(
        validated[
            "sell_event_count"
        ].ge(
            0
        ).all(),
        "Frozen replay input contains a negative SELL count.",
    )

    require(
        (
            validated[
                "buy_event_count"
            ]
            + validated[
                "sell_event_count"
            ]
        ).gt(
            0
        ).all(),
        "Frozen replay input contains an empty batch.",
    )

    if not validated.empty:
        require(
            int(
                validated[
                    "event_time_ns"
                ].iloc[0]
            )
            >= int(
                observation_start_ns
            ),
            "Frozen replay begins before its observation window.",
        )

        require(
            int(
                validated[
                    "event_time_ns"
                ].iloc[-1]
            )
            < int(
                observation_end_exclusive_ns
            ),
            "Frozen replay extends beyond its observation window.",
        )

    return validated


def replay_frozen_diagonal_hawkes(
    batch_frame: pd.DataFrame,
    parameters: FrozenDiagonalHawkesParameters,
    *,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
    initial_state: FrozenHawkesState | None = None,
) -> FrozenHawkesReplayResult:
    """
    Independently replay the frozen diagonal exponential Hawkes model.

    Each exact-time batch is scored against history strictly before
    the batch. Excitation from the current batch is applied only after
    both BUY and SELL event contributions have been scored.
    """
    start_ns = int(
        observation_start_ns
    )

    end_ns = int(
        observation_end_exclusive_ns
    )

    require(
        end_ns
        > start_ns,
        "Frozen replay interval has nonpositive duration.",
    )

    batches = validate_frozen_batch_frame(
        batch_frame,
        observation_start_ns=start_ns,
        observation_end_exclusive_ns=end_ns,
    )

    if initial_state is None:
        state = FrozenHawkesState(
            state_time_ns=start_ns,
            buy_excitation=0.0,
            sell_excitation=0.0,
        )
    else:
        state = initial_state

    require(
        int(
            state.state_time_ns
        )
        == start_ns,
        (
            "Initial excitation state is not expressed at the "
            "observation boundary."
        ),
    )

    require(
        math.isfinite(
            state.buy_excitation
        )
        and state.buy_excitation
        >= 0.0,
        "Initial BUY excitation is invalid.",
    )

    require(
        math.isfinite(
            state.sell_excitation
        )
        and state.sell_excitation
        >= 0.0,
        "Initial SELL excitation is invalid.",
    )

    previous_time_ns = start_ns

    post_buy_excitation = float(
        state.buy_excitation
    )

    post_sell_excitation = float(
        state.sell_excitation
    )

    cumulative_log_event_term = 0.0
    cumulative_compensator_buy = 0.0
    cumulative_compensator_sell = 0.0

    replay_rows: list[
        dict[str, Any]
    ] = []

    for batch_number, batch in enumerate(
        batches.itertuples(
            index=False
        ),
        start=1,
    ):
        event_time_ns = int(
            batch.event_time_ns
        )

        buy_event_count = int(
            batch.buy_event_count
        )

        sell_event_count = int(
            batch.sell_event_count
        )

        elapsed_ns = (
            event_time_ns
            - previous_time_ns
        )

        require(
            elapsed_ns
            >= 0,
            "Frozen replay encountered negative elapsed time.",
        )

        elapsed_seconds = (
            elapsed_ns
            / NANOSECONDS_PER_SECOND
        )

        buy_decay_factor = math.exp(
            -parameters.beta_buy
            * elapsed_seconds
        )

        sell_decay_factor = math.exp(
            -parameters.beta_sell
            * elapsed_seconds
        )

        pre_buy_excitation = (
            post_buy_excitation
            * buy_decay_factor
        )

        pre_sell_excitation = (
            post_sell_excitation
            * sell_decay_factor
        )

        lambda_buy_pre = (
            parameters.mu_buy
            + pre_buy_excitation
        )

        lambda_sell_pre = (
            parameters.mu_sell
            + pre_sell_excitation
        )

        require(
            math.isfinite(
                lambda_buy_pre
            )
            and lambda_buy_pre
            >= INTENSITY_FLOOR_PER_SECOND,
            (
                "Frozen replay produced an invalid BUY intensity "
                f"at {event_time_ns}."
            ),
        )

        require(
            math.isfinite(
                lambda_sell_pre
            )
            and lambda_sell_pre
            >= INTENSITY_FLOOR_PER_SECOND,
            (
                "Frozen replay produced an invalid SELL intensity "
                f"at {event_time_ns}."
            ),
        )

        interval_compensator_buy = (
            parameters.mu_buy
            * elapsed_seconds
            + exact_excitation_integral(
                post_buy_excitation,
                parameters.beta_buy,
                elapsed_seconds,
            )
        )

        interval_compensator_sell = (
            parameters.mu_sell
            * elapsed_seconds
            + exact_excitation_integral(
                post_sell_excitation,
                parameters.beta_sell,
                elapsed_seconds,
            )
        )

        log_event_term_buy = (
            buy_event_count
            * math.log(
                lambda_buy_pre
            )
        )

        log_event_term_sell = (
            sell_event_count
            * math.log(
                lambda_sell_pre
            )
        )

        next_post_buy_excitation = (
            pre_buy_excitation
            + buy_event_count
            * parameters.kappa_buy
            * parameters.beta_buy
        )

        next_post_sell_excitation = (
            pre_sell_excitation
            + sell_event_count
            * parameters.kappa_sell
            * parameters.beta_sell
        )

        cumulative_log_event_term += (
            log_event_term_buy
            + log_event_term_sell
        )

        cumulative_compensator_buy += (
            interval_compensator_buy
        )

        cumulative_compensator_sell += (
            interval_compensator_sell
        )

        replay_rows.append(
            {
                "batch_number": batch_number,
                "event_time_ns": event_time_ns,
                "elapsed_from_previous_batch_ns": (
                    elapsed_ns
                ),
                "elapsed_from_previous_batch_seconds": (
                    elapsed_seconds
                ),
                "buy_event_count": buy_event_count,
                "sell_event_count": sell_event_count,
                "pre_buy_excitation": (
                    pre_buy_excitation
                ),
                "pre_sell_excitation": (
                    pre_sell_excitation
                ),
                "lambda_buy_pre": (
                    lambda_buy_pre
                ),
                "lambda_sell_pre": (
                    lambda_sell_pre
                ),
                "interval_compensator_buy": (
                    interval_compensator_buy
                ),
                "interval_compensator_sell": (
                    interval_compensator_sell
                ),
                "log_event_term_buy": (
                    log_event_term_buy
                ),
                "log_event_term_sell": (
                    log_event_term_sell
                ),
                "post_buy_excitation": (
                    next_post_buy_excitation
                ),
                "post_sell_excitation": (
                    next_post_sell_excitation
                ),
                "strict_pre_batch_scoring": True,
                "zero_lag_within_batch_excitation": False,
            }
        )

        post_buy_excitation = (
            next_post_buy_excitation
        )

        post_sell_excitation = (
            next_post_sell_excitation
        )

        previous_time_ns = (
            event_time_ns
        )

    tail_elapsed_ns = (
        end_ns
        - previous_time_ns
    )

    require(
        tail_elapsed_ns
        >= 0,
        "Frozen replay observation end precedes its final batch.",
    )

    tail_elapsed_seconds = (
        tail_elapsed_ns
        / NANOSECONDS_PER_SECOND
    )

    cumulative_compensator_buy += (
        parameters.mu_buy
        * tail_elapsed_seconds
        + exact_excitation_integral(
            post_buy_excitation,
            parameters.beta_buy,
            tail_elapsed_seconds,
        )
    )

    cumulative_compensator_sell += (
        parameters.mu_sell
        * tail_elapsed_seconds
        + exact_excitation_integral(
            post_sell_excitation,
            parameters.beta_sell,
            tail_elapsed_seconds,
        )
    )

    final_buy_excitation = (
        post_buy_excitation
        * math.exp(
            -parameters.beta_buy
            * tail_elapsed_seconds
        )
    )

    final_sell_excitation = (
        post_sell_excitation
        * math.exp(
            -parameters.beta_sell
            * tail_elapsed_seconds
        )
    )

    total_compensator = (
        cumulative_compensator_buy
        + cumulative_compensator_sell
    )

    log_likelihood = (
        cumulative_log_event_term
        - total_compensator
    )

    require(
        math.isfinite(
            log_likelihood
        ),
        "Frozen replay produced a non-finite log likelihood.",
    )

    replay = pd.DataFrame(
        replay_rows
    )

    require(
        len(
            replay
        )
        == len(
            batches
        ),
        "Frozen replay lost an exact-time batch.",
    )

    if not replay.empty:
        replay[
            "cumulative_log_event_term"
        ] = (
            replay[
                [
                    "log_event_term_buy",
                    "log_event_term_sell",
                ]
            ]
            .sum(
                axis=1
            )
            .cumsum()
        )

        replay[
            "cumulative_compensator"
        ] = (
            replay[
                [
                    "interval_compensator_buy",
                    "interval_compensator_sell",
                ]
            ]
            .sum(
                axis=1
            )
            .cumsum()
        )

        replay[
            "cumulative_log_likelihood_before_tail"
        ] = (
            replay[
                "cumulative_log_event_term"
            ]
            - replay[
                "cumulative_compensator"
            ]
        )

    return FrozenHawkesReplayResult(
        model_id=parameters.model_id,
        observation_start_ns=start_ns,
        observation_end_exclusive_ns=end_ns,
        event_count=int(
            batches[
                [
                    "buy_event_count",
                    "sell_event_count",
                ]
            ]
            .to_numpy(
                dtype=np.int64
            )
            .sum()
        ),
        batch_count=len(
            batches
        ),
        log_event_term=float(
            cumulative_log_event_term
        ),
        compensator_buy=float(
            cumulative_compensator_buy
        ),
        compensator_sell=float(
            cumulative_compensator_sell
        ),
        total_compensator=float(
            total_compensator
        ),
        log_likelihood=float(
            log_likelihood
        ),
        negative_log_likelihood=float(
            -log_likelihood
        ),
        final_state=FrozenHawkesState(
            state_time_ns=end_ns,
            buy_excitation=float(
                final_buy_excitation
            ),
            sell_excitation=float(
                final_sell_excitation
            ),
        ),
        replay=replay,
    )


# ------------------------------------------------------------
# Partition-specific and continuous replay inputs
# ------------------------------------------------------------

DEVELOPMENT_BATCHES_FOR_FROZEN_REPLAY = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            "event_partition"
        ].eq(
            "DEVELOPMENT"
        ),
        [
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
    ]
    .copy()
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

CALIBRATION_BATCHES_FOR_FROZEN_REPLAY = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            "event_partition"
        ].eq(
            "CALIBRATION"
        ),
        [
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
    ]
    .copy()
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

ANALYTICAL_BATCHES_FOR_FROZEN_REPLAY = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        [
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ]
    ]
    .copy()
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Independent DEVELOPMENT replay
# ------------------------------------------------------------

FROZEN_DEVELOPMENT_REPLAY_RESULT = (
    replay_frozen_diagonal_hawkes(
        DEVELOPMENT_BATCHES_FOR_FROZEN_REPLAY,
        FROZEN_H1_PARAMETERS,
        observation_start_ns=(
            DEVELOPMENT_START_NS
        ),
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
        initial_state=None,
    )
)

require(
    FROZEN_DEVELOPMENT_REPLAY_RESULT.event_count
    == EXPECTED_DEVELOPMENT_EVENT_ROWS,
    "Independent DEVELOPMENT replay lost events.",
)

require(
    FROZEN_DEVELOPMENT_REPLAY_RESULT.batch_count
    == EXPECTED_DEVELOPMENT_BATCH_ROWS,
    "Independent DEVELOPMENT replay lost batches.",
)

development_fit_payload = require_mapping(
    FROZEN_SELECTED_MODEL_PACKAGE.get(
        "development_fit"
    ),
    label="frozen_model.development_fit",
)

require_close(
    FROZEN_DEVELOPMENT_REPLAY_RESULT.log_likelihood,
    development_fit_payload[
        "log_likelihood"
    ],
    label=(
        "Independent DEVELOPMENT log likelihood"
    ),
)

development_terminal_payload = require_mapping(
    FROZEN_SELECTED_MODEL_PACKAGE.get(
        "development_terminal_state"
    ),
    label=(
        "frozen_model.development_terminal_state"
    ),
)

require(
    FROZEN_DEVELOPMENT_REPLAY_RESULT
    .final_state
    .state_time_ns
    == int(
        development_terminal_payload[
            "state_time_ns"
        ]
    )
    == CALIBRATION_START_NS,
    (
        "Independent DEVELOPMENT terminal-state time "
        "does not reconcile."
    ),
)

require_close(
    FROZEN_DEVELOPMENT_REPLAY_RESULT
    .final_state
    .buy_excitation,
    development_terminal_payload[
        "buy_excitation"
    ],
    label=(
        "Independent DEVELOPMENT terminal BUY excitation"
    ),
)

require_close(
    FROZEN_DEVELOPMENT_REPLAY_RESULT
    .final_state
    .sell_excitation,
    development_terminal_payload[
        "sell_excitation"
    ],
    label=(
        "Independent DEVELOPMENT terminal SELL excitation"
    ),
)


# ------------------------------------------------------------
# Independent locked CALIBRATION replay
# ------------------------------------------------------------

FROZEN_CALIBRATION_REPLAY_RESULT = (
    replay_frozen_diagonal_hawkes(
        CALIBRATION_BATCHES_FOR_FROZEN_REPLAY,
        FROZEN_H1_PARAMETERS,
        observation_start_ns=(
            CALIBRATION_START_NS
        ),
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        initial_state=(
            FROZEN_DEVELOPMENT_REPLAY_RESULT
            .final_state
        ),
    )
)

require(
    FROZEN_CALIBRATION_REPLAY_RESULT.event_count
    == EXPECTED_CALIBRATION_EVENT_ROWS,
    "Independent CALIBRATION replay lost events.",
)

require(
    FROZEN_CALIBRATION_REPLAY_RESULT.batch_count
    == EXPECTED_CALIBRATION_BATCH_ROWS,
    "Independent CALIBRATION replay lost batches.",
)


# ------------------------------------------------------------
# Independent continuous replay
# ------------------------------------------------------------

FROZEN_CONTINUOUS_REPLAY_RESULT = (
    replay_frozen_diagonal_hawkes(
        ANALYTICAL_BATCHES_FOR_FROZEN_REPLAY,
        FROZEN_H1_PARAMETERS,
        observation_start_ns=(
            DEVELOPMENT_START_NS
        ),
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        initial_state=None,
    )
)

require_close(
    (
        FROZEN_DEVELOPMENT_REPLAY_RESULT
        .log_likelihood
        + FROZEN_CALIBRATION_REPLAY_RESULT
        .log_likelihood
    ),
    FROZEN_CONTINUOUS_REPLAY_RESULT.log_likelihood,
    label=(
        "Split versus continuous log likelihood"
    ),
)

require_close(
    (
        FROZEN_DEVELOPMENT_REPLAY_RESULT
        .compensator_buy
        + FROZEN_CALIBRATION_REPLAY_RESULT
        .compensator_buy
    ),
    FROZEN_CONTINUOUS_REPLAY_RESULT.compensator_buy,
    label=(
        "Split versus continuous BUY compensator"
    ),
)

require_close(
    (
        FROZEN_DEVELOPMENT_REPLAY_RESULT
        .compensator_sell
        + FROZEN_CALIBRATION_REPLAY_RESULT
        .compensator_sell
    ),
    FROZEN_CONTINUOUS_REPLAY_RESULT.compensator_sell,
    label=(
        "Split versus continuous SELL compensator"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT
    .final_state
    .buy_excitation,
    FROZEN_CONTINUOUS_REPLAY_RESULT
    .final_state
    .buy_excitation,
    label=(
        "Split versus continuous terminal BUY excitation"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT
    .final_state
    .sell_excitation,
    FROZEN_CONTINUOUS_REPLAY_RESULT
    .final_state
    .sell_excitation,
    label=(
        "Split versus continuous terminal SELL excitation"
    ),
)


# ------------------------------------------------------------
# Attach authoritative CALIBRATION batch identities
# ------------------------------------------------------------

calibration_batch_identity = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            "event_partition"
        ].eq(
            "CALIBRATION"
        ),
        [
            "primary_event_batch_id",
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_partition",
            "event_time_ns",
            "relative_batch_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "mixed_side_batch_flag",
            "simultaneous_batch_required_flag",
        ],
    ]
    .copy()
    .sort_values(
        [
            "event_time_ns",
            "primary_event_batch_number",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

INDEPENDENT_LOCKED_CALIBRATION_REPLAY = (
    calibration_batch_identity.merge(
        FROZEN_CALIBRATION_REPLAY_RESULT.replay,
        on=[
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

require(
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "_merge"
    ].eq(
        "both"
    ).all(),
    (
        "Independent CALIBRATION replay does not reconcile "
        "with authoritative batch identities."
    ),
)

INDEPENDENT_LOCKED_CALIBRATION_REPLAY = (
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY.drop(
        columns="_merge"
    )
    .sort_values(
        [
            "event_time_ns",
            "primary_event_batch_number",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
    "buy_excitation_share_pre"
] = (
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "pre_buy_excitation"
    ]
    / INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "lambda_buy_pre"
    ]
)

INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
    "sell_excitation_share_pre"
] = (
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "pre_sell_excitation"
    ]
    / INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "lambda_sell_pre"
    ]
)


# ------------------------------------------------------------
# Load persisted locked-CALIBRATION replay
# ------------------------------------------------------------

LOCKED_REPLAY_ROLE_PATH: Final[Path] = (
    NOTEBOOK07_REQUIRED_ARTIFACT_PATHS[
        "locked_calibration_replay"
    ]
)

LOCKED_REPLAY_ROLE_PACKAGE = load_json(
    LOCKED_REPLAY_ROLE_PATH
)

expect(
    LOCKED_REPLAY_ROLE_PACKAGE,
    "Notebook 07 locked-CALIBRATION replay role package",
    artifact_type=(
        "NOTEBOOK_07_LOCKED_CALIBRATION_REPLAY"
    ),
    schema_version=(
        "NOTEBOOK_07_LOCKED_CALIBRATION_REPLAY_V1"
    ),
    source_run_prefix=SOURCE_RUN_PREFIX,
    source_set_sha256=SOURCE_SET_SHA256,
    v0_1_run_id=V01_RUN_ID,
    run_config_sha256=RUN_CONFIG_SHA256,
    run_identity_sha256=RUN_IDENTITY_SHA256,
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    artifact_role=(
        "locked_calibration_replay"
    ),
    status="PASS",
)

locked_replay_components = require_mapping(
    LOCKED_REPLAY_ROLE_PACKAGE.get(
        "components"
    ),
    label="locked_replay_role.components",
)

PERSISTED_LOCKED_CALIBRATION_REPLAY = (
    dataframe_from_role_component(
        locked_replay_components.get(
            "locked_calibration_replay"
        ),
        label=(
            "locked_replay_role.components."
            "locked_calibration_replay"
        ),
    )
)

require(
    len(
        PERSISTED_LOCKED_CALIBRATION_REPLAY
    )
    == EXPECTED_CALIBRATION_BATCH_ROWS,
    "Persisted CALIBRATION replay has the wrong row count.",
)


# ------------------------------------------------------------
# Normalize persisted replay dtypes
# ------------------------------------------------------------

persisted_replay_integer_columns: Final[
    tuple[str, ...]
] = (
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_time_ns",
    "relative_batch_time_ns",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "batch_number",
    "elapsed_from_previous_batch_ns",
)

for integer_column in persisted_replay_integer_columns:
    PERSISTED_LOCKED_CALIBRATION_REPLAY[
        integer_column
    ] = parse_exact_int64(
        PERSISTED_LOCKED_CALIBRATION_REPLAY[
            integer_column
        ],
        label=(
            "PERSISTED_LOCKED_CALIBRATION_REPLAY."
            f"{integer_column}"
        ),
    )

for boolean_column in (
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "strict_pre_batch_scoring",
    "zero_lag_within_batch_excitation",
):
    PERSISTED_LOCKED_CALIBRATION_REPLAY[
        boolean_column
    ] = normalize_boolean(
        PERSISTED_LOCKED_CALIBRATION_REPLAY[
            boolean_column
        ],
        label=(
            "PERSISTED_LOCKED_CALIBRATION_REPLAY."
            f"{boolean_column}"
        ),
    )

PERSISTED_LOCKED_CALIBRATION_REPLAY[
    "event_partition"
] = (
    PERSISTED_LOCKED_CALIBRATION_REPLAY[
        "event_partition"
    ]
    .astype(
        "string"
    )
    .str.strip()
    .str.upper()
)

PERSISTED_LOCKED_CALIBRATION_REPLAY = (
    PERSISTED_LOCKED_CALIBRATION_REPLAY.sort_values(
        [
            "event_time_ns",
            "primary_event_batch_number",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Row-level replay reconciliation
# ------------------------------------------------------------

exact_replay_columns: Final[
    tuple[str, ...]
] = (
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_batch_time_ns",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "batch_number",
    "elapsed_from_previous_batch_ns",
    "strict_pre_batch_scoring",
    "zero_lag_within_batch_excitation",
)

float_replay_columns: Final[
    tuple[str, ...]
] = (
    "elapsed_from_previous_batch_seconds",
    "pre_buy_excitation",
    "pre_sell_excitation",
    "lambda_buy_pre",
    "lambda_sell_pre",
    "interval_compensator_buy",
    "interval_compensator_sell",
    "log_event_term_buy",
    "log_event_term_sell",
    "post_buy_excitation",
    "post_sell_excitation",
    "cumulative_log_event_term",
    "cumulative_compensator",
    "cumulative_log_likelihood_before_tail",
    "buy_excitation_share_pre",
    "sell_excitation_share_pre",
)

required_replay_columns = set(
    exact_replay_columns
).union(
    float_replay_columns
)

for frame_label, frame in (
    (
        "independent",
        INDEPENDENT_LOCKED_CALIBRATION_REPLAY,
    ),
    (
        "persisted",
        PERSISTED_LOCKED_CALIBRATION_REPLAY,
    ),
):
    missing_columns = sorted(
        required_replay_columns
        - set(
            frame.columns
        )
    )

    require(
        not missing_columns,
        (
            f"The {frame_label} CALIBRATION replay is "
            f"missing columns: {missing_columns}"
        ),
    )


for exact_column in exact_replay_columns:
    independent_values = (
        INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
            exact_column
        ]
    )

    persisted_values = (
        PERSISTED_LOCKED_CALIBRATION_REPLAY[
            exact_column
        ]
    )

    require(
        independent_values.eq(
            persisted_values
        ).all(),
        (
            "Independent and persisted CALIBRATION replay "
            f"differ in {exact_column!r}."
        ),
    )


replay_numeric_reconciliation_rows: list[
    dict[str, Any]
] = []

for float_column in float_replay_columns:
    independent_values = (
        INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
            float_column
        ].to_numpy(
            dtype=np.float64
        )
    )

    persisted_values = (
        PERSISTED_LOCKED_CALIBRATION_REPLAY[
            float_column
        ].to_numpy(
            dtype=np.float64
        )
    )

    require(
        np.isfinite(
            independent_values
        ).all(),
        (
            "Independent replay contains non-finite values "
            f"in {float_column!r}."
        ),
    )

    require(
        np.isfinite(
            persisted_values
        ).all(),
        (
            "Persisted replay contains non-finite values "
            f"in {float_column!r}."
        ),
    )

    maximum_absolute_difference = float(
        np.max(
            np.abs(
                independent_values
                - persisted_values
            )
        )
    )

    maximum_relative_difference = float(
        np.max(
            np.abs(
                independent_values
                - persisted_values
            )
            / np.maximum(
                np.maximum(
                    np.abs(
                        independent_values
                    ),
                    np.abs(
                        persisted_values
                    ),
                ),
                FLOAT_ABSOLUTE_TOLERANCE,
            )
        )
    )

    values_match = bool(
        np.allclose(
            independent_values,
            persisted_values,
            rtol=(
                FLOAT_RELATIVE_TOLERANCE
            ),
            atol=(
                FLOAT_ABSOLUTE_TOLERANCE
            ),
            equal_nan=False,
        )
    )

    require(
        values_match,
        (
            "Independent and persisted CALIBRATION replay "
            f"differ materially in {float_column!r}."
        ),
    )

    replay_numeric_reconciliation_rows.append(
        {
            "column": float_column,
            "row_count": len(
                independent_values
            ),
            "maximum_absolute_difference": (
                maximum_absolute_difference
            ),
            "maximum_relative_difference": (
                maximum_relative_difference
            ),
            "passed": values_match,
            "status": (
                "PASS"
                if values_match
                else "FAIL"
            ),
        }
    )

FROZEN_REPLAY_NUMERIC_RECONCILIATION = pd.DataFrame(
    replay_numeric_reconciliation_rows
)


# ------------------------------------------------------------
# Recompute the persisted replay semantic hash
# ------------------------------------------------------------

def portable_locked_calibration_records(
    replay_table: pd.DataFrame,
) -> list[dict[str, Any]]:
    """Return the exact Notebook 07 portable replay interface."""
    records: list[
        dict[str, Any]
    ] = []

    for row in replay_table.itertuples(
        index=False
    ):
        records.append(
            {
                "primary_event_batch_id": str(
                    row.primary_event_batch_id
                ),
                "primary_event_batch_number": int(
                    row.primary_event_batch_number
                ),
                "partition_batch_index": int(
                    row.partition_batch_index
                ),
                "event_time_ns": int(
                    row.event_time_ns
                ),
                "buy_event_count": int(
                    row.buy_event_count
                ),
                "sell_event_count": int(
                    row.sell_event_count
                ),
                "lambda_buy_pre": float(
                    row.lambda_buy_pre
                ),
                "lambda_sell_pre": float(
                    row.lambda_sell_pre
                ),
                "pre_buy_excitation": float(
                    row.pre_buy_excitation
                ),
                "pre_sell_excitation": float(
                    row.pre_sell_excitation
                ),
                "post_buy_excitation": float(
                    row.post_buy_excitation
                ),
                "post_sell_excitation": float(
                    row.post_sell_excitation
                ),
                "interval_compensator_buy": float(
                    row.interval_compensator_buy
                ),
                "interval_compensator_sell": float(
                    row.interval_compensator_sell
                ),
                "log_event_term_buy": float(
                    row.log_event_term_buy
                ),
                "log_event_term_sell": float(
                    row.log_event_term_sell
                ),
                "strict_pre_batch_scoring": bool(
                    row.strict_pre_batch_scoring
                ),
                "zero_lag_within_batch_excitation": bool(
                    row.zero_lag_within_batch_excitation
                ),
            }
        )

    return records


INDEPENDENT_LOCKED_CALIBRATION_REPLAY_SHA256: Final[str] = (
    canonical_json_sha256(
        {
            "schema_version": (
                "NOTEBOOK_07_LOCKED_CALIBRATION_"
                "HAWKES_REPLAY_V1"
            ),
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "selected_model_package_sha256": (
                FROZEN_SELECTED_MODEL_PACKAGE_SHA256
            ),
            "records": (
                portable_locked_calibration_records(
                    INDEPENDENT_LOCKED_CALIBRATION_REPLAY
                )
            ),
        }
    )
)

require(
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY_SHA256
    == locked_replay_components.get(
        "replay_sha256"
    )
    == LOCKED_REPLAY_ROLE_PACKAGE.get(
        "semantic_hashes",
        {},
    ).get(
        "replay_sha256"
    ),
    "The locked CALIBRATION replay semantic hash does not reconcile.",
)


# ------------------------------------------------------------
# Load and reconcile the locked-CALIBRATION package
# ------------------------------------------------------------

LOCKED_PACKAGE_ROLE_PATH: Final[Path] = (
    NOTEBOOK07_REQUIRED_ARTIFACT_PATHS[
        "locked_calibration_package"
    ]
)

LOCKED_PACKAGE_ROLE = load_json(
    LOCKED_PACKAGE_ROLE_PATH
)

expect(
    LOCKED_PACKAGE_ROLE,
    "Notebook 07 locked-CALIBRATION package role",
    artifact_type=(
        "NOTEBOOK_07_LOCKED_CALIBRATION_PACKAGE"
    ),
    schema_version=(
        "NOTEBOOK_07_LOCKED_CALIBRATION_PACKAGE_V1"
    ),
    source_run_prefix=SOURCE_RUN_PREFIX,
    source_set_sha256=SOURCE_SET_SHA256,
    v0_1_run_id=V01_RUN_ID,
    run_config_sha256=RUN_CONFIG_SHA256,
    run_identity_sha256=RUN_IDENTITY_SHA256,
    terminal_status=(
        EXPECTED_NOTEBOOK07_TERMINAL_STATUS
    ),
    selected_model_id=(
        EXPECTED_SELECTED_MODEL_ID
    ),
    artifact_role=(
        "locked_calibration_package"
    ),
    status="PASS",
)

locked_package_components = require_mapping(
    LOCKED_PACKAGE_ROLE.get(
        "components"
    ),
    label="locked_calibration_package_role.components",
)

FROZEN_LOCKED_CALIBRATION_PACKAGE = require_mapping(
    locked_package_components.get(
        "locked_calibration_package"
    ),
    label=(
        "locked_calibration_package_role.components."
        "locked_calibration_package"
    ),
)

FROZEN_LOCKED_CALIBRATION_PACKAGE_SHA256: Final[str] = (
    canonical_json_sha256(
        FROZEN_LOCKED_CALIBRATION_PACKAGE
    )
)

require(
    FROZEN_LOCKED_CALIBRATION_PACKAGE_SHA256
    == locked_package_components.get(
        "locked_calibration_package_sha256"
    )
    == LOCKED_PACKAGE_ROLE.get(
        "semantic_hashes",
        {},
    ).get(
        "locked_calibration_package_sha256"
    ),
    "The locked-CALIBRATION package semantic hash does not reconcile.",
)

expect(
    FROZEN_LOCKED_CALIBRATION_PACKAGE,
    "Frozen locked-CALIBRATION evaluation",
    artifact_type=(
        "NOTEBOOK_07_LOCKED_CALIBRATION_HAWKES_EVALUATION"
    ),
    schema_version=(
        "NOTEBOOK_07_LOCKED_CALIBRATION_HAWKES_EVALUATION_V1"
    ),
    producer="07_HAWKES_ESTIMATION.ipynb",
    source_run_prefix=SOURCE_RUN_PREFIX,
    v0_1_run_id=V01_RUN_ID,
    model_id=EXPECTED_SELECTED_MODEL_ID,
    selected_model_package_sha256=(
        FROZEN_SELECTED_MODEL_PACKAGE_SHA256
    ),
    replay_sha256=(
        INDEPENDENT_LOCKED_CALIBRATION_REPLAY_SHA256
    ),
    evaluation_partition="CALIBRATION",
    observation_start_ns=CALIBRATION_START_NS,
    observation_end_exclusive_ns=(
        CALIBRATION_END_EXCLUSIVE_NS
    ),
    event_count=EXPECTED_CALIBRATION_EVENT_ROWS,
    batch_count=EXPECTED_CALIBRATION_BATCH_ROWS,
    initial_state_source=(
        "TERMINAL_DEVELOPMENT_STATE"
    ),
    continuous_replay_reconciled=True,
    natural_parameters_unchanged=True,
    optimizer_vector_unchanged=True,
    model_package_unchanged=True,
    parameter_updates=0,
    reset_replay_authorized=False,
)

persisted_locked_score = require_mapping(
    FROZEN_LOCKED_CALIBRATION_PACKAGE.get(
        "locked_score"
    ),
    label="locked_calibration_package.locked_score",
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT.log_event_term,
    persisted_locked_score[
        "log_event_term"
    ],
    label=(
        "Locked CALIBRATION log-event term"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT.compensator_buy,
    persisted_locked_score[
        "compensator_buy"
    ],
    label=(
        "Locked CALIBRATION BUY compensator"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT.compensator_sell,
    persisted_locked_score[
        "compensator_sell"
    ],
    label=(
        "Locked CALIBRATION SELL compensator"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT.total_compensator,
    persisted_locked_score[
        "total_compensator"
    ],
    label=(
        "Locked CALIBRATION total compensator"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT.log_likelihood,
    persisted_locked_score[
        "log_likelihood"
    ],
    label=(
        "Locked CALIBRATION log likelihood"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT.negative_log_likelihood,
    persisted_locked_score[
        "negative_log_likelihood"
    ],
    label=(
        "Locked CALIBRATION negative log likelihood"
    ),
)

persisted_terminal_state = require_mapping(
    FROZEN_LOCKED_CALIBRATION_PACKAGE.get(
        "terminal_state"
    ),
    label="locked_calibration_package.terminal_state",
)

require(
    FROZEN_CALIBRATION_REPLAY_RESULT
    .final_state
    .state_time_ns
    == int(
        persisted_terminal_state[
            "state_time_ns"
        ]
    )
    == CALIBRATION_END_EXCLUSIVE_NS,
    "Locked CALIBRATION terminal-state time does not reconcile.",
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT
    .final_state
    .buy_excitation,
    persisted_terminal_state[
        "buy_excitation"
    ],
    label=(
        "Locked CALIBRATION terminal BUY excitation"
    ),
)

require_close(
    FROZEN_CALIBRATION_REPLAY_RESULT
    .final_state
    .sell_excitation,
    persisted_terminal_state[
        "sell_excitation"
    ],
    label=(
        "Locked CALIBRATION terminal SELL excitation"
    ),
)


# ------------------------------------------------------------
# Verify first CALIBRATION intensity inherited DEVELOPMENT state
# ------------------------------------------------------------

first_calibration_time_ns: Final[int] = int(
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "event_time_ns"
    ].iloc[0]
)

boundary_to_first_calibration_batch_seconds: Final[float] = (
    first_calibration_time_ns
    - CALIBRATION_START_NS
) / NANOSECONDS_PER_SECOND

expected_first_buy_excitation = (
    FROZEN_DEVELOPMENT_REPLAY_RESULT
    .final_state
    .buy_excitation
    * math.exp(
        -FROZEN_H1_PARAMETERS.beta_buy
        * boundary_to_first_calibration_batch_seconds
    )
)

expected_first_sell_excitation = (
    FROZEN_DEVELOPMENT_REPLAY_RESULT
    .final_state
    .sell_excitation
    * math.exp(
        -FROZEN_H1_PARAMETERS.beta_sell
        * boundary_to_first_calibration_batch_seconds
    )
)

require_close(
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "pre_buy_excitation"
    ].iloc[0],
    expected_first_buy_excitation,
    label=(
        "First CALIBRATION inherited BUY excitation"
    ),
)

require_close(
    INDEPENDENT_LOCKED_CALIBRATION_REPLAY[
        "pre_sell_excitation"
    ].iloc[0],
    expected_first_sell_excitation,
    label=(
        "First CALIBRATION inherited SELL excitation"
    ),
)


# ------------------------------------------------------------
# Verify zero parameter mutation
# ------------------------------------------------------------

frozen_parameter_vector_after = np.asarray(
    [
        FROZEN_H1_PARAMETERS.mu_buy,
        FROZEN_H1_PARAMETERS.mu_sell,
        FROZEN_H1_PARAMETERS.kappa_buy,
        FROZEN_H1_PARAMETERS.kappa_sell,
        FROZEN_H1_PARAMETERS.beta_buy,
        FROZEN_H1_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

require(
    np.array_equal(
        frozen_parameter_vector_before,
        frozen_parameter_vector_after,
    ),
    "Frozen Hawkes parameters changed during independent replay.",
)


# ------------------------------------------------------------
# Publish verified frozen-model authority
# ------------------------------------------------------------

FROZEN_MODEL_LOADED = True
FROZEN_REPLAY_RECONCILED = True
CALIBRATION_PARAMETER_UPDATES_PERFORMED = 0
FILESYSTEM_WRITES_PERFORMED = False

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was opened.",
)

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


# ------------------------------------------------------------
# Verification summaries
# ------------------------------------------------------------

FROZEN_H1_PARAMETER_SUMMARY = pd.DataFrame(
    [
        {
            "parameter": "mu_buy_per_second",
            "value": FROZEN_H1_PARAMETERS.mu_buy,
        },
        {
            "parameter": "mu_sell_per_second",
            "value": FROZEN_H1_PARAMETERS.mu_sell,
        },
        {
            "parameter": "kappa_buy_to_buy",
            "value": FROZEN_H1_PARAMETERS.kappa_buy,
        },
        {
            "parameter": "kappa_sell_to_sell",
            "value": FROZEN_H1_PARAMETERS.kappa_sell,
        },
        {
            "parameter": "kappa_buy_to_sell",
            "value": 0.0,
        },
        {
            "parameter": "kappa_sell_to_buy",
            "value": 0.0,
        },
        {
            "parameter": "beta_buy_per_second",
            "value": FROZEN_H1_PARAMETERS.beta_buy,
        },
        {
            "parameter": "beta_sell_per_second",
            "value": FROZEN_H1_PARAMETERS.beta_sell,
        },
        {
            "parameter": "spectral_radius",
            "value": FROZEN_H1_SPECTRAL_RADIUS,
        },
    ]
)

FROZEN_H1_REPLAY_SUMMARY = pd.DataFrame(
    [
        {
            "replay_scope": "DEVELOPMENT",
            "event_count": (
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .event_count
            ),
            "batch_count": (
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .batch_count
            ),
            "log_event_term": (
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .log_event_term
            ),
            "compensator_buy": (
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .compensator_buy
            ),
            "compensator_sell": (
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .compensator_sell
            ),
            "log_likelihood": (
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .log_likelihood
            ),
            "log_score_per_event": (
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .log_likelihood
                / FROZEN_DEVELOPMENT_REPLAY_RESULT
                .event_count
            ),
            "status": "PASS",
        },
        {
            "replay_scope": "CALIBRATION_LOCKED",
            "event_count": (
                FROZEN_CALIBRATION_REPLAY_RESULT
                .event_count
            ),
            "batch_count": (
                FROZEN_CALIBRATION_REPLAY_RESULT
                .batch_count
            ),
            "log_event_term": (
                FROZEN_CALIBRATION_REPLAY_RESULT
                .log_event_term
            ),
            "compensator_buy": (
                FROZEN_CALIBRATION_REPLAY_RESULT
                .compensator_buy
            ),
            "compensator_sell": (
                FROZEN_CALIBRATION_REPLAY_RESULT
                .compensator_sell
            ),
            "log_likelihood": (
                FROZEN_CALIBRATION_REPLAY_RESULT
                .log_likelihood
            ),
            "log_score_per_event": (
                FROZEN_CALIBRATION_REPLAY_RESULT
                .log_likelihood
                / FROZEN_CALIBRATION_REPLAY_RESULT
                .event_count
            ),
            "status": "PASS",
        },
        {
            "replay_scope": (
                "DEVELOPMENT_PLUS_CALIBRATION_CONTINUOUS"
            ),
            "event_count": (
                FROZEN_CONTINUOUS_REPLAY_RESULT
                .event_count
            ),
            "batch_count": (
                FROZEN_CONTINUOUS_REPLAY_RESULT
                .batch_count
            ),
            "log_event_term": (
                FROZEN_CONTINUOUS_REPLAY_RESULT
                .log_event_term
            ),
            "compensator_buy": (
                FROZEN_CONTINUOUS_REPLAY_RESULT
                .compensator_buy
            ),
            "compensator_sell": (
                FROZEN_CONTINUOUS_REPLAY_RESULT
                .compensator_sell
            ),
            "log_likelihood": (
                FROZEN_CONTINUOUS_REPLAY_RESULT
                .log_likelihood
            ),
            "log_score_per_event": (
                FROZEN_CONTINUOUS_REPLAY_RESULT
                .log_likelihood
                / FROZEN_CONTINUOUS_REPLAY_RESULT
                .event_count
            ),
            "status": "PASS",
        },
    ]
)

FROZEN_MODEL_VERIFICATION_SUMMARY = pd.DataFrame(
    [
        {
            "check_name": (
                "selected_model_package_semantic_hash"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "frozen_parameter_identity"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "stationarity"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "development_replay_matches_frozen_fit"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "development_terminal_state_matches"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_history_inherited"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "persisted_calibration_replay_rows_match"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "persisted_calibration_replay_hash_matches"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "locked_calibration_score_matches"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "split_and_continuous_replay_match"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "filesystem_writes_performed"
            ),
            "passed": True,
        },
    ]
)

display(
    FROZEN_H1_PARAMETER_SUMMARY
)

display(
    FROZEN_H1_REPLAY_SUMMARY
)

display(
    FROZEN_REPLAY_NUMERIC_RECONCILIATION
)

display(
    FROZEN_MODEL_VERIFICATION_SUMMARY
)

print(
    "Frozen H1 parameters loaded and independently replayed. "
    "DEVELOPMENT fit, inherited CALIBRATION state, persisted "
    "CALIBRATION replay, compensators, likelihoods, terminal "
    "states, semantic hashes, and continuous-history equivalence "
    "all reconcile. CALIBRATION parameter updates remain zero. "
    "VALIDATION and ENGINEERING_HOLDOUT remain unopened. "
    "No filesystem writes were performed."
)

,parameter,value
0,mu_buy_per_second,1.536124799
1,mu_sell_per_second,1.767916532
2,kappa_buy_to_buy,0.1892556591
3,kappa_sell_to_sell,0.1126637105
4,kappa_buy_to_sell,0
5,kappa_sell_to_buy,0
6,beta_buy_per_second,70.67941658
7,beta_sell_per_second,70.67941658
8,spectral_radius,0.1892556591


,replay_scope,event_count,batch_count,log_event_term,compensator_buy,compensator_sell,log_likelihood,log_score_per_event,status
0,DEVELOPMENT,7004,6859,"5,998.536946","3,414.000015","3,590.00007","-1,005.46314",-0.1435555596,PASS
1,CALIBRATION_LOCKED,2493,2400,"2,302.239061","1,339.253892","1,415.723094",-452.7379259,-0.1816036606,PASS
2,DEVELOPMENT_PLUS_CALIBRATION_CONTINUOUS,9497,9259,"8,300.776006","4,753.253908","5,005.723164","-1,458.201066",-0.1535433364,PASS


,column,row_count,maximum_absolute_difference,maximum_relative_difference,passed,status
0,elapsed_from_previous_batch_seconds,2400,0,0,True,PASS
1,pre_buy_excitation,2400,0,0,True,PASS
2,pre_sell_excitation,2400,0,0,True,PASS
3,lambda_buy_pre,2400,0,0,True,PASS
4,lambda_sell_pre,2400,0,0,True,PASS
5,interval_compensator_buy,2400,0,0,True,PASS
6,interval_compensator_sell,2400,0,0,True,PASS
7,log_event_term_buy,2400,0,0,True,PASS
8,log_event_term_sell,2400,0,0,True,PASS
9,post_buy_excitation,2400,0,0,True,PASS


,check_name,passed
0,selected_model_package_semantic_hash,True
1,frozen_parameter_identity,True
2,stationarity,True
3,development_replay_matches_frozen_fit,True
4,development_terminal_state_matches,True
5,calibration_history_inherited,True
6,persisted_calibration_replay_rows_match,True
7,persisted_calibration_replay_hash_matches,True
8,locked_calibration_score_matches,True
9,split_and_continuous_replay_match,True


Frozen H1 parameters loaded and independently replayed. DEVELOPMENT fit, inherited CALIBRATION state, persisted CALIBRATION replay, compensators, likelihoods, terminal states, semantic hashes, and continuous-history equivalence all reconcile. CALIBRATION parameter updates remain zero. VALIDATION and ENGINEERING_HOLDOUT remain unopened. No filesystem writes were performed.


In [8]:
# ============================================================
# Construct the canonical exact compensator path
#
# The split replay is authoritative for partition boundaries.
# The continuous replay is used only for cumulative reconciliation.
# Its first CALIBRATION interval spans the DEVELOPMENT tail plus
# the boundary-to-first-CALIBRATION interval, so those two interval
# compensators must be combined before comparison.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    FROZEN_MODEL_LOADED,
    "The frozen H1 model has not been loaded.",
)

require(
    FROZEN_REPLAY_RECONCILED,
    "The frozen H1 replay has not been reconciled.",
)

require(
    bool(
        globals().get(
            "ANALYTICAL_CONTENT_LOADED",
            False,
        )
    ),
    "Analytical DEVELOPMENT and CALIBRATION content is unavailable.",
)

require(
    not bool(
        globals().get(
            "EXACT_COMPENSATOR_PATH_BUILT",
            False,
        )
    ),
    "The exact compensator path is already built.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Exact numerical tolerances
# ------------------------------------------------------------

COMPENSATOR_ABSOLUTE_TOLERANCE: Final[float] = 1e-8
COMPENSATOR_RELATIVE_TOLERANCE: Final[float] = 1e-8


def require_array_close(
    observed: np.ndarray | pd.Series,
    expected: np.ndarray | pd.Series,
    *,
    label: str,
    rtol: float = COMPENSATOR_RELATIVE_TOLERANCE,
    atol: float = COMPENSATOR_ABSOLUTE_TOLERANCE,
) -> None:
    """Require two finite numerical arrays to agree."""
    observed_array = np.asarray(
        observed,
        dtype=np.float64,
    )

    expected_array = np.asarray(
        expected,
        dtype=np.float64,
    )

    require(
        observed_array.shape
        == expected_array.shape,
        (
            f"{label} shape mismatch: "
            f"{observed_array.shape} != {expected_array.shape}."
        ),
    )

    require(
        np.isfinite(
            observed_array
        ).all(),
        f"{label} observed array contains non-finite values.",
    )

    require(
        np.isfinite(
            expected_array
        ).all(),
        f"{label} expected array contains non-finite values.",
    )

    require(
        np.allclose(
            observed_array,
            expected_array,
            rtol=rtol,
            atol=atol,
            equal_nan=False,
        ),
        (
            f"{label} mismatch: "
            f"maximum absolute difference="
            f"{float(np.max(np.abs(observed_array - expected_array))):.12g}."
        ),
    )


# ------------------------------------------------------------
# Attach authoritative identities to a partition replay
# ------------------------------------------------------------

def identified_partition_replay(
    partition: str,
    replay_result: FrozenHawkesReplayResult,
) -> pd.DataFrame:
    """Attach Notebook 04 batch identities to one replay result."""
    identity = (
        PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
            PRIMARY_SCORING_BATCHES_ANALYTICAL[
                "event_partition"
            ].eq(
                partition
            ),
            [
                "primary_event_batch_id",
                "primary_event_batch_number",
                "partition_batch_index",
                "event_partition_order",
                "event_partition",
                "event_time_ns",
                "relative_batch_time_ns",
                "batch_event_count",
                "buy_event_count",
                "sell_event_count",
                "mixed_side_batch_flag",
                "simultaneous_batch_required_flag",
            ],
        ]
        .copy()
        .sort_values(
            [
                "event_time_ns",
                "primary_event_batch_number",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    replay = (
        replay_result.replay.copy()
        .sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        len(identity)
        == replay_result.batch_count
        == len(replay),
        f"{partition} identity and replay row counts differ.",
    )

    identified = identity.merge(
        replay,
        on=[
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    require(
        identified[
            "_merge"
        ].eq(
            "both"
        ).all(),
        f"{partition} replay does not reconcile with batch identities.",
    )

    identified = (
        identified.drop(
            columns="_merge"
        )
        .sort_values(
            [
                "event_time_ns",
                "primary_event_batch_number",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        identified[
            "batch_event_count"
        ].eq(
            identified[
                "buy_event_count"
            ]
            + identified[
                "sell_event_count"
            ]
        ).all(),
        f"{partition} identified replay does not conserve events.",
    )

    return identified


IDENTIFIED_DEVELOPMENT_REPLAY = identified_partition_replay(
    "DEVELOPMENT",
    FROZEN_DEVELOPMENT_REPLAY_RESULT,
)

IDENTIFIED_CALIBRATION_REPLAY = identified_partition_replay(
    "CALIBRATION",
    FROZEN_CALIBRATION_REPLAY_RESULT,
)


# ------------------------------------------------------------
# Build one partition-level exact compensator path
# ------------------------------------------------------------

def build_partition_compensator_path(
    identified_replay: pd.DataFrame,
    replay_result: FrozenHawkesReplayResult,
    parameters: FrozenDiagonalHawkesParameters,
    *,
    partition: str,
    initial_state: FrozenHawkesState,
) -> pd.DataFrame:
    """
    Build exact event intervals plus one explicit terminal-tail interval.

    Event intervals terminate at an observed exact-time batch.
    The terminal-tail interval contains no event and carries the
    compensator from the last batch to the partition boundary.
    """
    require(
        replay_result.observation_start_ns
        == initial_state.state_time_ns,
        (
            f"{partition} initial state is not expressed at "
            "the observation boundary."
        ),
    )

    require(
        replay_result.observation_end_exclusive_ns
        > replay_result.observation_start_ns,
        f"{partition} has a nonpositive observation interval.",
    )

    replay = (
        identified_replay.copy()
        .sort_values(
            [
                "event_time_ns",
                "primary_event_batch_number",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        len(replay)
        == replay_result.batch_count,
        f"{partition} replay row count is incorrect.",
    )

    event_count = len(
        replay
    )

    interval_start_ns = np.empty(
        event_count,
        dtype=np.int64,
    )

    interval_start_buy_excitation = np.empty(
        event_count,
        dtype=np.float64,
    )

    interval_start_sell_excitation = np.empty(
        event_count,
        dtype=np.float64,
    )

    interval_start_ns[0] = int(
        replay_result.observation_start_ns
    )

    interval_start_buy_excitation[0] = float(
        initial_state.buy_excitation
    )

    interval_start_sell_excitation[0] = float(
        initial_state.sell_excitation
    )

    if event_count > 1:
        interval_start_ns[1:] = (
            replay[
                "event_time_ns"
            ]
            .iloc[:-1]
            .to_numpy(
                dtype=np.int64
            )
        )

        interval_start_buy_excitation[1:] = (
            replay[
                "post_buy_excitation"
            ]
            .iloc[:-1]
            .to_numpy(
                dtype=np.float64
            )
        )

        interval_start_sell_excitation[1:] = (
            replay[
                "post_sell_excitation"
            ]
            .iloc[:-1]
            .to_numpy(
                dtype=np.float64
            )
        )

    event_end_ns = replay[
        "event_time_ns"
    ].to_numpy(
        dtype=np.int64
    )

    elapsed_ns = (
        event_end_ns
        - interval_start_ns
    )

    require(
        np.all(
            elapsed_ns
            >= 0
        ),
        f"{partition} contains a negative event interval.",
    )

    elapsed_seconds = (
        elapsed_ns.astype(
            np.float64
        )
        / NANOSECONDS_PER_SECOND
    )

    reconstructed_pre_buy = (
        interval_start_buy_excitation
        * np.exp(
            -parameters.beta_buy
            * elapsed_seconds
        )
    )

    reconstructed_pre_sell = (
        interval_start_sell_excitation
        * np.exp(
            -parameters.beta_sell
            * elapsed_seconds
        )
    )

    require_array_close(
        reconstructed_pre_buy,
        replay[
            "pre_buy_excitation"
        ],
        label=(
            f"{partition} reconstructed pre-BUY excitation"
        ),
    )

    require_array_close(
        reconstructed_pre_sell,
        replay[
            "pre_sell_excitation"
        ],
        label=(
            f"{partition} reconstructed pre-SELL excitation"
        ),
    )

    baseline_compensator_buy = (
        parameters.mu_buy
        * elapsed_seconds
    )

    baseline_compensator_sell = (
        parameters.mu_sell
        * elapsed_seconds
    )

    excitation_compensator_buy = np.asarray(
        [
            exact_excitation_integral(
                excitation_at_interval_start=(
                    start_excitation
                ),
                decay_rate=parameters.beta_buy,
                elapsed_seconds=duration_seconds,
            )
            for (
                start_excitation,
                duration_seconds,
            ) in zip(
                interval_start_buy_excitation,
                elapsed_seconds,
                strict=True,
            )
        ],
        dtype=np.float64,
    )

    excitation_compensator_sell = np.asarray(
        [
            exact_excitation_integral(
                excitation_at_interval_start=(
                    start_excitation
                ),
                decay_rate=parameters.beta_sell,
                elapsed_seconds=duration_seconds,
            )
            for (
                start_excitation,
                duration_seconds,
            ) in zip(
                interval_start_sell_excitation,
                elapsed_seconds,
                strict=True,
            )
        ],
        dtype=np.float64,
    )

    reconstructed_interval_compensator_buy = (
        baseline_compensator_buy
        + excitation_compensator_buy
    )

    reconstructed_interval_compensator_sell = (
        baseline_compensator_sell
        + excitation_compensator_sell
    )

    require_array_close(
        reconstructed_interval_compensator_buy,
        replay[
            "interval_compensator_buy"
        ],
        label=(
            f"{partition} exact BUY interval compensator"
        ),
    )

    require_array_close(
        reconstructed_interval_compensator_sell,
        replay[
            "interval_compensator_sell"
        ],
        label=(
            f"{partition} exact SELL interval compensator"
        ),
    )

    event_path = pd.DataFrame(
        {
            "event_partition": partition,
            "event_partition_order": (
                replay[
                    "event_partition_order"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "interval_kind": "EVENT_BATCH",
            "primary_event_batch_id": (
                replay[
                    "primary_event_batch_id"
                ].astype(
                    "string"
                )
            ),
            "primary_event_batch_number": (
                replay[
                    "primary_event_batch_number"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "partition_batch_index": (
                replay[
                    "partition_batch_index"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "event_time_ns": event_end_ns,
            "interval_start_ns": interval_start_ns,
            "interval_end_ns": event_end_ns,
            "elapsed_ns": elapsed_ns,
            "elapsed_seconds": elapsed_seconds,
            "batch_event_count": (
                replay[
                    "batch_event_count"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "buy_event_count": (
                replay[
                    "buy_event_count"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "sell_event_count": (
                replay[
                    "sell_event_count"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "mixed_side_batch_flag": (
                replay[
                    "mixed_side_batch_flag"
                ].to_numpy(
                    dtype=bool
                )
            ),
            "simultaneous_batch_required_flag": (
                replay[
                    "simultaneous_batch_required_flag"
                ].to_numpy(
                    dtype=bool
                )
            ),
            "interval_start_buy_excitation": (
                interval_start_buy_excitation
            ),
            "interval_start_sell_excitation": (
                interval_start_sell_excitation
            ),
            "pre_buy_excitation": (
                replay[
                    "pre_buy_excitation"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "pre_sell_excitation": (
                replay[
                    "pre_sell_excitation"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "lambda_buy_pre": (
                replay[
                    "lambda_buy_pre"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "lambda_sell_pre": (
                replay[
                    "lambda_sell_pre"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "post_buy_excitation": (
                replay[
                    "post_buy_excitation"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "post_sell_excitation": (
                replay[
                    "post_sell_excitation"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "baseline_compensator_buy": (
                baseline_compensator_buy
            ),
            "baseline_compensator_sell": (
                baseline_compensator_sell
            ),
            "excitation_compensator_buy": (
                excitation_compensator_buy
            ),
            "excitation_compensator_sell": (
                excitation_compensator_sell
            ),
            "interval_compensator_buy": (
                reconstructed_interval_compensator_buy
            ),
            "interval_compensator_sell": (
                reconstructed_interval_compensator_sell
            ),
            "log_event_term_buy": (
                replay[
                    "log_event_term_buy"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "log_event_term_sell": (
                replay[
                    "log_event_term_sell"
                ].to_numpy(
                    dtype=np.float64
                )
            ),
            "strict_pre_batch_scoring": True,
            "zero_lag_within_batch_excitation": False,
        }
    )

    if event_count == 0:
        tail_start_ns = int(
            replay_result.observation_start_ns
        )

        tail_start_buy_excitation = float(
            initial_state.buy_excitation
        )

        tail_start_sell_excitation = float(
            initial_state.sell_excitation
        )
    else:
        tail_start_ns = int(
            replay[
                "event_time_ns"
            ].iloc[-1]
        )

        tail_start_buy_excitation = float(
            replay[
                "post_buy_excitation"
            ].iloc[-1]
        )

        tail_start_sell_excitation = float(
            replay[
                "post_sell_excitation"
            ].iloc[-1]
        )

    tail_end_ns = int(
        replay_result.observation_end_exclusive_ns
    )

    tail_elapsed_ns = (
        tail_end_ns
        - tail_start_ns
    )

    require(
        tail_elapsed_ns
        >= 0,
        f"{partition} terminal-tail duration is negative.",
    )

    tail_elapsed_seconds = (
        tail_elapsed_ns
        / NANOSECONDS_PER_SECOND
    )

    tail_end_buy_excitation = (
        tail_start_buy_excitation
        * math.exp(
            -parameters.beta_buy
            * tail_elapsed_seconds
        )
    )

    tail_end_sell_excitation = (
        tail_start_sell_excitation
        * math.exp(
            -parameters.beta_sell
            * tail_elapsed_seconds
        )
    )

    require_close(
        tail_end_buy_excitation,
        replay_result
        .final_state
        .buy_excitation,
        label=(
            f"{partition} terminal-tail BUY state"
        ),
        relative_tolerance=(
            COMPENSATOR_RELATIVE_TOLERANCE
        ),
        absolute_tolerance=(
            COMPENSATOR_ABSOLUTE_TOLERANCE
        ),
    )

    require_close(
        tail_end_sell_excitation,
        replay_result
        .final_state
        .sell_excitation,
        label=(
            f"{partition} terminal-tail SELL state"
        ),
        relative_tolerance=(
            COMPENSATOR_RELATIVE_TOLERANCE
        ),
        absolute_tolerance=(
            COMPENSATOR_ABSOLUTE_TOLERANCE
        ),
    )

    tail_baseline_buy = (
        parameters.mu_buy
        * tail_elapsed_seconds
    )

    tail_baseline_sell = (
        parameters.mu_sell
        * tail_elapsed_seconds
    )

    tail_excitation_buy = exact_excitation_integral(
        excitation_at_interval_start=(
            tail_start_buy_excitation
        ),
        decay_rate=parameters.beta_buy,
        elapsed_seconds=tail_elapsed_seconds,
    )

    tail_excitation_sell = exact_excitation_integral(
        excitation_at_interval_start=(
            tail_start_sell_excitation
        ),
        decay_rate=parameters.beta_sell,
        elapsed_seconds=tail_elapsed_seconds,
    )

    tail_path = pd.DataFrame(
        [
            {
                "event_partition": partition,
                "event_partition_order": (
                    PARTITION_ORDER_MAP[
                        partition
                    ]
                ),
                "interval_kind": "PARTITION_TAIL",
                "primary_event_batch_id": pd.NA,
                "primary_event_batch_number": pd.NA,
                "partition_batch_index": pd.NA,
                "event_time_ns": pd.NA,
                "interval_start_ns": tail_start_ns,
                "interval_end_ns": tail_end_ns,
                "elapsed_ns": tail_elapsed_ns,
                "elapsed_seconds": tail_elapsed_seconds,
                "batch_event_count": 0,
                "buy_event_count": 0,
                "sell_event_count": 0,
                "mixed_side_batch_flag": False,
                "simultaneous_batch_required_flag": False,
                "interval_start_buy_excitation": (
                    tail_start_buy_excitation
                ),
                "interval_start_sell_excitation": (
                    tail_start_sell_excitation
                ),
                "pre_buy_excitation": (
                    tail_end_buy_excitation
                ),
                "pre_sell_excitation": (
                    tail_end_sell_excitation
                ),
                "lambda_buy_pre": (
                    parameters.mu_buy
                    + tail_end_buy_excitation
                ),
                "lambda_sell_pre": (
                    parameters.mu_sell
                    + tail_end_sell_excitation
                ),
                "post_buy_excitation": (
                    tail_end_buy_excitation
                ),
                "post_sell_excitation": (
                    tail_end_sell_excitation
                ),
                "baseline_compensator_buy": (
                    tail_baseline_buy
                ),
                "baseline_compensator_sell": (
                    tail_baseline_sell
                ),
                "excitation_compensator_buy": (
                    tail_excitation_buy
                ),
                "excitation_compensator_sell": (
                    tail_excitation_sell
                ),
                "interval_compensator_buy": (
                    tail_baseline_buy
                    + tail_excitation_buy
                ),
                "interval_compensator_sell": (
                    tail_baseline_sell
                    + tail_excitation_sell
                ),
                "log_event_term_buy": 0.0,
                "log_event_term_sell": 0.0,
                "strict_pre_batch_scoring": True,
                "zero_lag_within_batch_excitation": False,
            }
        ]
    )

    partition_path = pd.concat(
        [
            event_path,
            tail_path,
        ],
        ignore_index=True,
        sort=False,
    )

    partition_path[
        "interval_sequence_within_partition"
    ] = np.arange(
        1,
        len(
            partition_path
        )
        + 1,
        dtype=np.int64,
    )

    partition_path[
        "interval_total_compensator"
    ] = (
        partition_path[
            "interval_compensator_buy"
        ]
        + partition_path[
            "interval_compensator_sell"
        ]
    )

    partition_path[
        "interval_log_event_term"
    ] = (
        partition_path[
            "log_event_term_buy"
        ]
        + partition_path[
            "log_event_term_sell"
        ]
    )

    partition_path[
        "interval_log_likelihood"
    ] = (
        partition_path[
            "interval_log_event_term"
        ]
        - partition_path[
            "interval_total_compensator"
        ]
    )

    require_close(
        partition_path[
            "interval_compensator_buy"
        ].sum(),
        replay_result.compensator_buy,
        label=(
            f"{partition} total BUY compensator"
        ),
        relative_tolerance=(
            COMPENSATOR_RELATIVE_TOLERANCE
        ),
        absolute_tolerance=(
            COMPENSATOR_ABSOLUTE_TOLERANCE
        ),
    )

    require_close(
        partition_path[
            "interval_compensator_sell"
        ].sum(),
        replay_result.compensator_sell,
        label=(
            f"{partition} total SELL compensator"
        ),
        relative_tolerance=(
            COMPENSATOR_RELATIVE_TOLERANCE
        ),
        absolute_tolerance=(
            COMPENSATOR_ABSOLUTE_TOLERANCE
        ),
    )

    require_close(
        partition_path[
            "interval_log_event_term"
        ].sum(),
        replay_result.log_event_term,
        label=(
            f"{partition} total log-event term"
        ),
        relative_tolerance=(
            COMPENSATOR_RELATIVE_TOLERANCE
        ),
        absolute_tolerance=(
            COMPENSATOR_ABSOLUTE_TOLERANCE
        ),
    )

    require_close(
        partition_path[
            "interval_log_likelihood"
        ].sum(),
        replay_result.log_likelihood,
        label=(
            f"{partition} total log likelihood"
        ),
        relative_tolerance=(
            COMPENSATOR_RELATIVE_TOLERANCE
        ),
        absolute_tolerance=(
            COMPENSATOR_ABSOLUTE_TOLERANCE
        ),
    )

    return partition_path


DEVELOPMENT_INITIAL_STATE: Final[FrozenHawkesState] = (
    FrozenHawkesState(
        state_time_ns=DEVELOPMENT_START_NS,
        buy_excitation=0.0,
        sell_excitation=0.0,
    )
)

DEVELOPMENT_EXACT_COMPENSATOR_PATH = (
    build_partition_compensator_path(
        IDENTIFIED_DEVELOPMENT_REPLAY,
        FROZEN_DEVELOPMENT_REPLAY_RESULT,
        FROZEN_H1_PARAMETERS,
        partition="DEVELOPMENT",
        initial_state=DEVELOPMENT_INITIAL_STATE,
    )
)

CALIBRATION_EXACT_COMPENSATOR_PATH = (
    build_partition_compensator_path(
        IDENTIFIED_CALIBRATION_REPLAY,
        FROZEN_CALIBRATION_REPLAY_RESULT,
        FROZEN_H1_PARAMETERS,
        partition="CALIBRATION",
        initial_state=(
            FROZEN_DEVELOPMENT_REPLAY_RESULT
            .final_state
        ),
    )
)


# ------------------------------------------------------------
# Combine partition paths without crossing the frozen boundary
# ------------------------------------------------------------

EXACT_COMPENSATOR_PATH = pd.concat(
    [
        DEVELOPMENT_EXACT_COMPENSATOR_PATH,
        CALIBRATION_EXACT_COMPENSATOR_PATH,
    ],
    ignore_index=True,
    sort=False,
)

EXACT_COMPENSATOR_PATH = (
    EXACT_COMPENSATOR_PATH.sort_values(
        [
            "event_partition_order",
            "interval_sequence_within_partition",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

EXACT_COMPENSATOR_PATH[
    "global_interval_sequence"
] = np.arange(
    1,
    len(
        EXACT_COMPENSATOR_PATH
    )
    + 1,
    dtype=np.int64,
)

EXACT_COMPENSATOR_PATH[
    "cumulative_baseline_compensator_buy"
] = (
    EXACT_COMPENSATOR_PATH[
        "baseline_compensator_buy"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_baseline_compensator_sell"
] = (
    EXACT_COMPENSATOR_PATH[
        "baseline_compensator_sell"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_excitation_compensator_buy"
] = (
    EXACT_COMPENSATOR_PATH[
        "excitation_compensator_buy"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_excitation_compensator_sell"
] = (
    EXACT_COMPENSATOR_PATH[
        "excitation_compensator_sell"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_compensator_buy"
] = (
    EXACT_COMPENSATOR_PATH[
        "interval_compensator_buy"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_compensator_sell"
] = (
    EXACT_COMPENSATOR_PATH[
        "interval_compensator_sell"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_total_compensator"
] = (
    EXACT_COMPENSATOR_PATH[
        "interval_total_compensator"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_log_event_term_buy"
] = (
    EXACT_COMPENSATOR_PATH[
        "log_event_term_buy"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_log_event_term_sell"
] = (
    EXACT_COMPENSATOR_PATH[
        "log_event_term_sell"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_log_event_term"
] = (
    EXACT_COMPENSATOR_PATH[
        "interval_log_event_term"
    ].cumsum()
)

EXACT_COMPENSATOR_PATH[
    "cumulative_log_likelihood"
] = (
    EXACT_COMPENSATOR_PATH[
        "cumulative_log_event_term"
    ]
    - EXACT_COMPENSATOR_PATH[
        "cumulative_total_compensator"
    ]
)


# ------------------------------------------------------------
# Interval continuity and partition-boundary gates
# ------------------------------------------------------------

interval_starts = EXACT_COMPENSATOR_PATH[
    "interval_start_ns"
].to_numpy(
    dtype=np.int64
)

interval_ends = EXACT_COMPENSATOR_PATH[
    "interval_end_ns"
].to_numpy(
    dtype=np.int64
)

require(
    np.all(
        interval_ends
        >= interval_starts
    ),
    "The exact compensator path contains a negative interval.",
)

require(
    np.array_equal(
        interval_starts[1:],
        interval_ends[:-1],
    ),
    (
        "The exact compensator path contains a gap or overlap "
        "between consecutive intervals."
    ),
)

require(
    int(
        interval_starts[0]
    )
    == DEVELOPMENT_START_NS,
    "The exact compensator path starts at the wrong boundary.",
)

require(
    int(
        interval_ends[-1]
    )
    == CALIBRATION_END_EXCLUSIVE_NS,
    "The exact compensator path ends at the wrong boundary.",
)

development_tail_rows = (
    EXACT_COMPENSATOR_PATH.loc[
        EXACT_COMPENSATOR_PATH[
            "event_partition"
        ].eq(
            "DEVELOPMENT"
        )
        & EXACT_COMPENSATOR_PATH[
            "interval_kind"
        ].eq(
            "PARTITION_TAIL"
        )
    ]
)

calibration_tail_rows = (
    EXACT_COMPENSATOR_PATH.loc[
        EXACT_COMPENSATOR_PATH[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
        & EXACT_COMPENSATOR_PATH[
            "interval_kind"
        ].eq(
            "PARTITION_TAIL"
        )
    ]
)

require(
    len(
        development_tail_rows
    )
    == 1,
    "The DEVELOPMENT terminal tail is missing or duplicated.",
)

require(
    len(
        calibration_tail_rows
    )
    == 1,
    "The CALIBRATION terminal tail is missing or duplicated.",
)

development_tail_row = (
    development_tail_rows.iloc[0]
)

calibration_tail_row = (
    calibration_tail_rows.iloc[0]
)

require(
    int(
        development_tail_row[
            "interval_end_ns"
        ]
    )
    == CALIBRATION_START_NS,
    "The DEVELOPMENT tail does not end at CALIBRATION start.",
)

first_calibration_event_row = (
    EXACT_COMPENSATOR_PATH.loc[
        EXACT_COMPENSATOR_PATH[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
        & EXACT_COMPENSATOR_PATH[
            "interval_kind"
        ].eq(
            "EVENT_BATCH"
        )
    ]
    .iloc[0]
)

require(
    int(
        first_calibration_event_row[
            "interval_start_ns"
        ]
    )
    == CALIBRATION_START_NS,
    (
        "The first CALIBRATION event interval does not begin "
        "at the locked partition boundary."
    ),
)


# ------------------------------------------------------------
# Continuous replay event-level reconciliation
#
# Event states, intensities, event terms, and cumulative
# compensators must match. The first CALIBRATION interval itself
# is reconciled as:
#
# continuous interval
#   = DEVELOPMENT terminal tail
#   + locked CALIBRATION first interval
# ------------------------------------------------------------

continuous_replay = (
    FROZEN_CONTINUOUS_REPLAY_RESULT
    .replay
    .copy()
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

canonical_event_path = (
    EXACT_COMPENSATOR_PATH.loc[
        EXACT_COMPENSATOR_PATH[
            "interval_kind"
        ].eq(
            "EVENT_BATCH"
        )
    ]
    .copy()
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    len(
        canonical_event_path
    )
    == len(
        continuous_replay
    )
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "Continuous and canonical event-path row counts differ.",
)

continuous_event_reconciliation = (
    canonical_event_path.merge(
        continuous_replay[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
                "pre_buy_excitation",
                "pre_sell_excitation",
                "lambda_buy_pre",
                "lambda_sell_pre",
                "post_buy_excitation",
                "post_sell_excitation",
                "interval_compensator_buy",
                "interval_compensator_sell",
                "log_event_term_buy",
                "log_event_term_sell",
                "cumulative_log_event_term",
                "cumulative_compensator",
                "cumulative_log_likelihood_before_tail",
            ]
        ],
        on=[
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
        how="outer",
        validate="one_to_one",
        suffixes=(
            "_canonical",
            "_continuous",
        ),
        indicator=True,
    )
)

require(
    continuous_event_reconciliation[
        "_merge"
    ].eq(
        "both"
    ).all(),
    "Continuous replay event identities do not reconcile.",
)

continuous_event_reconciliation = (
    continuous_event_reconciliation.drop(
        columns="_merge"
    )
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

for state_column in (
    "pre_buy_excitation",
    "pre_sell_excitation",
    "lambda_buy_pre",
    "lambda_sell_pre",
    "post_buy_excitation",
    "post_sell_excitation",
    "log_event_term_buy",
    "log_event_term_sell",
):
    require_array_close(
        continuous_event_reconciliation[
            f"{state_column}_canonical"
        ],
        continuous_event_reconciliation[
            f"{state_column}_continuous"
        ],
        label=(
            "Canonical versus continuous "
            f"{state_column}"
        ),
    )


# ------------------------------------------------------------
# Boundary interval decomposition
# ------------------------------------------------------------

continuous_first_calibration_row = (
    continuous_event_reconciliation.loc[
        continuous_event_reconciliation[
            "event_time_ns"
        ].eq(
            int(
                first_calibration_event_row[
                    "event_time_ns"
                ]
            )
        )
    ]
)

require(
    len(
        continuous_first_calibration_row
    )
    == 1,
    "The first CALIBRATION event is missing from continuous replay.",
)

continuous_first_calibration_row = (
    continuous_first_calibration_row.iloc[0]
)

expected_continuous_boundary_buy_compensator = (
    float(
        development_tail_row[
            "interval_compensator_buy"
        ]
    )
    + float(
        first_calibration_event_row[
            "interval_compensator_buy"
        ]
    )
)

expected_continuous_boundary_sell_compensator = (
    float(
        development_tail_row[
            "interval_compensator_sell"
        ]
    )
    + float(
        first_calibration_event_row[
            "interval_compensator_sell"
        ]
    )
)

require_close(
    continuous_first_calibration_row[
        "interval_compensator_buy_continuous"
    ],
    expected_continuous_boundary_buy_compensator,
    label=(
        "Continuous first-CALIBRATION BUY interval "
        "boundary decomposition"
    ),
    relative_tolerance=(
        COMPENSATOR_RELATIVE_TOLERANCE
    ),
    absolute_tolerance=(
        COMPENSATOR_ABSOLUTE_TOLERANCE
    ),
)

require_close(
    continuous_first_calibration_row[
        "interval_compensator_sell_continuous"
    ],
    expected_continuous_boundary_sell_compensator,
    label=(
        "Continuous first-CALIBRATION SELL interval "
        "boundary decomposition"
    ),
    relative_tolerance=(
        COMPENSATOR_RELATIVE_TOLERANCE
    ),
    absolute_tolerance=(
        COMPENSATOR_ABSOLUTE_TOLERANCE
    ),
)


# ------------------------------------------------------------
# All non-boundary event intervals must match directly
# ------------------------------------------------------------

first_calibration_event_time_ns: Final[int] = int(
    first_calibration_event_row[
        "event_time_ns"
    ]
)

non_boundary_rows = (
    continuous_event_reconciliation.loc[
        ~continuous_event_reconciliation[
            "event_time_ns"
        ].eq(
            first_calibration_event_time_ns
        )
    ]
)

require_array_close(
    non_boundary_rows[
        "interval_compensator_buy_canonical"
    ],
    non_boundary_rows[
        "interval_compensator_buy_continuous"
    ],
    label=(
        "Non-boundary BUY interval compensators"
    ),
)

require_array_close(
    non_boundary_rows[
        "interval_compensator_sell_canonical"
    ],
    non_boundary_rows[
        "interval_compensator_sell_continuous"
    ],
    label=(
        "Non-boundary SELL interval compensators"
    ),
)


# ------------------------------------------------------------
# Cumulative event-time reconciliation
# ------------------------------------------------------------

require_array_close(
    continuous_event_reconciliation[
        "cumulative_log_event_term_canonical"
    ],
    continuous_event_reconciliation[
        "cumulative_log_event_term_continuous"
    ],
    label=(
        "Canonical versus continuous cumulative log-event term"
    ),
)

require_array_close(
    continuous_event_reconciliation[
        "cumulative_total_compensator"
    ],
    continuous_event_reconciliation[
        "cumulative_compensator"
    ],
    label=(
        "Canonical versus continuous cumulative compensator"
    ),
)

require_array_close(
    continuous_event_reconciliation[
        "cumulative_log_likelihood"
    ],
    continuous_event_reconciliation[
        "cumulative_log_likelihood_before_tail"
    ],
    label=(
        "Canonical versus continuous cumulative log likelihood"
    ),
)


# ------------------------------------------------------------
# Terminal totals and final-state reconciliation
# ------------------------------------------------------------

require_close(
    EXACT_COMPENSATOR_PATH[
        "interval_compensator_buy"
    ].sum(),
    FROZEN_CONTINUOUS_REPLAY_RESULT.compensator_buy,
    label="Full analytical BUY compensator",
    relative_tolerance=(
        COMPENSATOR_RELATIVE_TOLERANCE
    ),
    absolute_tolerance=(
        COMPENSATOR_ABSOLUTE_TOLERANCE
    ),
)

require_close(
    EXACT_COMPENSATOR_PATH[
        "interval_compensator_sell"
    ].sum(),
    FROZEN_CONTINUOUS_REPLAY_RESULT.compensator_sell,
    label="Full analytical SELL compensator",
    relative_tolerance=(
        COMPENSATOR_RELATIVE_TOLERANCE
    ),
    absolute_tolerance=(
        COMPENSATOR_ABSOLUTE_TOLERANCE
    ),
)

require_close(
    EXACT_COMPENSATOR_PATH[
        "interval_log_event_term"
    ].sum(),
    FROZEN_CONTINUOUS_REPLAY_RESULT.log_event_term,
    label="Full analytical log-event term",
    relative_tolerance=(
        COMPENSATOR_RELATIVE_TOLERANCE
    ),
    absolute_tolerance=(
        COMPENSATOR_ABSOLUTE_TOLERANCE
    ),
)

require_close(
    EXACT_COMPENSATOR_PATH[
        "interval_log_likelihood"
    ].sum(),
    FROZEN_CONTINUOUS_REPLAY_RESULT.log_likelihood,
    label="Full analytical log likelihood",
    relative_tolerance=(
        COMPENSATOR_RELATIVE_TOLERANCE
    ),
    absolute_tolerance=(
        COMPENSATOR_ABSOLUTE_TOLERANCE
    ),
)

require_close(
    EXACT_COMPENSATOR_PATH[
        "post_buy_excitation"
    ].iloc[-1],
    FROZEN_CONTINUOUS_REPLAY_RESULT
    .final_state
    .buy_excitation,
    label="Full analytical terminal BUY excitation",
)

require_close(
    EXACT_COMPENSATOR_PATH[
        "post_sell_excitation"
    ].iloc[-1],
    FROZEN_CONTINUOUS_REPLAY_RESULT
    .final_state
    .sell_excitation,
    label="Full analytical terminal SELL excitation",
)


# ------------------------------------------------------------
# Summary and audit tables
# ------------------------------------------------------------

EXACT_COMPENSATOR_PARTITION_SUMMARY = (
    EXACT_COMPENSATOR_PATH.groupby(
        [
            "event_partition_order",
            "event_partition",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        interval_count=(
            "global_interval_sequence",
            "size",
        ),
        event_batch_count=(
            "batch_event_count",
            lambda values: int(
                values.gt(
                    0
                ).sum()
            ),
        ),
        event_count=(
            "batch_event_count",
            "sum",
        ),
        buy_event_count=(
            "buy_event_count",
            "sum",
        ),
        sell_event_count=(
            "sell_event_count",
            "sum",
        ),
        observation_start_ns=(
            "interval_start_ns",
            "min",
        ),
        observation_end_exclusive_ns=(
            "interval_end_ns",
            "max",
        ),
        compensator_buy=(
            "interval_compensator_buy",
            "sum",
        ),
        compensator_sell=(
            "interval_compensator_sell",
            "sum",
        ),
        log_event_term=(
            "interval_log_event_term",
            "sum",
        ),
        log_likelihood=(
            "interval_log_likelihood",
            "sum",
        ),
    )
    .reset_index()
)

BOUNDARY_COMPENSATOR_RECONCILIATION = pd.DataFrame(
    [
        {
            "component": "BUY",
            "development_tail": float(
                development_tail_row[
                    "interval_compensator_buy"
                ]
            ),
            "calibration_first_interval": float(
                first_calibration_event_row[
                    "interval_compensator_buy"
                ]
            ),
            "combined_split_interval": (
                expected_continuous_boundary_buy_compensator
            ),
            "continuous_interval": float(
                continuous_first_calibration_row[
                    "interval_compensator_buy_continuous"
                ]
            ),
            "absolute_difference": abs(
                expected_continuous_boundary_buy_compensator
                - float(
                    continuous_first_calibration_row[
                        "interval_compensator_buy_continuous"
                    ]
                )
            ),
            "passed": True,
            "status": "PASS",
        },
        {
            "component": "SELL",
            "development_tail": float(
                development_tail_row[
                    "interval_compensator_sell"
                ]
            ),
            "calibration_first_interval": float(
                first_calibration_event_row[
                    "interval_compensator_sell"
                ]
            ),
            "combined_split_interval": (
                expected_continuous_boundary_sell_compensator
            ),
            "continuous_interval": float(
                continuous_first_calibration_row[
                    "interval_compensator_sell_continuous"
                ]
            ),
            "absolute_difference": abs(
                expected_continuous_boundary_sell_compensator
                - float(
                    continuous_first_calibration_row[
                        "interval_compensator_sell_continuous"
                    ]
                )
            ),
            "passed": True,
            "status": "PASS",
        },
    ]
)

EXACT_COMPENSATOR_VERIFICATION_SUMMARY = pd.DataFrame(
    [
        {
            "check_name": (
                "development_exact_interval_integrals"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_exact_interval_integrals"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "development_terminal_tail_explicit"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_terminal_tail_explicit"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "global_interval_continuity"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "boundary_interval_decomposition"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "non_boundary_interval_reconciliation"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "continuous_cumulative_compensator_reconciliation"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "continuous_cumulative_likelihood_reconciliation"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "terminal_state_reconciliation"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "filesystem_writes_performed"
            ),
            "passed": True,
        },
    ]
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

require(
    EXACT_COMPENSATOR_VERIFICATION_SUMMARY[
        "passed"
    ].all(),
    "At least one exact-compensator verification failed.",
)

EXACT_COMPENSATOR_PATH_BUILT = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was opened.",
)

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    EXACT_COMPENSATOR_PARTITION_SUMMARY
)

display(
    BOUNDARY_COMPENSATOR_RECONCILIATION
)

display(
    EXACT_COMPENSATOR_VERIFICATION_SUMMARY
)

print(
    "Canonical exact compensator path constructed. DEVELOPMENT "
    "and CALIBRATION terminal tails are represented explicitly. "
    "The continuous first-CALIBRATION interval reconciles to the "
    "sum of the DEVELOPMENT boundary tail and the locked "
    "CALIBRATION first interval. All other event intervals, "
    "cumulative compensators, likelihoods, and terminal states "
    "reconcile. No filesystem writes were performed."
)

C:\Users\Bryan\AppData\Local\Temp\ipykernel_14884\3916368398.py:869: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  partition_path = pd.concat(
C:\Users\Bryan\AppData\Local\Temp\ipykernel_14884\3916368398.py:869: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  partition_path = pd.concat(


,event_partition_order,event_partition,interval_count,event_batch_count,event_count,buy_event_count,sell_event_count,observation_start_ns,observation_end_exclusive_ns,compensator_buy,compensator_sell,log_event_term,log_likelihood
0,1,DEVELOPMENT,6860,6859,7004,3414,3590,1783665467531985400,1783667269391572100,"3,414.000015","3,590.00007","5,998.536946","-1,005.46314"
1,2,CALIBRATION,2401,2400,2493,1230,1263,1783667269391572100,1783667989690751200,"1,339.253892","1,415.723094","2,302.239061",-452.7379259


,component,development_tail,calibration_first_interval,combined_split_interval,continuous_interval,absolute_difference,passed,status
0,BUY,0.2448077544,1.7748538,2.019661555,2.019661555,0,True,PASS
1,SELL,0.3944099968,2.042669708,2.437079705,2.437079705,4.440892099e-16,True,PASS


,check_name,passed
0,development_exact_interval_integrals,True
1,calibration_exact_interval_integrals,True
2,development_terminal_tail_explicit,True
3,calibration_terminal_tail_explicit,True
4,global_interval_continuity,True
5,boundary_interval_decomposition,True
6,non_boundary_interval_reconciliation,True
7,continuous_cumulative_compensator_reconciliation,True
8,continuous_cumulative_likelihood_reconciliation,True
9,terminal_state_reconciliation,True


Canonical exact compensator path constructed. DEVELOPMENT and CALIBRATION terminal tails are represented explicitly. The continuous first-CALIBRATION interval reconciles to the sum of the DEVELOPMENT boundary tail and the locked CALIBRATION first interval. All other event intervals, cumulative compensators, likelihoods, and terminal states reconcile. No filesystem writes were performed.


In [9]:
# ============================================================
# Construct component-wise and pooled time-rescaling residuals
#
# Residuals are defined on distinct exact-time batches.
# Multiplicity is preserved as a diagnostic field and is never
# expanded into artificial zero-duration events.
#
# The first DEVELOPMENT residual for each scope is left-edge
# censored and excluded from primary goodness-of-fit summaries.
# CALIBRATION residuals inherit the preceding DEVELOPMENT event
# and compensator history across the frozen partition boundary.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "EXACT_COMPENSATOR_PATH_BUILT",
            False,
        )
    ),
    "The canonical exact compensator path has not been built.",
)

require(
    FROZEN_MODEL_LOADED,
    "The frozen H1 model has not been loaded.",
)

require(
    FROZEN_REPLAY_RECONCILED,
    "The frozen H1 replay has not been reconciled.",
)

require(
    not bool(
        globals().get(
            "TIME_RESCALING_RESIDUALS_BUILT",
            False,
        )
    ),
    "Time-rescaling residuals are already built.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Canonical event-time compensator states
# ------------------------------------------------------------

EVENT_COMPENSATOR_PATH = (
    EXACT_COMPENSATOR_PATH.loc[
        EXACT_COMPENSATOR_PATH[
            "interval_kind"
        ].eq(
            "EVENT_BATCH"
        )
    ]
    .copy()
    .sort_values(
        "global_interval_sequence",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    len(
        EVENT_COMPENSATOR_PATH
    )
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "The event compensator path has an incorrect row count.",
)

require(
    EVENT_COMPENSATOR_PATH[
        "event_time_ns"
    ].notna().all(),
    "An event-batch compensator row lacks an event time.",
)

EVENT_COMPENSATOR_PATH[
    "event_time_ns"
] = parse_exact_int64(
    EVENT_COMPENSATOR_PATH[
        "event_time_ns"
    ],
    label="EVENT_COMPENSATOR_PATH.event_time_ns",
)

require(
    EVENT_COMPENSATOR_PATH[
        "event_time_ns"
    ].is_unique,
    "The event compensator path contains duplicate timestamps.",
)

require(
    EVENT_COMPENSATOR_PATH[
        "event_time_ns"
    ].is_monotonic_increasing,
    "The event compensator path is not chronological.",
)

require(
    EVENT_COMPENSATOR_PATH[
        "cumulative_compensator_buy"
    ].is_monotonic_increasing,
    "The cumulative BUY compensator is not nondecreasing.",
)

require(
    EVENT_COMPENSATOR_PATH[
        "cumulative_compensator_sell"
    ].is_monotonic_increasing,
    "The cumulative SELL compensator is not nondecreasing.",
)

require(
    EVENT_COMPENSATOR_PATH[
        "cumulative_total_compensator"
    ].is_monotonic_increasing,
    "The cumulative total compensator is not nondecreasing.",
)


# ------------------------------------------------------------
# Residual construction helper
# ------------------------------------------------------------

RESIDUAL_COLUMNS: Final[tuple[str, ...]] = (
    "residual_scope",
    "residual_kind",
    "residual_index_global",
    "residual_index_within_destination_partition",
    "destination_event_partition",
    "destination_partition_order",
    "current_batch_id",
    "event_time_ns",
    "previous_relevant_batch_id",
    "previous_relevant_event_time_ns",
    "previous_relevant_event_partition",
    "interval_start_ns",
    "elapsed_ns",
    "elapsed_seconds",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "relevant_event_multiplicity",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "current_cumulative_compensator",
    "previous_cumulative_compensator",
    "tau",
    "u",
    "left_edge_censored",
    "cross_partition_interval",
    "relevant_multiplicity_gt_one",
    "exact_time_batch_multiplicity_gt_one",
    "primary_diagnostic_eligible",
    "coarsening_aware_simulation_required",
    "reference_distribution_role",
)


def build_batch_arrival_residuals(
    event_path: pd.DataFrame,
    *,
    residual_scope: str,
    residual_kind: str,
    relevant_count_column: str,
    cumulative_compensator_column: str,
    observation_start_ns: int,
) -> pd.DataFrame:
    """
    Construct compensator increments between relevant exact-time batches.

    For BUY and SELL, a relevant batch contains at least one event of
    that side. For POOLED, every exact-time batch is relevant.
    """
    required_columns = {
        "primary_event_batch_id",
        "event_partition",
        "event_partition_order",
        "event_time_ns",
        "batch_event_count",
        "buy_event_count",
        "sell_event_count",
        "mixed_side_batch_flag",
        "simultaneous_batch_required_flag",
        relevant_count_column,
        cumulative_compensator_column,
    }

    missing_columns = sorted(
        required_columns
        - set(
            event_path.columns
        )
    )

    require(
        not missing_columns,
        (
            f"{residual_scope} residual input is missing columns: "
            f"{missing_columns}"
        ),
    )

    selected = (
        event_path.loc[
            event_path[
                relevant_count_column
            ].gt(
                0
            )
        ]
        .copy()
        .sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        not selected.empty,
        f"No relevant batches exist for {residual_scope}.",
    )

    require(
        selected[
            "event_time_ns"
        ].is_unique,
        f"{residual_scope} relevant timestamps are not unique.",
    )

    current_cumulative = (
        selected[
            cumulative_compensator_column
        ].to_numpy(
            dtype=np.float64
        )
    )

    require(
        np.isfinite(
            current_cumulative
        ).all(),
        (
            f"{residual_scope} cumulative compensator "
            "contains non-finite values."
        ),
    )

    require(
        np.all(
            np.diff(
                current_cumulative
            )
            > 0.0
        ),
        (
            f"{residual_scope} cumulative compensator is not "
            "strictly increasing at relevant arrivals."
        ),
    )

    previous_cumulative = np.empty_like(
        current_cumulative
    )

    previous_cumulative[0] = 0.0
    previous_cumulative[1:] = (
        current_cumulative[:-1]
    )

    tau = (
        current_cumulative
        - previous_cumulative
    )

    require(
        np.isfinite(
            tau
        ).all(),
        f"{residual_scope} tau contains non-finite values.",
    )

    require(
        np.all(
            tau
            > 0.0
        ),
        f"{residual_scope} tau must be strictly positive.",
    )

    u = -np.expm1(
        -tau
    )

    require(
        np.isfinite(
            u
        ).all(),
        f"{residual_scope} PIT residuals contain non-finite values.",
    )

    require(
        np.all(
            u
            > 0.0
        )
        and np.all(
            u
            < 1.0
        ),
        (
            f"{residual_scope} PIT residuals must lie strictly "
            "inside (0, 1)."
        ),
    )

    previous_time = (
        selected[
            "event_time_ns"
        ]
        .shift(
            1
        )
        .astype(
            "Int64"
        )
    )

    interval_start = (
        previous_time.fillna(
            int(
                observation_start_ns
            )
        )
        .astype(
            "int64"
        )
    )

    elapsed_ns = (
        selected[
            "event_time_ns"
        ].to_numpy(
            dtype=np.int64
        )
        - interval_start.to_numpy(
            dtype=np.int64
        )
    )

    require(
        np.all(
            elapsed_ns
            > 0
        ),
        (
            f"{residual_scope} residual intervals must have "
            "strictly positive duration."
        ),
    )

    previous_partition = (
        selected[
            "event_partition"
        ]
        .shift(
            1
        )
        .astype(
            "string"
        )
    )

    cross_partition = (
        previous_partition.notna()
        & previous_partition.ne(
            selected[
                "event_partition"
            ].astype(
                "string"
            )
        )
    )

    previous_batch_id = (
        selected[
            "primary_event_batch_id"
        ]
        .shift(
            1
        )
        .astype(
            "string"
        )
    )

    left_edge_censored = np.zeros(
        len(
            selected
        ),
        dtype=bool,
    )

    left_edge_censored[0] = True

    within_partition_index = (
        selected.groupby(
            "event_partition",
            observed=True,
            sort=False,
        )
        .cumcount()
        .to_numpy(
            dtype=np.int64
        )
        + 1
    )

    result = pd.DataFrame(
        {
            "residual_scope": residual_scope,
            "residual_kind": residual_kind,
            "residual_index_global": np.arange(
                1,
                len(
                    selected
                )
                + 1,
                dtype=np.int64,
            ),
            "residual_index_within_destination_partition": (
                within_partition_index
            ),
            "destination_event_partition": (
                selected[
                    "event_partition"
                ]
                .astype(
                    "string"
                )
                .to_numpy()
            ),
            "destination_partition_order": (
                selected[
                    "event_partition_order"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "current_batch_id": (
                selected[
                    "primary_event_batch_id"
                ]
                .astype(
                    "string"
                )
                .to_numpy()
            ),
            "event_time_ns": (
                selected[
                    "event_time_ns"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "previous_relevant_batch_id": (
                previous_batch_id.to_numpy()
            ),
            "previous_relevant_event_time_ns": (
                previous_time.to_numpy()
            ),
            "previous_relevant_event_partition": (
                previous_partition.to_numpy()
            ),
            "interval_start_ns": (
                interval_start.to_numpy(
                    dtype=np.int64
                )
            ),
            "elapsed_ns": elapsed_ns,
            "elapsed_seconds": (
                elapsed_ns.astype(
                    np.float64
                )
                / NANOSECONDS_PER_SECOND
            ),
            "batch_event_count": (
                selected[
                    "batch_event_count"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "buy_event_count": (
                selected[
                    "buy_event_count"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "sell_event_count": (
                selected[
                    "sell_event_count"
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "relevant_event_multiplicity": (
                selected[
                    relevant_count_column
                ].to_numpy(
                    dtype=np.int64
                )
            ),
            "mixed_side_batch_flag": (
                selected[
                    "mixed_side_batch_flag"
                ].to_numpy(
                    dtype=bool
                )
            ),
            "simultaneous_batch_required_flag": (
                selected[
                    "simultaneous_batch_required_flag"
                ].to_numpy(
                    dtype=bool
                )
            ),
            "current_cumulative_compensator": (
                current_cumulative
            ),
            "previous_cumulative_compensator": (
                previous_cumulative
            ),
            "tau": tau,
            "u": u,
            "left_edge_censored": (
                left_edge_censored
            ),
            "cross_partition_interval": (
                cross_partition.to_numpy(
                    dtype=bool
                )
            ),
            "relevant_multiplicity_gt_one": (
                selected[
                    relevant_count_column
                ].gt(
                    1
                ).to_numpy(
                    dtype=bool
                )
            ),
            "exact_time_batch_multiplicity_gt_one": (
                selected[
                    "batch_event_count"
                ].gt(
                    1
                ).to_numpy(
                    dtype=bool
                )
            ),
            "primary_diagnostic_eligible": (
                ~left_edge_censored
            ),
            "coarsening_aware_simulation_required": True,
            "reference_distribution_role": (
                "DESCRIPTIVE_EXP1_REFERENCE_PENDING_"
                "COARSENING_AWARE_SIMULATION_ENVELOPE"
            ),
        },
        columns=list(
            RESIDUAL_COLUMNS
        ),
    )

    result[
        "previous_relevant_event_time_ns"
    ] = result[
        "previous_relevant_event_time_ns"
    ].astype(
        "Int64"
    )

    result[
        "event_time_ns"
    ] = result[
        "event_time_ns"
    ].astype(
        "int64"
    )

    result[
        "interval_start_ns"
    ] = result[
        "interval_start_ns"
    ].astype(
        "int64"
    )

    result[
        "elapsed_ns"
    ] = result[
        "elapsed_ns"
    ].astype(
        "int64"
    )

    for string_column in (
        "residual_scope",
        "residual_kind",
        "destination_event_partition",
        "current_batch_id",
        "previous_relevant_batch_id",
        "previous_relevant_event_partition",
        "reference_distribution_role",
    ):
        result[
            string_column
        ] = result[
            string_column
        ].astype(
            "string"
        )

    require(
        result[
            "left_edge_censored"
        ].sum()
        == 1,
        (
            f"{residual_scope} must contain exactly one "
            "left-edge-censored residual."
        ),
    )

    require(
        result.loc[
            result[
                "destination_event_partition"
            ].eq(
                "CALIBRATION"
            ),
            "left_edge_censored",
        ].sum()
        == 0,
        (
            f"{residual_scope} incorrectly marks a CALIBRATION "
            "residual as left-edge censored."
        ),
    )

    require(
        result[
            "cross_partition_interval"
        ].sum()
        == 1,
        (
            f"{residual_scope} must contain exactly one "
            "DEVELOPMENT-to-CALIBRATION residual interval."
        ),
    )

    require(
        set(
            result[
                "destination_event_partition"
            ]
        )
        == set(
            ANALYTICAL_PARTITIONS
        ),
        (
            f"{residual_scope} residuals have an unexpected "
            "destination partition set."
        ),
    )

    require(
        not set(
            result[
                "destination_event_partition"
            ]
        ).intersection(
            PROTECTED_PARTITIONS
        ),
        f"{residual_scope} residuals contain protected content.",
    )

    return result


# ------------------------------------------------------------
# Component and pooled residual tables
# ------------------------------------------------------------

BUY_TIME_RESCALING_RESIDUALS = (
    build_batch_arrival_residuals(
        EVENT_COMPENSATOR_PATH,
        residual_scope="BUY",
        residual_kind=(
            "COMPONENT_EXACT_TIME_BATCH_ARRIVAL"
        ),
        relevant_count_column=(
            "buy_event_count"
        ),
        cumulative_compensator_column=(
            "cumulative_compensator_buy"
        ),
        observation_start_ns=(
            DEVELOPMENT_START_NS
        ),
    )
)

SELL_TIME_RESCALING_RESIDUALS = (
    build_batch_arrival_residuals(
        EVENT_COMPENSATOR_PATH,
        residual_scope="SELL",
        residual_kind=(
            "COMPONENT_EXACT_TIME_BATCH_ARRIVAL"
        ),
        relevant_count_column=(
            "sell_event_count"
        ),
        cumulative_compensator_column=(
            "cumulative_compensator_sell"
        ),
        observation_start_ns=(
            DEVELOPMENT_START_NS
        ),
    )
)

POOLED_TIME_RESCALING_RESIDUALS = (
    build_batch_arrival_residuals(
        EVENT_COMPENSATOR_PATH,
        residual_scope="POOLED",
        residual_kind=(
            "SUPERPOSED_EXACT_TIME_BATCH_ARRIVAL"
        ),
        relevant_count_column=(
            "batch_event_count"
        ),
        cumulative_compensator_column=(
            "cumulative_total_compensator"
        ),
        observation_start_ns=(
            DEVELOPMENT_START_NS
        ),
    )
)


# ------------------------------------------------------------
# Combined residual table without dtype-ambiguous concatenation
# ------------------------------------------------------------

combined_residual_records: list[
    dict[str, Any]
] = []

for residual_table in (
    BUY_TIME_RESCALING_RESIDUALS,
    SELL_TIME_RESCALING_RESIDUALS,
    POOLED_TIME_RESCALING_RESIDUALS,
):
    combined_residual_records.extend(
        residual_table.to_dict(
            orient="records"
        )
    )

TIME_RESCALING_RESIDUALS = (
    pd.DataFrame.from_records(
        combined_residual_records,
        columns=list(
            RESIDUAL_COLUMNS
        ),
    )
)

TIME_RESCALING_RESIDUALS[
    "previous_relevant_event_time_ns"
] = TIME_RESCALING_RESIDUALS[
    "previous_relevant_event_time_ns"
].astype(
    "Int64"
)

for integer_column in (
    "residual_index_global",
    "residual_index_within_destination_partition",
    "destination_partition_order",
    "event_time_ns",
    "interval_start_ns",
    "elapsed_ns",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "relevant_event_multiplicity",
):
    TIME_RESCALING_RESIDUALS[
        integer_column
    ] = TIME_RESCALING_RESIDUALS[
        integer_column
    ].astype(
        "int64"
    )

for boolean_column in (
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "left_edge_censored",
    "cross_partition_interval",
    "relevant_multiplicity_gt_one",
    "exact_time_batch_multiplicity_gt_one",
    "primary_diagnostic_eligible",
    "coarsening_aware_simulation_required",
):
    TIME_RESCALING_RESIDUALS[
        boolean_column
    ] = TIME_RESCALING_RESIDUALS[
        boolean_column
    ].astype(
        bool
    )

for string_column in (
    "residual_scope",
    "residual_kind",
    "destination_event_partition",
    "current_batch_id",
    "previous_relevant_batch_id",
    "previous_relevant_event_partition",
    "reference_distribution_role",
):
    TIME_RESCALING_RESIDUALS[
        string_column
    ] = TIME_RESCALING_RESIDUALS[
        string_column
    ].astype(
        "string"
    )

TIME_RESCALING_RESIDUALS = (
    TIME_RESCALING_RESIDUALS.sort_values(
        [
            "residual_scope",
            "event_time_ns",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Residual-count and compensator reconciliation
# ------------------------------------------------------------

expected_buy_residual_count = int(
    EVENT_COMPENSATOR_PATH[
        "buy_event_count"
    ].gt(
        0
    ).sum()
)

expected_sell_residual_count = int(
    EVENT_COMPENSATOR_PATH[
        "sell_event_count"
    ].gt(
        0
    ).sum()
)

expected_pooled_residual_count = len(
    EVENT_COMPENSATOR_PATH
)

require(
    len(
        BUY_TIME_RESCALING_RESIDUALS
    )
    == expected_buy_residual_count,
    "BUY residual count does not reconcile.",
)

require(
    len(
        SELL_TIME_RESCALING_RESIDUALS
    )
    == expected_sell_residual_count,
    "SELL residual count does not reconcile.",
)

require(
    len(
        POOLED_TIME_RESCALING_RESIDUALS
    )
    == expected_pooled_residual_count,
    "Pooled residual count does not reconcile.",
)

require_close(
    BUY_TIME_RESCALING_RESIDUALS[
        "tau"
    ].sum(),
    BUY_TIME_RESCALING_RESIDUALS[
        "current_cumulative_compensator"
    ].iloc[-1],
    label="BUY residual compensator conservation",
)

require_close(
    SELL_TIME_RESCALING_RESIDUALS[
        "tau"
    ].sum(),
    SELL_TIME_RESCALING_RESIDUALS[
        "current_cumulative_compensator"
    ].iloc[-1],
    label="SELL residual compensator conservation",
)

require_close(
    POOLED_TIME_RESCALING_RESIDUALS[
        "tau"
    ].sum(),
    POOLED_TIME_RESCALING_RESIDUALS[
        "current_cumulative_compensator"
    ].iloc[-1],
    label="Pooled residual compensator conservation",
)


# ------------------------------------------------------------
# Right-censoring compensator ledger
# ------------------------------------------------------------

def build_right_censoring_record(
    partition_path: pd.DataFrame,
    *,
    partition: str,
    residual_scope: str,
    relevant_count_column: str,
    interval_compensator_column: str,
) -> dict[str, Any]:
    """Measure compensator mass after the final relevant arrival."""
    path = (
        partition_path.sort_values(
            "interval_sequence_within_partition",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    local_cumulative = (
        path[
            interval_compensator_column
        ].cumsum()
    )

    relevant_rows = path.loc[
        path[
            relevant_count_column
        ].gt(
            0
        )
        & path[
            "interval_kind"
        ].eq(
            "EVENT_BATCH"
        )
    ]

    total_compensator = float(
        path[
            interval_compensator_column
        ].sum()
    )

    if relevant_rows.empty:
        last_event_time_ns: int | pd._libs.missing.NAType = (
            pd.NA
        )

        compensator_through_last_event = 0.0
    else:
        last_relevant_index = int(
            relevant_rows.index[-1]
        )

        last_event_time_ns = int(
            path.loc[
                last_relevant_index,
                "event_time_ns",
            ]
        )

        compensator_through_last_event = float(
            local_cumulative.iloc[
                last_relevant_index
            ]
        )

    right_censoring_tau = (
        total_compensator
        - compensator_through_last_event
    )

    require(
        right_censoring_tau
        >= -COMPENSATOR_ABSOLUTE_TOLERANCE,
        (
            f"{partition} {residual_scope} right-censoring "
            "compensator is negative."
        ),
    )

    right_censoring_tau = max(
        0.0,
        right_censoring_tau,
    )

    return {
        "event_partition": partition,
        "partition_order": (
            PARTITION_ORDER_MAP[
                partition
            ]
        ),
        "residual_scope": residual_scope,
        "last_relevant_event_time_ns": (
            last_event_time_ns
        ),
        "partition_end_exclusive_ns": int(
            path[
                "interval_end_ns"
            ].iloc[-1]
        ),
        "partition_total_compensator": (
            total_compensator
        ),
        "compensator_through_last_relevant_event": (
            compensator_through_last_event
        ),
        "right_censoring_tau": (
            right_censoring_tau
        ),
        "right_censoring_survival_probability": float(
            math.exp(
                -right_censoring_tau
            )
        ),
        "status": "PASS",
    }


right_censoring_records: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    partition_path = (
        EXACT_COMPENSATOR_PATH.loc[
            EXACT_COMPENSATOR_PATH[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
        .copy()
    )

    right_censoring_records.append(
        build_right_censoring_record(
            partition_path,
            partition=partition_name,
            residual_scope="BUY",
            relevant_count_column=(
                "buy_event_count"
            ),
            interval_compensator_column=(
                "interval_compensator_buy"
            ),
        )
    )

    right_censoring_records.append(
        build_right_censoring_record(
            partition_path,
            partition=partition_name,
            residual_scope="SELL",
            relevant_count_column=(
                "sell_event_count"
            ),
            interval_compensator_column=(
                "interval_compensator_sell"
            ),
        )
    )

    right_censoring_records.append(
        build_right_censoring_record(
            partition_path,
            partition=partition_name,
            residual_scope="POOLED",
            relevant_count_column=(
                "batch_event_count"
            ),
            interval_compensator_column=(
                "interval_total_compensator"
            ),
        )
    )

TIME_RESCALING_RIGHT_CENSORING_LEDGER = (
    pd.DataFrame.from_records(
        right_censoring_records
    )
    .sort_values(
        [
            "partition_order",
            "residual_scope",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

TIME_RESCALING_RIGHT_CENSORING_LEDGER[
    "last_relevant_event_time_ns"
] = TIME_RESCALING_RIGHT_CENSORING_LEDGER[
    "last_relevant_event_time_ns"
].astype(
    "Int64"
)


# ------------------------------------------------------------
# Diagnostic summaries
# ------------------------------------------------------------

eligible_residuals = (
    TIME_RESCALING_RESIDUALS.loc[
        TIME_RESCALING_RESIDUALS[
            "primary_diagnostic_eligible"
        ]
    ]
    .copy()
)

TIME_RESCALING_RESIDUAL_SUMMARY = (
    TIME_RESCALING_RESIDUALS.groupby(
        [
            "residual_scope",
            "destination_partition_order",
            "destination_event_partition",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        residual_count=(
            "tau",
            "size",
        ),
        primary_eligible_count=(
            "primary_diagnostic_eligible",
            "sum",
        ),
        left_edge_censored_count=(
            "left_edge_censored",
            "sum",
        ),
        cross_partition_interval_count=(
            "cross_partition_interval",
            "sum",
        ),
        multiplicity_gt_one_count=(
            "relevant_multiplicity_gt_one",
            "sum",
        ),
        simultaneous_batch_count=(
            "exact_time_batch_multiplicity_gt_one",
            "sum",
        ),
        tau_sum=(
            "tau",
            "sum",
        ),
        tau_mean=(
            "tau",
            "mean",
        ),
        tau_variance=(
            "tau",
            lambda values: float(
                values.var(
                    ddof=1
                )
            ),
        ),
        tau_minimum=(
            "tau",
            "min",
        ),
        tau_maximum=(
            "tau",
            "max",
        ),
        u_minimum=(
            "u",
            "min",
        ),
        u_maximum=(
            "u",
            "max",
        ),
    )
    .reset_index()
)

PRIMARY_ELIGIBLE_RESIDUAL_SUMMARY = (
    eligible_residuals.groupby(
        [
            "residual_scope",
            "destination_partition_order",
            "destination_event_partition",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        residual_count=(
            "tau",
            "size",
        ),
        tau_mean=(
            "tau",
            "mean",
        ),
        tau_variance=(
            "tau",
            lambda values: float(
                values.var(
                    ddof=1
                )
            ),
        ),
        tau_median=(
            "tau",
            "median",
        ),
        tau_q95=(
            "tau",
            lambda values: float(
                values.quantile(
                    0.95
                )
            ),
        ),
        tau_q99=(
            "tau",
            lambda values: float(
                values.quantile(
                    0.99
                )
            ),
        ),
        u_mean=(
            "u",
            "mean",
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Formal verification matrix
# ------------------------------------------------------------

TIME_RESCALING_VERIFICATION_SUMMARY = pd.DataFrame(
    [
        {
            "check_name": (
                "buy_residual_count_reconciles"
            ),
            "passed": (
                len(
                    BUY_TIME_RESCALING_RESIDUALS
                )
                == expected_buy_residual_count
            ),
        },
        {
            "check_name": (
                "sell_residual_count_reconciles"
            ),
            "passed": (
                len(
                    SELL_TIME_RESCALING_RESIDUALS
                )
                == expected_sell_residual_count
            ),
        },
        {
            "check_name": (
                "pooled_residual_count_reconciles"
            ),
            "passed": (
                len(
                    POOLED_TIME_RESCALING_RESIDUALS
                )
                == expected_pooled_residual_count
            ),
        },
        {
            "check_name": (
                "all_tau_strictly_positive"
            ),
            "passed": bool(
                TIME_RESCALING_RESIDUALS[
                    "tau"
                ].gt(
                    0.0
                ).all()
            ),
        },
        {
            "check_name": (
                "all_uniform_transforms_inside_unit_interval"
            ),
            "passed": bool(
                TIME_RESCALING_RESIDUALS[
                    "u"
                ].gt(
                    0.0
                ).all()
                and TIME_RESCALING_RESIDUALS[
                    "u"
                ].lt(
                    1.0
                ).all()
            ),
        },
        {
            "check_name": (
                "one_left_edge_residual_per_scope"
            ),
            "passed": bool(
                TIME_RESCALING_RESIDUALS.groupby(
                    "residual_scope",
                    observed=True,
                )[
                    "left_edge_censored"
                ]
                .sum()
                .eq(
                    1
                )
                .all()
            ),
        },
        {
            "check_name": (
                "one_cross_partition_residual_per_scope"
            ),
            "passed": bool(
                TIME_RESCALING_RESIDUALS.groupby(
                    "residual_scope",
                    observed=True,
                )[
                    "cross_partition_interval"
                ]
                .sum()
                .eq(
                    1
                )
                .all()
            ),
        },
        {
            "check_name": (
                "multiplicity_preserved_without_event_expansion"
            ),
            "passed": bool(
                TIME_RESCALING_RESIDUALS[
                    "relevant_event_multiplicity"
                ].ge(
                    1
                ).all()
            ),
        },
        {
            "check_name": (
                "coarsening_aware_reference_required"
            ),
            "passed": bool(
                TIME_RESCALING_RESIDUALS[
                    "coarsening_aware_simulation_required"
                ].all()
            ),
        },
        {
            "check_name": (
                "right_censoring_tau_nonnegative"
            ),
            "passed": bool(
                TIME_RESCALING_RIGHT_CENSORING_LEDGER[
                    "right_censoring_tau"
                ].ge(
                    0.0
                ).all()
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    TIME_RESCALING_VERIFICATION_SUMMARY[
        "passed"
    ].all(),
    "At least one time-rescaling residual verification failed.",
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

TIME_RESCALING_RESIDUALS_BUILT = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    TIME_RESCALING_RESIDUAL_SUMMARY
)

display(
    PRIMARY_ELIGIBLE_RESIDUAL_SUMMARY
)

display(
    TIME_RESCALING_RIGHT_CENSORING_LEDGER
)

display(
    TIME_RESCALING_VERIFICATION_SUMMARY
)

print(
    "BUY, SELL, and pooled exact-time batch-arrival residuals "
    "constructed from the canonical compensator path. The first "
    "DEVELOPMENT residual in each scope is marked as left-edge "
    "censored. CALIBRATION residuals preserve inherited "
    "DEVELOPMENT history across the frozen boundary. Exact-time "
    "multiplicity remains intact and was not converted into "
    "zero-duration pseudo-events. Formal diagnostics must use "
    "coarsening-aware simulation envelopes. No filesystem writes "
    "were performed."
)

,residual_scope,destination_partition_order,destination_event_partition,residual_count,primary_eligible_count,left_edge_censored_count,cross_partition_interval_count,multiplicity_gt_one_count,simultaneous_batch_count,tau_sum,tau_mean,tau_variance,tau_minimum,tau_maximum,u_minimum,u_maximum
0,BUY,1,DEVELOPMENT,3331,3330,1,0,64,85,"3,412.973963",1.024609416,1.094237088,0.005779317497,12.39146669,0.005762649367,0.9999958461
1,BUY,2,CALIBRATION,1155,1155,0,1,53,57,"1,338.390906",1.158780005,1.594861185,0.01143951613,12.34853088,0.01137433365,0.9999956639
2,POOLED,1,DEVELOPMENT,6859,6858,1,0,111,111,"7,003.360868",1.021046926,0.9947398374,0.006416297823,8.963067697,0.006395757339,0.9998719472
3,POOLED,2,CALIBRATION,2400,2400,0,1,68,68,"2,754.376645",1.147656935,1.354008627,0.006770062419,10.34195407,0.006747197176,0.9999677488
4,SELL,1,DEVELOPMENT,3554,3553,1,0,26,52,"3,589.60566",1.010018475,0.9322805701,0.006337049108,7.856394685,0.00631701236,0.9996127324
5,SELL,2,CALIBRATION,1250,1250,0,1,11,16,"1,415.401864",1.132321492,1.111779563,0.005832638364,9.789958059,0.005815661552,0.9999439888


,residual_scope,destination_partition_order,destination_event_partition,residual_count,tau_mean,tau_variance,tau_median,tau_q95,tau_q99,u_mean
0,BUY,1,DEVELOPMENT,3330,1.024347418,1.094337069,0.6903253633,3.172370479,4.871577682,0.5101174686
1,BUY,2,CALIBRATION,1155,1.158780005,1.594861185,0.7612314207,3.687631391,5.379879337,0.5229131466
2,POOLED,1,DEVELOPMENT,6858,1.020600829,0.9935197499,0.7107736607,3.077209017,4.540780086,0.5108032798
3,POOLED,2,CALIBRATION,2400,1.147656935,1.354008627,0.7626387177,3.607729629,5.206202151,0.5292579917
4,SELL,1,DEVELOPMENT,3553,1.008954131,0.9285158346,0.699910082,2.917958165,4.357666668,0.5094955723
5,SELL,2,CALIBRATION,1250,1.132321492,1.111779563,0.819171951,3.151996533,4.432222212,0.5430980601


,event_partition,partition_order,residual_scope,last_relevant_event_time_ns,partition_end_exclusive_ns,partition_total_compensator,compensator_through_last_relevant_event,right_censoring_tau,right_censoring_survival_probability,status
0,DEVELOPMENT,1,BUY,1783667268846827100,1783667269391572100,"3,414.000015","3,412.973963",1.026051963,0.3584192234,PASS
1,DEVELOPMENT,1,POOLED,1783667269232205000,1783667269391572100,"7,004.000086","7,003.360868",0.6392177512,0.5277050593,PASS
2,DEVELOPMENT,1,SELL,1783667269232205000,1783667269391572100,"3,590.00007","3,589.60566",0.3944099968,0.6740776299,PASS
3,CALIBRATION,2,BUY,1783667988585894300,1783667989690751200,"1,339.253892","1,337.364854",1.889038013,0.1512172078,PASS
4,CALIBRATION,2,POOLED,1783667989349685300,1783667989690751200,"2,754.976987","2,753.737427",1.239559541,0.289511708,PASS
5,CALIBRATION,2,SELL,1783667989349685300,1783667989690751200,"1,415.723094","1,415.007454",0.7156397535,0.4888792495,PASS


,check_name,passed
0,buy_residual_count_reconciles,True
1,sell_residual_count_reconciles,True
2,pooled_residual_count_reconciles,True
3,all_tau_strictly_positive,True
4,all_uniform_transforms_inside_unit_interval,True
5,one_left_edge_residual_per_scope,True
6,one_cross_partition_residual_per_scope,True
7,multiplicity_preserved_without_event_expansion,True
8,coarsening_aware_reference_required,True
9,right_censoring_tau_nonnegative,True


BUY, SELL, and pooled exact-time batch-arrival residuals constructed from the canonical compensator path. The first DEVELOPMENT residual in each scope is marked as left-edge censored. CALIBRATION residuals preserve inherited DEVELOPMENT history across the frozen boundary. Exact-time multiplicity remains intact and was not converted into zero-duration pseudo-events. Formal diagnostics must use coarsening-aware simulation envelopes. No filesystem writes were performed.


In [10]:
# ============================================================
# Empirical residual distribution and serial-dependence diagnostics
#
# These diagnostics use the Exp(1) and Uniform(0,1) references
# descriptively. Raw p-values are report-only because parameters
# are estimated, event times are coarsened, and exact-time batches
# may carry multiplicity. Final adequacy decisions remain blocked
# until coarsening-aware simulation envelopes are available.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "TIME_RESCALING_RESIDUALS_BUILT",
            False,
        )
    ),
    "Time-rescaling residuals have not been built.",
)

require(
    bool(
        globals().get(
            "EXACT_COMPENSATOR_PATH_BUILT",
            False,
        )
    ),
    "The exact compensator path has not been built.",
)

require(
    not bool(
        globals().get(
            "EMPIRICAL_RESIDUAL_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Empirical residual diagnostics are already built.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Diagnostic configuration
# ------------------------------------------------------------

RESIDUAL_SCOPES: Final[tuple[str, str, str]] = (
    "BUY",
    "SELL",
    "POOLED",
)

RESIDUAL_DIAGNOSTIC_PARTITIONS: Final[
    tuple[str, str]
] = (
    "DEVELOPMENT",
    "CALIBRATION",
)

RESIDUAL_GROUP_COLUMNS: Final[
    tuple[str, str, str]
] = (
    "residual_scope",
    "destination_partition_order",
    "destination_event_partition",
)

QQ_PROBABILITIES: Final[np.ndarray] = np.linspace(
    0.01,
    0.99,
    99,
    dtype=np.float64,
)

NAIVE_ACF_STANDARD_ERROR_MULTIPLIER: Final[float] = 1.96
SERIAL_SUMMARY_LAG: Final[int] = 20

require(
    RESIDUAL_ACF_MAX_LAG >= max(
        LJUNG_BOX_LAGS
    ),
    (
        "The configured ACF maximum lag is below a "
        "requested Ljung-Box lag."
    ),
)

require(
    SERIAL_SUMMARY_LAG
    <= RESIDUAL_ACF_MAX_LAG,
    "The serial-summary lag exceeds the ACF maximum lag.",
)


# ------------------------------------------------------------
# Numerical helpers
# ------------------------------------------------------------

def uniform_ks_components(
    uniform_values: np.ndarray,
) -> tuple[float, float, float]:
    """Return the one-sample Uniform KS D+, D-, and D statistics."""
    values = np.asarray(
        uniform_values,
        dtype=np.float64,
    )

    require(
        values.ndim == 1,
        "KS input must be one-dimensional.",
    )

    require(
        len(values) >= 2,
        "At least two values are required for a KS statistic.",
    )

    require(
        np.isfinite(
            values
        ).all(),
        "KS input contains non-finite values.",
    )

    require(
        np.all(
            values > 0.0
        )
        and np.all(
            values < 1.0
        ),
        "Uniform KS input must lie strictly inside (0, 1).",
    )

    ordered = np.sort(
        values
    )

    sample_size = len(
        ordered
    )

    upper_empirical = (
        np.arange(
            1,
            sample_size + 1,
            dtype=np.float64,
        )
        / sample_size
    )

    lower_empirical = (
        np.arange(
            0,
            sample_size,
            dtype=np.float64,
        )
        / sample_size
    )

    d_plus = float(
        np.max(
            upper_empirical
            - ordered
        )
    )

    d_minus = float(
        np.max(
            ordered
            - lower_empirical
        )
    )

    d_statistic = max(
        d_plus,
        d_minus,
    )

    return (
        d_plus,
        d_minus,
        d_statistic,
    )


def uniform_cramer_von_mises_statistic(
    uniform_values: np.ndarray,
) -> float:
    """Return the one-sample Uniform Cramér-von Mises statistic."""
    values = np.asarray(
        uniform_values,
        dtype=np.float64,
    )

    require(
        values.ndim == 1,
        "Cramér-von Mises input must be one-dimensional.",
    )

    require(
        len(values) >= 2,
        (
            "At least two values are required for a "
            "Cramér-von Mises statistic."
        ),
    )

    ordered = np.sort(
        values
    )

    sample_size = len(
        ordered
    )

    theoretical_midpoints = (
        (
            2.0
            * np.arange(
                1,
                sample_size + 1,
                dtype=np.float64,
            )
            - 1.0
        )
        / (
            2.0
            * sample_size
        )
    )

    statistic = (
        1.0
        / (
            12.0
            * sample_size
        )
        + np.sum(
            (
                ordered
                - theoretical_midpoints
            )
            ** 2
        )
    )

    return float(
        statistic
    )


def sample_autocorrelation(
    values: np.ndarray,
    *,
    maximum_lag: int,
) -> np.ndarray:
    """Return the biased sample ACF from lag zero through maximum_lag."""
    series = np.asarray(
        values,
        dtype=np.float64,
    )

    require(
        series.ndim == 1,
        "ACF input must be one-dimensional.",
    )

    require(
        len(series)
        > maximum_lag,
        (
            "The residual sequence is too short for "
            f"maximum lag {maximum_lag}."
        ),
    )

    require(
        np.isfinite(
            series
        ).all(),
        "ACF input contains non-finite values.",
    )

    centered = (
        series
        - float(
            np.mean(
                series
            )
        )
    )

    denominator = float(
        np.dot(
            centered,
            centered,
        )
    )

    require(
        denominator > 0.0,
        "ACF input has zero variance.",
    )

    autocorrelations = np.empty(
        maximum_lag + 1,
        dtype=np.float64,
    )

    autocorrelations[0] = 1.0

    for lag in range(
        1,
        maximum_lag + 1,
    ):
        autocorrelations[
            lag
        ] = float(
            np.dot(
                centered[:-lag],
                centered[lag:],
            )
            / denominator
        )

    require(
        np.isfinite(
            autocorrelations
        ).all(),
        "The ACF calculation produced non-finite values.",
    )

    return autocorrelations


def ljung_box_statistic(
    autocorrelations: np.ndarray,
    *,
    sample_size: int,
    lag: int,
) -> tuple[float, float]:
    """Return the Ljung-Box Q statistic and chi-square p-value."""
    acf_values = np.asarray(
        autocorrelations,
        dtype=np.float64,
    )

    require(
        sample_size > lag,
        (
            f"Sample size {sample_size} is too small "
            f"for Ljung-Box lag {lag}."
        ),
    )

    require(
        len(
            acf_values
        )
        > lag,
        "The ACF vector does not cover the requested lag.",
    )

    lag_numbers = np.arange(
        1,
        lag + 1,
        dtype=np.float64,
    )

    selected_acf = acf_values[
        1 : lag + 1
    ]

    q_statistic = float(
        sample_size
        * (
            sample_size
            + 2.0
        )
        * np.sum(
            (
                selected_acf
                ** 2
            )
            / (
                sample_size
                - lag_numbers
            )
        )
    )

    p_value = float(
        stats.chi2.sf(
            q_statistic,
            df=lag,
        )
    )

    require(
        math.isfinite(
            q_statistic
        ),
        "The Ljung-Box statistic is non-finite.",
    )

    require(
        math.isfinite(
            p_value
        )
        and 0.0
        <= p_value
        <= 1.0,
        "The Ljung-Box p-value is invalid.",
    )

    return (
        q_statistic,
        p_value,
    )


# ------------------------------------------------------------
# Primary eligible residual interface
# ------------------------------------------------------------

PRIMARY_ELIGIBLE_TIME_RESCALING_RESIDUALS = (
    TIME_RESCALING_RESIDUALS.loc[
        TIME_RESCALING_RESIDUALS[
            "primary_diagnostic_eligible"
        ]
    ]
    .copy()
    .sort_values(
        [
            "residual_scope",
            "destination_partition_order",
            "event_time_ns",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    not PRIMARY_ELIGIBLE_TIME_RESCALING_RESIDUALS.empty,
    "No primary-eligible residuals are available.",
)

observed_group_keys = {
    (
        str(
            scope
        ),
        str(
            partition
        ),
    )
    for (
        scope,
        partition,
    ) in (
        PRIMARY_ELIGIBLE_TIME_RESCALING_RESIDUALS[
            [
                "residual_scope",
                "destination_event_partition",
            ]
        ]
        .drop_duplicates()
        .itertuples(
            index=False,
            name=None,
        )
    )
}

expected_group_keys = {
    (
        scope,
        partition,
    )
    for scope in RESIDUAL_SCOPES
    for partition in RESIDUAL_DIAGNOSTIC_PARTITIONS
}

require(
    observed_group_keys
    == expected_group_keys,
    (
        "The primary residual diagnostic groups are incorrect: "
        f"observed={sorted(observed_group_keys)}."
    ),
)


# ------------------------------------------------------------
# Build empirical diagnostics
# ------------------------------------------------------------

distribution_records: list[
    dict[str, Any]
] = []

acf_records: list[
    dict[str, Any]
] = []

ljung_box_records: list[
    dict[str, Any]
] = []

qq_records: list[
    dict[str, Any]
] = []

serial_summary_records: list[
    dict[str, Any]
] = []

acf_lookup: dict[
    tuple[str, str],
    np.ndarray,
] = {}

distribution_lookup: dict[
    tuple[str, str],
    dict[str, Any],
] = {}

grouped_residuals = (
    PRIMARY_ELIGIBLE_TIME_RESCALING_RESIDUALS.groupby(
        list(
            RESIDUAL_GROUP_COLUMNS
        ),
        observed=True,
        sort=True,
        dropna=False,
    )
)

for (
    residual_scope,
    partition_order,
    event_partition,
), group in grouped_residuals:
    scope = str(
        residual_scope
    )

    partition = str(
        event_partition
    )

    ordered_group = (
        group.sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    tau = ordered_group[
        "tau"
    ].to_numpy(
        dtype=np.float64
    )

    uniform_values = ordered_group[
        "u"
    ].to_numpy(
        dtype=np.float64
    )

    sample_size = len(
        tau
    )

    require(
        sample_size
        > RESIDUAL_ACF_MAX_LAG,
        (
            f"{scope} {partition} has insufficient residuals "
            "for the configured lag diagnostics."
        ),
    )

    require(
        np.isfinite(
            tau
        ).all()
        and np.all(
            tau > 0.0
        ),
        f"{scope} {partition} contains invalid tau values.",
    )

    require_array_close(
        uniform_values,
        -np.expm1(
            -tau
        ),
        label=(
            f"{scope} {partition} PIT transformation"
        ),
        rtol=1e-13,
        atol=1e-13,
    )

    (
        ks_d_plus,
        ks_d_minus,
        ks_d_uniform,
    ) = uniform_ks_components(
        uniform_values
    )

    ks_d_exponential = ks_d_uniform

    ks_asymptotic_p_value = float(
        stats.kstwo.sf(
            ks_d_uniform,
            sample_size,
        )
    )

    require(
        math.isfinite(
            ks_asymptotic_p_value
        )
        and 0.0
        <= ks_asymptotic_p_value
        <= 1.0,
        (
            f"{scope} {partition} produced an invalid "
            "KS p-value."
        ),
    )

    cvm_statistic = (
        uniform_cramer_von_mises_statistic(
            uniform_values
        )
    )

    tau_q95 = float(
        np.quantile(
            tau,
            0.95,
            method="linear",
        )
    )

    tau_q99 = float(
        np.quantile(
            tau,
            0.99,
            method="linear",
        )
    )

    theoretical_q95 = float(
        -math.log(
            0.05
        )
    )

    theoretical_q99 = float(
        -math.log(
            0.01
        )
    )

    distribution_record = {
        "residual_scope": scope,
        "partition_order": int(
            partition_order
        ),
        "event_partition": partition,
        "sample_size": sample_size,
        "tau_mean": float(
            np.mean(
                tau
            )
        ),
        "tau_variance": float(
            np.var(
                tau,
                ddof=1,
            )
        ),
        "tau_median": float(
            np.median(
                tau
            )
        ),
        "tau_q95": tau_q95,
        "tau_q99": tau_q99,
        "tau_q95_ratio_to_exp1": (
            tau_q95
            / theoretical_q95
        ),
        "tau_q99_ratio_to_exp1": (
            tau_q99
            / theoretical_q99
        ),
        "uniform_mean": float(
            np.mean(
                uniform_values
            )
        ),
        "uniform_variance": float(
            np.var(
                uniform_values,
                ddof=1,
            )
        ),
        "ks_d_plus": ks_d_plus,
        "ks_d_minus": ks_d_minus,
        "ks_d_uniform": ks_d_uniform,
        "ks_d_exponential": ks_d_exponential,
        "ks_asymptotic_p_value_report_only": (
            ks_asymptotic_p_value
        ),
        "cramer_von_mises_statistic": (
            cvm_statistic
        ),
        "absolute_tau_mean_error": abs(
            float(
                np.mean(
                    tau
                )
            )
            - 1.0
        ),
        "absolute_tau_variance_error": abs(
            float(
                np.var(
                    tau,
                    ddof=1,
                )
            )
            - 1.0
        ),
        "large_sample_flag": (
            sample_size
            >= 1_000
        ),
        "formal_p_values_are_acceptance_gates": False,
        "simulation_envelope_pending": True,
        "reference_role": (
            "DESCRIPTIVE_ONLY_PENDING_COARSENING_"
            "AWARE_SIMULATION"
        ),
        "status": "COMPUTED_REPORT_ONLY",
    }

    distribution_records.append(
        distribution_record
    )

    distribution_lookup[
        (
            scope,
            partition,
        )
    ] = distribution_record

    acf_values = sample_autocorrelation(
        tau,
        maximum_lag=RESIDUAL_ACF_MAX_LAG,
    )

    acf_lookup[
        (
            scope,
            partition,
        )
    ] = acf_values

    naive_confidence_bound = (
        NAIVE_ACF_STANDARD_ERROR_MULTIPLIER
        / math.sqrt(
            sample_size
        )
    )

    for lag, autocorrelation in enumerate(
        acf_values
    ):
        acf_records.append(
            {
                "residual_scope": scope,
                "partition_order": int(
                    partition_order
                ),
                "event_partition": partition,
                "sample_size": sample_size,
                "lag": int(
                    lag
                ),
                "autocorrelation": float(
                    autocorrelation
                ),
                "absolute_autocorrelation": abs(
                    float(
                        autocorrelation
                    )
                ),
                "naive_lower_bound": (
                    -naive_confidence_bound
                ),
                "naive_upper_bound": (
                    naive_confidence_bound
                ),
                "outside_naive_bound": bool(
                    lag > 0
                    and abs(
                        autocorrelation
                    )
                    > naive_confidence_bound
                ),
                "bound_role": (
                    "DESCRIPTIVE_ONLY_PENDING_SIMULATION_ENVELOPE"
                ),
            }
        )

    for lag in LJUNG_BOX_LAGS:
        (
            q_statistic,
            p_value,
        ) = ljung_box_statistic(
            acf_values,
            sample_size=sample_size,
            lag=lag,
        )

        ljung_box_records.append(
            {
                "residual_scope": scope,
                "partition_order": int(
                    partition_order
                ),
                "event_partition": partition,
                "sample_size": sample_size,
                "lag": int(
                    lag
                ),
                "ljung_box_q": q_statistic,
                "degrees_of_freedom_reference": int(
                    lag
                ),
                "chi_square_p_value_report_only": (
                    p_value
                ),
                "formal_p_value_is_acceptance_gate": False,
                "simulation_envelope_pending": True,
                "status": "COMPUTED_REPORT_ONLY",
            }
        )

    lag_one_to_twenty = acf_values[
        1 : SERIAL_SUMMARY_LAG + 1
    ]

    lag_one_to_fifty = acf_values[
        1 : RESIDUAL_ACF_MAX_LAG + 1
    ]

    serial_summary_records.append(
        {
            "residual_scope": scope,
            "partition_order": int(
                partition_order
            ),
            "event_partition": partition,
            "sample_size": sample_size,
            "lag_1_autocorrelation": float(
                acf_values[
                    1
                ]
            ),
            "maximum_absolute_acf_lag_1_to_20": float(
                np.max(
                    np.abs(
                        lag_one_to_twenty
                    )
                )
            ),
            "maximum_absolute_acf_lag_1_to_50": float(
                np.max(
                    np.abs(
                        lag_one_to_fifty
                    )
                )
            ),
            "lag_of_maximum_absolute_acf_1_to_50": int(
                np.argmax(
                    np.abs(
                        lag_one_to_fifty
                    )
                )
                + 1
            ),
            "naive_acf_bound": (
                naive_confidence_bound
            ),
            "outside_naive_bound_count_lag_1_to_20": int(
                np.sum(
                    np.abs(
                        lag_one_to_twenty
                    )
                    > naive_confidence_bound
                )
            ),
            "outside_naive_bound_count_lag_1_to_50": int(
                np.sum(
                    np.abs(
                        lag_one_to_fifty
                    )
                    > naive_confidence_bound
                )
            ),
            "simulation_envelope_pending": True,
            "status": "COMPUTED_REPORT_ONLY",
        }
    )

    empirical_tau_quantiles = np.quantile(
        tau,
        QQ_PROBABILITIES,
        method="linear",
    )

    empirical_uniform_quantiles = np.quantile(
        uniform_values,
        QQ_PROBABILITIES,
        method="linear",
    )

    theoretical_tau_quantiles = -np.log1p(
        -QQ_PROBABILITIES
    )

    for (
        probability,
        theoretical_tau,
        empirical_tau,
        empirical_uniform,
    ) in zip(
        QQ_PROBABILITIES,
        theoretical_tau_quantiles,
        empirical_tau_quantiles,
        empirical_uniform_quantiles,
        strict=True,
    ):
        qq_records.append(
            {
                "residual_scope": scope,
                "partition_order": int(
                    partition_order
                ),
                "event_partition": partition,
                "sample_size": sample_size,
                "probability": float(
                    probability
                ),
                "theoretical_exp1_quantile": float(
                    theoretical_tau
                ),
                "empirical_tau_quantile": float(
                    empirical_tau
                ),
                "tau_quantile_difference": float(
                    empirical_tau
                    - theoretical_tau
                ),
                "theoretical_uniform_quantile": float(
                    probability
                ),
                "empirical_uniform_quantile": float(
                    empirical_uniform
                ),
                "uniform_quantile_difference": float(
                    empirical_uniform
                    - probability
                ),
                "simulation_envelope_pending": True,
            }
        )


# ------------------------------------------------------------
# Materialize diagnostic tables
# ------------------------------------------------------------

EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS = (
    pd.DataFrame.from_records(
        distribution_records
    )
    .sort_values(
        [
            "residual_scope",
            "partition_order",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

EMPIRICAL_RESIDUAL_ACF = (
    pd.DataFrame.from_records(
        acf_records
    )
    .sort_values(
        [
            "residual_scope",
            "partition_order",
            "lag",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

EMPIRICAL_RESIDUAL_LJUNG_BOX = (
    pd.DataFrame.from_records(
        ljung_box_records
    )
    .sort_values(
        [
            "residual_scope",
            "partition_order",
            "lag",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

EMPIRICAL_RESIDUAL_QQ_COORDINATES = (
    pd.DataFrame.from_records(
        qq_records
    )
    .sort_values(
        [
            "residual_scope",
            "partition_order",
            "probability",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

EMPIRICAL_RESIDUAL_SERIAL_SUMMARY = (
    pd.DataFrame.from_records(
        serial_summary_records
    )
    .sort_values(
        [
            "residual_scope",
            "partition_order",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# DEVELOPMENT-to-CALIBRATION degradation summary
# ------------------------------------------------------------

degradation_records: list[
    dict[str, Any]
] = []

serial_lookup = {
    (
        str(
            row.residual_scope
        ),
        str(
            row.event_partition
        ),
    ): row
    for row in (
        EMPIRICAL_RESIDUAL_SERIAL_SUMMARY.itertuples(
            index=False
        )
    )
}

for scope in RESIDUAL_SCOPES:
    development_distribution = (
        distribution_lookup[
            (
                scope,
                "DEVELOPMENT",
            )
        ]
    )

    calibration_distribution = (
        distribution_lookup[
            (
                scope,
                "CALIBRATION",
            )
        ]
    )

    development_serial = (
        serial_lookup[
            (
                scope,
                "DEVELOPMENT",
            )
        ]
    )

    calibration_serial = (
        serial_lookup[
            (
                scope,
                "CALIBRATION",
            )
        ]
    )

    degradation_records.append(
        {
            "residual_scope": scope,
            "development_sample_size": (
                development_distribution[
                    "sample_size"
                ]
            ),
            "calibration_sample_size": (
                calibration_distribution[
                    "sample_size"
                ]
            ),
            "development_ks_d": (
                development_distribution[
                    "ks_d_uniform"
                ]
            ),
            "calibration_ks_d": (
                calibration_distribution[
                    "ks_d_uniform"
                ]
            ),
            "calibration_minus_development_ks_d": (
                calibration_distribution[
                    "ks_d_uniform"
                ]
                - development_distribution[
                    "ks_d_uniform"
                ]
            ),
            "development_absolute_tau_mean_error": (
                development_distribution[
                    "absolute_tau_mean_error"
                ]
            ),
            "calibration_absolute_tau_mean_error": (
                calibration_distribution[
                    "absolute_tau_mean_error"
                ]
            ),
            "calibration_minus_development_tau_mean_error": (
                calibration_distribution[
                    "absolute_tau_mean_error"
                ]
                - development_distribution[
                    "absolute_tau_mean_error"
                ]
            ),
            "development_max_abs_acf_1_to_20": (
                development_serial
                .maximum_absolute_acf_lag_1_to_20
            ),
            "calibration_max_abs_acf_1_to_20": (
                calibration_serial
                .maximum_absolute_acf_lag_1_to_20
            ),
            "calibration_minus_development_max_abs_acf_1_to_20": (
                calibration_serial
                .maximum_absolute_acf_lag_1_to_20
                - development_serial
                .maximum_absolute_acf_lag_1_to_20
            ),
            "formal_degradation_gate_authorized": False,
            "simulation_envelope_pending": True,
            "status": "DESCRIPTIVE_ONLY",
        }
    )

EMPIRICAL_RESIDUAL_DEGRADATION_SUMMARY = (
    pd.DataFrame.from_records(
        degradation_records
    )
    .sort_values(
        "residual_scope",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Reconciliation and verification
# ------------------------------------------------------------

require(
    len(
        EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS
    )
    == len(
        expected_group_keys
    ),
    "The residual distribution diagnostic row count is incorrect.",
)

require(
    len(
        EMPIRICAL_RESIDUAL_ACF
    )
    == (
        len(
            expected_group_keys
        )
        * (
            RESIDUAL_ACF_MAX_LAG
            + 1
        )
    ),
    "The residual ACF row count is incorrect.",
)

require(
    len(
        EMPIRICAL_RESIDUAL_LJUNG_BOX
    )
    == (
        len(
            expected_group_keys
        )
        * len(
            LJUNG_BOX_LAGS
        )
    ),
    "The Ljung-Box diagnostic row count is incorrect.",
)

require(
    len(
        EMPIRICAL_RESIDUAL_QQ_COORDINATES
    )
    == (
        len(
            expected_group_keys
        )
        * len(
            QQ_PROBABILITIES
        )
    ),
    "The residual Q-Q coordinate row count is incorrect.",
)

require(
    np.allclose(
        EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS[
            "ks_d_uniform"
        ].to_numpy(
            dtype=np.float64
        ),
        EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS[
            "ks_d_exponential"
        ].to_numpy(
            dtype=np.float64
        ),
        rtol=0.0,
        atol=1e-15,
    ),
    "Uniform and exponential KS distances do not reconcile.",
)

require(
    EMPIRICAL_RESIDUAL_ACF.loc[
        EMPIRICAL_RESIDUAL_ACF[
            "lag"
        ].eq(
            0
        ),
        "autocorrelation",
    ].eq(
        1.0
    ).all(),
    "At least one lag-zero autocorrelation differs from one.",
)

require(
    not EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS[
        "formal_p_values_are_acceptance_gates"
    ].any(),
    "A raw residual p-value was incorrectly made an acceptance gate.",
)

require(
    not EMPIRICAL_RESIDUAL_LJUNG_BOX[
        "formal_p_value_is_acceptance_gate"
    ].any(),
    "A Ljung-Box p-value was incorrectly made an acceptance gate.",
)

require(
    EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS[
        "simulation_envelope_pending"
    ].all(),
    "A distribution diagnostic incorrectly bypassed simulation envelopes.",
)

require(
    EMPIRICAL_RESIDUAL_SERIAL_SUMMARY[
        "simulation_envelope_pending"
    ].all(),
    "A serial diagnostic incorrectly bypassed simulation envelopes.",
)


EMPIRICAL_RESIDUAL_DIAGNOSTIC_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "six_scope_partition_groups_present"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "distribution_diagnostics_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "uniform_and_exponential_ks_reconcile"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "acf_lags_zero_through_fifty_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "lag_zero_acf_equals_one"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "ljung_box_lags_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "qq_coordinate_grid_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "raw_p_values_report_only"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "simulation_envelopes_still_required"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    EMPIRICAL_RESIDUAL_DIAGNOSTIC_VERIFICATION[
        "passed"
    ].all(),
    "At least one empirical residual diagnostic verification failed.",
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

EMPIRICAL_RESIDUAL_DIAGNOSTICS_BUILT = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS
)

display(
    EMPIRICAL_RESIDUAL_SERIAL_SUMMARY
)

display(
    EMPIRICAL_RESIDUAL_LJUNG_BOX
)

display(
    EMPIRICAL_RESIDUAL_DEGRADATION_SUMMARY
)

display(
    EMPIRICAL_RESIDUAL_DIAGNOSTIC_VERIFICATION
)

print(
    "Empirical BUY, SELL, and pooled residual diagnostics computed "
    "for DEVELOPMENT and locked CALIBRATION. KS distances, "
    "Cramér-von Mises statistics, Q-Q coordinates, ACF values "
    "through lag 50, and Ljung-Box statistics at lags 10, 20, "
    "and 50 are available. Raw p-values and naive ACF bounds are "
    "report-only. Final adequacy decisions remain blocked pending "
    "coarsening-aware simulation envelopes. No filesystem writes "
    "were performed."
)

,residual_scope,partition_order,event_partition,sample_size,tau_mean,tau_variance,tau_median,tau_q95,tau_q99,tau_q95_ratio_to_exp1,tau_q99_ratio_to_exp1,uniform_mean,uniform_variance,ks_d_plus,ks_d_minus,ks_d_uniform,ks_d_exponential,ks_asymptotic_p_value_report_only,cramer_von_mises_statistic,absolute_tau_mean_error,absolute_tau_variance_error,large_sample_flag,formal_p_values_are_acceptance_gates,simulation_envelope_pending,reference_role,status
0,BUY,1,DEVELOPMENT,3330,1.024347418,1.094337069,0.6903253633,3.172370479,4.871577682,1.058963281,1.057849653,0.5101174686,0.07683219889,0.01320601024,0.04942562989,0.04942562989,0.04942562989,1.650114373e-07,1.381475911,0.02434741778,0.09433706872,True,False,True,DESCRIPTIVE_ONLY_PENDING_COARSENING_AWARE_SIMU...,COMPUTED_REPORT_ONLY
1,BUY,2,CALIBRATION,1155,1.158780005,1.594861185,0.7612314207,3.687631391,5.379879337,1.2309616,1.168225955,0.5229131466,0.09207193542,0.02309474393,0.05897666468,0.05897666468,0.05897666468,0.0006200975584,1.069331742,0.1587800054,0.5948611846,True,False,True,DESCRIPTIVE_ONLY_PENDING_COARSENING_AWARE_SIMU...,COMPUTED_REPORT_ONLY
2,POOLED,1,DEVELOPMENT,6858,1.020600829,0.9935197499,0.7107736607,3.077209017,4.540780086,1.027197605,0.9860178673,0.5108032798,0.07947309893,0.001369633295,0.03087412143,0.03087412143,0.03087412143,4.102000797e-06,1.336526026,0.02060082945,0.006480250099,True,False,True,DESCRIPTIVE_ONLY_PENDING_COARSENING_AWARE_SIMU...,COMPUTED_REPORT_ONLY
3,POOLED,2,CALIBRATION,2400,1.147656935,1.354008627,0.7626387177,3.607729629,5.206202151,1.204289736,1.130512433,0.5292579917,0.0883582483,0.007166426537,0.05017457506,0.05017457506,0.05017457506,1.08654879e-05,2.58636667,0.1476569353,0.3540086273,True,False,True,DESCRIPTIVE_ONLY_PENDING_COARSENING_AWARE_SIMU...,COMPUTED_REPORT_ONLY
4,SELL,1,DEVELOPMENT,3553,1.008954131,0.9285158346,0.699910082,2.917958165,4.357666668,0.9740383649,0.9462552939,0.5094955723,0.07953037598,0.005128476934,0.03057856077,0.03057856077,0.03057856077,0.002547447864,0.5766399675,0.008954131368,0.07148416542,True,False,True,DESCRIPTIVE_ONLY_PENDING_COARSENING_AWARE_SIMU...,COMPUTED_REPORT_ONLY
5,SELL,2,CALIBRATION,1250,1.132321492,1.111779563,0.819171951,3.151996533,4.432222212,1.052162291,0.9624448247,0.5430980601,0.08263674445,0.004568540198,0.06917423779,0.06917423779,0.06917423779,1.205123278e-05,2.700686436,0.1323214916,0.1117795635,True,False,True,DESCRIPTIVE_ONLY_PENDING_COARSENING_AWARE_SIMU...,COMPUTED_REPORT_ONLY


,residual_scope,partition_order,event_partition,sample_size,lag_1_autocorrelation,maximum_absolute_acf_lag_1_to_20,maximum_absolute_acf_lag_1_to_50,lag_of_maximum_absolute_acf_1_to_50,naive_acf_bound,outside_naive_bound_count_lag_1_to_20,outside_naive_bound_count_lag_1_to_50,simulation_envelope_pending,status
0,BUY,1,DEVELOPMENT,3330,0.1462065372,0.1462065372,0.1462065372,1,0.03396518267,19,24,True,COMPUTED_REPORT_ONLY
1,BUY,2,CALIBRATION,1155,0.1769606004,0.1830735657,0.1830735657,3,0.05767200886,11,12,True,COMPUTED_REPORT_ONLY
2,POOLED,1,DEVELOPMENT,6858,0.09510970638,0.09510970638,0.09510970638,1,0.0236677695,20,34,True,COMPUTED_REPORT_ONLY
3,POOLED,2,CALIBRATION,2400,0.07421511606,0.1157581784,0.1157581784,6,0.04000833247,11,13,True,COMPUTED_REPORT_ONLY
4,SELL,1,DEVELOPMENT,3553,0.06629993476,0.06629993476,0.06629993476,1,0.03288201837,8,14,True,COMPUTED_REPORT_ONLY
5,SELL,2,CALIBRATION,1250,0.05986772615,0.09717201363,0.09717201363,9,0.05543717165,8,9,True,COMPUTED_REPORT_ONLY


,residual_scope,partition_order,event_partition,sample_size,lag,ljung_box_q,degrees_of_freedom_reference,chi_square_p_value_report_only,formal_p_value_is_acceptance_gate,simulation_envelope_pending,status
0,BUY,1,DEVELOPMENT,3330,10,301.440721,10,7.710056153e-59,False,True,COMPUTED_REPORT_ONLY
1,BUY,1,DEVELOPMENT,3330,20,418.5842406,20,2.828736581e-76,False,True,COMPUTED_REPORT_ONLY
2,BUY,1,DEVELOPMENT,3330,50,490.1764857,50,1.429249685e-73,False,True,COMPUTED_REPORT_ONLY
3,BUY,2,CALIBRATION,1155,10,164.1988708,10,4.397770977e-30,False,True,COMPUTED_REPORT_ONLY
4,BUY,2,CALIBRATION,1155,20,184.9596328,20,1.035392201e-28,False,True,COMPUTED_REPORT_ONLY
5,BUY,2,CALIBRATION,1155,50,220.8985425,50,2.402935093e-23,False,True,COMPUTED_REPORT_ONLY
6,POOLED,1,DEVELOPMENT,6858,10,306.2280832,10,7.493502971e-60,False,True,COMPUTED_REPORT_ONLY
7,POOLED,1,DEVELOPMENT,6858,20,384.4082109,20,3.480413493e-69,False,True,COMPUTED_REPORT_ONLY
8,POOLED,1,DEVELOPMENT,6858,50,512.9371824,50,4.827249327e-78,False,True,COMPUTED_REPORT_ONLY
9,POOLED,2,CALIBRATION,2400,10,119.9165415,10,5.261256366e-21,False,True,COMPUTED_REPORT_ONLY


,residual_scope,development_sample_size,calibration_sample_size,development_ks_d,calibration_ks_d,calibration_minus_development_ks_d,development_absolute_tau_mean_error,calibration_absolute_tau_mean_error,calibration_minus_development_tau_mean_error,development_max_abs_acf_1_to_20,calibration_max_abs_acf_1_to_20,calibration_minus_development_max_abs_acf_1_to_20,formal_degradation_gate_authorized,simulation_envelope_pending,status
0,BUY,3330,1155,0.04942562989,0.05897666468,0.009551034791,0.02434741778,0.1587800054,0.1344325876,0.1462065372,0.1830735657,0.0368670285,False,True,DESCRIPTIVE_ONLY
1,POOLED,6858,2400,0.03087412143,0.05017457506,0.01930045363,0.02060082945,0.1476569353,0.1270561059,0.09510970638,0.1157581784,0.02064847198,False,True,DESCRIPTIVE_ONLY
2,SELL,3553,1250,0.03057856077,0.06917423779,0.03859567702,0.008954131368,0.1323214916,0.1233673602,0.06629993476,0.09717201363,0.03087207888,False,True,DESCRIPTIVE_ONLY


,check_name,passed
0,six_scope_partition_groups_present,True
1,distribution_diagnostics_complete,True
2,uniform_and_exponential_ks_reconcile,True
3,acf_lags_zero_through_fifty_complete,True
4,lag_zero_acf_equals_one,True
5,ljung_box_lags_complete,True
6,qq_coordinate_grid_complete,True
7,raw_p_values_report_only,True
8,simulation_envelopes_still_required,True
9,calibration_parameter_updates_zero,True


Empirical BUY, SELL, and pooled residual diagnostics computed for DEVELOPMENT and locked CALIBRATION. KS distances, Cramér-von Mises statistics, Q-Q coordinates, ACF values through lag 50, and Ljung-Box statistics at lags 10, 20, and 50 are available. Raw p-values and naive ACF bounds are report-only. Final adequacy decisions remain blocked pending coarsening-aware simulation envelopes. No filesystem writes were performed.


In [11]:
# ============================================================
# Side-mark calibration and exact-time multiplicity diagnostics
#
# Conditional side probabilities use the frozen pre-batch
# intensities:
#
#   P(BUY | event time, pre-batch history)
#       = lambda_buy / (lambda_buy + lambda_sell)
#
# Every member of an exact-time batch receives the same pre-batch
# side probability. Mixed and multiplicity batches remain intact.
#
# Binomial standard errors and z-scores are descriptive only.
# Exact-time members are not assumed independent, and formal
# adequacy remains pending coarsening-aware simulation envelopes.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "EMPIRICAL_RESIDUAL_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Empirical residual diagnostics have not been built.",
)

require(
    bool(
        globals().get(
            "EXACT_COMPENSATOR_PATH_BUILT",
            False,
        )
    ),
    "The canonical exact compensator path has not been built.",
)

require(
    not bool(
        globals().get(
            "SIDE_MARK_MULTIPLICITY_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Side-mark and multiplicity diagnostics are already built.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Diagnostic configuration
# ------------------------------------------------------------

SIDE_MARK_PROBABILITY_BIN_COUNT: Final[int] = 10
INTENSITY_QUANTILE_BIN_COUNT: Final[int] = 5
MARK_PROBABILITY_FLOOR: Final[float] = 1e-15

require(
    SIDE_MARK_PROBABILITY_BIN_COUNT >= 2,
    "At least two side-probability bins are required.",
)

require(
    INTENSITY_QUANTILE_BIN_COUNT >= 2,
    "At least two intensity bins are required.",
)


# ------------------------------------------------------------
# Canonical event-batch diagnostic table
# ------------------------------------------------------------

required_mark_columns: Final[set[str]] = {
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "lambda_buy_pre",
    "lambda_sell_pre",
}

missing_mark_columns = sorted(
    required_mark_columns
    - set(
        EVENT_COMPENSATOR_PATH.columns
    )
)

require(
    not missing_mark_columns,
    (
        "The event compensator path is missing side-mark columns: "
        f"{missing_mark_columns}"
    ),
)

SIDE_MARK_BATCH_DIAGNOSTICS = (
    EVENT_COMPENSATOR_PATH.loc[
        :,
        [
            "primary_event_batch_id",
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_partition",
            "event_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "mixed_side_batch_flag",
            "simultaneous_batch_required_flag",
            "lambda_buy_pre",
            "lambda_sell_pre",
        ],
    ]
    .copy()
    .sort_values(
        [
            "event_partition_order",
            "event_time_ns",
            "primary_event_batch_number",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

for integer_column in (
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_time_ns",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
):
    SIDE_MARK_BATCH_DIAGNOSTICS[
        integer_column
    ] = parse_exact_int64(
        SIDE_MARK_BATCH_DIAGNOSTICS[
            integer_column
        ],
        label=(
            "SIDE_MARK_BATCH_DIAGNOSTICS."
            f"{integer_column}"
        ),
    )

for boolean_column in (
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
):
    SIDE_MARK_BATCH_DIAGNOSTICS[
        boolean_column
    ] = normalize_boolean(
        SIDE_MARK_BATCH_DIAGNOSTICS[
            boolean_column
        ],
        label=(
            "SIDE_MARK_BATCH_DIAGNOSTICS."
            f"{boolean_column}"
        ),
    )

SIDE_MARK_BATCH_DIAGNOSTICS[
    "event_partition"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "event_partition"
    ]
    .astype(
        "string"
    )
    .str.strip()
    .str.upper()
)

require(
    len(
        SIDE_MARK_BATCH_DIAGNOSTICS
    )
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "The side-mark table has an incorrect batch count.",
)

require(
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ].eq(
        SIDE_MARK_BATCH_DIAGNOSTICS[
            "buy_event_count"
        ]
        + SIDE_MARK_BATCH_DIAGNOSTICS[
            "sell_event_count"
        ]
    ).all(),
    "Side-mark batches do not conserve BUY and SELL events.",
)

require(
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ].gt(
        0
    ).all(),
    "The side-mark table contains an empty event batch.",
)

for intensity_column in (
    "lambda_buy_pre",
    "lambda_sell_pre",
):
    intensity_values = (
        SIDE_MARK_BATCH_DIAGNOSTICS[
            intensity_column
        ].to_numpy(
            dtype=np.float64
        )
    )

    require(
        np.isfinite(
            intensity_values
        ).all(),
        f"{intensity_column} contains non-finite values.",
    )

    require(
        np.all(
            intensity_values
            > INTENSITY_FLOOR_PER_SECOND
        ),
        f"{intensity_column} contains nonpositive values.",
    )


# ------------------------------------------------------------
# Frozen conditional side probabilities and mark scores
# ------------------------------------------------------------

SIDE_MARK_BATCH_DIAGNOSTICS[
    "lambda_total_pre"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "lambda_buy_pre"
    ]
    + SIDE_MARK_BATCH_DIAGNOSTICS[
        "lambda_sell_pre"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "predicted_buy_probability"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "lambda_buy_pre"
    ]
    / SIDE_MARK_BATCH_DIAGNOSTICS[
        "lambda_total_pre"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "predicted_sell_probability"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "lambda_sell_pre"
    ]
    / SIDE_MARK_BATCH_DIAGNOSTICS[
        "lambda_total_pre"
    ]
)

require(
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_buy_probability"
    ].between(
        0.0,
        1.0,
        inclusive="neither",
    ).all(),
    "A predicted BUY probability is outside (0, 1).",
)

require(
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_sell_probability"
    ].between(
        0.0,
        1.0,
        inclusive="neither",
    ).all(),
    "A predicted SELL probability is outside (0, 1).",
)

require_array_close(
    (
        SIDE_MARK_BATCH_DIAGNOSTICS[
            "predicted_buy_probability"
        ]
        + SIDE_MARK_BATCH_DIAGNOSTICS[
            "predicted_sell_probability"
        ]
    ),
    np.ones(
        len(
            SIDE_MARK_BATCH_DIAGNOSTICS
        ),
        dtype=np.float64,
    ),
    label="Conditional BUY and SELL probabilities",
    rtol=1e-14,
    atol=1e-14,
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "observed_buy_fraction"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "buy_event_count"
    ]
    / SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "expected_buy_event_count"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ]
    * SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_buy_probability"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "expected_sell_event_count"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ]
    * SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_sell_probability"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "buy_count_calibration_error"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "buy_event_count"
    ]
    - SIDE_MARK_BATCH_DIAGNOSTICS[
        "expected_buy_event_count"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "binomial_variance_reference"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ]
    * SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_buy_probability"
    ]
    * SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_sell_probability"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "buy_count_pearson_residual_report_only"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "buy_count_calibration_error"
    ]
    / np.sqrt(
        SIDE_MARK_BATCH_DIAGNOSTICS[
            "binomial_variance_reference"
        ]
    )
)

clipped_buy_probability = np.clip(
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_buy_probability"
    ].to_numpy(
        dtype=np.float64
    ),
    MARK_PROBABILITY_FLOOR,
    1.0 - MARK_PROBABILITY_FLOOR,
)

clipped_sell_probability = (
    1.0
    - clipped_buy_probability
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "mark_log_score"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "buy_event_count"
    ].to_numpy(
        dtype=np.float64
    )
    * np.log(
        clipped_buy_probability
    )
    + SIDE_MARK_BATCH_DIAGNOSTICS[
        "sell_event_count"
    ].to_numpy(
        dtype=np.float64
    )
    * np.log(
        clipped_sell_probability
    )
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "mark_log_score_per_event"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "mark_log_score"
    ]
    / SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ]
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "mark_brier_score_total"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "buy_event_count"
    ]
    * (
        1.0
        - SIDE_MARK_BATCH_DIAGNOSTICS[
            "predicted_buy_probability"
        ]
    )
    ** 2
    + SIDE_MARK_BATCH_DIAGNOSTICS[
        "sell_event_count"
    ]
    * SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_buy_probability"
    ]
    ** 2
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "mark_brier_score_per_event"
] = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "mark_brier_score_total"
    ]
    / SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ]
)

require(
    np.isfinite(
        SIDE_MARK_BATCH_DIAGNOSTICS[
            [
                "mark_log_score",
                "mark_log_score_per_event",
                "mark_brier_score_total",
                "mark_brier_score_per_event",
                "buy_count_pearson_residual_report_only",
            ]
        ].to_numpy(
            dtype=np.float64
        )
    ).all(),
    "A side-mark score or residual is non-finite.",
)


# ------------------------------------------------------------
# Exact-time batch classes
# ------------------------------------------------------------

single_event_flag = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ].eq(
        1
    )
)

same_side_multiplicity_flag = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_event_count"
    ].gt(
        1
    )
    & ~SIDE_MARK_BATCH_DIAGNOSTICS[
        "mixed_side_batch_flag"
    ]
)

mixed_side_flag = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "mixed_side_batch_flag"
    ]
)

require(
    (
        single_event_flag.astype(
            np.int8
        )
        + same_side_multiplicity_flag.astype(
            np.int8
        )
        + mixed_side_flag.astype(
            np.int8
        )
    ).eq(
        1
    ).all(),
    "The exact-time batch classes are not mutually exhaustive.",
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "batch_class"
] = np.select(
    [
        single_event_flag,
        same_side_multiplicity_flag,
        mixed_side_flag,
    ],
    [
        "SINGLE_EVENT_BATCH",
        "SAME_SIDE_MULTIPLICITY_BATCH",
        "MIXED_SIDE_BATCH",
    ],
    default="INVALID_BATCH_CLASS",
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "batch_class"
] = SIDE_MARK_BATCH_DIAGNOSTICS[
    "batch_class"
].astype(
    "string"
)

require(
    not SIDE_MARK_BATCH_DIAGNOSTICS[
        "batch_class"
    ].eq(
        "INVALID_BATCH_CLASS"
    ).any(),
    "An exact-time batch received an invalid class.",
)


# ------------------------------------------------------------
# Fixed conditional-probability bins
# ------------------------------------------------------------

probability_values = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "predicted_buy_probability"
    ].to_numpy(
        dtype=np.float64
    )
)

probability_bin_index_zero_based = np.minimum(
    np.floor(
        probability_values
        * SIDE_MARK_PROBABILITY_BIN_COUNT
    ).astype(
        np.int64
    ),
    SIDE_MARK_PROBABILITY_BIN_COUNT - 1,
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "buy_probability_bin_index"
] = (
    probability_bin_index_zero_based
    + 1
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "buy_probability_bin_lower"
] = (
    probability_bin_index_zero_based
    / SIDE_MARK_PROBABILITY_BIN_COUNT
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "buy_probability_bin_upper"
] = (
    (
        probability_bin_index_zero_based
        + 1
    )
    / SIDE_MARK_PROBABILITY_BIN_COUNT
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "buy_probability_bin_label"
] = pd.Series(
    [
        (
            f"[{lower:.1f}, {upper:.1f}"
            + (
                "]"
                if upper == 1.0
                else ")"
            )
        )
        for lower, upper in zip(
            SIDE_MARK_BATCH_DIAGNOSTICS[
                "buy_probability_bin_lower"
            ],
            SIDE_MARK_BATCH_DIAGNOSTICS[
                "buy_probability_bin_upper"
            ],
            strict=True,
        )
    ],
    dtype="string",
)


# ------------------------------------------------------------
# DEVELOPMENT-frozen total-intensity quantile bins
# ------------------------------------------------------------

development_total_intensity = (
    SIDE_MARK_BATCH_DIAGNOSTICS.loc[
        SIDE_MARK_BATCH_DIAGNOSTICS[
            "event_partition"
        ].eq(
            "DEVELOPMENT"
        ),
        "lambda_total_pre",
    ]
    .to_numpy(
        dtype=np.float64
    )
)

require(
    len(
        development_total_intensity
    )
    == EXPECTED_DEVELOPMENT_BATCH_ROWS,
    "The DEVELOPMENT intensity sample has the wrong size.",
)

raw_intensity_edges = np.quantile(
    development_total_intensity,
    np.linspace(
        0.0,
        1.0,
        INTENSITY_QUANTILE_BIN_COUNT + 1,
        dtype=np.float64,
    ),
    method="linear",
)

unique_intensity_edges = np.unique(
    raw_intensity_edges
)

require(
    len(
        unique_intensity_edges
    )
    >= 3,
    (
        "DEVELOPMENT total intensity does not support at least "
        "two distinct quantile bins."
    ),
)

intensity_interior_edges = (
    unique_intensity_edges[
        1:-1
    ]
)

intensity_bin_edges = np.concatenate(
    (
        np.asarray(
            [-np.inf],
            dtype=np.float64,
        ),
        intensity_interior_edges,
        np.asarray(
            [np.inf],
            dtype=np.float64,
        ),
    )
)

actual_intensity_bin_count: Final[int] = (
    len(
        intensity_bin_edges
    )
    - 1
)

all_total_intensity = (
    SIDE_MARK_BATCH_DIAGNOSTICS[
        "lambda_total_pre"
    ].to_numpy(
        dtype=np.float64
    )
)

intensity_bin_index_zero_based = np.searchsorted(
    intensity_interior_edges,
    all_total_intensity,
    side="right",
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "development_frozen_intensity_bin_index"
] = (
    intensity_bin_index_zero_based
    + 1
)

SIDE_MARK_BATCH_DIAGNOSTICS[
    "development_frozen_intensity_bin_label"
] = pd.Series(
    [
        (
            f"Q{bin_index}_OF_"
            f"{actual_intensity_bin_count}"
        )
        for bin_index in (
            intensity_bin_index_zero_based
            + 1
        )
    ],
    dtype="string",
)


# ------------------------------------------------------------
# Summary helper
# ------------------------------------------------------------

def summarize_side_mark_group(
    group: pd.DataFrame,
) -> dict[str, Any]:
    """Return event-weighted mark-calibration metrics."""
    batch_count = len(
        group
    )

    event_count = int(
        group[
            "batch_event_count"
        ].sum()
    )

    buy_event_count = int(
        group[
            "buy_event_count"
        ].sum()
    )

    sell_event_count = int(
        group[
            "sell_event_count"
        ].sum()
    )

    expected_buy_event_count = float(
        group[
            "expected_buy_event_count"
        ].sum()
    )

    binomial_variance_reference = float(
        group[
            "binomial_variance_reference"
        ].sum()
    )

    require(
        batch_count > 0,
        "A side-mark summary group is empty.",
    )

    require(
        event_count
        == buy_event_count
        + sell_event_count,
        "A side-mark summary group does not conserve events.",
    )

    require(
        event_count > 0,
        "A side-mark summary group contains no events.",
    )

    calibration_error = (
        buy_event_count
        - expected_buy_event_count
    )

    naive_standard_error = (
        math.sqrt(
            binomial_variance_reference
        )
        / event_count
    )

    calibration_z_report_only = (
        calibration_error
        / math.sqrt(
            binomial_variance_reference
        )
        if binomial_variance_reference
        > 0.0
        else float(
            "nan"
        )
    )

    return {
        "batch_count": batch_count,
        "event_count": event_count,
        "buy_event_count": buy_event_count,
        "sell_event_count": sell_event_count,
        "simultaneous_batch_count": int(
            group[
                "simultaneous_batch_required_flag"
            ].sum()
        ),
        "mixed_side_batch_count": int(
            group[
                "mixed_side_batch_flag"
            ].sum()
        ),
        "observed_buy_event_rate": (
            buy_event_count
            / event_count
        ),
        "predicted_buy_event_rate": (
            expected_buy_event_count
            / event_count
        ),
        "buy_rate_calibration_gap": (
            calibration_error
            / event_count
        ),
        "naive_buy_rate_standard_error_report_only": (
            naive_standard_error
        ),
        "naive_calibration_z_report_only": (
            calibration_z_report_only
        ),
        "mark_log_score": float(
            group[
                "mark_log_score"
            ].sum()
        ),
        "mark_log_score_per_event": float(
            group[
                "mark_log_score"
            ].sum()
            / event_count
        ),
        "mark_brier_score_per_event": float(
            group[
                "mark_brier_score_total"
            ].sum()
            / event_count
        ),
        "mean_absolute_batch_calibration_error": float(
            group[
                "buy_count_calibration_error"
            ]
            .abs()
            .mean()
        ),
        "maximum_absolute_pearson_residual_report_only": float(
            group[
                "buy_count_pearson_residual_report_only"
            ]
            .abs()
            .max()
        ),
        "formal_binomial_inference_authorized": False,
        "simulation_envelope_pending": True,
        "status": "COMPUTED_REPORT_ONLY",
    }


# ------------------------------------------------------------
# Partition-level side-mark summary
# ------------------------------------------------------------

partition_summary_records: list[
    dict[str, Any]
] = []

for (
    partition_order,
    partition_name,
), group in (
    SIDE_MARK_BATCH_DIAGNOSTICS.groupby(
        [
            "event_partition_order",
            "event_partition",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
):
    partition_summary_records.append(
        {
            "partition_order": int(
                partition_order
            ),
            "event_partition": str(
                partition_name
            ),
            **summarize_side_mark_group(
                group
            ),
        }
    )

SIDE_MARK_PARTITION_SUMMARY = (
    pd.DataFrame.from_records(
        partition_summary_records
    )
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Batch-class side-mark summary
# ------------------------------------------------------------

batch_class_summary_records: list[
    dict[str, Any]
] = []

for (
    partition_order,
    partition_name,
    batch_class,
), group in (
    SIDE_MARK_BATCH_DIAGNOSTICS.groupby(
        [
            "event_partition_order",
            "event_partition",
            "batch_class",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
):
    batch_class_summary_records.append(
        {
            "partition_order": int(
                partition_order
            ),
            "event_partition": str(
                partition_name
            ),
            "batch_class": str(
                batch_class
            ),
            **summarize_side_mark_group(
                group
            ),
        }
    )

SIDE_MARK_BATCH_CLASS_SUMMARY = (
    pd.DataFrame.from_records(
        batch_class_summary_records
    )
    .sort_values(
        [
            "partition_order",
            "batch_class",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Conditional side-probability reliability table
# ------------------------------------------------------------

reliability_records: list[
    dict[str, Any]
] = []

for (
    partition_order,
    partition_name,
    bin_index,
    bin_label,
), group in (
    SIDE_MARK_BATCH_DIAGNOSTICS.groupby(
        [
            "event_partition_order",
            "event_partition",
            "buy_probability_bin_index",
            "buy_probability_bin_label",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
):
    summary = summarize_side_mark_group(
        group
    )

    reliability_records.append(
        {
            "partition_order": int(
                partition_order
            ),
            "event_partition": str(
                partition_name
            ),
            "buy_probability_bin_index": int(
                bin_index
            ),
            "buy_probability_bin_label": str(
                bin_label
            ),
            "bin_lower": float(
                group[
                    "buy_probability_bin_lower"
                ].iloc[0]
            ),
            "bin_upper": float(
                group[
                    "buy_probability_bin_upper"
                ].iloc[0]
            ),
            "minimum_predicted_buy_probability": float(
                group[
                    "predicted_buy_probability"
                ].min()
            ),
            "maximum_predicted_buy_probability": float(
                group[
                    "predicted_buy_probability"
                ].max()
            ),
            **summary,
        }
    )

SIDE_MARK_RELIABILITY_TABLE = (
    pd.DataFrame.from_records(
        reliability_records
    )
    .sort_values(
        [
            "partition_order",
            "buy_probability_bin_index",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Overall exact-time multiplicity distribution
# ------------------------------------------------------------

batch_multiplicity_records: list[
    dict[str, Any]
] = []

for (
    partition_order,
    partition_name,
), partition_group in (
    SIDE_MARK_BATCH_DIAGNOSTICS.groupby(
        [
            "event_partition_order",
            "event_partition",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
):
    partition_batch_count = len(
        partition_group
    )

    partition_event_count = int(
        partition_group[
            "batch_event_count"
        ].sum()
    )

    for multiplicity, multiplicity_group in (
        partition_group.groupby(
            "batch_event_count",
            observed=True,
            sort=True,
            dropna=False,
        )
    ):
        multiplicity_value = int(
            multiplicity
        )

        observed_batch_count = len(
            multiplicity_group
        )

        observed_event_count = int(
            multiplicity_group[
                "batch_event_count"
            ].sum()
        )

        batch_multiplicity_records.append(
            {
                "partition_order": int(
                    partition_order
                ),
                "event_partition": str(
                    partition_name
                ),
                "batch_multiplicity": (
                    multiplicity_value
                ),
                "observed_batch_count": (
                    observed_batch_count
                ),
                "observed_event_count": (
                    observed_event_count
                ),
                "batch_probability": (
                    observed_batch_count
                    / partition_batch_count
                ),
                "event_share": (
                    observed_event_count
                    / partition_event_count
                ),
                "buy_event_count": int(
                    multiplicity_group[
                        "buy_event_count"
                    ].sum()
                ),
                "sell_event_count": int(
                    multiplicity_group[
                        "sell_event_count"
                    ].sum()
                ),
                "mixed_side_batch_count": int(
                    multiplicity_group[
                        "mixed_side_batch_flag"
                    ].sum()
                ),
                "simulation_envelope_pending": True,
                "status": "OBSERVED_DISTRIBUTION",
            }
        )

BATCH_MULTIPLICITY_DISTRIBUTION = (
    pd.DataFrame.from_records(
        batch_multiplicity_records
    )
    .sort_values(
        [
            "partition_order",
            "batch_multiplicity",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Side-specific exact-time multiplicity distributions
# ------------------------------------------------------------

side_multiplicity_records: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    partition_order = (
        PARTITION_ORDER_MAP[
            partition_name
        ]
    )

    partition_group = (
        SIDE_MARK_BATCH_DIAGNOSTICS.loc[
            SIDE_MARK_BATCH_DIAGNOSTICS[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    for (
        side_name,
        side_count_column,
    ) in (
        (
            "BUY",
            "buy_event_count",
        ),
        (
            "SELL",
            "sell_event_count",
        ),
    ):
        relevant_group = (
            partition_group.loc[
                partition_group[
                    side_count_column
                ].gt(
                    0
                )
            ]
        )

        relevant_batch_count = len(
            relevant_group
        )

        relevant_event_count = int(
            relevant_group[
                side_count_column
            ].sum()
        )

        require(
            relevant_batch_count > 0,
            (
                f"{partition_name} contains no "
                f"{side_name} batches."
            ),
        )

        for multiplicity, multiplicity_group in (
            relevant_group.groupby(
                side_count_column,
                observed=True,
                sort=True,
                dropna=False,
            )
        ):
            multiplicity_value = int(
                multiplicity
            )

            observed_batch_count = len(
                multiplicity_group
            )

            observed_side_event_count = int(
                multiplicity_group[
                    side_count_column
                ].sum()
            )

            side_multiplicity_records.append(
                {
                    "partition_order": int(
                        partition_order
                    ),
                    "event_partition": (
                        partition_name
                    ),
                    "event_side": side_name,
                    "side_multiplicity": (
                        multiplicity_value
                    ),
                    "observed_relevant_batch_count": (
                        observed_batch_count
                    ),
                    "observed_side_event_count": (
                        observed_side_event_count
                    ),
                    "relevant_batch_probability": (
                        observed_batch_count
                        / relevant_batch_count
                    ),
                    "side_event_share": (
                        observed_side_event_count
                        / relevant_event_count
                    ),
                    "mixed_side_batch_count": int(
                        multiplicity_group[
                            "mixed_side_batch_flag"
                        ].sum()
                    ),
                    "simulation_envelope_pending": True,
                    "status": "OBSERVED_DISTRIBUTION",
                }
            )

SIDE_MULTIPLICITY_DISTRIBUTION = (
    pd.DataFrame.from_records(
        side_multiplicity_records
    )
    .sort_values(
        [
            "partition_order",
            "event_side",
            "side_multiplicity",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Multiplicity conditional on frozen predicted total intensity
# ------------------------------------------------------------

intensity_multiplicity_records: list[
    dict[str, Any]
] = []

for (
    partition_order,
    partition_name,
    intensity_bin_index,
    intensity_bin_label,
), group in (
    SIDE_MARK_BATCH_DIAGNOSTICS.groupby(
        [
            "event_partition_order",
            "event_partition",
            "development_frozen_intensity_bin_index",
            "development_frozen_intensity_bin_label",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
):
    batch_count = len(
        group
    )

    event_count = int(
        group[
            "batch_event_count"
        ].sum()
    )

    intensity_multiplicity_records.append(
        {
            "partition_order": int(
                partition_order
            ),
            "event_partition": str(
                partition_name
            ),
            "development_frozen_intensity_bin_index": int(
                intensity_bin_index
            ),
            "development_frozen_intensity_bin_label": str(
                intensity_bin_label
            ),
            "batch_count": batch_count,
            "event_count": event_count,
            "minimum_total_intensity": float(
                group[
                    "lambda_total_pre"
                ].min()
            ),
            "mean_total_intensity": float(
                group[
                    "lambda_total_pre"
                ].mean()
            ),
            "maximum_total_intensity": float(
                group[
                    "lambda_total_pre"
                ].max()
            ),
            "mean_batch_multiplicity": float(
                group[
                    "batch_event_count"
                ].mean()
            ),
            "batch_multiplicity_variance": float(
                group[
                    "batch_event_count"
                ].var(
                    ddof=1
                )
            )
            if batch_count > 1
            else 0.0,
            "multiple_event_batch_rate": float(
                group[
                    "batch_event_count"
                ].gt(
                    1
                ).mean()
            ),
            "mixed_side_batch_rate": float(
                group[
                    "mixed_side_batch_flag"
                ].mean()
            ),
            "maximum_batch_multiplicity": int(
                group[
                    "batch_event_count"
                ].max()
            ),
            "mean_buy_multiplicity": float(
                group[
                    "buy_event_count"
                ].mean()
            ),
            "mean_sell_multiplicity": float(
                group[
                    "sell_event_count"
                ].mean()
            ),
            "simulation_envelope_pending": True,
            "status": "COMPUTED_REPORT_ONLY",
        }
    )

MULTIPLICITY_BY_FROZEN_INTENSITY_BIN = (
    pd.DataFrame.from_records(
        intensity_multiplicity_records
    )
    .sort_values(
        [
            "partition_order",
            "development_frozen_intensity_bin_index",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Scope ledger
# ------------------------------------------------------------

MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER = pd.DataFrame(
    [
        {
            "diagnostic": (
                "POSITIVE_EXACT_TIME_BATCH_MULTIPLICITY"
            ),
            "computed": True,
            "primary_table": (
                "BATCH_MULTIPLICITY_DISTRIBUTION"
            ),
            "zero_count_cells_available": False,
            "requires_common_1ms_grid": False,
            "requires_simulation_envelope": True,
            "status": "COMPUTED",
        },
        {
            "diagnostic": (
                "SIDE_SPECIFIC_EXACT_TIME_MULTIPLICITY"
            ),
            "computed": True,
            "primary_table": (
                "SIDE_MULTIPLICITY_DISTRIBUTION"
            ),
            "zero_count_cells_available": False,
            "requires_common_1ms_grid": False,
            "requires_simulation_envelope": True,
            "status": "COMPUTED",
        },
        {
            "diagnostic": (
                "ZERO_ONE_MULTIPLE_COUNT_CELL_PROBABILITIES"
            ),
            "computed": False,
            "primary_table": (
                "PENDING_COMMON_1MS_COUNT_GRID"
            ),
            "zero_count_cells_available": False,
            "requires_common_1ms_grid": True,
            "requires_simulation_envelope": True,
            "status": "DEFERRED_TO_COMMON_GRID",
        },
        {
            "diagnostic": (
                "MULTIPLICITY_CONDITIONAL_ON_PREDICTED_INTENSITY"
            ),
            "computed": True,
            "primary_table": (
                "MULTIPLICITY_BY_FROZEN_INTENSITY_BIN"
            ),
            "zero_count_cells_available": False,
            "requires_common_1ms_grid": False,
            "requires_simulation_envelope": True,
            "status": "COMPUTED",
        },
    ]
)


# ------------------------------------------------------------
# Reconciliation checks
# ------------------------------------------------------------

for partition_name in ANALYTICAL_PARTITIONS:
    partition_batches = (
        SIDE_MARK_BATCH_DIAGNOSTICS.loc[
            SIDE_MARK_BATCH_DIAGNOSTICS[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    partition_reliability = (
        SIDE_MARK_RELIABILITY_TABLE.loc[
            SIDE_MARK_RELIABILITY_TABLE[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    partition_multiplicity = (
        BATCH_MULTIPLICITY_DISTRIBUTION.loc[
            BATCH_MULTIPLICITY_DISTRIBUTION[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    require(
        int(
            partition_reliability[
                "event_count"
            ].sum()
        )
        == int(
            partition_batches[
                "batch_event_count"
            ].sum()
        ),
        (
            f"{partition_name} reliability bins do not "
            "conserve events."
        ),
    )

    require(
        int(
            partition_reliability[
                "batch_count"
            ].sum()
        )
        == len(
            partition_batches
        ),
        (
            f"{partition_name} reliability bins do not "
            "conserve batches."
        ),
    )

    require_close(
        partition_multiplicity[
            "batch_probability"
        ].sum(),
        1.0,
        label=(
            f"{partition_name} multiplicity batch probabilities"
        ),
        relative_tolerance=1e-12,
        absolute_tolerance=1e-12,
    )

    require_close(
        partition_multiplicity[
            "event_share"
        ].sum(),
        1.0,
        label=(
            f"{partition_name} multiplicity event shares"
        ),
        relative_tolerance=1e-12,
        absolute_tolerance=1e-12,
    )

    for (
        side_name,
        side_count_column,
    ) in (
        (
            "BUY",
            "buy_event_count",
        ),
        (
            "SELL",
            "sell_event_count",
        ),
    ):
        side_distribution = (
            SIDE_MULTIPLICITY_DISTRIBUTION.loc[
                SIDE_MULTIPLICITY_DISTRIBUTION[
                    "event_partition"
                ].eq(
                    partition_name
                )
                & SIDE_MULTIPLICITY_DISTRIBUTION[
                    "event_side"
                ].eq(
                    side_name
                )
            ]
        )

        require_close(
            side_distribution[
                "relevant_batch_probability"
            ].sum(),
            1.0,
            label=(
                f"{partition_name} {side_name} "
                "relevant-batch probabilities"
            ),
            relative_tolerance=1e-12,
            absolute_tolerance=1e-12,
        )

        require_close(
            side_distribution[
                "side_event_share"
            ].sum(),
            1.0,
            label=(
                f"{partition_name} {side_name} "
                "event shares"
            ),
            relative_tolerance=1e-12,
            absolute_tolerance=1e-12,
        )

        require(
            int(
                side_distribution[
                    "observed_side_event_count"
                ].sum()
            )
            == int(
                partition_batches[
                    side_count_column
                ].sum()
            ),
            (
                f"{partition_name} {side_name} multiplicity "
                "distribution does not conserve events."
            ),
        )


# ------------------------------------------------------------
# Verification matrix
# ------------------------------------------------------------

SIDE_MARK_MULTIPLICITY_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "side_mark_batch_count_reconciles"
            ),
            "passed": (
                len(
                    SIDE_MARK_BATCH_DIAGNOSTICS
                )
                == EXPECTED_ANALYTICAL_BATCH_ROWS
            ),
        },
        {
            "check_name": (
                "side_counts_conserve_batch_events"
            ),
            "passed": bool(
                SIDE_MARK_BATCH_DIAGNOSTICS[
                    "batch_event_count"
                ].eq(
                    SIDE_MARK_BATCH_DIAGNOSTICS[
                        "buy_event_count"
                    ]
                    + SIDE_MARK_BATCH_DIAGNOSTICS[
                        "sell_event_count"
                    ]
                ).all()
            ),
        },
        {
            "check_name": (
                "side_probabilities_strictly_inside_unit_interval"
            ),
            "passed": bool(
                SIDE_MARK_BATCH_DIAGNOSTICS[
                    "predicted_buy_probability"
                ].between(
                    0.0,
                    1.0,
                    inclusive="neither",
                ).all()
                and SIDE_MARK_BATCH_DIAGNOSTICS[
                    "predicted_sell_probability"
                ].between(
                    0.0,
                    1.0,
                    inclusive="neither",
                ).all()
            ),
        },
        {
            "check_name": (
                "side_probabilities_sum_to_one"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "mark_scores_are_finite"
            ),
            "passed": bool(
                np.isfinite(
                    SIDE_MARK_BATCH_DIAGNOSTICS[
                        [
                            "mark_log_score",
                            "mark_brier_score_total",
                        ]
                    ].to_numpy(
                        dtype=np.float64
                    )
                ).all()
            ),
        },
        {
            "check_name": (
                "batch_classes_mutually_exhaustive"
            ),
            "passed": bool(
                not SIDE_MARK_BATCH_DIAGNOSTICS[
                    "batch_class"
                ].eq(
                    "INVALID_BATCH_CLASS"
                ).any()
            ),
        },
        {
            "check_name": (
                "reliability_bins_conserve_events"
            ),
            "passed": bool(
                SIDE_MARK_RELIABILITY_TABLE[
                    "event_count"
                ].sum()
                == SIDE_MARK_BATCH_DIAGNOSTICS[
                    "batch_event_count"
                ].sum()
            ),
        },
        {
            "check_name": (
                "batch_multiplicity_probabilities_reconcile"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "side_multiplicity_probabilities_reconcile"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "intensity_bins_frozen_from_development"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "formal_binomial_inference_disabled"
            ),
            "passed": bool(
                not SIDE_MARK_PARTITION_SUMMARY[
                    "formal_binomial_inference_authorized"
                ].any()
            ),
        },
        {
            "check_name": (
                "simulation_envelopes_still_required"
            ),
            "passed": bool(
                SIDE_MARK_PARTITION_SUMMARY[
                    "simulation_envelope_pending"
                ].all()
                and BATCH_MULTIPLICITY_DISTRIBUTION[
                    "simulation_envelope_pending"
                ].all()
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    SIDE_MARK_MULTIPLICITY_VERIFICATION[
        "passed"
    ].all(),
    (
        "At least one side-mark or multiplicity "
        "verification failed."
    ),
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

SIDE_MARK_MULTIPLICITY_DIAGNOSTICS_BUILT = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    SIDE_MARK_PARTITION_SUMMARY
)

display(
    SIDE_MARK_BATCH_CLASS_SUMMARY
)

display(
    SIDE_MARK_RELIABILITY_TABLE
)

display(
    BATCH_MULTIPLICITY_DISTRIBUTION
)

display(
    SIDE_MULTIPLICITY_DISTRIBUTION
)

display(
    MULTIPLICITY_BY_FROZEN_INTENSITY_BIN
)

display(
    MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER
)

display(
    SIDE_MARK_MULTIPLICITY_VERIFICATION
)

print(
    "Frozen conditional BUY/SELL mark probabilities, event-weighted "
    "mark scores, reliability bins, exact-time batch classes, and "
    "observed multiplicity distributions were constructed for "
    "DEVELOPMENT and locked CALIBRATION. DEVELOPMENT total-intensity "
    "quantiles were frozen before applying the multiplicity analysis "
    "to CALIBRATION. Mixed and same-side multiplicity batches remain "
    "intact. Zero-count cell probabilities are correctly deferred to "
    "the common 1 ms grid. Formal adequacy remains pending "
    "coarsening-aware simulation envelopes. No filesystem writes "
    "were performed."
)

,partition_order,event_partition,batch_count,event_count,buy_event_count,sell_event_count,simultaneous_batch_count,mixed_side_batch_count,observed_buy_event_rate,predicted_buy_event_rate,buy_rate_calibration_gap,naive_buy_rate_standard_error_report_only,naive_calibration_z_report_only,mark_log_score,mark_log_score_per_event,mark_brier_score_per_event,mean_absolute_batch_calibration_error,maximum_absolute_pearson_residual_report_only,formal_binomial_inference_authorized,simulation_envelope_pending,status
0,1,DEVELOPMENT,6859,7004,3414,3590,111,26,0.487435751,0.4918473438,-0.004411592843,0.00564962654,-0.7808645071,"-4,562.678094",-0.6514389055,0.2315567026,0.462903981,4.709540167,False,True,COMPUTED_REPORT_ONLY
1,2,CALIBRATION,2400,2493,1230,1263,68,5,0.4933814681,0.5067221189,-0.01334065084,0.009373500472,-1.423230401,"-1,570.024015",-0.6297729701,0.2226033779,0.457903874,6.73664393,False,True,COMPUTED_REPORT_ONLY


,partition_order,event_partition,batch_class,batch_count,event_count,buy_event_count,sell_event_count,simultaneous_batch_count,mixed_side_batch_count,observed_buy_event_rate,predicted_buy_event_rate,buy_rate_calibration_gap,naive_buy_rate_standard_error_report_only,naive_calibration_z_report_only,mark_log_score,mark_log_score_per_event,mark_brier_score_per_event,mean_absolute_batch_calibration_error,maximum_absolute_pearson_residual_report_only,formal_binomial_inference_authorized,simulation_envelope_pending,status
0,1,DEVELOPMENT,MIXED_SIDE_BATCH,26,60,34,26,26,26,0.5666666667,0.625147796,-0.05848112938,0.04700371582,-1.244180983,-56.95662265,-0.9492770442,0.3070483863,0.5318525172,4.509952876,False,True,COMPUTED_REPORT_ONLY
1,1,DEVELOPMENT,SAME_SIDE_MULTIPLICITY_BATCH,85,196,134,62,85,0,0.6836734694,0.6237736906,0.05989977881,0.02407261706,2.488295255,-47.61477727,-0.2429325371,0.07065445773,0.4248234803,2.703295512,False,True,COMPUTED_REPORT_ONLY
2,1,DEVELOPMENT,SINGLE_EVENT_BATCH,6748,6748,3246,3502,0,0,0.4810314167,0.4868302142,-0.005798797445,0.005807102501,-0.9985698451,"-4,458.106694",-0.6606560009,0.2355589757,0.4631179971,4.709540167,False,True,COMPUTED_REPORT_ONLY
3,2,CALIBRATION,MIXED_SIDE_BATCH,5,11,6,5,5,5,0.5454545455,0.8045087488,-0.2590542033,0.1050257264,-2.466578544,-11.53156431,-1.048324029,0.3369055067,0.5979808494,2.567497433,False,True,COMPUTED_REPORT_ONLY
4,2,CALIBRATION,SAME_SIDE_MULTIPLICITY_BATCH,63,150,126,24,63,0,0.84,0.7149379992,0.1250620008,0.02983917685,4.191201433,-44.60758586,-0.2973839057,0.09271761234,0.5387478181,2.146675316,False,True,COMPUTED_REPORT_ONLY
5,2,CALIBRATION,SINGLE_EVENT_BATCH,2332,2332,1098,1234,0,0,0.4708404803,0.4919245053,-0.02108402505,0.009822627481,-2.146475074,"-1,513.884864",-0.6491787583,0.2304187902,0.4554195029,6.73664393,False,True,COMPUTED_REPORT_ONLY


,partition_order,event_partition,buy_probability_bin_index,buy_probability_bin_label,bin_lower,bin_upper,minimum_predicted_buy_probability,maximum_predicted_buy_probability,batch_count,event_count,buy_event_count,sell_event_count,simultaneous_batch_count,mixed_side_batch_count,observed_buy_event_rate,predicted_buy_event_rate,buy_rate_calibration_gap,naive_buy_rate_standard_error_report_only,naive_calibration_z_report_only,mark_log_score,mark_log_score_per_event,mark_brier_score_per_event,mean_absolute_batch_calibration_error,maximum_absolute_pearson_residual_report_only,formal_binomial_inference_authorized,simulation_envelope_pending,status
0,1,DEVELOPMENT,1,"[0.0, 0.1)",0,0.1,0.04172709032,0.09967682479,72,83,6,77,8,2,0.07228915663,0.06889958576,0.003389570865,0.02775162465,0.1221395471,-22.84123851,-0.2751956447,0.06890095036,0.1502095672,4.588915109,False,True,COMPUTED_REPORT_ONLY
1,1,DEVELOPMENT,2,"[0.1, 0.2)",0.1,0.2,0.1003623128,0.1992641615,209,227,33,194,13,2,0.1453744493,0.1526855165,-0.007311067188,0.02380627718,-0.3071066985,-91.82148996,-0.4044999558,0.1217410229,0.2691630581,2.894934231,False,True,COMPUTED_REPORT_ONLY
2,1,DEVELOPMENT,3,"[0.2, 0.3)",0.2,0.3,0.2006145061,0.2998115073,187,196,66,130,7,2,0.3367346939,0.2468834212,0.08985127268,0.03073048082,2.923848579,-128.063724,-0.653386347,0.2291273071,0.4281549617,2.703295512,False,True,COMPUTED_REPORT_ONLY
3,1,DEVELOPMENT,4,"[0.3, 0.4)",0.3,0.4,0.3005088524,0.3984587858,188,193,75,118,5,2,0.3886010363,0.3511458873,0.03745514892,0.03429799082,1.092050818,-129.8069464,-0.6725748516,0.2395796819,0.4711286573,2.001592099,False,True,COMPUTED_REPORT_ONLY
4,1,DEVELOPMENT,5,"[0.4, 0.5)",0.4,0.5,0.4005855935,0.499753051,5131,5149,2416,2733,18,7,0.4692173238,0.4638337513,0.00538357242,0.006948663996,0.7747636702,"-3,560.753087",-0.6915426465,0.2491962306,0.4983013951,1.586335304,False,True,COMPUTED_REPORT_ONLY
5,1,DEVELOPMENT,6,"[0.5, 0.6)",0.5,0.6,0.5001358898,0.5998708134,233,237,121,116,3,0,0.5105485232,0.5371084943,-0.02655997105,0.03233680701,-0.821354163,-164.9756485,-0.6960997827,0.2514538558,0.5078487158,1.692691348,False,True,COMPUTED_REPORT_ONLY
6,1,DEVELOPMENT,7,"[0.6, 0.7)",0.6,0.7,0.6003916653,0.6988053214,138,139,73,66,1,0,0.5251798561,0.652751216,-0.1275713599,0.04030317319,-3.165293197,-100.9577267,-0.7263145807,0.2655648085,0.4949093695,1.734895032,False,True,COMPUTED_REPORT_ONLY
7,1,DEVELOPMENT,8,"[0.7, 0.8)",0.7,0.8,0.7003425239,0.7991021257,147,150,89,61,3,1,0.5933333333,0.74477174,-0.1514384066,0.0355200385,-4.263464034,-109.8458581,-0.7323057204,0.2647716501,0.4598392075,1.985646522,False,True,COMPUTED_REPORT_ONLY
8,1,DEVELOPMENT,9,"[0.8, 0.9)",0.8,0.9,0.8014818898,0.899954047,273,285,211,74,10,3,0.7403508772,0.859913732,-0.1195628548,0.02049475336,-5.833827454,-176.1049333,-0.6179120465,0.205144162,0.3363990641,2.984334108,False,True,COMPUTED_REPORT_ONLY
9,1,DEVELOPMENT,10,"[0.9, 1.0]",0.9,1,0.9004008145,0.985151386,281,345,324,21,43,7,0.9391304348,0.9427881716,-0.003657736794,0.01244743585,-0.293854641,-77.50744175,-0.2246592515,0.05657978407,0.1299183722,4.709540167,False,True,COMPUTED_REPORT_ONLY


,partition_order,event_partition,batch_multiplicity,observed_batch_count,observed_event_count,batch_probability,event_share,buy_event_count,sell_event_count,mixed_side_batch_count,simulation_envelope_pending,status
0,1,DEVELOPMENT,1,6748,6748,0.9838168829,0.9634494575,3246,3502,0,True,OBSERVED_DISTRIBUTION
1,1,DEVELOPMENT,2,91,182,0.01326724012,0.02598515134,119,63,21,True,OBSERVED_DISTRIBUTION
2,1,DEVELOPMENT,3,10,30,0.001457938475,0.004283266705,25,5,2,True,OBSERVED_DISTRIBUTION
3,1,DEVELOPMENT,4,7,28,0.001020556932,0.003997715591,13,15,3,True,OBSERVED_DISTRIBUTION
4,1,DEVELOPMENT,5,2,10,0.000291587695,0.001427755568,5,5,0,True,OBSERVED_DISTRIBUTION
5,1,DEVELOPMENT,6,1,6,0.0001457938475,0.0008566533409,6,0,0,True,OBSERVED_DISTRIBUTION
6,2,CALIBRATION,1,2332,2332,0.9716666667,0.9354191737,1098,1234,0,True,OBSERVED_DISTRIBUTION
7,2,CALIBRATION,2,49,98,0.02041666667,0.03931006819,76,22,4,True,OBSERVED_DISTRIBUTION
8,2,CALIBRATION,3,13,39,0.005416666667,0.01564380265,32,7,1,True,OBSERVED_DISTRIBUTION
9,2,CALIBRATION,4,6,24,0.0025,0.009626955475,24,0,0,True,OBSERVED_DISTRIBUTION


,partition_order,event_partition,event_side,side_multiplicity,observed_relevant_batch_count,observed_side_event_count,relevant_batch_probability,side_event_share,mixed_side_batch_count,simulation_envelope_pending,status
0,1,DEVELOPMENT,BUY,1,3267,3267,0.9807865506,0.9569420035,21,True,OBSERVED_DISTRIBUTION
1,1,DEVELOPMENT,BUY,2,51,102,0.0153107175,0.02987697715,2,True,OBSERVED_DISTRIBUTION
2,1,DEVELOPMENT,BUY,3,10,30,0.003002101471,0.008787346221,3,True,OBSERVED_DISTRIBUTION
3,1,DEVELOPMENT,BUY,4,1,4,0.0003002101471,0.001171646163,0,True,OBSERVED_DISTRIBUTION
4,1,DEVELOPMENT,BUY,5,1,5,0.0003002101471,0.001464557704,0,True,OBSERVED_DISTRIBUTION
5,1,DEVELOPMENT,BUY,6,1,6,0.0003002101471,0.001757469244,0,True,OBSERVED_DISTRIBUTION
6,1,DEVELOPMENT,SELL,1,3528,3528,0.9926842994,0.982729805,26,True,OBSERVED_DISTRIBUTION
7,1,DEVELOPMENT,SELL,2,21,42,0.005908835115,0.01169916435,0,True,OBSERVED_DISTRIBUTION
8,1,DEVELOPMENT,SELL,3,1,3,0.0002813731007,0.0008356545961,0,True,OBSERVED_DISTRIBUTION
9,1,DEVELOPMENT,SELL,4,3,12,0.0008441193022,0.003342618384,0,True,OBSERVED_DISTRIBUTION


,partition_order,event_partition,development_frozen_intensity_bin_index,development_frozen_intensity_bin_label,batch_count,event_count,minimum_total_intensity,mean_total_intensity,maximum_total_intensity,mean_batch_multiplicity,batch_multiplicity_variance,multiple_event_batch_rate,mixed_side_batch_rate,maximum_batch_multiplicity,mean_buy_multiplicity,mean_sell_multiplicity,simulation_envelope_pending,status
0,1,DEVELOPMENT,1,Q1_OF_5,1372,1374,3.30404133,3.30404133,3.30404133,1.001457726,0.00145666269,0.001457725948,0.0007288629738,2,0.4511661808,0.5502915452,True,COMPUTED_REPORT_ONLY
1,1,DEVELOPMENT,2,Q2_OF_5,1372,1378,3.30404133,3.304041392,3.304042044,1.004373178,0.004357228981,0.004373177843,0.002186588921,2,0.4854227405,0.5189504373,True,COMPUTED_REPORT_ONLY
2,1,DEVELOPMENT,3,Q3_OF_5,1371,1376,3.304042047,3.304778281,3.30959473,1.003646973,0.003636324916,0.003646973012,0.001458789205,2,0.47702407,0.526622903,True,COMPUTED_REPORT_ONLY
3,1,DEVELOPMENT,4,Q4_OF_5,1372,1381,3.309648067,3.575175925,4.696135867,1.006559767,0.007980278701,0.00583090379,0.001457725948,3,0.4781341108,0.528425656,True,COMPUTED_REPORT_ONLY
4,1,DEVELOPMENT,5,Q5_OF_5,1372,1495,4.71198791,16.97489603,130.6450194,1.089650146,0.1575295639,0.06559766764,0.01311953353,6,0.5969387755,0.4927113703,True,COMPUTED_REPORT_ONLY
5,2,CALIBRATION,1,Q1_OF_5,582,585,3.30404133,3.30404133,3.30404133,1.005154639,0.008579235942,0.003436426117,0.001718213058,3,0.4347079038,0.5704467354,True,COMPUTED_REPORT_ONLY
6,2,CALIBRATION,2,Q2_OF_5,454,460,3.30404133,3.304041397,3.304042044,1.013215859,0.0219000107,0.008810572687,0,3,0.486784141,0.5264317181,True,COMPUTED_REPORT_ONLY
7,2,CALIBRATION,3,Q3_OF_5,417,427,3.304042076,3.304880668,3.309636636,1.023980815,0.04269276886,0.01678657074,0,4,0.4316546763,0.5923261391,True,COMPUTED_REPORT_ONLY
8,2,CALIBRATION,4,Q4_OF_5,452,463,3.30966512,3.554952205,4.694700079,1.024336283,0.04153503522,0.01769911504,0,4,0.4623893805,0.5619469027,True,COMPUTED_REPORT_ONLY
9,2,CALIBRATION,5,Q5_OF_5,495,558,4.741603156,19.62502572,105.6173445,1.127272727,0.192270887,0.09494949495,0.008080808081,4,0.7414141414,0.3858585859,True,COMPUTED_REPORT_ONLY


,diagnostic,computed,primary_table,zero_count_cells_available,requires_common_1ms_grid,requires_simulation_envelope,status
0,POSITIVE_EXACT_TIME_BATCH_MULTIPLICITY,True,BATCH_MULTIPLICITY_DISTRIBUTION,False,False,True,COMPUTED
1,SIDE_SPECIFIC_EXACT_TIME_MULTIPLICITY,True,SIDE_MULTIPLICITY_DISTRIBUTION,False,False,True,COMPUTED
2,ZERO_ONE_MULTIPLE_COUNT_CELL_PROBABILITIES,False,PENDING_COMMON_1MS_COUNT_GRID,False,True,True,DEFERRED_TO_COMMON_GRID
3,MULTIPLICITY_CONDITIONAL_ON_PREDICTED_INTENSITY,True,MULTIPLICITY_BY_FROZEN_INTENSITY_BIN,False,False,True,COMPUTED


,check_name,passed
0,side_mark_batch_count_reconciles,True
1,side_counts_conserve_batch_events,True
2,side_probabilities_strictly_inside_unit_interval,True
3,side_probabilities_sum_to_one,True
4,mark_scores_are_finite,True
5,batch_classes_mutually_exhaustive,True
6,reliability_bins_conserve_events,True
7,batch_multiplicity_probabilities_reconcile,True
8,side_multiplicity_probabilities_reconcile,True
9,intensity_bins_frozen_from_development,True


Frozen conditional BUY/SELL mark probabilities, event-weighted mark scores, reliability bins, exact-time batch classes, and observed multiplicity distributions were constructed for DEVELOPMENT and locked CALIBRATION. DEVELOPMENT total-intensity quantiles were frozen before applying the multiplicity analysis to CALIBRATION. Mixed and same-side multiplicity batches remain intact. Zero-count cell probabilities are correctly deferred to the common 1 ms grid. Formal adequacy remains pending coarsening-aware simulation envelopes. No filesystem writes were performed.


In [13]:
# ============================================================
# Build the authoritative common 1 ms count grid and project
# the frozen H1 intensity onto exact half-open grid cells
#
# Grid cells:
#     [cell_start_ns, cell_end_ns)
#
# Observed timestamps remain exact integer nanoseconds.
# No rounding, jitter, event removal, or within-batch ordering
# is introduced.
#
# This cell establishes:
#   - observed BUY, SELL, and pooled counts;
#   - exact frozen-H1 expected counts per cell;
#   - Poisson count scores on a common comparison space;
#   - Pearson and deviance residual arrays;
#   - zero / one / multiple-count calibration.
#
# Cross-side residual dependence and Notebook 06 baseline
# comparison are performed in subsequent cells.
# ============================================================
# Required SciPy namespace for Poisson log-score evaluation
from scipy import special

require(
    callable(
        special.gammaln
    ),
    "scipy.special.gammaln is unavailable.",
)

# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "SIDE_MARK_MULTIPLICITY_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Side-mark and multiplicity diagnostics have not been built.",
)

require(
    bool(
        globals().get(
            "EXACT_COMPENSATOR_PATH_BUILT",
            False,
        )
    ),
    "The canonical exact compensator path has not been built.",
)

require(
    FROZEN_MODEL_LOADED,
    "The frozen H1 model has not been loaded.",
)

require(
    FROZEN_REPLAY_RECONCILED,
    "The frozen H1 replay has not been reconciled.",
)

require(
    not bool(
        globals().get(
            "HAWKES_COMMON_COUNT_GRID_BUILT",
            False,
        )
    ),
    "The frozen-H1 common count grid is already built.",
)

require(
    COUNT_GRID_WIDTH_NS
    == NANOSECONDS_PER_MILLISECOND,
    "The Notebook 08 count grid must remain exactly one millisecond.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED
    == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Immutable grid container
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class FrozenHawkesCountGrid:
    """One exact 1 ms partition grid scored by frozen H1."""

    event_partition: str
    partition_order: int
    contract_start_ns: int
    contract_end_exclusive_ns: int
    grid_width_ns: int
    cell_start_ns: np.ndarray
    cell_end_ns: np.ndarray
    exposure_ns: np.ndarray
    buy_counts: np.ndarray
    sell_counts: np.ndarray
    expected_buy_counts: np.ndarray
    expected_sell_counts: np.ndarray
    buy_pearson_residuals: np.ndarray
    sell_pearson_residuals: np.ndarray
    buy_deviance_residuals: np.ndarray
    sell_deviance_residuals: np.ndarray

    @property
    def cell_count(self) -> int:
        """Return the number of exact grid cells."""
        return int(
            self.cell_start_ns.size
        )

    @property
    def pooled_counts(self) -> np.ndarray:
        """Return observed pooled counts."""
        return (
            self.buy_counts
            + self.sell_counts
        )

    @property
    def expected_pooled_counts(self) -> np.ndarray:
        """Return expected pooled counts."""
        return (
            self.expected_buy_counts
            + self.expected_sell_counts
        )

    @property
    def exposure_seconds(self) -> float:
        """Return total partition exposure in seconds."""
        return float(
            self.exposure_ns.sum(
                dtype=np.int64
            )
            / NANOSECONDS_PER_SECOND
        )


# ------------------------------------------------------------
# Numerical count-score helpers
# ------------------------------------------------------------

def poisson_log_score_terms(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
) -> np.ndarray:
    """Return complete cell-level Poisson log-score terms."""
    observed = np.asarray(
        observed_counts,
        dtype=np.int64,
    )

    expected = np.asarray(
        expected_counts,
        dtype=np.float64,
    )

    require(
        observed.ndim == 1,
        "Observed counts must be one-dimensional.",
    )

    require(
        expected.ndim == 1,
        "Expected counts must be one-dimensional.",
    )

    require(
        observed.shape
        == expected.shape,
        "Observed and expected count arrays differ in shape.",
    )

    require(
        np.all(
            observed
            >= 0
        ),
        "Observed counts contain negative values.",
    )

    require(
        np.isfinite(
            expected
        ).all(),
        "Expected counts contain non-finite values.",
    )

    require(
        np.all(
            expected
            > 0.0
        ),
        "Expected counts must be strictly positive.",
    )

    terms = (
        observed.astype(
            np.float64
        )
        * np.log(
            expected
        )
        - expected
        - special.gammaln(
            observed.astype(
                np.float64
            )
            + 1.0
        )
    )

    require(
        np.isfinite(
            terms
        ).all(),
        "Poisson log-score terms contain non-finite values.",
    )

    return terms


def poisson_pearson_residuals(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
) -> np.ndarray:
    """Return cell-level Poisson Pearson residuals."""
    observed = np.asarray(
        observed_counts,
        dtype=np.float64,
    )

    expected = np.asarray(
        expected_counts,
        dtype=np.float64,
    )

    require(
        observed.shape
        == expected.shape,
        "Pearson residual arrays differ in shape.",
    )

    require(
        np.all(
            observed
            >= 0.0
        ),
        "Pearson observed counts contain negative values.",
    )

    require(
        np.isfinite(
            expected
        ).all()
        and np.all(
            expected
            > 0.0
        ),
        (
            "Pearson expected counts must be finite "
            "and strictly positive."
        ),
    )

    residuals = (
        observed
        - expected
    ) / np.sqrt(
        expected
    )

    require(
        np.isfinite(
            residuals
        ).all(),
        "Pearson residuals contain non-finite values.",
    )

    return residuals


def poisson_deviance_residuals_count_grid(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
) -> np.ndarray:
    """Return signed cell-level Poisson deviance residuals."""
    observed = np.asarray(
        observed_counts,
        dtype=np.float64,
    )

    expected = np.asarray(
        expected_counts,
        dtype=np.float64,
    )

    require(
        observed.shape
        == expected.shape,
        "Deviance residual arrays differ in shape.",
    )

    require(
        np.all(
            observed
            >= 0.0
        ),
        "Deviance observed counts contain negative values.",
    )

    require(
        np.isfinite(
            expected
        ).all()
        and np.all(
            expected
            > 0.0
        ),
        (
            "Deviance expected counts must be finite "
            "and strictly positive."
        ),
    )

    positive_observation = (
        observed
        > 0.0
    )

    deviance_component = np.empty_like(
        expected,
        dtype=np.float64,
    )

    deviance_component[
        ~positive_observation
    ] = expected[
        ~positive_observation
    ]

    deviance_component[
        positive_observation
    ] = (
        observed[
            positive_observation
        ]
        * np.log(
            observed[
                positive_observation
            ]
            / expected[
                positive_observation
            ]
        )
        - (
            observed[
                positive_observation
            ]
            - expected[
                positive_observation
            ]
        )
    )

    deviance_component = np.maximum(
        deviance_component,
        0.0,
    )

    residuals = (
        np.sign(
            observed
            - expected
        )
        * np.sqrt(
            2.0
            * deviance_component
        )
    )

    require(
        np.isfinite(
            residuals
        ).all(),
        "Deviance residuals contain non-finite values.",
    )

    return residuals


# ------------------------------------------------------------
# Exact cumulative compensator evaluation at arbitrary times
# ------------------------------------------------------------

def cumulative_frozen_h1_compensator_at_times(
    evaluation_times_ns: np.ndarray,
    *,
    partition_path: pd.DataFrame,
    parameters: FrozenDiagonalHawkesParameters,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
    initial_state: FrozenHawkesState,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Evaluate exact cumulative BUY and SELL compensators.

    Events exactly at an evaluation boundary are excluded from the
    excitation state at that boundary. This is implemented through
    searchsorted(..., side="left"), consistent with predictable
    intensities and half-open grid cells.
    """
    times_ns = np.asarray(
        evaluation_times_ns,
        dtype=np.int64,
    )

    require(
        times_ns.ndim
        == 1,
        "Compensator evaluation times must be one-dimensional.",
    )

    require(
        len(
            times_ns
        )
        >= 2,
        "At least two compensator evaluation times are required.",
    )

    require(
        np.all(
            times_ns[1:]
            >= times_ns[:-1]
        ),
        "Compensator evaluation times are not nondecreasing.",
    )

    require(
        int(
            times_ns[0]
        )
        >= int(
            observation_start_ns
        ),
        "Compensator evaluation begins before the partition.",
    )

    require(
        int(
            times_ns[-1]
        )
        <= int(
            observation_end_exclusive_ns
        ),
        "Compensator evaluation extends beyond the partition.",
    )

    require(
        int(
            initial_state.state_time_ns
        )
        == int(
            observation_start_ns
        ),
        (
            "The initial excitation state is not expressed "
            "at the partition boundary."
        ),
    )

    event_rows = (
        partition_path.loc[
            partition_path[
                "interval_kind"
            ].eq(
                "EVENT_BATCH"
            )
        ]
        .copy()
        .sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        not event_rows.empty,
        "The partition contains no event batches.",
    )

    event_times_ns = (
        event_rows[
            "event_time_ns"
        ].to_numpy(
            dtype=np.int64
        )
    )

    require(
        np.all(
            event_times_ns[1:]
            > event_times_ns[:-1]
        ),
        "Partition event-batch times are not strictly increasing.",
    )

    cumulative_buy_at_event = (
        event_rows[
            "interval_compensator_buy"
        ]
        .cumsum()
        .to_numpy(
            dtype=np.float64
        )
    )

    cumulative_sell_at_event = (
        event_rows[
            "interval_compensator_sell"
        ]
        .cumsum()
        .to_numpy(
            dtype=np.float64
        )
    )

    post_buy_excitation = (
        event_rows[
            "post_buy_excitation"
        ].to_numpy(
            dtype=np.float64
        )
    )

    post_sell_excitation = (
        event_rows[
            "post_sell_excitation"
        ].to_numpy(
            dtype=np.float64
        )
    )

    previous_event_index = (
        np.searchsorted(
            event_times_ns,
            times_ns,
            side="left",
        )
        - 1
    )

    no_previous_event = (
        previous_event_index
        < 0
    )

    has_previous_event = (
        ~no_previous_event
    )

    cumulative_buy = np.empty(
        len(
            times_ns
        ),
        dtype=np.float64,
    )

    cumulative_sell = np.empty(
        len(
            times_ns
        ),
        dtype=np.float64,
    )

    if no_previous_event.any():
        initial_elapsed_seconds = (
            times_ns[
                no_previous_event
            ]
            - int(
                observation_start_ns
            )
        ).astype(
            np.float64
        ) / NANOSECONDS_PER_SECOND

        cumulative_buy[
            no_previous_event
        ] = (
            parameters.mu_buy
            * initial_elapsed_seconds
            + float(
                initial_state.buy_excitation
            )
            * (
                -np.expm1(
                    -parameters.beta_buy
                    * initial_elapsed_seconds
                )
            )
            / parameters.beta_buy
        )

        cumulative_sell[
            no_previous_event
        ] = (
            parameters.mu_sell
            * initial_elapsed_seconds
            + float(
                initial_state.sell_excitation
            )
            * (
                -np.expm1(
                    -parameters.beta_sell
                    * initial_elapsed_seconds
                )
            )
            / parameters.beta_sell
        )

    if has_previous_event.any():
        selected_event_indices = (
            previous_event_index[
                has_previous_event
            ]
        )

        elapsed_after_event_seconds = (
            times_ns[
                has_previous_event
            ]
            - event_times_ns[
                selected_event_indices
            ]
        ).astype(
            np.float64
        ) / NANOSECONDS_PER_SECOND

        cumulative_buy[
            has_previous_event
        ] = (
            cumulative_buy_at_event[
                selected_event_indices
            ]
            + parameters.mu_buy
            * elapsed_after_event_seconds
            + post_buy_excitation[
                selected_event_indices
            ]
            * (
                -np.expm1(
                    -parameters.beta_buy
                    * elapsed_after_event_seconds
                )
            )
            / parameters.beta_buy
        )

        cumulative_sell[
            has_previous_event
        ] = (
            cumulative_sell_at_event[
                selected_event_indices
            ]
            + parameters.mu_sell
            * elapsed_after_event_seconds
            + post_sell_excitation[
                selected_event_indices
            ]
            * (
                -np.expm1(
                    -parameters.beta_sell
                    * elapsed_after_event_seconds
                )
            )
            / parameters.beta_sell
        )

    require(
        np.isfinite(
            cumulative_buy
        ).all(),
        "Cumulative BUY compensator contains non-finite values.",
    )

    require(
        np.isfinite(
            cumulative_sell
        ).all(),
        "Cumulative SELL compensator contains non-finite values.",
    )

    require(
        np.all(
            np.diff(
                cumulative_buy
            )
            >= 0.0
        ),
        "Cumulative BUY compensator is decreasing.",
    )

    require(
        np.all(
            np.diff(
                cumulative_sell
            )
            >= 0.0
        ),
        "Cumulative SELL compensator is decreasing.",
    )

    return (
        cumulative_buy,
        cumulative_sell,
    )


# ------------------------------------------------------------
# Build one exact partition count grid
# ------------------------------------------------------------

def build_frozen_h1_count_grid(
    *,
    partition_name: str,
    partition_path: pd.DataFrame,
    replay_result: FrozenHawkesReplayResult,
    initial_state: FrozenHawkesState,
    parameters: FrozenDiagonalHawkesParameters,
) -> FrozenHawkesCountGrid:
    """Build and score one exact half-open 1 ms partition grid."""
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        f"Unauthorized count-grid partition: {partition_name}",
    )

    contract_start_ns = int(
        replay_result.observation_start_ns
    )

    contract_end_exclusive_ns = int(
        replay_result.observation_end_exclusive_ns
    )

    contract_duration_ns = (
        contract_end_exclusive_ns
        - contract_start_ns
    )

    require(
        contract_duration_ns
        > 0,
        f"{partition_name} has nonpositive contract duration.",
    )

    cell_count = int(
        (
            contract_duration_ns
            + COUNT_GRID_WIDTH_NS
            - 1
        )
        // COUNT_GRID_WIDTH_NS
    )

    require(
        cell_count
        > 0,
        f"{partition_name} produced no count-grid cells.",
    )

    cell_offsets_ns = (
        np.arange(
            cell_count,
            dtype=np.int64,
        )
        * COUNT_GRID_WIDTH_NS
    )

    cell_start_ns = (
        contract_start_ns
        + cell_offsets_ns
    )

    cell_end_ns = np.minimum(
        cell_start_ns
        + COUNT_GRID_WIDTH_NS,
        contract_end_exclusive_ns,
    ).astype(
        np.int64,
        copy=False,
    )

    exposure_ns = (
        cell_end_ns
        - cell_start_ns
    ).astype(
        np.int64,
        copy=False,
    )

    require(
        np.all(
            exposure_ns
            > 0
        ),
        f"{partition_name} contains a nonpositive grid exposure.",
    )

    require(
        np.all(
            exposure_ns
            <= COUNT_GRID_WIDTH_NS
        ),
        f"{partition_name} contains an oversized grid exposure.",
    )

    require(
        int(
            exposure_ns.sum(
                dtype=np.int64
            )
        )
        == contract_duration_ns,
        f"{partition_name} grid exposure does not conserve time.",
    )

    grid_boundaries_ns = np.empty(
        cell_count + 1,
        dtype=np.int64,
    )

    grid_boundaries_ns[:-1] = (
        cell_start_ns
    )

    grid_boundaries_ns[-1] = (
        contract_end_exclusive_ns
    )

    require(
        int(
            grid_boundaries_ns[0]
        )
        == contract_start_ns,
        f"{partition_name} grid begins at the wrong boundary.",
    )

    require(
        int(
            grid_boundaries_ns[-1]
        )
        == contract_end_exclusive_ns,
        f"{partition_name} grid ends at the wrong boundary.",
    )

    partition_batches = (
        PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
            PRIMARY_SCORING_BATCHES_ANALYTICAL[
                "event_partition"
            ].eq(
                partition_name
            ),
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ],
        ]
        .copy()
        .sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        len(
            partition_batches
        )
        == replay_result.batch_count,
        (
            f"{partition_name} count-grid input has the "
            "wrong batch count."
        ),
    )

    event_times_ns = (
        partition_batches[
            "event_time_ns"
        ].to_numpy(
            dtype=np.int64
        )
    )

    event_cell_indices = (
        (
            event_times_ns
            - contract_start_ns
        )
        // COUNT_GRID_WIDTH_NS
    ).astype(
        np.int64
    )

    require(
        np.all(
            event_cell_indices
            >= 0
        ),
        f"{partition_name} contains an event before grid start.",
    )

    require(
        np.all(
            event_cell_indices
            < cell_count
        ),
        f"{partition_name} contains an event outside the grid.",
    )

    buy_counts = np.zeros(
        cell_count,
        dtype=np.int32,
    )

    sell_counts = np.zeros(
        cell_count,
        dtype=np.int32,
    )

    np.add.at(
        buy_counts,
        event_cell_indices,
        partition_batches[
            "buy_event_count"
        ].to_numpy(
            dtype=np.int32
        ),
    )

    np.add.at(
        sell_counts,
        event_cell_indices,
        partition_batches[
            "sell_event_count"
        ].to_numpy(
            dtype=np.int32
        ),
    )

    require(
        int(
            buy_counts.sum(
                dtype=np.int64
            )
        )
        == int(
            partition_batches[
                "buy_event_count"
            ].sum()
        ),
        f"{partition_name} BUY counts do not reconcile.",
    )

    require(
        int(
            sell_counts.sum(
                dtype=np.int64
            )
        )
        == int(
            partition_batches[
                "sell_event_count"
            ].sum()
        ),
        f"{partition_name} SELL counts do not reconcile.",
    )

    (
        cumulative_buy_at_boundaries,
        cumulative_sell_at_boundaries,
    ) = cumulative_frozen_h1_compensator_at_times(
        grid_boundaries_ns,
        partition_path=partition_path,
        parameters=parameters,
        observation_start_ns=contract_start_ns,
        observation_end_exclusive_ns=(
            contract_end_exclusive_ns
        ),
        initial_state=initial_state,
    )

    expected_buy_counts = np.diff(
        cumulative_buy_at_boundaries
    )

    expected_sell_counts = np.diff(
        cumulative_sell_at_boundaries
    )

    require(
        expected_buy_counts.shape
        == (
            cell_count,
        ),
        f"{partition_name} BUY expected-count shape is invalid.",
    )

    require(
        expected_sell_counts.shape
        == (
            cell_count,
        ),
        f"{partition_name} SELL expected-count shape is invalid.",
    )

    require(
        np.isfinite(
            expected_buy_counts
        ).all()
        and np.all(
            expected_buy_counts
            > 0.0
        ),
        (
            f"{partition_name} BUY expected counts are not "
            "finite and strictly positive."
        ),
    )

    require(
        np.isfinite(
            expected_sell_counts
        ).all()
        and np.all(
            expected_sell_counts
            > 0.0
        ),
        (
            f"{partition_name} SELL expected counts are not "
            "finite and strictly positive."
        ),
    )

    require_close(
        expected_buy_counts.sum(
            dtype=np.float64
        ),
        replay_result.compensator_buy,
        label=(
            f"{partition_name} grid BUY compensator"
        ),
        relative_tolerance=1e-9,
        absolute_tolerance=1e-8,
    )

    require_close(
        expected_sell_counts.sum(
            dtype=np.float64
        ),
        replay_result.compensator_sell,
        label=(
            f"{partition_name} grid SELL compensator"
        ),
        relative_tolerance=1e-9,
        absolute_tolerance=1e-8,
    )

    buy_pearson_residuals = (
        poisson_pearson_residuals(
            buy_counts,
            expected_buy_counts,
        )
    )

    sell_pearson_residuals = (
        poisson_pearson_residuals(
            sell_counts,
            expected_sell_counts,
        )
    )

    buy_deviance_residuals = (
        poisson_deviance_residuals_count_grid(
            buy_counts,
            expected_buy_counts,
        )
    )

    sell_deviance_residuals = (
        poisson_deviance_residuals_count_grid(
            sell_counts,
            expected_sell_counts,
        )
    )

    return FrozenHawkesCountGrid(
        event_partition=partition_name,
        partition_order=(
            PARTITION_ORDER_MAP[
                partition_name
            ]
        ),
        contract_start_ns=contract_start_ns,
        contract_end_exclusive_ns=(
            contract_end_exclusive_ns
        ),
        grid_width_ns=COUNT_GRID_WIDTH_NS,
        cell_start_ns=cell_start_ns,
        cell_end_ns=cell_end_ns,
        exposure_ns=exposure_ns,
        buy_counts=buy_counts,
        sell_counts=sell_counts,
        expected_buy_counts=expected_buy_counts,
        expected_sell_counts=expected_sell_counts,
        buy_pearson_residuals=(
            buy_pearson_residuals
        ),
        sell_pearson_residuals=(
            sell_pearson_residuals
        ),
        buy_deviance_residuals=(
            buy_deviance_residuals
        ),
        sell_deviance_residuals=(
            sell_deviance_residuals
        ),
    )


# ------------------------------------------------------------
# Construct DEVELOPMENT and locked CALIBRATION grids
# ------------------------------------------------------------

HAWKES_COMMON_COUNT_GRIDS: dict[
    str,
    FrozenHawkesCountGrid,
] = {
    "DEVELOPMENT": (
        build_frozen_h1_count_grid(
            partition_name="DEVELOPMENT",
            partition_path=(
                DEVELOPMENT_EXACT_COMPENSATOR_PATH
            ),
            replay_result=(
                FROZEN_DEVELOPMENT_REPLAY_RESULT
            ),
            initial_state=(
                DEVELOPMENT_INITIAL_STATE
            ),
            parameters=(
                FROZEN_H1_PARAMETERS
            ),
        )
    ),
    "CALIBRATION": (
        build_frozen_h1_count_grid(
            partition_name="CALIBRATION",
            partition_path=(
                CALIBRATION_EXACT_COMPENSATOR_PATH
            ),
            replay_result=(
                FROZEN_CALIBRATION_REPLAY_RESULT
            ),
            initial_state=(
                FROZEN_DEVELOPMENT_REPLAY_RESULT
                .final_state
            ),
            parameters=(
                FROZEN_H1_PARAMETERS
            ),
        )
    ),
}


# ------------------------------------------------------------
# Grid-level reconciliation
# ------------------------------------------------------------

grid_audit_records: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    grid = HAWKES_COMMON_COUNT_GRIDS[
        partition_name
    ]

    expected_partition_counts = (
        EXPECTED_ANALYTICAL_COUNTS_BY_PARTITION[
            partition_name
        ]
    )

    observed_buy_count = int(
        grid.buy_counts.sum(
            dtype=np.int64
        )
    )

    observed_sell_count = int(
        grid.sell_counts.sum(
            dtype=np.int64
        )
    )

    observed_pooled_count = (
        observed_buy_count
        + observed_sell_count
    )

    require(
        observed_pooled_count
        == expected_partition_counts[
            "event_count"
        ],
        (
            f"{partition_name} pooled grid count differs "
            "from the authoritative event count."
        ),
    )

    require(
        int(
            np.count_nonzero(
                grid.pooled_counts
            )
        )
        <= expected_partition_counts[
            "batch_count"
        ],
        (
            f"{partition_name} occupied 1 ms cells exceed "
            "the number of exact-time batches."
        ),
    )

    grid_audit_records.append(
        {
            "partition_order": (
                grid.partition_order
            ),
            "event_partition": (
                partition_name
            ),
            "grid_width_ns": (
                grid.grid_width_ns
            ),
            "grid_width_ms": (
                grid.grid_width_ns
                / NANOSECONDS_PER_MILLISECOND
            ),
            "contract_start_ns": (
                grid.contract_start_ns
            ),
            "contract_end_exclusive_ns": (
                grid.contract_end_exclusive_ns
            ),
            "contract_duration_ns": int(
                grid.exposure_ns.sum(
                    dtype=np.int64
                )
            ),
            "cell_count": (
                grid.cell_count
            ),
            "full_width_cell_count": int(
                np.sum(
                    grid.exposure_ns
                    == COUNT_GRID_WIDTH_NS
                )
            ),
            "partial_width_cell_count": int(
                np.sum(
                    grid.exposure_ns
                    < COUNT_GRID_WIDTH_NS
                )
            ),
            "occupied_pooled_cell_count": int(
                np.count_nonzero(
                    grid.pooled_counts
                )
            ),
            "observed_buy_event_count": (
                observed_buy_count
            ),
            "observed_sell_event_count": (
                observed_sell_count
            ),
            "observed_pooled_event_count": (
                observed_pooled_count
            ),
            "predicted_buy_event_count": float(
                grid.expected_buy_counts.sum(
                    dtype=np.float64
                )
            ),
            "predicted_sell_event_count": float(
                grid.expected_sell_counts.sum(
                    dtype=np.float64
                )
            ),
            "predicted_pooled_event_count": float(
                grid.expected_pooled_counts.sum(
                    dtype=np.float64
                )
            ),
            "timestamp_mutation_performed": False,
            "event_removal_performed": False,
            "within_batch_order_imposed": False,
            "status": "PASS",
        }
    )

HAWKES_COMMON_COUNT_GRID_AUDIT = (
    pd.DataFrame.from_records(
        grid_audit_records
    )
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Common-grid Hawkes count-score table
# ------------------------------------------------------------

count_score_records: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    grid = HAWKES_COMMON_COUNT_GRIDS[
        partition_name
    ]

    exposure_seconds = (
        grid.exposure_seconds
    )

    buy_score_terms = (
        poisson_log_score_terms(
            grid.buy_counts,
            grid.expected_buy_counts,
        )
    )

    sell_score_terms = (
        poisson_log_score_terms(
            grid.sell_counts,
            grid.expected_sell_counts,
        )
    )

    pooled_score_terms = (
        poisson_log_score_terms(
            grid.pooled_counts,
            grid.expected_pooled_counts,
        )
    )

    component_contracts = (
        (
            "BUY",
            grid.buy_counts,
            grid.expected_buy_counts,
            buy_score_terms,
            "COMMON_1MS_SIDE_POISSON_COUNT",
        ),
        (
            "SELL",
            grid.sell_counts,
            grid.expected_sell_counts,
            sell_score_terms,
            "COMMON_1MS_SIDE_POISSON_COUNT",
        ),
        (
            "POOLED",
            grid.pooled_counts,
            grid.expected_pooled_counts,
            pooled_score_terms,
            "COMMON_1MS_POOLED_POISSON_COUNT",
        ),
    )

    for (
        component,
        observed_counts,
        expected_counts,
        score_terms,
        score_space,
    ) in component_contracts:
        observed_event_count = int(
            observed_counts.sum(
                dtype=np.int64
            )
        )

        predicted_event_count = float(
            expected_counts.sum(
                dtype=np.float64
            )
        )

        log_score_total = float(
            score_terms.sum(
                dtype=np.float64
            )
        )

        count_score_records.append(
            {
                "model_id": (
                    EXPECTED_SELECTED_MODEL_ID
                ),
                "model_family": (
                    EXPECTED_SELECTED_MODEL_FAMILY
                ),
                "fitted_partition": (
                    "DEVELOPMENT"
                ),
                "scored_partition": (
                    partition_name
                ),
                "partition_order": (
                    grid.partition_order
                ),
                "score_space": (
                    score_space
                ),
                "component": (
                    component
                ),
                "cell_count": (
                    grid.cell_count
                ),
                "exposure_seconds": (
                    exposure_seconds
                ),
                "observed_event_count": (
                    observed_event_count
                ),
                "predicted_event_count": (
                    predicted_event_count
                ),
                "observed_minus_predicted": (
                    observed_event_count
                    - predicted_event_count
                ),
                "log_score_total": (
                    log_score_total
                ),
                "log_score_per_second": (
                    log_score_total
                    / exposure_seconds
                ),
                "log_score_per_observed_event": (
                    log_score_total
                    / observed_event_count
                ),
                "calibration_parameter_updates": 0,
                "strict_pre_batch_history": True,
                "common_grid_comparison_authorized": True,
                "status": "PASS",
            }
        )

    aggregate_observed_event_count = int(
        grid.pooled_counts.sum(
            dtype=np.int64
        )
    )

    aggregate_predicted_event_count = float(
        grid.expected_pooled_counts.sum(
            dtype=np.float64
        )
    )

    aggregate_log_score = float(
        buy_score_terms.sum(
            dtype=np.float64
        )
        + sell_score_terms.sum(
            dtype=np.float64
        )
    )

    count_score_records.append(
        {
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "model_family": (
                EXPECTED_SELECTED_MODEL_FAMILY
            ),
            "fitted_partition": (
                "DEVELOPMENT"
            ),
            "scored_partition": (
                partition_name
            ),
            "partition_order": (
                grid.partition_order
            ),
            "score_space": (
                "COMMON_1MS_BUY_AND_SELL_POISSON_COUNT"
            ),
            "component": (
                "BUY_AND_SELL_AGGREGATE"
            ),
            "cell_count": int(
                2
                * grid.cell_count
            ),
            "exposure_seconds": (
                exposure_seconds
            ),
            "observed_event_count": (
                aggregate_observed_event_count
            ),
            "predicted_event_count": (
                aggregate_predicted_event_count
            ),
            "observed_minus_predicted": (
                aggregate_observed_event_count
                - aggregate_predicted_event_count
            ),
            "log_score_total": (
                aggregate_log_score
            ),
            "log_score_per_second": (
                aggregate_log_score
                / exposure_seconds
            ),
            "log_score_per_observed_event": (
                aggregate_log_score
                / aggregate_observed_event_count
            ),
            "calibration_parameter_updates": 0,
            "strict_pre_batch_history": True,
            "common_grid_comparison_authorized": True,
            "status": "PASS",
        }
    )

HAWKES_COMMON_COUNT_SCORE_TABLE = (
    pd.DataFrame.from_records(
        count_score_records
    )
    .sort_values(
        [
            "partition_order",
            "component",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Common-grid count residual summaries
# ------------------------------------------------------------

count_residual_records: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    grid = HAWKES_COMMON_COUNT_GRIDS[
        partition_name
    ]

    component_arrays = (
        (
            "BUY",
            grid.buy_counts,
            grid.expected_buy_counts,
            grid.buy_pearson_residuals,
            grid.buy_deviance_residuals,
        ),
        (
            "SELL",
            grid.sell_counts,
            grid.expected_sell_counts,
            grid.sell_pearson_residuals,
            grid.sell_deviance_residuals,
        ),
        (
            "POOLED",
            grid.pooled_counts,
            grid.expected_pooled_counts,
            poisson_pearson_residuals(
                grid.pooled_counts,
                grid.expected_pooled_counts,
            ),
            poisson_deviance_residuals_count_grid(
                grid.pooled_counts,
                grid.expected_pooled_counts,
            ),
        ),
    )

    for (
        component,
        observed_counts,
        expected_counts,
        pearson_residuals,
        deviance_residuals,
    ) in component_arrays:
        raw_residuals = (
            observed_counts.astype(
                np.float64
            )
            - expected_counts
        )

        observed_mean = float(
            np.mean(
                observed_counts
            )
        )

        observed_variance = float(
            np.var(
                observed_counts,
                ddof=1,
            )
        )

        count_residual_records.append(
            {
                "model_id": (
                    EXPECTED_SELECTED_MODEL_ID
                ),
                "event_partition": (
                    partition_name
                ),
                "partition_order": (
                    grid.partition_order
                ),
                "component": (
                    component
                ),
                "grid_width_ms": 1,
                "cell_count": (
                    grid.cell_count
                ),
                "observed_event_count": int(
                    observed_counts.sum(
                        dtype=np.int64
                    )
                ),
                "predicted_event_count": float(
                    expected_counts.sum(
                        dtype=np.float64
                    )
                ),
                "observed_count_mean": (
                    observed_mean
                ),
                "observed_count_variance": (
                    observed_variance
                ),
                "observed_fano_ratio": (
                    observed_variance
                    / observed_mean
                    if observed_mean
                    > 0.0
                    else float(
                        "nan"
                    )
                ),
                "raw_residual_mean": float(
                    np.mean(
                        raw_residuals
                    )
                ),
                "raw_residual_variance": float(
                    np.var(
                        raw_residuals,
                        ddof=1,
                    )
                ),
                "pearson_residual_mean": float(
                    np.mean(
                        pearson_residuals
                    )
                ),
                "pearson_residual_variance": float(
                    np.var(
                        pearson_residuals,
                        ddof=1,
                    )
                ),
                "deviance_residual_mean": float(
                    np.mean(
                        deviance_residuals
                    )
                ),
                "deviance_residual_variance": float(
                    np.var(
                        deviance_residuals,
                        ddof=1,
                    )
                ),
                "maximum_absolute_pearson_residual": float(
                    np.max(
                        np.abs(
                            pearson_residuals
                        )
                    )
                ),
                "maximum_absolute_deviance_residual": float(
                    np.max(
                        np.abs(
                            deviance_residuals
                        )
                    )
                ),
                "simulation_envelope_pending": True,
                "status": "COMPUTED_REPORT_ONLY",
            }
        )

HAWKES_COMMON_COUNT_RESIDUAL_SUMMARY = (
    pd.DataFrame.from_records(
        count_residual_records
    )
    .sort_values(
        [
            "partition_order",
            "component",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Zero / one / multiple count-cell calibration
# ------------------------------------------------------------

count_category_records: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    grid = HAWKES_COMMON_COUNT_GRIDS[
        partition_name
    ]

    component_arrays = (
        (
            "BUY",
            grid.buy_counts,
            grid.expected_buy_counts,
        ),
        (
            "SELL",
            grid.sell_counts,
            grid.expected_sell_counts,
        ),
        (
            "POOLED",
            grid.pooled_counts,
            grid.expected_pooled_counts,
        ),
    )

    for (
        component,
        observed_counts,
        expected_counts,
    ) in component_arrays:
        probability_zero = np.exp(
            -expected_counts
        )

        probability_one = (
            expected_counts
            * probability_zero
        )

        probability_multiple = np.maximum(
            0.0,
            1.0
            - probability_zero
            - probability_one,
        )

        require_array_close(
            (
                probability_zero
                + probability_one
                + probability_multiple
            ),
            np.ones(
                grid.cell_count,
                dtype=np.float64,
            ),
            label=(
                f"{partition_name} {component} "
                "Poisson count-category probabilities"
            ),
            rtol=1e-13,
            atol=1e-13,
        )

        category_contracts = (
            (
                "ZERO",
                observed_counts
                == 0,
                probability_zero,
            ),
            (
                "ONE",
                observed_counts
                == 1,
                probability_one,
            ),
            (
                "MULTIPLE",
                observed_counts
                >= 2,
                probability_multiple,
            ),
        )

        for (
            count_category,
            observed_indicator,
            expected_probability,
        ) in category_contracts:
            observed_cell_count = int(
                observed_indicator.sum()
            )

            expected_cell_count = float(
                expected_probability.sum(
                    dtype=np.float64
                )
            )

            count_category_records.append(
                {
                    "model_id": (
                        EXPECTED_SELECTED_MODEL_ID
                    ),
                    "event_partition": (
                        partition_name
                    ),
                    "partition_order": (
                        grid.partition_order
                    ),
                    "component": (
                        component
                    ),
                    "grid_width_ms": 1,
                    "count_category": (
                        count_category
                    ),
                    "cell_count": (
                        grid.cell_count
                    ),
                    "observed_cell_count": (
                        observed_cell_count
                    ),
                    "expected_cell_count": (
                        expected_cell_count
                    ),
                    "observed_cell_rate": (
                        observed_cell_count
                        / grid.cell_count
                    ),
                    "expected_cell_rate": (
                        expected_cell_count
                        / grid.cell_count
                    ),
                    "observed_minus_expected_cell_rate": (
                        observed_cell_count
                        / grid.cell_count
                        - expected_cell_count
                        / grid.cell_count
                    ),
                    "simulation_envelope_pending": True,
                    "status": "COMPUTED_REPORT_ONLY",
                }
            )

HAWKES_COUNT_CATEGORY_CALIBRATION = (
    pd.DataFrame.from_records(
        count_category_records
    )
    .sort_values(
        [
            "partition_order",
            "component",
            "count_category",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Update multiplicity diagnostic scope ledger
# ------------------------------------------------------------

zero_count_scope_mask = (
    MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER[
        "diagnostic"
    ].eq(
        "ZERO_ONE_MULTIPLE_COUNT_CELL_PROBABILITIES"
    )
)

require(
    int(
        zero_count_scope_mask.sum()
    )
    == 1,
    "The zero/one/multiple scope ledger row is missing.",
)

MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER.loc[
    zero_count_scope_mask,
    "computed",
] = True

MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER.loc[
    zero_count_scope_mask,
    "primary_table",
] = "HAWKES_COUNT_CATEGORY_CALIBRATION"

MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER.loc[
    zero_count_scope_mask,
    "zero_count_cells_available",
] = True

MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER.loc[
    zero_count_scope_mask,
    "status",
] = "COMPUTED_ON_COMMON_1MS_GRID"


# ------------------------------------------------------------
# Verification matrix
# ------------------------------------------------------------

for partition_name in ANALYTICAL_PARTITIONS:
    grid = HAWKES_COMMON_COUNT_GRIDS[
        partition_name
    ]

    partition_category_table = (
        HAWKES_COUNT_CATEGORY_CALIBRATION.loc[
            HAWKES_COUNT_CATEGORY_CALIBRATION[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    for component in (
        "BUY",
        "SELL",
        "POOLED",
    ):
        component_categories = (
            partition_category_table.loc[
                partition_category_table[
                    "component"
                ].eq(
                    component
                )
            ]
        )

        require(
            len(
                component_categories
            )
            == 3,
            (
                f"{partition_name} {component} does not contain "
                "all three count categories."
            ),
        )

        require(
            int(
                component_categories[
                    "observed_cell_count"
                ].sum()
            )
            == grid.cell_count,
            (
                f"{partition_name} {component} observed count "
                "categories do not conserve cells."
            ),
        )

        require_close(
            component_categories[
                "expected_cell_count"
            ].sum(),
            grid.cell_count,
            label=(
                f"{partition_name} {component} expected "
                "count-category cells"
            ),
            relative_tolerance=1e-11,
            absolute_tolerance=1e-7,
        )


HAWKES_COMMON_COUNT_GRID_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "one_millisecond_grid_width_preserved"
            ),
            "passed": bool(
                all(
                    grid.grid_width_ns
                    == NANOSECONDS_PER_MILLISECOND
                    for grid in (
                        HAWKES_COMMON_COUNT_GRIDS.values()
                    )
                )
            ),
        },
        {
            "check_name": (
                "half_open_grid_exposure_conserved"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "observed_buy_sell_counts_reconcile"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "grid_expected_counts_reconcile_to_exact_compensators"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "all_expected_counts_strictly_positive"
            ),
            "passed": bool(
                all(
                    np.all(
                        grid.expected_buy_counts
                        > 0.0
                    )
                    and np.all(
                        grid.expected_sell_counts
                        > 0.0
                    )
                    for grid in (
                        HAWKES_COMMON_COUNT_GRIDS.values()
                    )
                )
            ),
        },
        {
            "check_name": (
                "poisson_count_scores_finite"
            ),
            "passed": bool(
                np.isfinite(
                    HAWKES_COMMON_COUNT_SCORE_TABLE[
                        "log_score_total"
                    ].to_numpy(
                        dtype=np.float64
                    )
                ).all()
            ),
        },
        {
            "check_name": (
                "pearson_and_deviance_residuals_finite"
            ),
            "passed": bool(
                np.isfinite(
                    HAWKES_COMMON_COUNT_RESIDUAL_SUMMARY[
                        [
                            "pearson_residual_mean",
                            "pearson_residual_variance",
                            "deviance_residual_mean",
                            "deviance_residual_variance",
                        ]
                    ].to_numpy(
                        dtype=np.float64
                    )
                ).all()
            ),
        },
        {
            "check_name": (
                "zero_one_multiple_categories_conserve_cells"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "timestamp_mutation_absent"
            ),
            "passed": bool(
                not HAWKES_COMMON_COUNT_GRID_AUDIT[
                    "timestamp_mutation_performed"
                ].any()
            ),
        },
        {
            "check_name": (
                "event_removal_absent"
            ),
            "passed": bool(
                not HAWKES_COMMON_COUNT_GRID_AUDIT[
                    "event_removal_performed"
                ].any()
            ),
        },
        {
            "check_name": (
                "within_batch_order_not_imposed"
            ),
            "passed": bool(
                not HAWKES_COMMON_COUNT_GRID_AUDIT[
                    "within_batch_order_imposed"
                ].any()
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    HAWKES_COMMON_COUNT_GRID_VERIFICATION[
        "passed"
    ].all(),
    (
        "At least one frozen-H1 common count-grid "
        "verification failed."
    ),
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

HAWKES_COMMON_COUNT_GRID_BUILT = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is False,
    "This cell performed a filesystem write.",
)


display(
    HAWKES_COMMON_COUNT_GRID_AUDIT
)

display(
    HAWKES_COMMON_COUNT_SCORE_TABLE
)

display(
    HAWKES_COMMON_COUNT_RESIDUAL_SUMMARY
)

display(
    HAWKES_COUNT_CATEGORY_CALIBRATION
)

display(
    MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER
)

display(
    HAWKES_COMMON_COUNT_GRID_VERIFICATION
)

print(
    "The authoritative one-millisecond half-open count grid was "
    "constructed for DEVELOPMENT and locked CALIBRATION. Exact "
    "integer-nanosecond event timestamps were preserved. Frozen-H1 "
    "expected BUY and SELL counts reconcile to the exact partition "
    "compensators. Common-grid Poisson scores, Pearson and deviance "
    "residuals, and zero/one/multiple count-cell calibration are "
    "available. Cross-side residual diagnostics and comparison with "
    "the verified Notebook 06 baselines remain pending. No filesystem "
    "writes were performed."
)

,partition_order,event_partition,grid_width_ns,grid_width_ms,contract_start_ns,contract_end_exclusive_ns,contract_duration_ns,cell_count,full_width_cell_count,partial_width_cell_count,occupied_pooled_cell_count,observed_buy_event_count,observed_sell_event_count,observed_pooled_event_count,predicted_buy_event_count,predicted_sell_event_count,predicted_pooled_event_count,timestamp_mutation_performed,event_removal_performed,within_batch_order_imposed,status
0,1,DEVELOPMENT,1000000,1,1783665467531985400,1783667269391572100,1801859586700,1801860,1801859,1,6854,3414,3590,7004,"3,414.000015","3,590.00007","7,004.000086",False,False,False,PASS
1,2,CALIBRATION,1000000,1,1783667269391572100,1783667989690751200,720299179100,720300,720299,1,2396,1230,1263,2493,"1,339.253892","1,415.723094","2,754.976987",False,False,False,PASS


,model_id,model_family,fitted_partition,scored_partition,partition_order,score_space,component,cell_count,exposure_seconds,observed_event_count,predicted_event_count,observed_minus_predicted,log_score_total,log_score_per_second,log_score_per_observed_event,calibration_parameter_updates,strict_pre_batch_history,common_grid_comparison_authorized,status
0,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,DEVELOPMENT,1,COMMON_1MS_SIDE_POISSON_COUNT,BUY,1801860,"1,801.859587",3414,"3,414.000015",-1.538985407e-05,"-19,457.58476",-10.79861322,-5.69935113,0,True,True,PASS
1,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,DEVELOPMENT,1,COMMON_1MS_BUY_AND_SELL_POISSON_COUNT,BUY_AND_SELL_AGGREGATE,3603720,"1,801.859587",7004,"7,004.000086",-8.551491192e-05,"-41,486.19596",-23.02410036,-5.923214729,0,True,True,PASS
2,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,DEVELOPMENT,1,COMMON_1MS_POOLED_POISSON_COUNT,POOLED,1801860,"1,801.859587",7004,"7,004.000086",-8.551491192e-05,"-39,613.37267",-21.98471677,-5.655821341,0,True,True,PASS
3,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,DEVELOPMENT,1,COMMON_1MS_SIDE_POISSON_COUNT,SELL,1801860,"1,801.859587",3590,"3,590.00007",-7.012505421e-05,"-22,028.6112",-12.22548714,-6.1361034,0,True,True,PASS
4,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,CALIBRATION,2,COMMON_1MS_SIDE_POISSON_COUNT,BUY,720300,720.2991791,1230,"1,339.253892",-109.2538923,"-7,003.281796",-9.722740216,-5.693725037,0,True,True,PASS
5,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,CALIBRATION,2,COMMON_1MS_BUY_AND_SELL_POISSON_COUNT,BUY_AND_SELL_AGGREGATE,1440600,720.2991791,2493,"2,754.976987",-261.9769865,"-14,914.46158",-20.70592611,-5.982535733,0,True,True,PASS
6,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,CALIBRATION,2,COMMON_1MS_POOLED_POISSON_COUNT,POOLED,720300,720.2991791,2493,"2,754.976987",-261.9769865,"-14,280.89518",-19.82633826,-5.728397583,0,True,True,PASS
7,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,DEVELOPMENT,CALIBRATION,2,COMMON_1MS_SIDE_POISSON_COUNT,SELL,720300,720.2991791,1263,"1,415.723094",-152.7230942,"-7,911.179787",-10.9831859,-6.263800306,0,True,True,PASS


,model_id,event_partition,partition_order,component,grid_width_ms,cell_count,observed_event_count,predicted_event_count,observed_count_mean,observed_count_variance,observed_fano_ratio,raw_residual_mean,raw_residual_variance,pearson_residual_mean,pearson_residual_variance,deviance_residual_mean,deviance_residual_variance,maximum_absolute_pearson_residual,maximum_absolute_deviance_residual,simulation_envelope_pending,status
0,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,BUY,1,1801860,3414,"3,414.000015",0.001894708801,0.002018765877,1.065475537,-8.541093132e-12,0.001980475977,-0.02075773122,0.2724637976,-0.05392531082,0.01496898924,35.04996656,6.301362329,True,COMPUTED_REPORT_ONLY
1,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,POOLED,1,1801860,7004,"7,004.000086",0.003887094447,0.00409730994,1.054080367,-4.745924116e-11,0.004038472673,-0.02002379331,0.4715161433,-0.07562835521,0.03059641578,31.41958304,6.281249756,True,COMPUTED_REPORT_ONLY
2,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,SELL,1,1801860,3590,"3,590.00007",0.001992385646,0.002048355246,1.02809175,-3.891814803e-11,0.002029472832,-0.01709955139,0.3896801461,-0.05624273655,0.01733382695,33.03437212,6.079297718,True,COMPUTED_REPORT_ONLY
3,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,BUY,1,720300,1230,"1,339.253892",0.001707621824,0.002015690186,1.180407838,-0.0001516783178,0.001973385196,-0.02344478596,0.2350705583,-0.0540936073,0.01326364963,52.88493178,6.335341308,True,COMPUTED_REPORT_ONLY
4,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,POOLED,1,720300,2493,"2,754.976987",0.003461057893,0.00382115147,1.104041478,-0.0003637053818,0.003761823629,-0.02473598239,0.4153589005,-0.07631147,0.02710419632,42.4453224,6.294850506,True,COMPUTED_REPORT_ONLY
5,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,SELL,1,720300,1263,"1,415.723094",0.001753436068,0.001792013331,1.022000952,-0.000212027064,0.001776079687,-0.02011426289,0.340575654,-0.05663588454,0.01527754639,30.72889167,5.069838122,True,COMPUTED_REPORT_ONLY


,model_id,event_partition,partition_order,component,grid_width_ms,count_category,cell_count,observed_cell_count,expected_cell_count,observed_cell_rate,expected_cell_rate,observed_minus_expected_cell_rate,simulation_envelope_pending,status
0,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,BUY,1,MULTIPLE,1801860,67,6.70599816,3.718379896e-05,3.721708767e-06,3.34620902e-05,True,COMPUTED_REPORT_ONLY
1,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,BUY,1,ONE,1801860,3261,"3,400.546585",0.001809796544,0.001887242397,-7.744585311e-05,True,COMPUTED_REPORT_ONLY
2,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,BUY,1,ZERO,1801860,1798532,"1,798,452.747",0.9981530197,0.9981090359,4.398376292e-05,True,COMPUTED_REPORT_ONLY
3,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,POOLED,1,MULTIPLE,1801860,116,18.12246369,6.437792059e-05,1.005764248e-05,5.432027811e-05,True,COMPUTED_REPORT_ONLY
4,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,POOLED,1,ONE,1801860,6738,"6,967.681612",0.003739469215,0.003866938392,-0.0001274691772,True,COMPUTED_REPORT_ONLY
5,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,POOLED,1,ZERO,1801860,1795006,"1,794,874.196",0.9961961529,0.996123004,7.314889907e-05,True,COMPUTED_REPORT_ONLY
6,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,SELL,1,MULTIPLE,1801860,28,4.559439153,1.553949807e-05,2.530406998e-06,1.300909108e-05,True,COMPUTED_REPORT_ONLY
7,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,SELL,1,ONE,1801860,3524,"3,580.873633",0.001955756829,0.001987320676,-3.156384663e-05,True,COMPUTED_REPORT_ONLY
8,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,SELL,1,ZERO,1801860,1798308,"1,798,274.567",0.9980287037,0.9980101489,1.855475555e-05,True,COMPUTED_REPORT_ONLY
9,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,BUY,1,MULTIPLE,720300,53,2.776223586,7.358045259e-05,3.854260151e-06,6.972619244e-05,True,COMPUTED_REPORT_ONLY


,diagnostic,computed,primary_table,zero_count_cells_available,requires_common_1ms_grid,requires_simulation_envelope,status
0,POSITIVE_EXACT_TIME_BATCH_MULTIPLICITY,True,BATCH_MULTIPLICITY_DISTRIBUTION,False,False,True,COMPUTED
1,SIDE_SPECIFIC_EXACT_TIME_MULTIPLICITY,True,SIDE_MULTIPLICITY_DISTRIBUTION,False,False,True,COMPUTED
2,ZERO_ONE_MULTIPLE_COUNT_CELL_PROBABILITIES,True,HAWKES_COUNT_CATEGORY_CALIBRATION,True,True,True,COMPUTED_ON_COMMON_1MS_GRID
3,MULTIPLICITY_CONDITIONAL_ON_PREDICTED_INTENSITY,True,MULTIPLICITY_BY_FROZEN_INTENSITY_BIN,False,False,True,COMPUTED


,check_name,passed
0,one_millisecond_grid_width_preserved,True
1,half_open_grid_exposure_conserved,True
2,observed_buy_sell_counts_reconcile,True
3,grid_expected_counts_reconcile_to_exact_compen...,True
4,all_expected_counts_strictly_positive,True
5,poisson_count_scores_finite,True
6,pearson_and_deviance_residuals_finite,True
7,zero_one_multiple_categories_conserve_cells,True
8,timestamp_mutation_absent,True
9,event_removal_absent,True


The authoritative one-millisecond half-open count grid was constructed for DEVELOPMENT and locked CALIBRATION. Exact integer-nanosecond event timestamps were preserved. Frozen-H1 expected BUY and SELL counts reconcile to the exact partition compensators. Common-grid Poisson scores, Pearson and deviance residuals, and zero/one/multiple count-cell calibration are available. Cross-side residual diagnostics and comparison with the verified Notebook 06 baselines remain pending. No filesystem writes were performed.


In [14]:
# ============================================================
# Cross-side residual dependence diagnostics on the common 1 ms grid
#
# Positive lag:
#     BUY residual at cell t versus SELL residual at cell t + lag
#
# Negative lag:
#     SELL residual at cell t versus BUY residual at cell t + |lag|
#
# Naive correlation bounds, chi-square references, coincidence
# z-scores, and portmanteau p-values are descriptive only.
# Final conclusions about omitted BUY↔SELL excitation remain
# blocked pending coarsening-aware parametric simulation.
# ============================================================

from collections.abc import Mapping
from scipy import stats


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    "HAWKES_COMMON_COUNT_GRIDS" in globals(),
    "The authoritative common 1 ms Hawkes grids are unavailable.",
)

require(
    isinstance(
        HAWKES_COMMON_COUNT_GRIDS,
        Mapping,
    ),
    "HAWKES_COMMON_COUNT_GRIDS must be a mapping.",
)

require(
    set(
        ANALYTICAL_PARTITIONS
    ).issubset(
        HAWKES_COMMON_COUNT_GRIDS.keys()
    ),
    "A required analytical partition is missing from the common grid.",
)

require(
    not bool(
        globals().get(
            "CROSS_SIDE_RESIDUAL_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Cross-side residual diagnostics are already built.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Diagnostic configuration
# ------------------------------------------------------------

CROSS_SIDE_MAX_LAG_CELLS = 50

CROSS_SIDE_PORTMANTEAU_LAGS = (
    10,
    20,
    50,
)

NAIVE_CROSS_CORRELATION_MULTIPLIER = 1.96

require(
    CROSS_SIDE_MAX_LAG_CELLS
    >= max(
        CROSS_SIDE_PORTMANTEAU_LAGS
    ),
    "The maximum cross-side lag is below a requested portmanteau lag.",
)


# ------------------------------------------------------------
# Numerical helpers
# ------------------------------------------------------------

def require_finite_vector(
    values,
    *,
    label,
):
    """Return a finite one-dimensional float64 array."""
    array = np.asarray(
        values,
        dtype=np.float64,
    )

    require(
        array.ndim == 1,
        f"{label} must be one-dimensional.",
    )

    require(
        len(
            array
        )
        > CROSS_SIDE_MAX_LAG_CELLS,
        (
            f"{label} is too short for maximum lag "
            f"{CROSS_SIDE_MAX_LAG_CELLS}."
        ),
    )

    require(
        np.isfinite(
            array
        ).all(),
        f"{label} contains non-finite values.",
    )

    return array


def lagged_cross_correlation(
    centered_buy,
    centered_sell,
    *,
    lag,
):
    """
    Compute one pairwise-normalized cross-correlation.

    lag > 0:
        BUY_t versus SELL_(t + lag)

    lag < 0:
        SELL_t versus BUY_(t + |lag|)
    """
    lag_value = int(
        lag
    )

    require(
        abs(
            lag_value
        )
        <= CROSS_SIDE_MAX_LAG_CELLS,
        "Requested cross-correlation lag exceeds the configured maximum.",
    )

    if lag_value > 0:
        buy_slice = centered_buy[
            :-lag_value
        ]

        sell_slice = centered_sell[
            lag_value:
        ]

    elif lag_value < 0:
        offset = -lag_value

        buy_slice = centered_buy[
            offset:
        ]

        sell_slice = centered_sell[
            :-offset
        ]

    else:
        buy_slice = centered_buy
        sell_slice = centered_sell

    require(
        len(
            buy_slice
        )
        == len(
            sell_slice
        ),
        "Cross-correlation slices have different lengths.",
    )

    numerator = float(
        np.dot(
            buy_slice,
            sell_slice,
        )
    )

    buy_energy = float(
        np.dot(
            buy_slice,
            buy_slice,
        )
    )

    sell_energy = float(
        np.dot(
            sell_slice,
            sell_slice,
        )
    )

    denominator = math.sqrt(
        buy_energy
        * sell_energy
    )

    require(
        denominator > 0.0,
        "Cross-correlation denominator is nonpositive.",
    )

    correlation = (
        numerator
        / denominator
    )

    require(
        math.isfinite(
            correlation
        ),
        "Cross-correlation is non-finite.",
    )

    require(
        abs(
            correlation
        )
        <= 1.0
        + 1e-12,
        "Cross-correlation lies outside [-1, 1].",
    )

    return (
        float(
            np.clip(
                correlation,
                -1.0,
                1.0,
            )
        ),
        len(
            buy_slice
        ),
    )


def cross_portmanteau_statistic(
    correlations,
    *,
    sample_size,
    lag_count,
):
    """Return a directional cross-correlation portmanteau statistic."""
    selected = np.asarray(
        correlations,
        dtype=np.float64,
    )

    require(
        selected.ndim == 1,
        "Portmanteau correlations must be one-dimensional.",
    )

    require(
        len(
            selected
        )
        == lag_count,
        "Portmanteau correlation count does not match its lag count.",
    )

    require(
        sample_size
        > lag_count,
        "The sample is too short for the requested portmanteau lag.",
    )

    lag_numbers = np.arange(
        1,
        lag_count + 1,
        dtype=np.float64,
    )

    statistic = float(
        sample_size
        * (
            sample_size
            + 2.0
        )
        * np.sum(
            (
                selected
                ** 2
            )
            / (
                sample_size
                - lag_numbers
            )
        )
    )

    p_value = float(
        stats.chi2.sf(
            statistic,
            df=lag_count,
        )
    )

    require(
        math.isfinite(
            statistic
        ),
        "The cross-side portmanteau statistic is non-finite.",
    )

    require(
        math.isfinite(
            p_value
        )
        and 0.0
        <= p_value
        <= 1.0,
        "The cross-side portmanteau p-value is invalid.",
    )

    return (
        statistic,
        p_value,
    )


# ------------------------------------------------------------
# Build common-grid cross-side diagnostics
# ------------------------------------------------------------

cross_correlation_records = []
portmanteau_records = []
coincidence_records = []
summary_records = []

cross_correlation_lookup = {}


for partition_name in ANALYTICAL_PARTITIONS:
    grid = HAWKES_COMMON_COUNT_GRIDS[
        partition_name
    ]

    buy_counts = require_finite_vector(
        grid.buy_counts,
        label=(
            f"{partition_name} BUY counts"
        ),
    )

    sell_counts = require_finite_vector(
        grid.sell_counts,
        label=(
            f"{partition_name} SELL counts"
        ),
    )

    expected_buy_counts = require_finite_vector(
        grid.expected_buy_counts,
        label=(
            f"{partition_name} expected BUY counts"
        ),
    )

    expected_sell_counts = require_finite_vector(
        grid.expected_sell_counts,
        label=(
            f"{partition_name} expected SELL counts"
        ),
    )

    cell_count = len(
        buy_counts
    )

    require(
        len(
            sell_counts
        )
        == cell_count,
        f"{partition_name} BUY and SELL grid lengths differ.",
    )

    require(
        len(
            expected_buy_counts
        )
        == cell_count,
        f"{partition_name} expected BUY grid length is incorrect.",
    )

    require(
        len(
            expected_sell_counts
        )
        == cell_count,
        f"{partition_name} expected SELL grid length is incorrect.",
    )

    require(
        np.all(
            buy_counts
            >= 0.0
        ),
        f"{partition_name} BUY counts contain negative values.",
    )

    require(
        np.all(
            sell_counts
            >= 0.0
        ),
        f"{partition_name} SELL counts contain negative values.",
    )

    require(
        np.all(
            expected_buy_counts
            > 0.0
        ),
        f"{partition_name} expected BUY counts are not strictly positive.",
    )

    require(
        np.all(
            expected_sell_counts
            > 0.0
        ),
        f"{partition_name} expected SELL counts are not strictly positive.",
    )

    buy_pearson_residuals = (
        buy_counts
        - expected_buy_counts
    ) / np.sqrt(
        expected_buy_counts
    )

    sell_pearson_residuals = (
        sell_counts
        - expected_sell_counts
    ) / np.sqrt(
        expected_sell_counts
    )

    require(
        np.isfinite(
            buy_pearson_residuals
        ).all(),
        f"{partition_name} BUY Pearson residuals are non-finite.",
    )

    require(
        np.isfinite(
            sell_pearson_residuals
        ).all(),
        f"{partition_name} SELL Pearson residuals are non-finite.",
    )

    centered_buy_residuals = (
        buy_pearson_residuals
        - float(
            np.mean(
                buy_pearson_residuals
            )
        )
    )

    centered_sell_residuals = (
        sell_pearson_residuals
        - float(
            np.mean(
                sell_pearson_residuals
            )
        )
    )

    require(
        float(
            np.dot(
                centered_buy_residuals,
                centered_buy_residuals,
            )
        )
        > 0.0,
        f"{partition_name} BUY Pearson residuals have zero variance.",
    )

    require(
        float(
            np.dot(
                centered_sell_residuals,
                centered_sell_residuals,
            )
        )
        > 0.0,
        f"{partition_name} SELL Pearson residuals have zero variance.",
    )

    partition_ccf_records = []

    for lag_cells in range(
        -CROSS_SIDE_MAX_LAG_CELLS,
        CROSS_SIDE_MAX_LAG_CELLS + 1,
    ):
        (
            correlation,
            effective_pair_count,
        ) = lagged_cross_correlation(
            centered_buy_residuals,
            centered_sell_residuals,
            lag=lag_cells,
        )

        naive_bound = (
            NAIVE_CROSS_CORRELATION_MULTIPLIER
            / math.sqrt(
                effective_pair_count
            )
        )

        if lag_cells > 0:
            direction = (
                "BUY_LEADS_SELL"
            )

        elif lag_cells < 0:
            direction = (
                "SELL_LEADS_BUY"
            )

        else:
            direction = (
                "CONTEMPORANEOUS"
            )

        record = {
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "event_partition": (
                partition_name
            ),
            "partition_order": int(
                PARTITION_ORDER_MAP[
                    partition_name
                ]
            ),
            "grid_width_ms": 1,
            "lag_cells": int(
                lag_cells
            ),
            "lag_milliseconds": int(
                lag_cells
            ),
            "direction": direction,
            "effective_pair_count": int(
                effective_pair_count
            ),
            "pearson_residual_cross_correlation": (
                correlation
            ),
            "absolute_cross_correlation": abs(
                correlation
            ),
            "naive_lower_bound": (
                -naive_bound
            ),
            "naive_upper_bound": (
                naive_bound
            ),
            "outside_naive_bound": bool(
                abs(
                    correlation
                )
                > naive_bound
            ),
            "formal_acceptance_gate_authorized": False,
            "simulation_envelope_pending": True,
            "status": (
                "COMPUTED_REPORT_ONLY"
            ),
        }

        cross_correlation_records.append(
            record
        )

        partition_ccf_records.append(
            record
        )

        cross_correlation_lookup[
            (
                partition_name,
                lag_cells,
            )
        ] = correlation

    partition_ccf = pd.DataFrame.from_records(
        partition_ccf_records
    )

    nonzero_partition_ccf = (
        partition_ccf.loc[
            partition_ccf[
                "lag_cells"
            ].ne(
                0
            )
        ]
    )

    maximum_nonzero_row = (
        nonzero_partition_ccf.loc[
            nonzero_partition_ccf[
                "absolute_cross_correlation"
            ].idxmax()
        ]
    )

    positive_lag_rows = (
        partition_ccf.loc[
            partition_ccf[
                "lag_cells"
            ].gt(
                0
            )
        ]
    )

    negative_lag_rows = (
        partition_ccf.loc[
            partition_ccf[
                "lag_cells"
            ].lt(
                0
            )
        ]
    )

    maximum_positive_row = (
        positive_lag_rows.loc[
            positive_lag_rows[
                "absolute_cross_correlation"
            ].idxmax()
        ]
    )

    maximum_negative_row = (
        negative_lag_rows.loc[
            negative_lag_rows[
                "absolute_cross_correlation"
            ].idxmax()
        ]
    )

    summary_records.append(
        {
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "event_partition": (
                partition_name
            ),
            "partition_order": int(
                PARTITION_ORDER_MAP[
                    partition_name
                ]
            ),
            "cell_count": int(
                cell_count
            ),
            "lag_zero_cross_correlation": float(
                cross_correlation_lookup[
                    (
                        partition_name,
                        0,
                    )
                ]
            ),
            "maximum_absolute_nonzero_cross_correlation": float(
                maximum_nonzero_row[
                    "absolute_cross_correlation"
                ]
            ),
            "lag_of_maximum_absolute_nonzero_correlation_ms": int(
                maximum_nonzero_row[
                    "lag_milliseconds"
                ]
            ),
            "maximum_buy_leads_sell_absolute_correlation": float(
                maximum_positive_row[
                    "absolute_cross_correlation"
                ]
            ),
            "buy_leads_sell_lag_of_maximum_ms": int(
                maximum_positive_row[
                    "lag_milliseconds"
                ]
            ),
            "maximum_sell_leads_buy_absolute_correlation": float(
                maximum_negative_row[
                    "absolute_cross_correlation"
                ]
            ),
            "sell_leads_buy_lag_of_maximum_ms": int(
                abs(
                    maximum_negative_row[
                        "lag_milliseconds"
                    ]
                )
            ),
            "outside_naive_bound_count_all_lags": int(
                partition_ccf[
                    "outside_naive_bound"
                ].sum()
            ),
            "outside_naive_bound_count_nonzero_lags": int(
                nonzero_partition_ccf[
                    "outside_naive_bound"
                ].sum()
            ),
            "formal_cross_excitation_gate_authorized": False,
            "simulation_envelope_pending": True,
            "status": (
                "COMPUTED_REPORT_ONLY"
            ),
        }
    )

    # --------------------------------------------------------
    # Directional portmanteau summaries
    # --------------------------------------------------------

    for lag_count in CROSS_SIDE_PORTMANTEAU_LAGS:
        buy_leads_sell_correlations = np.asarray(
            [
                cross_correlation_lookup[
                    (
                        partition_name,
                        lag,
                    )
                ]
                for lag in range(
                    1,
                    lag_count + 1,
                )
            ],
            dtype=np.float64,
        )

        sell_leads_buy_correlations = np.asarray(
            [
                cross_correlation_lookup[
                    (
                        partition_name,
                        -lag,
                    )
                ]
                for lag in range(
                    1,
                    lag_count + 1,
                )
            ],
            dtype=np.float64,
        )

        for (
            direction,
            directional_correlations,
        ) in (
            (
                "BUY_LEADS_SELL",
                buy_leads_sell_correlations,
            ),
            (
                "SELL_LEADS_BUY",
                sell_leads_buy_correlations,
            ),
        ):
            (
                q_statistic,
                p_value,
            ) = cross_portmanteau_statistic(
                directional_correlations,
                sample_size=cell_count,
                lag_count=lag_count,
            )

            portmanteau_records.append(
                {
                    "model_id": (
                        EXPECTED_SELECTED_MODEL_ID
                    ),
                    "event_partition": (
                        partition_name
                    ),
                    "partition_order": int(
                        PARTITION_ORDER_MAP[
                            partition_name
                        ]
                    ),
                    "direction": (
                        direction
                    ),
                    "maximum_lag_ms": int(
                        lag_count
                    ),
                    "cell_count": int(
                        cell_count
                    ),
                    "cross_portmanteau_q": (
                        q_statistic
                    ),
                    "chi_square_degrees_of_freedom_reference": int(
                        lag_count
                    ),
                    "chi_square_p_value_report_only": (
                        p_value
                    ),
                    "formal_p_value_is_acceptance_gate": False,
                    "simulation_envelope_pending": True,
                    "status": (
                        "COMPUTED_REPORT_ONLY"
                    ),
                }
            )

    # --------------------------------------------------------
    # Same-cell BUY/SELL coincidence diagnostics
    # --------------------------------------------------------

    buy_positive_probability = -np.expm1(
        -expected_buy_counts
    )

    sell_positive_probability = -np.expm1(
        -expected_sell_counts
    )

    independent_both_positive_probability = (
        buy_positive_probability
        * sell_positive_probability
    )

    observed_both_positive_cells = int(
        np.sum(
            (
                buy_counts
                > 0.0
            )
            & (
                sell_counts
                > 0.0
            )
        )
    )

    expected_both_positive_cells = float(
        np.sum(
            independent_both_positive_probability
        )
    )

    coincidence_variance_reference = float(
        np.sum(
            independent_both_positive_probability
            * (
                1.0
                - independent_both_positive_probability
            )
        )
    )

    require(
        coincidence_variance_reference
        > 0.0,
        (
            f"{partition_name} same-cell coincidence "
            "variance reference is nonpositive."
        ),
    )

    coincidence_z_report_only = (
        observed_both_positive_cells
        - expected_both_positive_cells
    ) / math.sqrt(
        coincidence_variance_reference
    )

    observed_count_cross_product = float(
        np.dot(
            buy_counts,
            sell_counts,
        )
    )

    expected_count_cross_product = float(
        np.dot(
            expected_buy_counts,
            expected_sell_counts,
        )
    )

    residual_cross_product = float(
        np.dot(
            buy_counts
            - expected_buy_counts,
            sell_counts
            - expected_sell_counts,
        )
    )

    coincidence_records.append(
        {
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "event_partition": (
                partition_name
            ),
            "partition_order": int(
                PARTITION_ORDER_MAP[
                    partition_name
                ]
            ),
            "grid_width_ms": 1,
            "cell_count": int(
                cell_count
            ),
            "observed_both_positive_cell_count": (
                observed_both_positive_cells
            ),
            "expected_both_positive_cell_count_under_conditional_independence": (
                expected_both_positive_cells
            ),
            "observed_minus_expected_both_positive_cells": (
                observed_both_positive_cells
                - expected_both_positive_cells
            ),
            "observed_both_positive_cell_rate": (
                observed_both_positive_cells
                / cell_count
            ),
            "expected_both_positive_cell_rate": (
                expected_both_positive_cells
                / cell_count
            ),
            "coincidence_variance_reference": (
                coincidence_variance_reference
            ),
            "coincidence_z_score_report_only": (
                coincidence_z_report_only
            ),
            "observed_buy_sell_count_cross_product": (
                observed_count_cross_product
            ),
            "expected_buy_sell_count_cross_product_under_conditional_independence": (
                expected_count_cross_product
            ),
            "observed_minus_expected_count_cross_product": (
                observed_count_cross_product
                - expected_count_cross_product
            ),
            "raw_residual_cross_product": (
                residual_cross_product
            ),
            "formal_independence_gate_authorized": False,
            "simulation_envelope_pending": True,
            "status": (
                "COMPUTED_REPORT_ONLY"
            ),
        }
    )


# ------------------------------------------------------------
# Materialize diagnostic tables
# ------------------------------------------------------------

HAWKES_CROSS_SIDE_RESIDUAL_CCF = (
    pd.DataFrame.from_records(
        cross_correlation_records
    )
    .sort_values(
        [
            "partition_order",
            "lag_cells",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

HAWKES_CROSS_SIDE_PORTMANTEAU = (
    pd.DataFrame.from_records(
        portmanteau_records
    )
    .sort_values(
        [
            "partition_order",
            "direction",
            "maximum_lag_ms",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS = (
    pd.DataFrame.from_records(
        coincidence_records
    )
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY = (
    pd.DataFrame.from_records(
        summary_records
    )
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# DEVELOPMENT-to-CALIBRATION comparison
# ------------------------------------------------------------

development_cross_summary = (
    HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY.loc[
        HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY[
            "event_partition"
        ].eq(
            "DEVELOPMENT"
        )
    ]
    .iloc[0]
)

calibration_cross_summary = (
    HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY.loc[
        HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .iloc[0]
)

development_coincidence_summary = (
    HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS.loc[
        HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS[
            "event_partition"
        ].eq(
            "DEVELOPMENT"
        )
    ]
    .iloc[0]
)

calibration_coincidence_summary = (
    HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS.loc[
        HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .iloc[0]
)

HAWKES_CROSS_SIDE_DEGRADATION_SUMMARY = pd.DataFrame(
    [
        {
            "metric": (
                "LAG_ZERO_ABSOLUTE_CROSS_CORRELATION"
            ),
            "development_value": abs(
                float(
                    development_cross_summary[
                        "lag_zero_cross_correlation"
                    ]
                )
            ),
            "calibration_value": abs(
                float(
                    calibration_cross_summary[
                        "lag_zero_cross_correlation"
                    ]
                )
            ),
            "calibration_minus_development": (
                abs(
                    float(
                        calibration_cross_summary[
                            "lag_zero_cross_correlation"
                        ]
                    )
                )
                - abs(
                    float(
                        development_cross_summary[
                            "lag_zero_cross_correlation"
                        ]
                    )
                )
            ),
            "formal_degradation_gate_authorized": False,
            "simulation_envelope_pending": True,
            "status": "DESCRIPTIVE_ONLY",
        },
        {
            "metric": (
                "MAXIMUM_ABSOLUTE_NONZERO_CROSS_CORRELATION"
            ),
            "development_value": float(
                development_cross_summary[
                    "maximum_absolute_nonzero_cross_correlation"
                ]
            ),
            "calibration_value": float(
                calibration_cross_summary[
                    "maximum_absolute_nonzero_cross_correlation"
                ]
            ),
            "calibration_minus_development": (
                float(
                    calibration_cross_summary[
                        "maximum_absolute_nonzero_cross_correlation"
                    ]
                )
                - float(
                    development_cross_summary[
                        "maximum_absolute_nonzero_cross_correlation"
                    ]
                )
            ),
            "formal_degradation_gate_authorized": False,
            "simulation_envelope_pending": True,
            "status": "DESCRIPTIVE_ONLY",
        },
        {
            "metric": (
                "ABSOLUTE_SAME_CELL_COINCIDENCE_Z_REPORT_ONLY"
            ),
            "development_value": abs(
                float(
                    development_coincidence_summary[
                        "coincidence_z_score_report_only"
                    ]
                )
            ),
            "calibration_value": abs(
                float(
                    calibration_coincidence_summary[
                        "coincidence_z_score_report_only"
                    ]
                )
            ),
            "calibration_minus_development": (
                abs(
                    float(
                        calibration_coincidence_summary[
                            "coincidence_z_score_report_only"
                        ]
                    )
                )
                - abs(
                    float(
                        development_coincidence_summary[
                            "coincidence_z_score_report_only"
                        ]
                    )
                )
            ),
            "formal_degradation_gate_authorized": False,
            "simulation_envelope_pending": True,
            "status": "DESCRIPTIVE_ONLY",
        },
    ]
)


# ------------------------------------------------------------
# Verification matrix
# ------------------------------------------------------------

expected_ccf_row_count = (
    len(
        ANALYTICAL_PARTITIONS
    )
    * (
        2
        * CROSS_SIDE_MAX_LAG_CELLS
        + 1
    )
)

expected_portmanteau_row_count = (
    len(
        ANALYTICAL_PARTITIONS
    )
    * 2
    * len(
        CROSS_SIDE_PORTMANTEAU_LAGS
    )
)

require(
    len(
        HAWKES_CROSS_SIDE_RESIDUAL_CCF
    )
    == expected_ccf_row_count,
    "The cross-side CCF row count is incorrect.",
)

require(
    len(
        HAWKES_CROSS_SIDE_PORTMANTEAU
    )
    == expected_portmanteau_row_count,
    "The cross-side portmanteau row count is incorrect.",
)

require(
    len(
        HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS
    )
    == len(
        ANALYTICAL_PARTITIONS
    ),
    "The same-cell coincidence diagnostic row count is incorrect.",
)

require(
    HAWKES_CROSS_SIDE_RESIDUAL_CCF[
        "pearson_residual_cross_correlation"
    ].between(
        -1.0,
        1.0,
        inclusive="both",
    ).all(),
    "A cross-side residual correlation lies outside [-1, 1].",
)

require(
    np.isfinite(
        HAWKES_CROSS_SIDE_PORTMANTEAU[
            [
                "cross_portmanteau_q",
                "chi_square_p_value_report_only",
            ]
        ].to_numpy(
            dtype=np.float64
        )
    ).all(),
    "A cross-side portmanteau value is non-finite.",
)

require(
    np.isfinite(
        HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS[
            [
                "expected_both_positive_cell_count_under_conditional_independence",
                "coincidence_z_score_report_only",
                "raw_residual_cross_product",
            ]
        ].to_numpy(
            dtype=np.float64
        )
    ).all(),
    "A same-cell coincidence diagnostic is non-finite.",
)

require(
    not HAWKES_CROSS_SIDE_RESIDUAL_CCF[
        "formal_acceptance_gate_authorized"
    ].any(),
    "A naive cross-correlation bound was incorrectly made an acceptance gate.",
)

require(
    not HAWKES_CROSS_SIDE_PORTMANTEAU[
        "formal_p_value_is_acceptance_gate"
    ].any(),
    "A cross-side portmanteau p-value was incorrectly made an acceptance gate.",
)

require(
    not HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS[
        "formal_independence_gate_authorized"
    ].any(),
    "A same-cell independence diagnostic was incorrectly made an acceptance gate.",
)

require(
    HAWKES_CROSS_SIDE_RESIDUAL_CCF[
        "simulation_envelope_pending"
    ].all(),
    "A cross-side residual diagnostic bypassed simulation envelopes.",
)

require(
    HAWKES_CROSS_SIDE_PORTMANTEAU[
        "simulation_envelope_pending"
    ].all(),
    "A cross-side portmanteau diagnostic bypassed simulation envelopes.",
)

require(
    HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS[
        "simulation_envelope_pending"
    ].all(),
    "A coincidence diagnostic bypassed simulation envelopes.",
)


HAWKES_CROSS_SIDE_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "common_grid_partitions_present"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "cross_correlation_lags_minus_50_to_plus_50_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "directional_portmanteau_lags_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "same_cell_coincidence_diagnostics_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "all_cross_correlations_finite_and_bounded"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "all_portmanteau_statistics_finite"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "all_coincidence_metrics_finite"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "naive_bounds_and_p_values_report_only"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "cross_excitation_conclusion_not_yet_authorized"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "simulation_envelopes_still_required"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    HAWKES_CROSS_SIDE_VERIFICATION[
        "passed"
    ].all(),
    "At least one cross-side residual verification failed.",
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

CROSS_SIDE_RESIDUAL_DIAGNOSTICS_BUILT = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY
)

display(
    HAWKES_CROSS_SIDE_PORTMANTEAU
)

display(
    HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS
)

display(
    HAWKES_CROSS_SIDE_DEGRADATION_SUMMARY
)

display(
    HAWKES_CROSS_SIDE_VERIFICATION
)

print(
    "Common-grid BUY/SELL Pearson-residual cross-correlations were "
    "computed from -50 ms through +50 ms for DEVELOPMENT and locked "
    "CALIBRATION. Directional BUY-leading-SELL and SELL-leading-BUY "
    "portmanteau summaries and same-cell coincidence diagnostics are "
    "available. Naive bounds, chi-square references, z-scores, and "
    "p-values remain report-only. No conclusion about omitted cross-"
    "excitation is authorized until coarsening-aware parametric "
    "simulation envelopes are constructed. No filesystem writes were "
    "performed."
)

,model_id,event_partition,partition_order,cell_count,lag_zero_cross_correlation,maximum_absolute_nonzero_cross_correlation,lag_of_maximum_absolute_nonzero_correlation_ms,maximum_buy_leads_sell_absolute_correlation,buy_leads_sell_lag_of_maximum_ms,maximum_sell_leads_buy_absolute_correlation,sell_leads_buy_lag_of_maximum_ms,outside_naive_bound_count_all_lags,outside_naive_bound_count_nonzero_lags,formal_cross_excitation_gate_authorized,simulation_envelope_pending,status
0,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,1801860,0.004470133242,0.003186212527,26,0.003186212527,26,0.00295836645,49,10,9,False,True,COMPUTED_REPORT_ONLY
1,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,720300,0.0010996749,0.003683879649,7,0.003683879649,7,0.003459608473,41,7,7,False,True,COMPUTED_REPORT_ONLY


,model_id,event_partition,partition_order,direction,maximum_lag_ms,cell_count,cross_portmanteau_q,chi_square_degrees_of_freedom_reference,chi_square_p_value_report_only,formal_p_value_is_acceptance_gate,simulation_envelope_pending,status
0,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,BUY_LEADS_SELL,10,1801860,24.03813517,10,0.007499809317,False,True,COMPUTED_REPORT_ONLY
1,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,BUY_LEADS_SELL,20,1801860,35.95096086,20,0.01558648068,False,True,COMPUTED_REPORT_ONLY
2,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,BUY_LEADS_SELL,50,1801860,90.09854803,50,0.0004385612036,False,True,COMPUTED_REPORT_ONLY
3,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,SELL_LEADS_BUY,10,1801860,8.8782928,10,0.5436917661,False,True,COMPUTED_REPORT_ONLY
4,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,SELL_LEADS_BUY,20,1801860,12.97534186,20,0.8784395091,False,True,COMPUTED_REPORT_ONLY
5,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,SELL_LEADS_BUY,50,1801860,53.64350595,50,0.3364416926,False,True,COMPUTED_REPORT_ONLY
6,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,BUY_LEADS_SELL,10,720300,25.01020225,10,0.005326200893,False,True,COMPUTED_REPORT_ONLY
7,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,BUY_LEADS_SELL,20,720300,35.74689465,20,0.01646862431,False,True,COMPUTED_REPORT_ONLY
8,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,BUY_LEADS_SELL,50,720300,74.21261998,50,0.01470583474,False,True,COMPUTED_REPORT_ONLY
9,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,SELL_LEADS_BUY,10,720300,6.268421992,10,0.7922299376,False,True,COMPUTED_REPORT_ONLY


,model_id,event_partition,partition_order,grid_width_ms,cell_count,observed_both_positive_cell_count,expected_both_positive_cell_count_under_conditional_independence,observed_minus_expected_both_positive_cells,observed_both_positive_cell_rate,expected_both_positive_cell_rate,coincidence_variance_reference,coincidence_z_score_report_only,observed_buy_sell_count_cross_product,expected_buy_sell_count_cross_product_under_conditional_independence,observed_minus_expected_count_cross_product,raw_residual_cross_product,formal_independence_gate_authorized,simulation_envelope_pending,status
0,H1_DIAGONAL_SHARED_DECAY,DEVELOPMENT,1,1,1801860,26,6.881579599,19.1184204,1.442953393e-05,3.819153319e-06,6.881461897,7.288055185,34,6.906298225,27.09370178,25.69798996,False,True,COMPUTED_REPORT_ONLY
1,H1_DIAGONAL_SHARED_DECAY,CALIBRATION,2,1,720300,6,2.650813152,3.349186848,8.329862557e-06,3.680151536e-06,2.650776687,2.057087693,7,2.660150708,4.339849292,4.474160509,False,True,COMPUTED_REPORT_ONLY


,metric,development_value,calibration_value,calibration_minus_development,formal_degradation_gate_authorized,simulation_envelope_pending,status
0,LAG_ZERO_ABSOLUTE_CROSS_CORRELATION,0.004470133242,0.0010996749,-0.003370458342,False,True,DESCRIPTIVE_ONLY
1,MAXIMUM_ABSOLUTE_NONZERO_CROSS_CORRELATION,0.003186212527,0.003683879649,0.0004976671219,False,True,DESCRIPTIVE_ONLY
2,ABSOLUTE_SAME_CELL_COINCIDENCE_Z_REPORT_ONLY,7.288055185,2.057087693,-5.230967492,False,True,DESCRIPTIVE_ONLY


,check_name,passed
0,common_grid_partitions_present,True
1,cross_correlation_lags_minus_50_to_plus_50_com...,True
2,directional_portmanteau_lags_complete,True
3,same_cell_coincidence_diagnostics_complete,True
4,all_cross_correlations_finite_and_bounded,True
5,all_portmanteau_statistics_finite,True
6,all_coincidence_metrics_finite,True
7,naive_bounds_and_p_values_report_only,True
8,cross_excitation_conclusion_not_yet_authorized,True
9,simulation_envelopes_still_required,True


Common-grid BUY/SELL Pearson-residual cross-correlations were computed from -50 ms through +50 ms for DEVELOPMENT and locked CALIBRATION. Directional BUY-leading-SELL and SELL-leading-BUY portmanteau summaries and same-cell coincidence diagnostics are available. Naive bounds, chi-square references, z-scores, and p-values remain report-only. No conclusion about omitted cross-excitation is authorized until coarsening-aware parametric simulation envelopes are constructed. No filesystem writes were performed.


In [15]:
# ============================================================
# Freeze and validate the coarsening-aware H1 simulation engine
#
# The simulator generates continuous-time events from the frozen
# diagonal H1 model, carries the terminal DEVELOPMENT excitation
# state into CALIBRATION, and then applies the authorized
# partition-relative 1 ms floor-coarsening contract.
#
# This cell validates the engine only. It does not yet construct
# the full parametric diagnostic envelopes.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "CROSS_SIDE_RESIDUAL_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Cross-side residual diagnostics have not been built.",
)

require(
    bool(
        globals().get(
            "TIME_RESCALING_RESIDUALS_BUILT",
            False,
        )
    ),
    "Observed time-rescaling residuals have not been built.",
)

require(
    bool(
        globals().get(
            "SIDE_MARK_MULTIPLICITY_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Observed side-mark and multiplicity diagnostics are unavailable.",
)

require(
    "HAWKES_COMMON_COUNT_GRIDS" in globals(),
    "The authoritative common 1 ms grids are unavailable.",
)

require(
    not bool(
        globals().get(
            "COARSENING_AWARE_SIMULATION_ENGINE_VALIDATED",
            False,
        )
    ),
    "The coarsening-aware simulation engine is already validated.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Frozen simulation contract
# ------------------------------------------------------------

SIMULATION_MODEL_ID: Final[str] = (
    EXPECTED_SELECTED_MODEL_ID
)

SIMULATION_MODEL_FAMILY: Final[str] = (
    EXPECTED_SELECTED_MODEL_FAMILY
)

SIMULATION_INITIAL_STATE_SOURCE: Final[str] = (
    "ZERO_EXCITATION_AT_DEVELOPMENT_LEFT_BOUNDARY"
)

SIMULATION_CALIBRATION_STATE_SOURCE: Final[str] = (
    "SIMULATED_TERMINAL_DEVELOPMENT_STATE"
)

SIMULATION_TIMESTAMP_GENERATION: Final[str] = (
    "CONTINUOUS_TIME_OGATA_THINNING"
)

SIMULATION_COARSENING_METHOD: Final[str] = (
    "PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID"
)

SIMULATION_BATCHING_METHOD: Final[str] = (
    "AGGREGATE_BUY_AND_SELL_COUNTS_AT_COARSENED_TIME"
)

SIMULATION_WITHIN_BATCH_ORDER_IMPOSED: Final[bool] = False
SIMULATION_TIMESTAMP_JITTER_PERFORMED: Final[bool] = False
SIMULATION_EVENT_REMOVAL_PERFORMED: Final[bool] = False
SIMULATION_CALIBRATION_PARAMETER_UPDATES: Final[int] = 0

SIMULATION_PILOT_REPLICATES: Final[int] = 5
SIMULATION_PILOT_SEED_OFFSET: Final[int] = 80_000
SIMULATION_MAX_ACCEPTED_EVENTS_PER_PARTITION: Final[int] = 1_000_000

require(
    COUNT_GRID_WIDTH_NS
    == 1_000_000,
    "The simulation coarsening width must be exactly one millisecond.",
)

require(
    DEVELOPMENT_END_EXCLUSIVE_NS
    == CALIBRATION_START_NS,
    "DEVELOPMENT and CALIBRATION are not contiguous.",
)

require(
    FROZEN_H1_PARAMETERS.model_id
    == SIMULATION_MODEL_ID,
    "The simulation model identity does not match frozen H1.",
)


# ------------------------------------------------------------
# Immutable simulation results
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class SimulatedHawkesPartitionResult:
    """One simulated partition and its exact terminal state."""

    event_partition: str
    partition_start_ns: int
    partition_end_exclusive_ns: int
    initial_state: FrozenHawkesState
    terminal_state: FrozenHawkesState
    proposal_count: int
    rejection_count: int
    accepted_event_count: int
    events: pd.DataFrame


@dataclass(frozen=True, slots=True)
class SimulatedHawkesPathResult:
    """Continuous DEVELOPMENT-to-CALIBRATION simulated path."""

    seed: int
    development: SimulatedHawkesPartitionResult
    calibration: SimulatedHawkesPartitionResult
    events: pd.DataFrame


# ------------------------------------------------------------
# Continuous-time Ogata thinning engine
# ------------------------------------------------------------

SIMULATED_EVENT_COLUMNS: Final[tuple[str, ...]] = (
    "simulation_seed",
    "event_partition",
    "event_partition_order",
    "partition_event_number",
    "global_simulated_event_number",
    "event_time_ns",
    "continuous_time_seconds_from_partition_start",
    "event_side",
    "event_side_code",
    "lambda_buy_pre",
    "lambda_sell_pre",
    "buy_excitation_pre",
    "sell_excitation_pre",
    "buy_excitation_post",
    "sell_excitation_post",
)


def empty_simulated_event_frame() -> pd.DataFrame:
    """Return an empty simulated-event table with stable dtypes."""
    return pd.DataFrame(
        {
            "simulation_seed": pd.Series(
                dtype="int64"
            ),
            "event_partition": pd.Series(
                dtype="string"
            ),
            "event_partition_order": pd.Series(
                dtype="int64"
            ),
            "partition_event_number": pd.Series(
                dtype="int64"
            ),
            "global_simulated_event_number": pd.Series(
                dtype="int64"
            ),
            "event_time_ns": pd.Series(
                dtype="int64"
            ),
            "continuous_time_seconds_from_partition_start": pd.Series(
                dtype="float64"
            ),
            "event_side": pd.Series(
                dtype="string"
            ),
            "event_side_code": pd.Series(
                dtype="int64"
            ),
            "lambda_buy_pre": pd.Series(
                dtype="float64"
            ),
            "lambda_sell_pre": pd.Series(
                dtype="float64"
            ),
            "buy_excitation_pre": pd.Series(
                dtype="float64"
            ),
            "sell_excitation_pre": pd.Series(
                dtype="float64"
            ),
            "buy_excitation_post": pd.Series(
                dtype="float64"
            ),
            "sell_excitation_post": pd.Series(
                dtype="float64"
            ),
        },
        columns=list(
            SIMULATED_EVENT_COLUMNS
        ),
    )


def simulate_diagonal_hawkes_partition(
    parameters: FrozenDiagonalHawkesParameters,
    *,
    event_partition: str,
    partition_start_ns: int,
    partition_end_exclusive_ns: int,
    initial_state: FrozenHawkesState,
    simulation_seed: int,
    rng: np.random.Generator,
) -> SimulatedHawkesPartitionResult:
    """
    Simulate one diagonal bivariate exponential Hawkes interval.

    The current total intensity is a valid thinning upper bound
    because both excitation states decay monotonically between
    accepted events.
    """
    partition_name = str(
        event_partition
    )

    start_ns = int(
        partition_start_ns
    )

    end_ns = int(
        partition_end_exclusive_ns
    )

    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        f"Unauthorized simulation partition: {partition_name}",
    )

    require(
        end_ns
        > start_ns,
        f"{partition_name} simulation interval has nonpositive duration.",
    )

    require(
        initial_state.state_time_ns
        == start_ns,
        (
            f"{partition_name} initial state is not expressed "
            "at its partition boundary."
        ),
    )

    require(
        math.isfinite(
            initial_state.buy_excitation
        )
        and initial_state.buy_excitation
        >= 0.0,
        f"{partition_name} initial BUY excitation is invalid.",
    )

    require(
        math.isfinite(
            initial_state.sell_excitation
        )
        and initial_state.sell_excitation
        >= 0.0,
        f"{partition_name} initial SELL excitation is invalid.",
    )

    duration_ns = (
        end_ns
        - start_ns
    )

    duration_seconds = (
        duration_ns
        / NANOSECONDS_PER_SECOND
    )

    current_time_seconds = 0.0

    buy_excitation = float(
        initial_state.buy_excitation
    )

    sell_excitation = float(
        initial_state.sell_excitation
    )

    proposal_count = 0
    rejection_count = 0

    event_records: list[
        dict[str, Any]
    ] = []

    while True:
        lambda_buy_upper = (
            parameters.mu_buy
            + buy_excitation
        )

        lambda_sell_upper = (
            parameters.mu_sell
            + sell_excitation
        )

        total_intensity_upper = (
            lambda_buy_upper
            + lambda_sell_upper
        )

        require(
            math.isfinite(
                total_intensity_upper
            )
            and total_intensity_upper
            > 0.0,
            (
                f"{partition_name} simulation produced an "
                "invalid thinning upper bound."
            ),
        )

        waiting_time_seconds = float(
            rng.exponential(
                scale=(
                    1.0
                    / total_intensity_upper
                )
            )
        )

        require(
            math.isfinite(
                waiting_time_seconds
            )
            and waiting_time_seconds
            >= 0.0,
            (
                f"{partition_name} simulation produced an "
                "invalid waiting time."
            ),
        )

        candidate_time_seconds = (
            current_time_seconds
            + waiting_time_seconds
        )

        if candidate_time_seconds >= duration_seconds:
            remaining_seconds = (
                duration_seconds
                - current_time_seconds
            )

            require(
                remaining_seconds
                >= 0.0,
                (
                    f"{partition_name} terminal decay interval "
                    "is negative."
                ),
            )

            buy_excitation *= math.exp(
                -parameters.beta_buy
                * remaining_seconds
            )

            sell_excitation *= math.exp(
                -parameters.beta_sell
                * remaining_seconds
            )

            current_time_seconds = (
                duration_seconds
            )

            break

        elapsed_seconds = (
            candidate_time_seconds
            - current_time_seconds
        )

        require(
            elapsed_seconds
            >= 0.0,
            (
                f"{partition_name} simulation encountered "
                "negative elapsed time."
            ),
        )

        buy_excitation *= math.exp(
            -parameters.beta_buy
            * elapsed_seconds
        )

        sell_excitation *= math.exp(
            -parameters.beta_sell
            * elapsed_seconds
        )

        current_time_seconds = (
            candidate_time_seconds
        )

        lambda_buy_candidate = (
            parameters.mu_buy
            + buy_excitation
        )

        lambda_sell_candidate = (
            parameters.mu_sell
            + sell_excitation
        )

        total_intensity_candidate = (
            lambda_buy_candidate
            + lambda_sell_candidate
        )

        require(
            total_intensity_candidate
            <= total_intensity_upper
            * (
                1.0
                + 1e-12
            ),
            (
                f"{partition_name} thinning upper bound "
                "was violated."
            ),
        )

        acceptance_probability = (
            total_intensity_candidate
            / total_intensity_upper
        )

        require(
            math.isfinite(
                acceptance_probability
            )
            and 0.0
            <= acceptance_probability
            <= 1.0
            + 1e-12,
            (
                f"{partition_name} simulation produced an "
                "invalid acceptance probability."
            ),
        )

        proposal_count += 1

        if float(
            rng.random()
        ) > min(
            1.0,
            acceptance_probability,
        ):
            rejection_count += 1
            continue

        buy_probability = (
            lambda_buy_candidate
            / total_intensity_candidate
        )

        require(
            0.0
            < buy_probability
            < 1.0,
            (
                f"{partition_name} simulation produced an "
                "invalid BUY mark probability."
            ),
        )

        buy_excitation_pre = (
            buy_excitation
        )

        sell_excitation_pre = (
            sell_excitation
        )

        if float(
            rng.random()
        ) < buy_probability:
            event_side = "BUY"
            event_side_code = 0

            buy_excitation += (
                parameters.kappa_buy
                * parameters.beta_buy
            )

        else:
            event_side = "SELL"
            event_side_code = 1

            sell_excitation += (
                parameters.kappa_sell
                * parameters.beta_sell
            )

        event_offset_ns = int(
            math.floor(
                current_time_seconds
                * NANOSECONDS_PER_SECOND
            )
        )

        require(
            0
            <= event_offset_ns
            < duration_ns,
            (
                f"{partition_name} simulated event offset "
                "lies outside the partition."
            ),
        )

        event_time_ns = (
            start_ns
            + event_offset_ns
        )

        require(
            start_ns
            <= event_time_ns
            < end_ns,
            (
                f"{partition_name} simulated event time "
                "lies outside the partition."
            ),
        )

        partition_event_number = (
            len(
                event_records
            )
            + 1
        )

        event_records.append(
            {
                "simulation_seed": int(
                    simulation_seed
                ),
                "event_partition": (
                    partition_name
                ),
                "event_partition_order": int(
                    PARTITION_ORDER_MAP[
                        partition_name
                    ]
                ),
                "partition_event_number": int(
                    partition_event_number
                ),
                "global_simulated_event_number": 0,
                "event_time_ns": int(
                    event_time_ns
                ),
                "continuous_time_seconds_from_partition_start": float(
                    current_time_seconds
                ),
                "event_side": (
                    event_side
                ),
                "event_side_code": int(
                    event_side_code
                ),
                "lambda_buy_pre": float(
                    lambda_buy_candidate
                ),
                "lambda_sell_pre": float(
                    lambda_sell_candidate
                ),
                "buy_excitation_pre": float(
                    buy_excitation_pre
                ),
                "sell_excitation_pre": float(
                    sell_excitation_pre
                ),
                "buy_excitation_post": float(
                    buy_excitation
                ),
                "sell_excitation_post": float(
                    sell_excitation
                ),
            }
        )

        require(
            len(
                event_records
            )
            <= SIMULATION_MAX_ACCEPTED_EVENTS_PER_PARTITION,
            (
                f"{partition_name} simulation exceeded the "
                "accepted-event safety limit."
            ),
        )

    terminal_state = FrozenHawkesState(
        state_time_ns=end_ns,
        buy_excitation=float(
            buy_excitation
        ),
        sell_excitation=float(
            sell_excitation
        ),
    )

    if event_records:
        events = pd.DataFrame.from_records(
            event_records,
            columns=list(
                SIMULATED_EVENT_COLUMNS
            ),
        )
    else:
        events = empty_simulated_event_frame()

    require(
        len(
            events
        )
        == len(
            event_records
        ),
        f"{partition_name} simulation lost events.",
    )

    if not events.empty:
        events = (
            events.sort_values(
                [
                    "continuous_time_seconds_from_partition_start",
                    "partition_event_number",
                ],
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        require(
            events[
                "continuous_time_seconds_from_partition_start"
            ].is_monotonic_increasing,
            (
                f"{partition_name} continuous event times "
                "are not nondecreasing."
            ),
        )

        require(
            events[
                "event_time_ns"
            ].is_monotonic_increasing,
            (
                f"{partition_name} integer event times "
                "are not nondecreasing."
            ),
        )

        require(
            events[
                "event_side"
            ].isin(
                EVENT_SIDES
            ).all(),
            f"{partition_name} simulation produced an invalid side.",
        )

    require(
        proposal_count
        >= len(
            events
        ),
        f"{partition_name} accepted more events than proposals.",
    )

    require(
        rejection_count
        == proposal_count
        - len(
            events
        ),
        f"{partition_name} proposal accounting does not reconcile.",
    )

    return SimulatedHawkesPartitionResult(
        event_partition=partition_name,
        partition_start_ns=start_ns,
        partition_end_exclusive_ns=end_ns,
        initial_state=initial_state,
        terminal_state=terminal_state,
        proposal_count=int(
            proposal_count
        ),
        rejection_count=int(
            rejection_count
        ),
        accepted_event_count=len(
            events
        ),
        events=events,
    )


def simulate_full_frozen_h1_path(
    *,
    simulation_seed: int,
) -> SimulatedHawkesPathResult:
    """Simulate DEVELOPMENT and inherited-state CALIBRATION."""
    seed_value = int(
        simulation_seed
    )

    rng = np.random.default_rng(
        seed_value
    )

    development_initial_state = FrozenHawkesState(
        state_time_ns=(
            DEVELOPMENT_START_NS
        ),
        buy_excitation=0.0,
        sell_excitation=0.0,
    )

    development_result = (
        simulate_diagonal_hawkes_partition(
            FROZEN_H1_PARAMETERS,
            event_partition="DEVELOPMENT",
            partition_start_ns=(
                DEVELOPMENT_START_NS
            ),
            partition_end_exclusive_ns=(
                DEVELOPMENT_END_EXCLUSIVE_NS
            ),
            initial_state=(
                development_initial_state
            ),
            simulation_seed=seed_value,
            rng=rng,
        )
    )

    require(
        development_result
        .terminal_state
        .state_time_ns
        == CALIBRATION_START_NS,
        (
            "The simulated DEVELOPMENT terminal state is not "
            "expressed at the CALIBRATION boundary."
        ),
    )

    calibration_result = (
        simulate_diagonal_hawkes_partition(
            FROZEN_H1_PARAMETERS,
            event_partition="CALIBRATION",
            partition_start_ns=(
                CALIBRATION_START_NS
            ),
            partition_end_exclusive_ns=(
                CALIBRATION_END_EXCLUSIVE_NS
            ),
            initial_state=(
                development_result
                .terminal_state
            ),
            simulation_seed=seed_value,
            rng=rng,
        )
    )

    combined_records: list[
        dict[str, Any]
    ] = []

    for frame in (
        development_result.events,
        calibration_result.events,
    ):
        combined_records.extend(
            frame.to_dict(
                orient="records"
            )
        )

    if combined_records:
        combined_events = pd.DataFrame.from_records(
            combined_records,
            columns=list(
                SIMULATED_EVENT_COLUMNS
            ),
        )

        combined_events = (
            combined_events.sort_values(
                [
                    "event_partition_order",
                    "continuous_time_seconds_from_partition_start",
                    "partition_event_number",
                ],
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        combined_events[
            "global_simulated_event_number"
        ] = np.arange(
            1,
            len(
                combined_events
            )
            + 1,
            dtype=np.int64,
        )

    else:
        combined_events = (
            empty_simulated_event_frame()
        )

    require(
        len(
            combined_events
        )
        == (
            development_result
            .accepted_event_count
            + calibration_result
            .accepted_event_count
        ),
        "The combined simulated path does not conserve events.",
    )

    require(
        not set(
            combined_events[
                "event_partition"
            ]
        ).intersection(
            PROTECTED_PARTITIONS
        ),
        "The simulated path contains a protected partition.",
    )

    return SimulatedHawkesPathResult(
        seed=seed_value,
        development=development_result,
        calibration=calibration_result,
        events=combined_events,
    )


# ------------------------------------------------------------
# Authorized partition-relative 1 ms coarsening
# ------------------------------------------------------------

SIMULATED_COARSENED_BATCH_COLUMNS: Final[
    tuple[str, ...]
] = (
    "simulation_seed",
    "event_partition",
    "event_partition_order",
    "event_time_ns",
    "buy_event_count",
    "sell_event_count",
    "batch_event_count",
    "source_continuous_event_count",
    "source_exact_ns_batch_count",
    "first_source_event_time_ns",
    "last_source_event_time_ns",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "timestamp_floor_distance_max_ns",
    "timestamp_mutation_performed",
    "event_removal_performed",
    "within_batch_order_imposed",
)


def empty_simulated_batch_frame() -> pd.DataFrame:
    """Return an empty coarsened-batch table with stable dtypes."""
    return pd.DataFrame(
        {
            "simulation_seed": pd.Series(
                dtype="int64"
            ),
            "event_partition": pd.Series(
                dtype="string"
            ),
            "event_partition_order": pd.Series(
                dtype="int64"
            ),
            "event_time_ns": pd.Series(
                dtype="int64"
            ),
            "buy_event_count": pd.Series(
                dtype="int64"
            ),
            "sell_event_count": pd.Series(
                dtype="int64"
            ),
            "batch_event_count": pd.Series(
                dtype="int64"
            ),
            "source_continuous_event_count": pd.Series(
                dtype="int64"
            ),
            "source_exact_ns_batch_count": pd.Series(
                dtype="int64"
            ),
            "first_source_event_time_ns": pd.Series(
                dtype="int64"
            ),
            "last_source_event_time_ns": pd.Series(
                dtype="int64"
            ),
            "mixed_side_batch_flag": pd.Series(
                dtype="bool"
            ),
            "simultaneous_batch_required_flag": pd.Series(
                dtype="bool"
            ),
            "timestamp_floor_distance_max_ns": pd.Series(
                dtype="int64"
            ),
            "timestamp_mutation_performed": pd.Series(
                dtype="bool"
            ),
            "event_removal_performed": pd.Series(
                dtype="bool"
            ),
            "within_batch_order_imposed": pd.Series(
                dtype="bool"
            ),
        },
        columns=list(
            SIMULATED_COARSENED_BATCH_COLUMNS
        ),
    )


def coarsen_simulated_partition_to_1ms(
    simulated_events: pd.DataFrame,
    *,
    event_partition: str,
    partition_start_ns: int,
    partition_end_exclusive_ns: int,
    simulation_seed: int,
) -> pd.DataFrame:
    """Floor simulated timestamps to the partition-relative 1 ms grid."""
    partition_name = str(
        event_partition
    )

    start_ns = int(
        partition_start_ns
    )

    end_ns = int(
        partition_end_exclusive_ns
    )

    source = (
        simulated_events.loc[
            simulated_events[
                "event_partition"
            ].eq(
                partition_name
            ),
            [
                "event_time_ns",
                "event_side",
            ],
        ]
        .copy()
        .sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    if source.empty:
        return empty_simulated_batch_frame()

    source[
        "event_time_ns"
    ] = parse_exact_int64(
        source[
            "event_time_ns"
        ],
        label=(
            f"SIMULATED_{partition_name}."
            "event_time_ns"
        ),
    )

    source[
        "event_side"
    ] = (
        source[
            "event_side"
        ]
        .astype(
            "string"
        )
        .str.strip()
        .str.upper()
    )

    require(
        source[
            "event_side"
        ].isin(
            EVENT_SIDES
        ).all(),
        (
            f"{partition_name} simulated coarsening input "
            "contains an invalid side."
        ),
    )

    source_event_times_ns = (
        source[
            "event_time_ns"
        ].to_numpy(
            dtype=np.int64
        )
    )

    require(
        np.all(
            source_event_times_ns
            >= start_ns
        ),
        (
            f"{partition_name} simulated coarsening input "
            "contains an event before partition start."
        ),
    )

    require(
        np.all(
            source_event_times_ns
            < end_ns
        ),
        (
            f"{partition_name} simulated coarsening input "
            "contains an event outside the partition."
        ),
    )

    relative_time_ns = (
        source_event_times_ns
        - np.int64(
            start_ns
        )
    )

    coarsened_relative_time_ns = (
        relative_time_ns
        // np.int64(
            COUNT_GRID_WIDTH_NS
        )
    ) * np.int64(
        COUNT_GRID_WIDTH_NS
    )

    coarsened_event_time_ns = (
        np.int64(
            start_ns
        )
        + coarsened_relative_time_ns
    )

    require(
        np.all(
            coarsened_event_time_ns
            <= source_event_times_ns
        ),
        "Floor coarsening moved a simulated event forward.",
    )

    require(
        np.all(
            coarsened_event_time_ns
            >= start_ns
        )
        and np.all(
            coarsened_event_time_ns
            < end_ns
        ),
        "Floor coarsening moved a simulated event outside its partition.",
    )

    source[
        "coarsened_event_time_ns"
    ] = (
        coarsened_event_time_ns
    )

    source[
        "buy_event_count"
    ] = (
        source[
            "event_side"
        ]
        .eq(
            "BUY"
        )
        .astype(
            np.int64
        )
    )

    source[
        "sell_event_count"
    ] = (
        source[
            "event_side"
        ]
        .eq(
            "SELL"
        )
        .astype(
            np.int64
        )
    )

    source[
        "timestamp_floor_distance_ns"
    ] = (
        source_event_times_ns
        - coarsened_event_time_ns
    )

    coarsened = (
        source.groupby(
            "coarsened_event_time_ns",
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            buy_event_count=(
                "buy_event_count",
                "sum",
            ),
            sell_event_count=(
                "sell_event_count",
                "sum",
            ),
            source_continuous_event_count=(
                "event_time_ns",
                "size",
            ),
            source_exact_ns_batch_count=(
                "event_time_ns",
                "nunique",
            ),
            first_source_event_time_ns=(
                "event_time_ns",
                "min",
            ),
            last_source_event_time_ns=(
                "event_time_ns",
                "max",
            ),
            timestamp_floor_distance_max_ns=(
                "timestamp_floor_distance_ns",
                "max",
            ),
        )
        .reset_index()
        .rename(
            columns={
                "coarsened_event_time_ns": (
                    "event_time_ns"
                )
            }
        )
    )

    for integer_column in (
        "event_time_ns",
        "buy_event_count",
        "sell_event_count",
        "source_continuous_event_count",
        "source_exact_ns_batch_count",
        "first_source_event_time_ns",
        "last_source_event_time_ns",
        "timestamp_floor_distance_max_ns",
    ):
        coarsened[
            integer_column
        ] = parse_exact_int64(
            coarsened[
                integer_column
            ],
            label=(
                f"COARSENED_{partition_name}."
                f"{integer_column}"
            ),
        )

    coarsened[
        "batch_event_count"
    ] = (
        coarsened[
            "buy_event_count"
        ]
        + coarsened[
            "sell_event_count"
        ]
    )

    coarsened[
        "mixed_side_batch_flag"
    ] = (
        coarsened[
            "buy_event_count"
        ].gt(
            0
        )
        & coarsened[
            "sell_event_count"
        ].gt(
            0
        )
    )

    coarsened[
        "simultaneous_batch_required_flag"
    ] = (
        coarsened[
            "batch_event_count"
        ].gt(
            1
        )
    )

    coarsened.insert(
        0,
        "simulation_seed",
        int(
            simulation_seed
        ),
    )

    coarsened.insert(
        1,
        "event_partition",
        partition_name,
    )

    coarsened.insert(
        2,
        "event_partition_order",
        int(
            PARTITION_ORDER_MAP[
                partition_name
            ]
        ),
    )

    coarsened[
        "timestamp_mutation_performed"
    ] = False

    coarsened[
        "event_removal_performed"
    ] = False

    coarsened[
        "within_batch_order_imposed"
    ] = False

    coarsened = coarsened.loc[
        :,
        list(
            SIMULATED_COARSENED_BATCH_COLUMNS
        ),
    ]

    coarsened = (
        coarsened.sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        coarsened[
            "event_time_ns"
        ].is_unique,
        (
            f"{partition_name} coarsened simulated "
            "batch times are not unique."
        ),
    )

    require(
        coarsened[
            "event_time_ns"
        ].is_monotonic_increasing,
        (
            f"{partition_name} coarsened simulated "
            "batches are not chronological."
        ),
    )

    require(
        int(
            coarsened[
                "batch_event_count"
            ].sum()
        )
        == len(
            source
        ),
        (
            f"{partition_name} coarsening does not "
            "conserve simulated events."
        ),
    )

    require(
        int(
            coarsened[
                "buy_event_count"
            ].sum()
        )
        == int(
            source[
                "buy_event_count"
            ].sum()
        ),
        (
            f"{partition_name} coarsening does not "
            "conserve BUY events."
        ),
    )

    require(
        int(
            coarsened[
                "sell_event_count"
            ].sum()
        )
        == int(
            source[
                "sell_event_count"
            ].sum()
        ),
        (
            f"{partition_name} coarsening does not "
            "conserve SELL events."
        ),
    )

    require(
        coarsened[
            "timestamp_floor_distance_max_ns"
        ].between(
            0,
            COUNT_GRID_WIDTH_NS - 1,
            inclusive="both",
        ).all(),
        "A simulated timestamp moved by at least one full grid cell.",
    )

    require(
        not coarsened[
            "timestamp_mutation_performed"
        ].any(),
        "Timestamp mutation was incorrectly recorded.",
    )

    require(
        not coarsened[
            "event_removal_performed"
        ].any(),
        "Event removal was incorrectly recorded.",
    )

    require(
        not coarsened[
            "within_batch_order_imposed"
        ].any(),
        "An artificial within-batch order was imposed.",
    )

    return coarsened


def coarsen_full_simulated_path_to_1ms(
    path_result: SimulatedHawkesPathResult,
) -> pd.DataFrame:
    """Coarsen both simulated analytical partitions."""
    batch_records: list[
        dict[str, Any]
    ] = []

    for (
        partition_name,
        partition_start_ns,
        partition_end_ns,
    ) in (
        (
            "DEVELOPMENT",
            DEVELOPMENT_START_NS,
            DEVELOPMENT_END_EXCLUSIVE_NS,
        ),
        (
            "CALIBRATION",
            CALIBRATION_START_NS,
            CALIBRATION_END_EXCLUSIVE_NS,
        ),
    ):
        partition_batches = (
            coarsen_simulated_partition_to_1ms(
                path_result.events,
                event_partition=(
                    partition_name
                ),
                partition_start_ns=(
                    partition_start_ns
                ),
                partition_end_exclusive_ns=(
                    partition_end_ns
                ),
                simulation_seed=(
                    path_result.seed
                ),
            )
        )

        batch_records.extend(
            partition_batches.to_dict(
                orient="records"
            )
        )

    if not batch_records:
        return empty_simulated_batch_frame()

    combined_batches = (
        pd.DataFrame.from_records(
            batch_records,
            columns=list(
                SIMULATED_COARSENED_BATCH_COLUMNS
            ),
        )
        .sort_values(
            [
                "event_partition_order",
                "event_time_ns",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        int(
            combined_batches[
                "batch_event_count"
            ].sum()
        )
        == len(
            path_result.events
        ),
        "The full coarsened path does not conserve events.",
    )

    return combined_batches


# ------------------------------------------------------------
# Deterministic reproducibility validation
# ------------------------------------------------------------

SIMULATION_REPRODUCIBILITY_SEED: Final[int] = (
    RANDOM_SEED
    + SIMULATION_PILOT_SEED_OFFSET
)

reproducibility_path_a = (
    simulate_full_frozen_h1_path(
        simulation_seed=(
            SIMULATION_REPRODUCIBILITY_SEED
        )
    )
)

reproducibility_path_b = (
    simulate_full_frozen_h1_path(
        simulation_seed=(
            SIMULATION_REPRODUCIBILITY_SEED
        )
    )
)

pd.testing.assert_frame_equal(
    reproducibility_path_a.events,
    reproducibility_path_b.events,
    check_dtype=True,
    check_exact=True,
)

require(
    reproducibility_path_a
    .development
    .terminal_state
    == reproducibility_path_b
    .development
    .terminal_state,
    "Repeated DEVELOPMENT simulation is not deterministic.",
)

require(
    reproducibility_path_a
    .calibration
    .terminal_state
    == reproducibility_path_b
    .calibration
    .terminal_state,
    "Repeated CALIBRATION simulation is not deterministic.",
)

reproducibility_batches_a = (
    coarsen_full_simulated_path_to_1ms(
        reproducibility_path_a
    )
)

reproducibility_batches_b = (
    coarsen_full_simulated_path_to_1ms(
        reproducibility_path_b
    )
)

pd.testing.assert_frame_equal(
    reproducibility_batches_a,
    reproducibility_batches_b,
    check_dtype=True,
    check_exact=True,
)

different_seed_path = (
    simulate_full_frozen_h1_path(
        simulation_seed=(
            SIMULATION_REPRODUCIBILITY_SEED
            + 1
        )
    )
)

require(
    not reproducibility_path_a.events.equals(
        different_seed_path.events
    ),
    "Different simulation seeds produced identical paths.",
)


# ------------------------------------------------------------
# Pilot simulation run
# ------------------------------------------------------------

pilot_path_results: dict[
    int,
    SimulatedHawkesPathResult,
] = {}

pilot_batch_results: dict[
    int,
    pd.DataFrame,
] = {}

pilot_summary_records: list[
    dict[str, Any]
] = []

pilot_seeds = tuple(
    RANDOM_SEED
    + SIMULATION_PILOT_SEED_OFFSET
    + replicate_index
    for replicate_index in range(
        SIMULATION_PILOT_REPLICATES
    )
)

for (
    replicate_index,
    simulation_seed,
) in enumerate(
    pilot_seeds,
    start=1,
):
    path_result = (
        simulate_full_frozen_h1_path(
            simulation_seed=(
                simulation_seed
            )
        )
    )

    coarsened_batches = (
        coarsen_full_simulated_path_to_1ms(
            path_result
        )
    )

    pilot_path_results[
        simulation_seed
    ] = path_result

    pilot_batch_results[
        simulation_seed
    ] = coarsened_batches

    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        partition_events = (
            path_result.events.loc[
                path_result.events[
                    "event_partition"
                ].eq(
                    partition_name
                )
            ]
        )

        partition_batches = (
            coarsened_batches.loc[
                coarsened_batches[
                    "event_partition"
                ].eq(
                    partition_name
                )
            ]
        )

        buy_event_count = int(
            partition_events[
                "event_side"
            ].eq(
                "BUY"
            ).sum()
        )

        sell_event_count = int(
            partition_events[
                "event_side"
            ].eq(
                "SELL"
            ).sum()
        )

        total_event_count = len(
            partition_events
        )

        require(
            total_event_count
            == buy_event_count
            + sell_event_count,
            (
                f"Pilot replicate {replicate_index} "
                f"{partition_name} does not conserve sides."
            ),
        )

        require(
            int(
                partition_batches[
                    "batch_event_count"
                ].sum()
            )
            == total_event_count,
            (
                f"Pilot replicate {replicate_index} "
                f"{partition_name} does not conserve "
                "events after coarsening."
            ),
        )

        pilot_summary_records.append(
            {
                "replicate_index": int(
                    replicate_index
                ),
                "simulation_seed": int(
                    simulation_seed
                ),
                "event_partition": (
                    partition_name
                ),
                "partition_order": int(
                    PARTITION_ORDER_MAP[
                        partition_name
                    ]
                ),
                "simulated_event_count": int(
                    total_event_count
                ),
                "simulated_buy_event_count": int(
                    buy_event_count
                ),
                "simulated_sell_event_count": int(
                    sell_event_count
                ),
                "coarsened_batch_count": int(
                    len(
                        partition_batches
                    )
                ),
                "simultaneous_batch_count": int(
                    partition_batches[
                        "simultaneous_batch_required_flag"
                    ].sum()
                ),
                "mixed_side_batch_count": int(
                    partition_batches[
                        "mixed_side_batch_flag"
                    ].sum()
                ),
                "maximum_batch_multiplicity": int(
                    partition_batches[
                        "batch_event_count"
                    ].max()
                ),
                "proposal_count": int(
                    (
                        path_result.development
                        if partition_name
                        == "DEVELOPMENT"
                        else path_result.calibration
                    )
                    .proposal_count
                ),
                "rejection_count": int(
                    (
                        path_result.development
                        if partition_name
                        == "DEVELOPMENT"
                        else path_result.calibration
                    )
                    .rejection_count
                ),
                "timestamp_mutation_performed": False,
                "event_removal_performed": False,
                "within_batch_order_imposed": False,
                "status": "PASS",
            }
        )

COARSENING_AWARE_SIMULATION_PILOT_SUMMARY = (
    pd.DataFrame.from_records(
        pilot_summary_records
    )
    .sort_values(
        [
            "replicate_index",
            "partition_order",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

PILOT_SIMULATION_PATHS = (
    pilot_path_results
)

PILOT_COARSENED_BATCHES = (
    pilot_batch_results
)


# ------------------------------------------------------------
# Stationary-rate reference
# ------------------------------------------------------------

FROZEN_H1_STATIONARY_RATE_REFERENCE = pd.DataFrame(
    [
        {
            "event_side": "BUY",
            "baseline_rate_per_second": (
                FROZEN_H1_PARAMETERS.mu_buy
            ),
            "branching_mass": (
                FROZEN_H1_PARAMETERS.kappa_buy
            ),
            "stationary_rate_per_second": (
                FROZEN_H1_PARAMETERS.mu_buy
                / (
                    1.0
                    - FROZEN_H1_PARAMETERS.kappa_buy
                )
            ),
        },
        {
            "event_side": "SELL",
            "baseline_rate_per_second": (
                FROZEN_H1_PARAMETERS.mu_sell
            ),
            "branching_mass": (
                FROZEN_H1_PARAMETERS.kappa_sell
            ),
            "stationary_rate_per_second": (
                FROZEN_H1_PARAMETERS.mu_sell
                / (
                    1.0
                    - FROZEN_H1_PARAMETERS.kappa_sell
                )
            ),
        },
    ]
)

FROZEN_H1_STATIONARY_RATE_REFERENCE[
    "status"
] = "REFERENCE_ONLY"


# ------------------------------------------------------------
# Frozen simulation contract table
# ------------------------------------------------------------

COARSENING_AWARE_SIMULATION_CONTRACT = pd.DataFrame(
    [
        {
            "field": "model_id",
            "value": SIMULATION_MODEL_ID,
        },
        {
            "field": "model_family",
            "value": SIMULATION_MODEL_FAMILY,
        },
        {
            "field": "timestamp_generation",
            "value": SIMULATION_TIMESTAMP_GENERATION,
        },
        {
            "field": "development_initial_state",
            "value": SIMULATION_INITIAL_STATE_SOURCE,
        },
        {
            "field": "calibration_initial_state",
            "value": SIMULATION_CALIBRATION_STATE_SOURCE,
        },
        {
            "field": "coarsening_method",
            "value": SIMULATION_COARSENING_METHOD,
        },
        {
            "field": "coarsening_width_ns",
            "value": COUNT_GRID_WIDTH_NS,
        },
        {
            "field": "batching_method",
            "value": SIMULATION_BATCHING_METHOD,
        },
        {
            "field": "timestamp_jitter_performed",
            "value": SIMULATION_TIMESTAMP_JITTER_PERFORMED,
        },
        {
            "field": "event_removal_performed",
            "value": SIMULATION_EVENT_REMOVAL_PERFORMED,
        },
        {
            "field": "within_batch_order_imposed",
            "value": SIMULATION_WITHIN_BATCH_ORDER_IMPOSED,
        },
        {
            "field": "calibration_parameter_updates",
            "value": SIMULATION_CALIBRATION_PARAMETER_UPDATES,
        },
        {
            "field": "pilot_replicates",
            "value": SIMULATION_PILOT_REPLICATES,
        },
        {
            "field": "planned_envelope_replicates",
            "value": N_MODEL_ENVELOPE_REPLICATES,
        },
        {
            "field": "protected_partitions_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": False,
        },
    ]
)


# ------------------------------------------------------------
# Validation matrix
# ------------------------------------------------------------

simulation_pilot_group_count = (
    SIMULATION_PILOT_REPLICATES
    * len(
        ANALYTICAL_PARTITIONS
    )
)

require(
    len(
        COARSENING_AWARE_SIMULATION_PILOT_SUMMARY
    )
    == simulation_pilot_group_count,
    "The simulation pilot summary has an incorrect row count.",
)

require(
    COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
        "simulated_event_count"
    ].gt(
        0
    ).all(),
    "A pilot partition contains no simulated events.",
)

require(
    COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
        "coarsened_batch_count"
    ].gt(
        0
    ).all(),
    "A pilot partition contains no coarsened batches.",
)

require(
    COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
        "simulated_event_count"
    ].eq(
        COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
            "simulated_buy_event_count"
        ]
        + COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
            "simulated_sell_event_count"
        ]
    ).all(),
    "Pilot BUY and SELL events do not conserve totals.",
)

require(
    not COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
        "timestamp_mutation_performed"
    ].any(),
    "Pilot simulation mutated timestamps.",
)

require(
    not COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
        "event_removal_performed"
    ].any(),
    "Pilot simulation removed events.",
)

require(
    not COARSENING_AWARE_SIMULATION_PILOT_SUMMARY[
        "within_batch_order_imposed"
    ].any(),
    "Pilot simulation imposed within-batch ordering.",
)


COARSENING_AWARE_SIMULATION_ENGINE_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "frozen_h1_parameters_used"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "ogata_upper_bound_validated"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "same_seed_path_reproducible"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "same_seed_coarsening_reproducible"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "different_seed_path_changes"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "development_starts_with_zero_excitation"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_inherits_simulated_development_state"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "partition_relative_one_ms_floor_preserved"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "coarsening_conserves_all_events"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "coarsening_conserves_buy_sell_counts"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "timestamp_jitter_absent"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "event_removal_absent"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "within_batch_order_not_imposed"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    COARSENING_AWARE_SIMULATION_ENGINE_VERIFICATION[
        "passed"
    ].all(),
    "At least one simulation-engine verification failed.",
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

MODEL_ENVELOPE_SIMULATION_CONTRACT_FROZEN = True
COARSENING_AWARE_SIMULATION_ENGINE_VALIDATED = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    COARSENING_AWARE_SIMULATION_CONTRACT
)

display(
    FROZEN_H1_STATIONARY_RATE_REFERENCE
)

display(
    COARSENING_AWARE_SIMULATION_PILOT_SUMMARY
)

display(
    COARSENING_AWARE_SIMULATION_ENGINE_VERIFICATION
)

print(
    "The frozen diagonal-H1 parametric simulation engine was "
    "validated using continuous-time Ogata thinning. Simulated "
    "DEVELOPMENT begins with zero excitation and its exact terminal "
    "state is inherited by CALIBRATION. Partition-relative 1 ms "
    "floor coarsening conserves all BUY and SELL events without "
    "timestamp jitter, event removal, or artificial within-batch "
    "ordering. Deterministic pilot paths passed reproducibility and "
    "contract checks. Full coarsening-aware diagnostic envelopes "
    "are now authorized. No filesystem writes were performed."
)

,field,value
0,model_id,H1_DIAGONAL_SHARED_DECAY
1,model_family,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES
2,timestamp_generation,CONTINUOUS_TIME_OGATA_THINNING
3,development_initial_state,ZERO_EXCITATION_AT_DEVELOPMENT_LEFT_BOUNDARY
4,calibration_initial_state,SIMULATED_TERMINAL_DEVELOPMENT_STATE
5,coarsening_method,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID
6,coarsening_width_ns,1000000
7,batching_method,AGGREGATE_BUY_AND_SELL_COUNTS_AT_COARSENED_TIME
8,timestamp_jitter_performed,False
9,event_removal_performed,False


,event_side,baseline_rate_per_second,branching_mass,stationary_rate_per_second,status
0,BUY,1.536124799,0.1892556591,1.894709246,REFERENCE_ONLY
1,SELL,1.767916532,0.1126637105,1.992386148,REFERENCE_ONLY


,replicate_index,simulation_seed,event_partition,partition_order,simulated_event_count,simulated_buy_event_count,simulated_sell_event_count,coarsened_batch_count,simultaneous_batch_count,mixed_side_batch_count,maximum_batch_multiplicity,proposal_count,rejection_count,timestamp_mutation_performed,event_removal_performed,within_batch_order_imposed,status
0,1,20340720,DEVELOPMENT,1,7084,3506,3578,7023,61,7,2,12364,5280,False,False,False,PASS
1,1,20340720,CALIBRATION,2,2777,1356,1421,2755,22,1,2,4826,2049,False,False,False,PASS
2,2,20340721,DEVELOPMENT,1,6844,3395,3449,6804,40,6,2,11969,5125,False,False,False,PASS
3,2,20340721,CALIBRATION,2,2713,1372,1341,2694,19,1,2,4793,2080,False,False,False,PASS
4,3,20340722,DEVELOPMENT,1,7056,3495,3561,7000,56,10,2,12221,5165,False,False,False,PASS
5,3,20340722,CALIBRATION,2,2783,1333,1450,2763,19,4,3,4801,2018,False,False,False,PASS
6,4,20340723,DEVELOPMENT,1,6967,3394,3573,6917,48,2,3,12042,5075,False,False,False,PASS
7,4,20340723,CALIBRATION,2,2681,1287,1394,2664,17,2,2,4697,2016,False,False,False,PASS
8,5,20340724,DEVELOPMENT,1,7045,3457,3588,6986,59,8,2,12224,5179,False,False,False,PASS
9,5,20340724,CALIBRATION,2,2803,1385,1418,2783,20,1,2,4911,2108,False,False,False,PASS


,check_name,passed
0,frozen_h1_parameters_used,True
1,ogata_upper_bound_validated,True
2,same_seed_path_reproducible,True
3,same_seed_coarsening_reproducible,True
4,different_seed_path_changes,True
5,development_starts_with_zero_excitation,True
6,calibration_inherits_simulated_development_state,True
7,partition_relative_one_ms_floor_preserved,True
8,coarsening_conserves_all_events,True
9,coarsening_conserves_buy_sell_counts,True


The frozen diagonal-H1 parametric simulation engine was validated using continuous-time Ogata thinning. Simulated DEVELOPMENT begins with zero excitation and its exact terminal state is inherited by CALIBRATION. Partition-relative 1 ms floor coarsening conserves all BUY and SELL events without timestamp jitter, event removal, or artificial within-batch ordering. Deterministic pilot paths passed reproducibility and contract checks. Full coarsening-aware diagnostic envelopes are now authorized. No filesystem writes were performed.


In [16]:
# ============================================================
# Construct full coarsening-aware parametric diagnostic envelopes
#
# Each replicate:
#   1. simulates continuous-time DEVELOPMENT and CALIBRATION;
#   2. carries simulated DEVELOPMENT state into CALIBRATION;
#   3. floors timestamps to the authorized partition-relative
#      one-millisecond grid;
#   4. replays the frozen H1 model under strict pre-batch rules;
#   5. computes the same residual, mark, count, and multiplicity
#      statistics used on the observed path.
#
# No model parameters are refitted or updated.
# ============================================================

import gc
import time


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "COARSENING_AWARE_SIMULATION_ENGINE_VALIDATED",
            False,
        )
    ),
    "The coarsening-aware simulation engine is not validated.",
)

require(
    bool(
        globals().get(
            "MODEL_ENVELOPE_SIMULATION_CONTRACT_FROZEN",
            False,
        )
    ),
    "The simulation-envelope contract is not frozen.",
)

require(
    bool(
        globals().get(
            "EMPIRICAL_RESIDUAL_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Observed empirical residual diagnostics are unavailable.",
)

require(
    bool(
        globals().get(
            "SIDE_MARK_MULTIPLICITY_DIAGNOSTICS_BUILT",
            False,
        )
    ),
    "Observed mark and multiplicity diagnostics are unavailable.",
)

require(
    not bool(
        globals().get(
            "COARSENING_AWARE_MODEL_ENVELOPES_BUILT",
            False,
        )
    ),
    "Coarsening-aware model envelopes are already built.",
)

require(
    N_MODEL_ENVELOPE_REPLICATES
    >= 100,
    "At least 100 model-envelope replicates are required.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Frozen envelope configuration
# ------------------------------------------------------------

MODEL_ENVELOPE_BASE_SEED: Final[int] = (
    RANDOM_SEED
    + 1_000_000
)

MODEL_ENVELOPE_PROGRESS_INTERVAL: Final[int] = 100

MODEL_ENVELOPE_LOWER_QUANTILE: Final[float] = 0.025
MODEL_ENVELOPE_MEDIAN_QUANTILE: Final[float] = 0.500
MODEL_ENVELOPE_UPPER_QUANTILE: Final[float] = 0.975

MODEL_ENVELOPE_TAIL_DIRECTIONS: Final[
    Mapping[str, str]
] = {
    "ABS_TAU_MEAN_ERROR": "UPPER",
    "ABS_TAU_VARIANCE_ERROR": "UPPER",
    "KS_D_UNIFORM": "UPPER",
    "CRAMER_VON_MISES": "UPPER",
    "MAX_ABS_ACF_1_TO_20": "UPPER",
    "MAX_ABS_ACF_1_TO_50": "UPPER",
    "LJUNG_BOX_Q_10": "UPPER",
    "LJUNG_BOX_Q_20": "UPPER",
    "LJUNG_BOX_Q_50": "UPPER",
    "ABS_BUY_RATE_CALIBRATION_GAP": "UPPER",
    "MARK_NEGATIVE_LOG_SCORE_PER_EVENT": "UPPER",
    "MARK_BRIER_SCORE_PER_EVENT": "UPPER",
    "MULTIPLE_EVENT_BATCH_RATE": "UPPER",
    "MIXED_SIDE_BATCH_RATE": "UPPER",
    "MAX_BATCH_MULTIPLICITY": "UPPER",
    "MULTIPLE_RELEVANT_BATCH_RATE": "UPPER",
    "MAX_SIDE_MULTIPLICITY": "UPPER",
    "EVENT_COUNT": "TWO_SIDED",
    "BATCH_COUNT": "TWO_SIDED",
    "BUY_EVENT_COUNT": "TWO_SIDED",
    "SELL_EVENT_COUNT": "TWO_SIDED",
}

require(
    set(
        MODEL_ENVELOPE_TAIL_DIRECTIONS.values()
    ).issubset(
        {
            "UPPER",
            "LOWER",
            "TWO_SIDED",
        }
    ),
    "The model-envelope tail-direction registry is invalid.",
)


# ------------------------------------------------------------
# Metric-record helper
# ------------------------------------------------------------

def append_diagnostic_metric(
    records: list[dict[str, Any]],
    *,
    event_partition: str,
    diagnostic_scope: str,
    diagnostic_family: str,
    metric_name: str,
    metric_value: float,
    replicate_index: int | None = None,
    simulation_seed: int | None = None,
) -> None:
    """Append one finite diagnostic metric under the frozen registry."""
    require(
        metric_name
        in MODEL_ENVELOPE_TAIL_DIRECTIONS,
        f"Unregistered diagnostic metric: {metric_name}",
    )

    numeric_value = float(
        metric_value
    )

    require(
        math.isfinite(
            numeric_value
        ),
        (
            f"Diagnostic metric {metric_name} is non-finite "
            f"for {event_partition} {diagnostic_scope}."
        ),
    )

    records.append(
        {
            "replicate_index": (
                pd.NA
                if replicate_index is None
                else int(
                    replicate_index
                )
            ),
            "simulation_seed": (
                pd.NA
                if simulation_seed is None
                else int(
                    simulation_seed
                )
            ),
            "event_partition": str(
                event_partition
            ),
            "partition_order": int(
                PARTITION_ORDER_MAP[
                    event_partition
                ]
            ),
            "diagnostic_scope": str(
                diagnostic_scope
            ),
            "diagnostic_family": str(
                diagnostic_family
            ),
            "metric_name": str(
                metric_name
            ),
            "tail_direction": (
                MODEL_ENVELOPE_TAIL_DIRECTIONS[
                    metric_name
                ]
            ),
            "metric_value": numeric_value,
        }
    )


# ------------------------------------------------------------
# Observed metric registry
# ------------------------------------------------------------

observed_metric_records: list[
    dict[str, Any]
] = []


# Residual distribution and serial metrics
for distribution_row in (
    EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS
    .itertuples(
        index=False
    )
):
    partition_name = str(
        distribution_row.event_partition
    )

    residual_scope = str(
        distribution_row.residual_scope
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope=residual_scope,
        diagnostic_family=(
            "TIME_RESCALING_RESIDUAL"
        ),
        metric_name=(
            "ABS_TAU_MEAN_ERROR"
        ),
        metric_value=(
            distribution_row
            .absolute_tau_mean_error
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope=residual_scope,
        diagnostic_family=(
            "TIME_RESCALING_RESIDUAL"
        ),
        metric_name=(
            "ABS_TAU_VARIANCE_ERROR"
        ),
        metric_value=(
            distribution_row
            .absolute_tau_variance_error
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope=residual_scope,
        diagnostic_family=(
            "TIME_RESCALING_RESIDUAL"
        ),
        metric_name="KS_D_UNIFORM",
        metric_value=(
            distribution_row
            .ks_d_uniform
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope=residual_scope,
        diagnostic_family=(
            "TIME_RESCALING_RESIDUAL"
        ),
        metric_name=(
            "CRAMER_VON_MISES"
        ),
        metric_value=(
            distribution_row
            .cramer_von_mises_statistic
        ),
    )


for serial_row in (
    EMPIRICAL_RESIDUAL_SERIAL_SUMMARY
    .itertuples(
        index=False
    )
):
    partition_name = str(
        serial_row.event_partition
    )

    residual_scope = str(
        serial_row.residual_scope
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope=residual_scope,
        diagnostic_family=(
            "TIME_RESCALING_RESIDUAL"
        ),
        metric_name=(
            "MAX_ABS_ACF_1_TO_20"
        ),
        metric_value=(
            serial_row
            .maximum_absolute_acf_lag_1_to_20
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope=residual_scope,
        diagnostic_family=(
            "TIME_RESCALING_RESIDUAL"
        ),
        metric_name=(
            "MAX_ABS_ACF_1_TO_50"
        ),
        metric_value=(
            serial_row
            .maximum_absolute_acf_lag_1_to_50
        ),
    )


for ljung_box_row in (
    EMPIRICAL_RESIDUAL_LJUNG_BOX
    .itertuples(
        index=False
    )
):
    append_diagnostic_metric(
        observed_metric_records,
        event_partition=str(
            ljung_box_row.event_partition
        ),
        diagnostic_scope=str(
            ljung_box_row.residual_scope
        ),
        diagnostic_family=(
            "TIME_RESCALING_RESIDUAL"
        ),
        metric_name=(
            f"LJUNG_BOX_Q_"
            f"{int(ljung_box_row.lag)}"
        ),
        metric_value=(
            ljung_box_row
            .ljung_box_q
        ),
    )


# Side-mark metrics
for mark_row in (
    SIDE_MARK_PARTITION_SUMMARY
    .itertuples(
        index=False
    )
):
    partition_name = str(
        mark_row.event_partition
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope="MARKS",
        diagnostic_family=(
            "SIDE_MARK_CALIBRATION"
        ),
        metric_name=(
            "ABS_BUY_RATE_CALIBRATION_GAP"
        ),
        metric_value=abs(
            float(
                mark_row
                .buy_rate_calibration_gap
            )
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope="MARKS",
        diagnostic_family=(
            "SIDE_MARK_CALIBRATION"
        ),
        metric_name=(
            "MARK_NEGATIVE_LOG_SCORE_PER_EVENT"
        ),
        metric_value=(
            -float(
                mark_row
                .mark_log_score_per_event
            )
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope="MARKS",
        diagnostic_family=(
            "SIDE_MARK_CALIBRATION"
        ),
        metric_name=(
            "MARK_BRIER_SCORE_PER_EVENT"
        ),
        metric_value=(
            mark_row
            .mark_brier_score_per_event
        ),
    )


# Count and multiplicity metrics
for partition_name in ANALYTICAL_PARTITIONS:
    observed_partition_batches = (
        SIDE_MARK_BATCH_DIAGNOSTICS.loc[
            SIDE_MARK_BATCH_DIAGNOSTICS[
                "event_partition"
            ].eq(
                partition_name
            )
        ]
    )

    require(
        not observed_partition_batches.empty,
        (
            f"No observed batches exist for "
            f"{partition_name}."
        ),
    )

    observed_batch_count = len(
        observed_partition_batches
    )

    observed_event_count = int(
        observed_partition_batches[
            "batch_event_count"
        ].sum()
    )

    observed_buy_count = int(
        observed_partition_batches[
            "buy_event_count"
        ].sum()
    )

    observed_sell_count = int(
        observed_partition_batches[
            "sell_event_count"
        ].sum()
    )

    for metric_name, metric_value in (
        (
            "EVENT_COUNT",
            observed_event_count,
        ),
        (
            "BATCH_COUNT",
            observed_batch_count,
        ),
        (
            "BUY_EVENT_COUNT",
            observed_buy_count,
        ),
        (
            "SELL_EVENT_COUNT",
            observed_sell_count,
        ),
    ):
        append_diagnostic_metric(
            observed_metric_records,
            event_partition=partition_name,
            diagnostic_scope="ALL",
            diagnostic_family=(
                "PATH_COUNTS"
            ),
            metric_name=metric_name,
            metric_value=metric_value,
        )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope="POOLED",
        diagnostic_family=(
            "EXACT_TIME_MULTIPLICITY"
        ),
        metric_name=(
            "MULTIPLE_EVENT_BATCH_RATE"
        ),
        metric_value=float(
            observed_partition_batches[
                "batch_event_count"
            ].gt(
                1
            ).mean()
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope="POOLED",
        diagnostic_family=(
            "EXACT_TIME_MULTIPLICITY"
        ),
        metric_name=(
            "MIXED_SIDE_BATCH_RATE"
        ),
        metric_value=float(
            observed_partition_batches[
                "mixed_side_batch_flag"
            ].mean()
        ),
    )

    append_diagnostic_metric(
        observed_metric_records,
        event_partition=partition_name,
        diagnostic_scope="POOLED",
        diagnostic_family=(
            "EXACT_TIME_MULTIPLICITY"
        ),
        metric_name=(
            "MAX_BATCH_MULTIPLICITY"
        ),
        metric_value=float(
            observed_partition_batches[
                "batch_event_count"
            ].max()
        ),
    )

    for (
        side_name,
        side_count_column,
    ) in (
        (
            "BUY",
            "buy_event_count",
        ),
        (
            "SELL",
            "sell_event_count",
        ),
    ):
        observed_relevant_batches = (
            observed_partition_batches.loc[
                observed_partition_batches[
                    side_count_column
                ].gt(
                    0
                )
            ]
        )

        require(
            not observed_relevant_batches.empty,
            (
                f"No observed {side_name} batches exist "
                f"for {partition_name}."
            ),
        )

        append_diagnostic_metric(
            observed_metric_records,
            event_partition=partition_name,
            diagnostic_scope=side_name,
            diagnostic_family=(
                "EXACT_TIME_MULTIPLICITY"
            ),
            metric_name=(
                "MULTIPLE_RELEVANT_BATCH_RATE"
            ),
            metric_value=float(
                observed_relevant_batches[
                    side_count_column
                ].gt(
                    1
                ).mean()
            ),
        )

        append_diagnostic_metric(
            observed_metric_records,
            event_partition=partition_name,
            diagnostic_scope=side_name,
            diagnostic_family=(
                "EXACT_TIME_MULTIPLICITY"
            ),
            metric_name=(
                "MAX_SIDE_MULTIPLICITY"
            ),
            metric_value=float(
                observed_relevant_batches[
                    side_count_column
                ].max()
            ),
        )


OBSERVED_COARSENING_AWARE_DIAGNOSTIC_METRICS = (
    pd.DataFrame.from_records(
        observed_metric_records
    )
    .drop(
        columns=[
            "replicate_index",
            "simulation_seed",
        ]
    )
    .sort_values(
        [
            "partition_order",
            "diagnostic_family",
            "diagnostic_scope",
            "metric_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

observed_metric_key_columns: Final[
    tuple[str, ...]
] = (
    "event_partition",
    "diagnostic_scope",
    "diagnostic_family",
    "metric_name",
    "tail_direction",
)

require(
    not OBSERVED_COARSENING_AWARE_DIAGNOSTIC_METRICS.duplicated(
        subset=list(
            observed_metric_key_columns
        ),
        keep=False,
    ).any(),
    "The observed diagnostic metric registry contains duplicates.",
)


# ------------------------------------------------------------
# Simulated metric extraction
# ------------------------------------------------------------

def simulated_diagnostic_metric_records(
    path_result: SimulatedHawkesPathResult,
    coarsened_batches: pd.DataFrame,
    *,
    replicate_index: int,
) -> list[dict[str, Any]]:
    """Compute all registered envelope metrics for one replicate."""
    simulation_seed = int(
        path_result.seed
    )

    batches = (
        coarsened_batches.loc[
            :,
            [
                "event_partition",
                "event_partition_order",
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
                "batch_event_count",
                "mixed_side_batch_flag",
                "simultaneous_batch_required_flag",
            ],
        ]
        .copy()
        .sort_values(
            [
                "event_partition_order",
                "event_time_ns",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    require(
        not batches.empty,
        (
            f"Simulation replicate {replicate_index} "
            "contains no coarsened batches."
        ),
    )

    require(
        set(
            batches[
                "event_partition"
            ]
        )
        == set(
            ANALYTICAL_PARTITIONS
        ),
        (
            f"Simulation replicate {replicate_index} "
            "does not contain both analytical partitions."
        ),
    )

    require(
        batches[
            "event_time_ns"
        ].is_unique,
        (
            f"Simulation replicate {replicate_index} "
            "contains duplicate global batch times."
        ),
    )

    replay_result = (
        replay_frozen_diagonal_hawkes(
            batches[
                [
                    "event_time_ns",
                    "buy_event_count",
                    "sell_event_count",
                ]
            ],
            FROZEN_H1_PARAMETERS,
            observation_start_ns=(
                DEVELOPMENT_START_NS
            ),
            observation_end_exclusive_ns=(
                CALIBRATION_END_EXCLUSIVE_NS
            ),
            initial_state=None,
        )
    )

    replay = (
        replay_result.replay.merge(
            batches,
            on=[
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ],
            how="outer",
            validate="one_to_one",
            indicator=True,
        )
    )

    require(
        replay[
            "_merge"
        ].eq(
            "both"
        ).all(),
        (
            f"Simulation replicate {replicate_index} "
            "replay does not reconcile with coarsened batches."
        ),
    )

    replay = (
        replay.drop(
            columns="_merge"
        )
        .sort_values(
            "event_time_ns",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    replay[
        "cumulative_compensator_buy"
    ] = replay[
        "interval_compensator_buy"
    ].cumsum()

    replay[
        "cumulative_compensator_sell"
    ] = replay[
        "interval_compensator_sell"
    ].cumsum()

    replay[
        "cumulative_total_compensator"
    ] = (
        replay[
            "cumulative_compensator_buy"
        ]
        + replay[
            "cumulative_compensator_sell"
        ]
    )

    metric_records: list[
        dict[str, Any]
    ] = []

    # --------------------------------------------------------
    # Time-rescaling residual metrics
    # --------------------------------------------------------

    for (
        residual_scope,
        relevant_count_column,
        cumulative_compensator_column,
    ) in (
        (
            "BUY",
            "buy_event_count",
            "cumulative_compensator_buy",
        ),
        (
            "SELL",
            "sell_event_count",
            "cumulative_compensator_sell",
        ),
        (
            "POOLED",
            "batch_event_count",
            "cumulative_total_compensator",
        ),
    ):
        relevant = (
            replay.loc[
                replay[
                    relevant_count_column
                ].gt(
                    0
                ),
                [
                    "event_partition",
                    "event_partition_order",
                    "event_time_ns",
                    cumulative_compensator_column,
                ],
            ]
            .copy()
            .sort_values(
                "event_time_ns",
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        require(
            len(
                relevant
            )
            > RESIDUAL_ACF_MAX_LAG
            + 1,
            (
                f"Simulation replicate {replicate_index} "
                f"{residual_scope} residual sample is too short."
            ),
        )

        cumulative_values = (
            relevant[
                cumulative_compensator_column
            ].to_numpy(
                dtype=np.float64
            )
        )

        tau = np.diff(
            np.concatenate(
                (
                    np.asarray(
                        [0.0],
                        dtype=np.float64,
                    ),
                    cumulative_values,
                )
            )
        )

        require(
            len(
                tau
            )
            == len(
                relevant
            ),
            "Simulated residual differencing lost an arrival.",
        )

        # The first DEVELOPMENT residual is left-edge censored.
        eligible = relevant.iloc[
            1:
        ].copy()

        eligible[
            "tau"
        ] = tau[
            1:
        ]

        require(
            eligible[
                "tau"
            ].gt(
                0.0
            ).all(),
            (
                f"Simulation replicate {replicate_index} "
                f"{residual_scope} contains a nonpositive "
                "eligible residual."
            ),
        )

        for partition_name in (
            ANALYTICAL_PARTITIONS
        ):
            partition_tau = (
                eligible.loc[
                    eligible[
                        "event_partition"
                    ].eq(
                        partition_name
                    ),
                    "tau",
                ]
                .to_numpy(
                    dtype=np.float64
                )
            )

            require(
                len(
                    partition_tau
                )
                > RESIDUAL_ACF_MAX_LAG,
                (
                    f"Simulation replicate {replicate_index} "
                    f"{partition_name} {residual_scope} "
                    "residual sample is too short."
                ),
            )

            uniform_values = -np.expm1(
                -partition_tau
            )

            require(
                np.all(
                    uniform_values
                    > 0.0
                )
                and np.all(
                    uniform_values
                    < 1.0
                ),
                "A simulated PIT residual lies outside (0, 1).",
            )

            (
                _,
                _,
                ks_d,
            ) = uniform_ks_components(
                uniform_values
            )

            cvm_statistic = (
                uniform_cramer_von_mises_statistic(
                    uniform_values
                )
            )

            acf_values = sample_autocorrelation(
                partition_tau,
                maximum_lag=(
                    RESIDUAL_ACF_MAX_LAG
                ),
            )

            for metric_name, metric_value in (
                (
                    "ABS_TAU_MEAN_ERROR",
                    abs(
                        float(
                            np.mean(
                                partition_tau
                            )
                        )
                        - 1.0
                    ),
                ),
                (
                    "ABS_TAU_VARIANCE_ERROR",
                    abs(
                        float(
                            np.var(
                                partition_tau,
                                ddof=1,
                            )
                        )
                        - 1.0
                    ),
                ),
                (
                    "KS_D_UNIFORM",
                    ks_d,
                ),
                (
                    "CRAMER_VON_MISES",
                    cvm_statistic,
                ),
                (
                    "MAX_ABS_ACF_1_TO_20",
                    float(
                        np.max(
                            np.abs(
                                acf_values[
                                    1:21
                                ]
                            )
                        )
                    ),
                ),
                (
                    "MAX_ABS_ACF_1_TO_50",
                    float(
                        np.max(
                            np.abs(
                                acf_values[
                                    1:51
                                ]
                            )
                        )
                    ),
                ),
            ):
                append_diagnostic_metric(
                    metric_records,
                    event_partition=(
                        partition_name
                    ),
                    diagnostic_scope=(
                        residual_scope
                    ),
                    diagnostic_family=(
                        "TIME_RESCALING_RESIDUAL"
                    ),
                    metric_name=metric_name,
                    metric_value=metric_value,
                    replicate_index=(
                        replicate_index
                    ),
                    simulation_seed=(
                        simulation_seed
                    ),
                )

            for lag in LJUNG_BOX_LAGS:
                (
                    q_statistic,
                    _,
                ) = ljung_box_statistic(
                    acf_values,
                    sample_size=len(
                        partition_tau
                    ),
                    lag=lag,
                )

                append_diagnostic_metric(
                    metric_records,
                    event_partition=(
                        partition_name
                    ),
                    diagnostic_scope=(
                        residual_scope
                    ),
                    diagnostic_family=(
                        "TIME_RESCALING_RESIDUAL"
                    ),
                    metric_name=(
                        f"LJUNG_BOX_Q_{lag}"
                    ),
                    metric_value=(
                        q_statistic
                    ),
                    replicate_index=(
                        replicate_index
                    ),
                    simulation_seed=(
                        simulation_seed
                    ),
                )

    # --------------------------------------------------------
    # Partition-level mark, count, and multiplicity metrics
    # --------------------------------------------------------

    replay[
        "lambda_total_pre"
    ] = (
        replay[
            "lambda_buy_pre"
        ]
        + replay[
            "lambda_sell_pre"
        ]
    )

    replay[
        "predicted_buy_probability"
    ] = (
        replay[
            "lambda_buy_pre"
        ]
        / replay[
            "lambda_total_pre"
        ]
    )

    require(
        replay[
            "predicted_buy_probability"
        ].between(
            0.0,
            1.0,
            inclusive="neither",
        ).all(),
        "A simulated BUY mark probability lies outside (0, 1).",
    )

    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        partition_replay = (
            replay.loc[
                replay[
                    "event_partition"
                ].eq(
                    partition_name
                )
            ]
        )

        require(
            not partition_replay.empty,
            (
                f"Simulation replicate {replicate_index} "
                f"contains no {partition_name} replay rows."
            ),
        )

        batch_count = len(
            partition_replay
        )

        event_count = int(
            partition_replay[
                "batch_event_count"
            ].sum()
        )

        buy_event_count = int(
            partition_replay[
                "buy_event_count"
            ].sum()
        )

        sell_event_count = int(
            partition_replay[
                "sell_event_count"
            ].sum()
        )

        require(
            event_count
            == buy_event_count
            + sell_event_count,
            "A simulated partition does not conserve side counts.",
        )

        for metric_name, metric_value in (
            (
                "EVENT_COUNT",
                event_count,
            ),
            (
                "BATCH_COUNT",
                batch_count,
            ),
            (
                "BUY_EVENT_COUNT",
                buy_event_count,
            ),
            (
                "SELL_EVENT_COUNT",
                sell_event_count,
            ),
        ):
            append_diagnostic_metric(
                metric_records,
                event_partition=partition_name,
                diagnostic_scope="ALL",
                diagnostic_family=(
                    "PATH_COUNTS"
                ),
                metric_name=metric_name,
                metric_value=metric_value,
                replicate_index=(
                    replicate_index
                ),
                simulation_seed=(
                    simulation_seed
                ),
            )

        expected_buy_events = float(
            np.sum(
                partition_replay[
                    "batch_event_count"
                ]
                * partition_replay[
                    "predicted_buy_probability"
                ]
            )
        )

        buy_rate_gap = (
            buy_event_count
            - expected_buy_events
        ) / event_count

        buy_probability = np.clip(
            partition_replay[
                "predicted_buy_probability"
            ].to_numpy(
                dtype=np.float64
            ),
            MARK_PROBABILITY_FLOOR,
            1.0
            - MARK_PROBABILITY_FLOOR,
        )

        sell_probability = (
            1.0
            - buy_probability
        )

        mark_log_score = float(
            np.sum(
                partition_replay[
                    "buy_event_count"
                ].to_numpy(
                    dtype=np.float64
                )
                * np.log(
                    buy_probability
                )
                + partition_replay[
                    "sell_event_count"
                ].to_numpy(
                    dtype=np.float64
                )
                * np.log(
                    sell_probability
                )
            )
        )

        mark_brier_total = float(
            np.sum(
                partition_replay[
                    "buy_event_count"
                ].to_numpy(
                    dtype=np.float64
                )
                * (
                    1.0
                    - buy_probability
                )
                ** 2
                + partition_replay[
                    "sell_event_count"
                ].to_numpy(
                    dtype=np.float64
                )
                * buy_probability
                ** 2
            )
        )

        for metric_name, metric_value in (
            (
                "ABS_BUY_RATE_CALIBRATION_GAP",
                abs(
                    buy_rate_gap
                ),
            ),
            (
                "MARK_NEGATIVE_LOG_SCORE_PER_EVENT",
                -mark_log_score
                / event_count,
            ),
            (
                "MARK_BRIER_SCORE_PER_EVENT",
                mark_brier_total
                / event_count,
            ),
        ):
            append_diagnostic_metric(
                metric_records,
                event_partition=partition_name,
                diagnostic_scope="MARKS",
                diagnostic_family=(
                    "SIDE_MARK_CALIBRATION"
                ),
                metric_name=metric_name,
                metric_value=metric_value,
                replicate_index=(
                    replicate_index
                ),
                simulation_seed=(
                    simulation_seed
                ),
            )

        for metric_name, metric_value in (
            (
                "MULTIPLE_EVENT_BATCH_RATE",
                float(
                    partition_replay[
                        "batch_event_count"
                    ].gt(
                        1
                    ).mean()
                ),
            ),
            (
                "MIXED_SIDE_BATCH_RATE",
                float(
                    partition_replay[
                        "mixed_side_batch_flag"
                    ].mean()
                ),
            ),
            (
                "MAX_BATCH_MULTIPLICITY",
                float(
                    partition_replay[
                        "batch_event_count"
                    ].max()
                ),
            ),
        ):
            append_diagnostic_metric(
                metric_records,
                event_partition=partition_name,
                diagnostic_scope="POOLED",
                diagnostic_family=(
                    "EXACT_TIME_MULTIPLICITY"
                ),
                metric_name=metric_name,
                metric_value=metric_value,
                replicate_index=(
                    replicate_index
                ),
                simulation_seed=(
                    simulation_seed
                ),
            )

        for (
            side_name,
            side_count_column,
        ) in (
            (
                "BUY",
                "buy_event_count",
            ),
            (
                "SELL",
                "sell_event_count",
            ),
        ):
            relevant_side_batches = (
                partition_replay.loc[
                    partition_replay[
                        side_count_column
                    ].gt(
                        0
                    )
                ]
            )

            require(
                not relevant_side_batches.empty,
                (
                    f"Simulation replicate {replicate_index} "
                    f"contains no {partition_name} "
                    f"{side_name} batches."
                ),
            )

            append_diagnostic_metric(
                metric_records,
                event_partition=partition_name,
                diagnostic_scope=side_name,
                diagnostic_family=(
                    "EXACT_TIME_MULTIPLICITY"
                ),
                metric_name=(
                    "MULTIPLE_RELEVANT_BATCH_RATE"
                ),
                metric_value=float(
                    relevant_side_batches[
                        side_count_column
                    ].gt(
                        1
                    ).mean()
                ),
                replicate_index=(
                    replicate_index
                ),
                simulation_seed=(
                    simulation_seed
                ),
            )

            append_diagnostic_metric(
                metric_records,
                event_partition=partition_name,
                diagnostic_scope=side_name,
                diagnostic_family=(
                    "EXACT_TIME_MULTIPLICITY"
                ),
                metric_name=(
                    "MAX_SIDE_MULTIPLICITY"
                ),
                metric_value=float(
                    relevant_side_batches[
                        side_count_column
                    ].max()
                ),
                replicate_index=(
                    replicate_index
                ),
                simulation_seed=(
                    simulation_seed
                ),
            )

    return metric_records


# ------------------------------------------------------------
# Execute frozen envelope simulations
# ------------------------------------------------------------

envelope_metric_records: list[
    dict[str, Any]
] = []

envelope_replicate_summary_records: list[
    dict[str, Any]
] = []

MODEL_ENVELOPE_SEEDS: Final[tuple[int, ...]] = tuple(
    MODEL_ENVELOPE_BASE_SEED
    + replicate_index
    for replicate_index in range(
        1,
        N_MODEL_ENVELOPE_REPLICATES
        + 1,
    )
)

require(
    len(
        set(
            MODEL_ENVELOPE_SEEDS
        )
    )
    == N_MODEL_ENVELOPE_REPLICATES,
    "Model-envelope simulation seeds are not unique.",
)

simulation_start_time = time.perf_counter()

for (
    replicate_index,
    simulation_seed,
) in enumerate(
    MODEL_ENVELOPE_SEEDS,
    start=1,
):
    try:
        simulated_path = (
            simulate_full_frozen_h1_path(
                simulation_seed=(
                    simulation_seed
                )
            )
        )

        simulated_batches = (
            coarsen_full_simulated_path_to_1ms(
                simulated_path
            )
        )

        replicate_metrics = (
            simulated_diagnostic_metric_records(
                simulated_path,
                simulated_batches,
                replicate_index=(
                    replicate_index
                ),
            )
        )

    except Exception as exc:
        raise RuntimeError(
            (
                "Coarsening-aware envelope simulation failed "
                f"at replicate={replicate_index}, "
                f"seed={simulation_seed}."
            )
        ) from exc

    envelope_metric_records.extend(
        replicate_metrics
    )

    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        partition_events = (
            simulated_path.events.loc[
                simulated_path.events[
                    "event_partition"
                ].eq(
                    partition_name
                )
            ]
        )

        partition_batches = (
            simulated_batches.loc[
                simulated_batches[
                    "event_partition"
                ].eq(
                    partition_name
                )
            ]
        )

        envelope_replicate_summary_records.append(
            {
                "replicate_index": int(
                    replicate_index
                ),
                "simulation_seed": int(
                    simulation_seed
                ),
                "event_partition": (
                    partition_name
                ),
                "partition_order": int(
                    PARTITION_ORDER_MAP[
                        partition_name
                    ]
                ),
                "event_count": int(
                    len(
                        partition_events
                    )
                ),
                "buy_event_count": int(
                    partition_events[
                        "event_side"
                    ].eq(
                        "BUY"
                    ).sum()
                ),
                "sell_event_count": int(
                    partition_events[
                        "event_side"
                    ].eq(
                        "SELL"
                    ).sum()
                ),
                "coarsened_batch_count": int(
                    len(
                        partition_batches
                    )
                ),
                "multiple_event_batch_count": int(
                    partition_batches[
                        "batch_event_count"
                    ].gt(
                        1
                    ).sum()
                ),
                "mixed_side_batch_count": int(
                    partition_batches[
                        "mixed_side_batch_flag"
                    ].sum()
                ),
                "maximum_batch_multiplicity": int(
                    partition_batches[
                        "batch_event_count"
                    ].max()
                ),
                "status": "PASS",
            }
        )

    if (
        replicate_index
        % MODEL_ENVELOPE_PROGRESS_INTERVAL
        == 0
        or replicate_index
        == N_MODEL_ENVELOPE_REPLICATES
    ):
        elapsed_seconds = (
            time.perf_counter()
            - simulation_start_time
        )

        print(
            f"Completed {replicate_index:,}/"
            f"{N_MODEL_ENVELOPE_REPLICATES:,} "
            "coarsening-aware replicates "
            f"in {elapsed_seconds:,.1f} seconds."
        )

        gc.collect()

    del (
        simulated_path,
        simulated_batches,
        replicate_metrics,
    )


MODEL_ENVELOPE_RUNTIME_SECONDS: Final[float] = (
    time.perf_counter()
    - simulation_start_time
)


# ------------------------------------------------------------
# Materialize replicate-level results
# ------------------------------------------------------------

COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS = (
    pd.DataFrame.from_records(
        envelope_metric_records
    )
    .sort_values(
        [
            "replicate_index",
            "partition_order",
            "diagnostic_family",
            "diagnostic_scope",
            "metric_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_SUMMARY = (
    pd.DataFrame.from_records(
        envelope_replicate_summary_records
    )
    .sort_values(
        [
            "replicate_index",
            "partition_order",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

for integer_column in (
    "replicate_index",
    "simulation_seed",
    "partition_order",
):
    COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS[
        integer_column
    ] = parse_exact_int64(
        COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS[
            integer_column
        ],
        label=(
            "COARSENING_AWARE_MODEL_ENVELOPE_"
            f"REPLICATE_METRICS.{integer_column}"
        ),
    )

require(
    np.isfinite(
        COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS[
            "metric_value"
        ].to_numpy(
            dtype=np.float64
        )
    ).all(),
    "At least one simulated envelope metric is non-finite.",
)


# ------------------------------------------------------------
# Metric-key reconciliation
# ------------------------------------------------------------

observed_metric_keys = {
    tuple(
        row
    )
    for row in (
        OBSERVED_COARSENING_AWARE_DIAGNOSTIC_METRICS[
            list(
                observed_metric_key_columns
            )
        ]
        .itertuples(
            index=False,
            name=None,
        )
    )
}

simulated_metric_keys = {
    tuple(
        row
    )
    for row in (
        COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS[
            list(
                observed_metric_key_columns
            )
        ]
        .drop_duplicates()
        .itertuples(
            index=False,
            name=None,
        )
    )
}

require(
    simulated_metric_keys
    == observed_metric_keys,
    (
        "Observed and simulated diagnostic metric registries "
        "do not match."
    ),
)

expected_metric_rows = (
    N_MODEL_ENVELOPE_REPLICATES
    * len(
        observed_metric_keys
    )
)

require(
    len(
        COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS
    )
    == expected_metric_rows,
    (
        "The simulated diagnostic metric row count is incorrect: "
        f"expected={expected_metric_rows:,}; "
        "observed="
        f"{len(COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS):,}."
    ),
)

metric_group_sizes = (
    COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS
    .groupby(
        list(
            observed_metric_key_columns
        ),
        observed=True,
        sort=True,
        dropna=False,
    )
    .size()
)

require(
    metric_group_sizes.eq(
        N_MODEL_ENVELOPE_REPLICATES
    ).all(),
    (
        "At least one diagnostic metric lacks the full "
        "simulation replicate count."
    ),
)


# ------------------------------------------------------------
# Build observed-versus-simulated envelopes
# ------------------------------------------------------------

observed_value_lookup = {
    tuple(
        getattr(
            row,
            column_name,
        )
        for column_name in (
            observed_metric_key_columns
        )
    ): float(
        row.metric_value
    )
    for row in (
        OBSERVED_COARSENING_AWARE_DIAGNOSTIC_METRICS
        .itertuples(
            index=False
        )
    )
}

envelope_summary_records: list[
    dict[str, Any]
] = []

for metric_key, group in (
    COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS
    .groupby(
        list(
            observed_metric_key_columns
        ),
        observed=True,
        sort=True,
        dropna=False,
    )
):
    (
        partition_name,
        diagnostic_scope,
        diagnostic_family,
        metric_name,
        tail_direction,
    ) = tuple(
        metric_key
    )

    simulated_values = (
        group[
            "metric_value"
        ].to_numpy(
            dtype=np.float64
        )
    )

    require(
        len(
            simulated_values
        )
        == N_MODEL_ENVELOPE_REPLICATES,
        "An envelope group has an incorrect replicate count.",
    )

    observed_value = (
        observed_value_lookup[
            (
                partition_name,
                diagnostic_scope,
                diagnostic_family,
                metric_name,
                tail_direction,
            )
        ]
    )

    (
        lower_quantile,
        median_quantile,
        upper_quantile,
    ) = np.quantile(
        simulated_values,
        [
            MODEL_ENVELOPE_LOWER_QUANTILE,
            MODEL_ENVELOPE_MEDIAN_QUANTILE,
            MODEL_ENVELOPE_UPPER_QUANTILE,
        ],
        method="linear",
    )

    lower_tail_probability = (
        1.0
        + float(
            np.sum(
                simulated_values
                <= observed_value
            )
        )
    ) / (
        N_MODEL_ENVELOPE_REPLICATES
        + 1.0
    )

    upper_tail_probability = (
        1.0
        + float(
            np.sum(
                simulated_values
                >= observed_value
            )
        )
    ) / (
        N_MODEL_ENVELOPE_REPLICATES
        + 1.0
    )

    if tail_direction == "UPPER":
        directional_tail_probability = (
            upper_tail_probability
        )

        outside_directional_envelope = bool(
            observed_value
            > upper_quantile
        )

    elif tail_direction == "LOWER":
        directional_tail_probability = (
            lower_tail_probability
        )

        outside_directional_envelope = bool(
            observed_value
            < lower_quantile
        )

    else:
        directional_tail_probability = min(
            1.0,
            2.0
            * min(
                lower_tail_probability,
                upper_tail_probability,
            ),
        )

        outside_directional_envelope = bool(
            observed_value
            < lower_quantile
            or observed_value
            > upper_quantile
        )

    envelope_summary_records.append(
        {
            "event_partition": str(
                partition_name
            ),
            "partition_order": int(
                PARTITION_ORDER_MAP[
                    str(
                        partition_name
                    )
                ]
            ),
            "diagnostic_scope": str(
                diagnostic_scope
            ),
            "diagnostic_family": str(
                diagnostic_family
            ),
            "metric_name": str(
                metric_name
            ),
            "tail_direction": str(
                tail_direction
            ),
            "simulation_replicates": int(
                N_MODEL_ENVELOPE_REPLICATES
            ),
            "observed_value": float(
                observed_value
            ),
            "simulation_mean": float(
                np.mean(
                    simulated_values
                )
            ),
            "simulation_standard_deviation": float(
                np.std(
                    simulated_values,
                    ddof=1,
                )
            ),
            "simulation_minimum": float(
                np.min(
                    simulated_values
                )
            ),
            "simulation_q025": float(
                lower_quantile
            ),
            "simulation_median": float(
                median_quantile
            ),
            "simulation_q975": float(
                upper_quantile
            ),
            "simulation_maximum": float(
                np.max(
                    simulated_values
                )
            ),
            "observed_minus_simulation_median": float(
                observed_value
                - median_quantile
            ),
            "lower_tail_probability_report_only": float(
                lower_tail_probability
            ),
            "upper_tail_probability_report_only": float(
                upper_tail_probability
            ),
            "directional_tail_probability_report_only": float(
                directional_tail_probability
            ),
            "outside_two_sided_95pct_envelope": bool(
                observed_value
                < lower_quantile
                or observed_value
                > upper_quantile
            ),
            "outside_directional_95pct_envelope": (
                outside_directional_envelope
            ),
            "envelope_interpretation": (
                "OUTSIDE_MODEL_ENVELOPE"
                if outside_directional_envelope
                else "INSIDE_MODEL_ENVELOPE"
            ),
            "parameter_refit_performed": False,
            "calibration_parameter_updates": 0,
            "status": "PASS_ENVELOPE_COMPUTED",
        }
    )


COARSENING_AWARE_DIAGNOSTIC_ENVELOPES = (
    pd.DataFrame.from_records(
        envelope_summary_records
    )
    .sort_values(
        [
            "partition_order",
            "diagnostic_family",
            "diagnostic_scope",
            "metric_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Replicate distribution summary
# ------------------------------------------------------------

COARSENING_AWARE_REPLICATE_COUNT_SUMMARY = (
    COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_SUMMARY
    .groupby(
        [
            "partition_order",
            "event_partition",
        ],
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        replicate_count=(
            "replicate_index",
            "nunique",
        ),
        event_count_mean=(
            "event_count",
            "mean",
        ),
        event_count_standard_deviation=(
            "event_count",
            "std",
        ),
        event_count_q025=(
            "event_count",
            lambda values: float(
                values.quantile(
                    MODEL_ENVELOPE_LOWER_QUANTILE
                )
            ),
        ),
        event_count_median=(
            "event_count",
            "median",
        ),
        event_count_q975=(
            "event_count",
            lambda values: float(
                values.quantile(
                    MODEL_ENVELOPE_UPPER_QUANTILE
                )
            ),
        ),
        batch_count_mean=(
            "coarsened_batch_count",
            "mean",
        ),
        multiple_event_batch_count_mean=(
            "multiple_event_batch_count",
            "mean",
        ),
        mixed_side_batch_count_mean=(
            "mixed_side_batch_count",
            "mean",
        ),
        maximum_batch_multiplicity_maximum=(
            "maximum_batch_multiplicity",
            "max",
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Verification matrix
# ------------------------------------------------------------

require(
    COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_SUMMARY[
        "replicate_index"
    ].nunique()
    == N_MODEL_ENVELOPE_REPLICATES,
    "The replicate summary has an incorrect replicate count.",
)

require(
    COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_SUMMARY[
        "simulation_seed"
    ].nunique()
    == N_MODEL_ENVELOPE_REPLICATES,
    "The replicate summary has an incorrect seed count.",
)

require(
    len(
        COARSENING_AWARE_DIAGNOSTIC_ENVELOPES
    )
    == len(
        OBSERVED_COARSENING_AWARE_DIAGNOSTIC_METRICS
    ),
    "The final envelope table has an incorrect row count.",
)

require(
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "simulation_replicates"
    ].eq(
        N_MODEL_ENVELOPE_REPLICATES
    ).all(),
    "An envelope does not contain the full replicate count.",
)

require(
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "simulation_q025"
    ].le(
        COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
            "simulation_median"
        ]
    ).all(),
    "An envelope lower bound exceeds its median.",
)

require(
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "simulation_median"
    ].le(
        COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
            "simulation_q975"
        ]
    ).all(),
    "An envelope median exceeds its upper bound.",
)

require(
    not COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "parameter_refit_performed"
    ].any(),
    "A simulation replicate refitted model parameters.",
)

require(
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "calibration_parameter_updates"
    ].eq(
        0
    ).all(),
    "A simulation replicate updated CALIBRATION parameters.",
)


COARSENING_AWARE_MODEL_ENVELOPE_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "one_thousand_unique_simulation_seeds"
            ),
            "passed": (
                len(
                    MODEL_ENVELOPE_SEEDS
                )
                == N_MODEL_ENVELOPE_REPLICATES
            ),
        },
        {
            "check_name": (
                "all_simulation_replicates_completed"
            ),
            "passed": (
                COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_SUMMARY[
                    "replicate_index"
                ].nunique()
                == N_MODEL_ENVELOPE_REPLICATES
            ),
        },
        {
            "check_name": (
                "observed_and_simulated_metric_registries_match"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "every_metric_has_full_replicate_count"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "all_replicate_metrics_finite"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "all_envelope_quantiles_ordered"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "no_parameter_refitting_performed"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    COARSENING_AWARE_MODEL_ENVELOPE_VERIFICATION[
        "passed"
    ].all(),
    "At least one model-envelope verification failed.",
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

COARSENING_AWARE_MODEL_ENVELOPES_BUILT = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    COARSENING_AWARE_REPLICATE_COUNT_SUMMARY
)

display(
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES.loc[
        COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
)

display(
    COARSENING_AWARE_MODEL_ENVELOPE_VERIFICATION
)

print(
    f"Completed {N_MODEL_ENVELOPE_REPLICATES:,} frozen-H1 "
    "coarsening-aware parametric simulations in "
    f"{MODEL_ENVELOPE_RUNTIME_SECONDS:,.1f} seconds. "
    "Observed residual-distribution, serial-dependence, side-mark, "
    "path-count, and exact-time multiplicity statistics were compared "
    "with their model-generated 95% envelopes. No parameters were "
    "refitted, CALIBRATION parameter updates remained zero, protected "
    "partitions remained unopened, and no filesystem writes were "
    "performed."
)

Completed 100/1,000 coarsening-aware replicates in 25.4 seconds.
Completed 200/1,000 coarsening-aware replicates in 53.4 seconds.
Completed 300/1,000 coarsening-aware replicates in 85.4 seconds.
Completed 400/1,000 coarsening-aware replicates in 113.5 seconds.
Completed 500/1,000 coarsening-aware replicates in 146.8 seconds.
Completed 600/1,000 coarsening-aware replicates in 177.2 seconds.
Completed 700/1,000 coarsening-aware replicates in 206.4 seconds.
Completed 800/1,000 coarsening-aware replicates in 235.6 seconds.
Completed 900/1,000 coarsening-aware replicates in 264.7 seconds.
Completed 1,000/1,000 coarsening-aware replicates in 294.1 seconds.


,partition_order,event_partition,replicate_count,event_count_mean,event_count_standard_deviation,event_count_q025,event_count_median,event_count_q975,batch_count_mean,multiple_event_batch_count_mean,mixed_side_batch_count_mean,maximum_batch_multiplicity_maximum
0,1,DEVELOPMENT,1000,"6,999.7",95.86706679,"6,815.975","6,997","7,192.025","6,946.861",52.325,6.762,4
1,2,CALIBRATION,1000,"2,801.082",64.24821483,"2,672.95","2,802","2,919","2,779.983",20.918,2.706,3


,event_partition,partition_order,diagnostic_scope,diagnostic_family,metric_name,tail_direction,simulation_replicates,observed_value,simulation_mean,simulation_standard_deviation,simulation_minimum,simulation_q025,simulation_median,simulation_q975,simulation_maximum,observed_minus_simulation_median,lower_tail_probability_report_only,upper_tail_probability_report_only,directional_tail_probability_report_only,outside_two_sided_95pct_envelope,outside_directional_95pct_envelope,envelope_interpretation,parameter_refit_performed,calibration_parameter_updates,status
41,CALIBRATION,2,BUY,EXACT_TIME_MULTIPLICITY,MAX_SIDE_MULTIPLICITY,UPPER,1000,4,2.103,0.3041109723,2,2,2,3,3,2,1,0.000999000999,0.000999000999,True,True,OUTSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
42,CALIBRATION,2,BUY,EXACT_TIME_MULTIPLICITY,MULTIPLE_RELEVANT_BATCH_RATE,UPPER,1000,0.04588744589,0.008147625122,0.002521759749,0.0007457121551,0.003560874704,0.008036532093,0.01353279031,0.01692420898,0.03785091379,1,0.000999000999,0.000999000999,True,True,OUTSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
43,CALIBRATION,2,POOLED,EXACT_TIME_MULTIPLICITY,MAX_BATCH_MULTIPLICITY,UPPER,1000,4,2.167,0.3731624985,2,2,2,3,3,2,1,0.000999000999,0.000999000999,True,True,OUTSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
44,CALIBRATION,2,POOLED,EXACT_TIME_MULTIPLICITY,MIXED_SIDE_BATCH_RATE,UPPER,1000,0.002083333333,0.0009734420701,0.0005942744263,0,0,0.001042934153,0.002227337328,0.003571428571,0.00104039918,0.9450549451,0.05594405594,0.05594405594,False,False,INSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
45,CALIBRATION,2,POOLED,EXACT_TIME_MULTIPLICITY,MULTIPLE_EVENT_BATCH_RATE,UPPER,1000,0.02833333333,0.007523103613,0.001740870977,0.00217944061,0.0042281439,0.007395082523,0.0109822277,0.01292466765,0.02093825081,1,0.000999000999,0.000999000999,True,True,OUTSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
46,CALIBRATION,2,SELL,EXACT_TIME_MULTIPLICITY,MAX_SIDE_MULTIPLICITY,UPPER,1000,3,2.049,0.2159760303,2,2,2,3,3,1,1,0.04995004995,0.04995004995,False,False,INSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
47,CALIBRATION,2,SELL,EXACT_TIME_MULTIPLICITY,MULTIPLE_RELEVANT_BATCH_RATE,UPPER,1000,0.0088,0.005042585207,0.001860251001,0.0007007708479,0.001420379043,0.004900245613,0.008910671448,0.01091405184,0.003899754387,0.971028971,0.02997002997,0.02997002997,False,False,INSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
48,CALIBRATION,2,ALL,PATH_COUNTS,BATCH_COUNT,TWO_SIDED,1000,"2,400","2,779.983",63.44154428,"2,577","2,651.975","2,781.5","2,898","2,982",-381.5,0.000999000999,1,0.001998001998,True,True,OUTSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
49,CALIBRATION,2,ALL,PATH_COUNTS,BUY_EVENT_COUNT,TWO_SIDED,1000,"1,230","1,362.699",46.17577938,"1,192","1,270.975","1,362","1,450.025","1,504",-132,0.004995004995,0.996003996,0.00999000999,True,True,OUTSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED
50,CALIBRATION,2,ALL,PATH_COUNTS,EVENT_COUNT,TWO_SIDED,1000,"2,493","2,801.082",64.24821483,"2,598","2,672.95","2,802","2,919","2,994",-309,0.000999000999,1,0.001998001998,True,True,OUTSIDE_MODEL_ENVELOPE,False,0,PASS_ENVELOPE_COMPUTED


,check_name,passed
0,one_thousand_unique_simulation_seeds,True
1,all_simulation_replicates_completed,True
2,observed_and_simulated_metric_registries_match,True
3,every_metric_has_full_replicate_count,True
4,all_replicate_metrics_finite,True
5,all_envelope_quantiles_ordered,True
6,no_parameter_refitting_performed,True
7,calibration_parameter_updates_zero,True
8,protected_partitions_unopened,True
9,filesystem_writes_absent,True


Completed 1,000 frozen-H1 coarsening-aware parametric simulations in 294.2 seconds. Observed residual-distribution, serial-dependence, side-mark, path-count, and exact-time multiplicity statistics were compared with their model-generated 95% envelopes. No parameters were refitted, CALIBRATION parameter updates remained zero, protected partitions remained unopened, and no filesystem writes were performed.


In [20]:
# ============================================================
# Normalize common-grid diagnostic inputs
#
# grid.exposure_seconds is the scalar total partition exposure.
# Per-cell exposure is reconstructed exactly from the integer-
# nanosecond half-open grid contract.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    "HAWKES_COMMON_COUNT_GRIDS"
    in globals(),
    "The authoritative Hawkes common count grids are unavailable.",
)

require(
    not bool(
        globals().get(
            "COMMON_GRID_DIAGNOSTIC_INPUTS_NORMALIZED",
            False,
        )
    ),
    "Common-grid diagnostic inputs are already normalized.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED
    == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Immutable normalized container
# ------------------------------------------------------------

@dataclass(
    frozen=True,
    slots=True,
)
class CommonGridDiagnosticInputs:
    event_partition: str
    partition_order: int
    grid_width_ns: int
    contract_start_ns: int
    contract_end_exclusive_ns: int
    contract_duration_ns: int
    cell_count: int
    exposure_ns: np.ndarray
    exposure_seconds: np.ndarray
    total_exposure_seconds: float
    buy_counts: np.ndarray
    sell_counts: np.ndarray
    pooled_counts: np.ndarray
    expected_buy_counts: np.ndarray
    expected_sell_counts: np.ndarray
    expected_pooled_counts: np.ndarray


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def diagnostic_exact_ceil_division(
    numerator: int,
    denominator: int,
) -> int:
    """Return exact ceil(numerator / denominator)."""
    require(
        numerator >= 0,
        "Ceiling-division numerator must be nonnegative.",
    )

    require(
        denominator > 0,
        "Ceiling-division denominator must be positive.",
    )

    return -(
        -numerator
        // denominator
    )


def immutable_one_dimensional_array(
    values: Any,
    *,
    dtype: np.dtype | str,
    expected_length: int,
    label: str,
) -> np.ndarray:
    """Return a finite immutable one-dimensional array."""
    array = np.asarray(
        values,
        dtype=dtype,
    )

    require(
        array.ndim == 1,
        f"{label} must be one-dimensional.",
    )

    require(
        array.shape
        == (
            expected_length,
        ),
        (
            f"{label} shape differs from the registered "
            "grid cell count."
        ),
    )

    require(
        np.isfinite(
            array.astype(
                np.float64,
                copy=False,
            )
        ).all(),
        f"{label} contains non-finite values.",
    )

    frozen = array.copy()

    frozen.setflags(
        write=False
    )

    return frozen


def reconstruct_cell_exposure_ns(
    *,
    contract_start_ns: int,
    contract_end_exclusive_ns: int,
    grid_width_ns: int,
    cell_count: int,
    partition_name: str,
) -> np.ndarray:
    """Reconstruct exact cell exposure from half-open boundaries."""
    duration_ns = (
        contract_end_exclusive_ns
        - contract_start_ns
    )

    require(
        duration_ns > 0,
        (
            f"{partition_name} contract duration "
            "must be positive."
        ),
    )

    expected_cell_count = (
        diagnostic_exact_ceil_division(
            duration_ns,
            grid_width_ns,
        )
    )

    require(
        expected_cell_count
        == cell_count,
        (
            f"{partition_name} registered cell count differs "
            "from exact integer-nanosecond grid geometry."
        ),
    )

    exposure_ns = np.full(
        cell_count,
        grid_width_ns,
        dtype=np.int64,
    )

    final_cell_exposure_ns = (
        duration_ns
        - grid_width_ns
        * (
            cell_count
            - 1
        )
    )

    require(
        0
        < final_cell_exposure_ns
        <= grid_width_ns,
        (
            f"{partition_name} final cell exposure "
            "is invalid."
        ),
    )

    exposure_ns[
        -1
    ] = final_cell_exposure_ns

    require(
        int(
            exposure_ns.sum(
                dtype=np.int64
            )
        )
        == duration_ns,
        (
            f"{partition_name} cell exposure does not "
            "conserve the contract duration."
        ),
    )

    exposure_ns.setflags(
        write=False
    )

    return exposure_ns


# ------------------------------------------------------------
# Normalize each analytical grid
# ------------------------------------------------------------

normalized_grid_inputs: dict[
    str,
    CommonGridDiagnosticInputs,
] = {}

normalization_summary_records: list[
    dict[str, Any]
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    grid = HAWKES_COMMON_COUNT_GRIDS[
        partition_name
    ]

    cell_count = int(
        grid.cell_count
    )

    grid_width_ns = int(
        grid.grid_width_ns
    )

    contract_start_ns = int(
        grid.contract_start_ns
    )

    contract_end_exclusive_ns = int(
        grid.contract_end_exclusive_ns
    )

    contract_duration_ns = (
        contract_end_exclusive_ns
        - contract_start_ns
    )

    require(
        grid_width_ns
        == COUNT_GRID_WIDTH_NS,
        (
            f"{partition_name} grid width differs from "
            "the frozen one-millisecond contract."
        ),
    )

    exposure_ns = (
        reconstruct_cell_exposure_ns(
            contract_start_ns=(
                contract_start_ns
            ),
            contract_end_exclusive_ns=(
                contract_end_exclusive_ns
            ),
            grid_width_ns=(
                grid_width_ns
            ),
            cell_count=(
                cell_count
            ),
            partition_name=(
                partition_name
            ),
        )
    )

    exposure_seconds = (
        exposure_ns.astype(
            np.float64
        )
        / NANOSECONDS_PER_SECOND
    )

    exposure_seconds.setflags(
        write=False
    )

    registered_total_exposure_array = np.asarray(
        grid.exposure_seconds,
        dtype=np.float64,
    )

    require(
        registered_total_exposure_array.size
        == 1,
        (
            f"{partition_name} registered exposure_seconds "
            "must be one scalar total exposure."
        ),
    )

    registered_total_exposure_seconds = float(
        registered_total_exposure_array.reshape(
            -1
        )[
            0
        ]
    )

    reconstructed_total_exposure_seconds = float(
        exposure_seconds.sum(
            dtype=np.float64
        )
    )

    require(
        math.isclose(
            reconstructed_total_exposure_seconds,
            registered_total_exposure_seconds,
            rel_tol=1e-13,
            abs_tol=1e-12,
        ),
        (
            f"{partition_name} reconstructed cell exposure "
            "does not reconcile with the registered scalar "
            "partition exposure."
        ),
    )

    buy_counts = (
        immutable_one_dimensional_array(
            grid.buy_counts,
            dtype=np.int64,
            expected_length=cell_count,
            label=(
                f"{partition_name} buy_counts"
            ),
        )
    )

    sell_counts = (
        immutable_one_dimensional_array(
            grid.sell_counts,
            dtype=np.int64,
            expected_length=cell_count,
            label=(
                f"{partition_name} sell_counts"
            ),
        )
    )

    pooled_counts = (
        immutable_one_dimensional_array(
            grid.pooled_counts,
            dtype=np.int64,
            expected_length=cell_count,
            label=(
                f"{partition_name} pooled_counts"
            ),
        )
    )

    expected_buy_counts = (
        immutable_one_dimensional_array(
            grid.expected_buy_counts,
            dtype=np.float64,
            expected_length=cell_count,
            label=(
                f"{partition_name} expected_buy_counts"
            ),
        )
    )

    expected_sell_counts = (
        immutable_one_dimensional_array(
            grid.expected_sell_counts,
            dtype=np.float64,
            expected_length=cell_count,
            label=(
                f"{partition_name} expected_sell_counts"
            ),
        )
    )

    expected_pooled_counts = (
        immutable_one_dimensional_array(
            grid.expected_pooled_counts,
            dtype=np.float64,
            expected_length=cell_count,
            label=(
                f"{partition_name} expected_pooled_counts"
            ),
        )
    )

    require(
        np.all(
            buy_counts
            >= 0
        ),
        (
            f"{partition_name} BUY counts contain "
            "negative values."
        ),
    )

    require(
        np.all(
            sell_counts
            >= 0
        ),
        (
            f"{partition_name} SELL counts contain "
            "negative values."
        ),
    )

    require(
        np.array_equal(
            pooled_counts,
            buy_counts
            + sell_counts,
        ),
        (
            f"{partition_name} pooled counts do not "
            "equal BUY plus SELL counts."
        ),
    )

    require(
        np.all(
            expected_buy_counts
            > 0.0
        ),
        (
            f"{partition_name} expected BUY counts "
            "are not strictly positive."
        ),
    )

    require(
        np.all(
            expected_sell_counts
            > 0.0
        ),
        (
            f"{partition_name} expected SELL counts "
            "are not strictly positive."
        ),
    )

    require(
        np.all(
            expected_pooled_counts
            > 0.0
        ),
        (
            f"{partition_name} expected pooled counts "
            "are not strictly positive."
        ),
    )

    require(
        np.allclose(
            expected_pooled_counts,
            expected_buy_counts
            + expected_sell_counts,
            rtol=1e-13,
            atol=1e-15,
        ),
        (
            f"{partition_name} expected pooled counts "
            "do not equal expected BUY plus SELL counts."
        ),
    )

    normalized_grid_inputs[
        partition_name
    ] = CommonGridDiagnosticInputs(
        event_partition=(
            partition_name
        ),
        partition_order=int(
            PARTITION_ORDER_MAP[
                partition_name
            ]
        ),
        grid_width_ns=(
            grid_width_ns
        ),
        contract_start_ns=(
            contract_start_ns
        ),
        contract_end_exclusive_ns=(
            contract_end_exclusive_ns
        ),
        contract_duration_ns=(
            contract_duration_ns
        ),
        cell_count=(
            cell_count
        ),
        exposure_ns=(
            exposure_ns
        ),
        exposure_seconds=(
            exposure_seconds
        ),
        total_exposure_seconds=(
            reconstructed_total_exposure_seconds
        ),
        buy_counts=(
            buy_counts
        ),
        sell_counts=(
            sell_counts
        ),
        pooled_counts=(
            pooled_counts
        ),
        expected_buy_counts=(
            expected_buy_counts
        ),
        expected_sell_counts=(
            expected_sell_counts
        ),
        expected_pooled_counts=(
            expected_pooled_counts
        ),
    )

    normalization_summary_records.append(
        {
            "partition_order": int(
                PARTITION_ORDER_MAP[
                    partition_name
                ]
            ),
            "event_partition": (
                partition_name
            ),
            "grid_width_ns": (
                grid_width_ns
            ),
            "cell_count": (
                cell_count
            ),
            "contract_duration_ns": (
                contract_duration_ns
            ),
            "full_width_cell_count": int(
                np.sum(
                    exposure_ns
                    == grid_width_ns
                )
            ),
            "partial_width_cell_count": int(
                np.sum(
                    exposure_ns
                    < grid_width_ns
                )
            ),
            "final_cell_exposure_ns": int(
                exposure_ns[
                    -1
                ]
            ),
            "registered_total_exposure_seconds": (
                registered_total_exposure_seconds
            ),
            "reconstructed_total_exposure_seconds": (
                reconstructed_total_exposure_seconds
            ),
            "observed_buy_event_count": int(
                buy_counts.sum(
                    dtype=np.int64
                )
            ),
            "observed_sell_event_count": int(
                sell_counts.sum(
                    dtype=np.int64
                )
            ),
            "observed_pooled_event_count": int(
                pooled_counts.sum(
                    dtype=np.int64
                )
            ),
            "predicted_buy_event_count": float(
                expected_buy_counts.sum(
                    dtype=np.float64
                )
            ),
            "predicted_sell_event_count": float(
                expected_sell_counts.sum(
                    dtype=np.float64
                )
            ),
            "predicted_pooled_event_count": float(
                expected_pooled_counts.sum(
                    dtype=np.float64
                )
            ),
            "status": "PASS",
        }
    )


# ------------------------------------------------------------
# Publish normalized interface
# ------------------------------------------------------------

HAWKES_COMMON_GRID_DIAGNOSTIC_INPUTS: Final[
    Mapping[
        str,
        CommonGridDiagnosticInputs,
    ]
] = dict(
    normalized_grid_inputs
)

COMMON_GRID_DIAGNOSTIC_INPUT_SUMMARY = (
    pd.DataFrame.from_records(
        normalization_summary_records
    )
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Verification matrix
# ------------------------------------------------------------

COMMON_GRID_DIAGNOSTIC_INPUT_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "all_analytical_partitions_normalized"
            ),
            "passed": (
                set(
                    HAWKES_COMMON_GRID_DIAGNOSTIC_INPUTS
                )
                == set(
                    ANALYTICAL_PARTITIONS
                )
            ),
        },
        {
            "check_name": (
                "cell_exposure_vectors_match_grid_shape"
            ),
            "passed": all(
                inputs.exposure_seconds.shape
                == (
                    inputs.cell_count,
                )
                for inputs in (
                    HAWKES_COMMON_GRID_DIAGNOSTIC_INPUTS
                    .values()
                )
            ),
        },
        {
            "check_name": (
                "integer_nanosecond_exposure_conserved"
            ),
            "passed": all(
                int(
                    inputs.exposure_ns.sum(
                        dtype=np.int64
                    )
                )
                == inputs.contract_duration_ns
                for inputs in (
                    HAWKES_COMMON_GRID_DIAGNOSTIC_INPUTS
                    .values()
                )
            ),
        },
        {
            "check_name": (
                "scalar_total_exposure_reconciled"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "observed_buy_sell_counts_reconcile"
            ),
            "passed": all(
                np.array_equal(
                    inputs.pooled_counts,
                    inputs.buy_counts
                    + inputs.sell_counts,
                )
                for inputs in (
                    HAWKES_COMMON_GRID_DIAGNOSTIC_INPUTS
                    .values()
                )
            ),
        },
        {
            "check_name": (
                "expected_buy_sell_counts_reconcile"
            ),
            "passed": all(
                np.allclose(
                    inputs.expected_pooled_counts,
                    inputs.expected_buy_counts
                    + inputs.expected_sell_counts,
                    rtol=1e-13,
                    atol=1e-15,
                )
                for inputs in (
                    HAWKES_COMMON_GRID_DIAGNOSTIC_INPUTS
                    .values()
                )
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    COMMON_GRID_DIAGNOSTIC_INPUT_VERIFICATION[
        "passed"
    ].all(),
    (
        "At least one common-grid diagnostic-input "
        "normalization check failed."
    ),
)


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

COMMON_GRID_DIAGNOSTIC_INPUTS_NORMALIZED = True
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is False,
    "This cell performed a filesystem write.",
)


display(
    COMMON_GRID_DIAGNOSTIC_INPUT_SUMMARY
)

display(
    COMMON_GRID_DIAGNOSTIC_INPUT_VERIFICATION
)

print(
    "Common-grid diagnostic inputs were normalized successfully. "
    "The registered grid exposure_seconds field was retained as a "
    "scalar partition total, while exact per-cell exposure vectors "
    "were reconstructed from integer-nanosecond half-open contract "
    "geometry. BUY, SELL, pooled, and expected-count arrays all "
    "match the registered cell counts. No filesystem writes were "
    "performed."
)

,partition_order,event_partition,grid_width_ns,cell_count,contract_duration_ns,full_width_cell_count,partial_width_cell_count,final_cell_exposure_ns,registered_total_exposure_seconds,reconstructed_total_exposure_seconds,observed_buy_event_count,observed_sell_event_count,observed_pooled_event_count,predicted_buy_event_count,predicted_sell_event_count,predicted_pooled_event_count,status
0,1,DEVELOPMENT,1000000,1801860,1801859586700,1801859,1,586700,"1,801.859587","1,801.859587",3414,3590,7004,"3,414.000015","3,590.00007","7,004.000086",PASS
1,2,CALIBRATION,1000000,720300,720299179100,720299,1,179100,720.2991791,720.2991791,1230,1263,2493,"1,339.253892","1,415.723094","2,754.976987",PASS


,check_name,passed
0,all_analytical_partitions_normalized,True
1,cell_exposure_vectors_match_grid_shape,True
2,integer_nanosecond_exposure_conserved,True
3,scalar_total_exposure_reconciled,True
4,observed_buy_sell_counts_reconcile,True
5,expected_buy_sell_counts_reconcile,True
6,calibration_parameter_updates_zero,True
7,protected_partitions_unopened,True
8,filesystem_writes_absent,True


Common-grid diagnostic inputs were normalized successfully. The registered grid exposure_seconds field was retained as a scalar partition total, while exact per-cell exposure vectors were reconstructed from integer-nanosecond half-open contract geometry. BUY, SELL, pooled, and expected-count arrays all match the registered cell counts. No filesystem writes were performed.


In [25]:
# ============================================================
# Verified Notebook 06 baseline readback and locked H1 comparison
#
# LOCKED_CALIBRATION_COUNT_BASELINE_RANKING was a runtime table
# in Notebook 06 but was not persisted by its table-discovery
# registry. It is therefore reconstructed deterministically from
# the persisted UNIFIED_COUNT_BASELINE_SCORE_TABLE.
#
# H1 and every baseline are compared in the identical native
# bivariate BUY/SELL one-millisecond Poisson-count score space.
# Formal uncertainty uses Notebook 06's exact complete
# CALIBRATION block geometry and paired resampling.
# ============================================================

import hashlib
import json
from pathlib import Path


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "COMMON_GRID_DIAGNOSTIC_INPUTS_NORMALIZED",
            False,
        )
    ),
    "Common-grid diagnostic inputs have not been normalized.",
)

require(
    bool(
        globals().get(
            "COARSENING_AWARE_MODEL_ENVELOPES_BUILT",
            False,
        )
    ),
    "Coarsening-aware model envelopes have not been built.",
)

require(
    isinstance(
        globals().get(
            "nb06_handoff"
        ),
        dict,
    ),
    "The verified Notebook 06 handoff is unavailable.",
)

require(
    not bool(
        globals().get(
            "HAWKES_BASELINE_COUNT_COMPARISON_BUILT",
            False,
        )
    ),
    "The Hawkes-versus-baseline count comparison is already built.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Local verification helpers
# ------------------------------------------------------------

def local_file_sha256(
    source_path: Path,
) -> str:
    """Return the SHA-256 digest of one file."""
    digest = hashlib.sha256()

    with source_path.open(
        "rb"
    ) as source_file:
        for chunk in iter(
            lambda: source_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def local_canonical_json_sha256(
    payload: dict[str, Any],
) -> str:
    """Hash an already JSON-compatible mapping canonically."""
    serialized = json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        ensure_ascii=False,
        allow_nan=False,
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        serialized
    ).hexdigest()


def local_path_is_inside(
    candidate_path: Path,
    parent_path: Path,
) -> bool:
    """Return whether candidate is inside parent."""
    try:
        candidate_path.resolve(
            strict=False
        ).relative_to(
            parent_path.resolve(
                strict=False
            )
        )

        return True

    except ValueError:
        return False


def normalize_loaded_boolean(
    values: pd.Series,
    *,
    label: str,
) -> pd.Series:
    """Normalize a persisted Boolean column."""
    if pd.api.types.is_bool_dtype(
        values.dtype
    ):
        return values.astype(
            bool
        )

    normalized = (
        values.astype(
            "string"
        )
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
    )

    require(
        normalized.notna().all(),
        f"{label} contains an invalid Boolean value.",
    )

    return normalized.astype(
        bool
    )


# ------------------------------------------------------------
# Resolve and verify Notebook 06 output manifest
# ------------------------------------------------------------

nb06_control_artifacts = (
    nb06_handoff.get(
        "control_artifacts"
    )
)

require(
    isinstance(
        nb06_control_artifacts,
        dict,
    ),
    "Notebook 06 handoff lacks control_artifacts.",
)

nb06_manifest_reference = (
    nb06_control_artifacts.get(
        "output_manifest"
    )
)

require(
    isinstance(
        nb06_manifest_reference,
        dict,
    ),
    "Notebook 06 handoff lacks its output-manifest reference.",
)

NB06_OUTPUT_MANIFEST_PATH = Path(
    str(
        nb06_manifest_reference[
            "path"
        ]
    )
)

require(
    NB06_OUTPUT_MANIFEST_PATH.is_file(),
    (
        "Notebook 06 output manifest does not exist: "
        f"{NB06_OUTPUT_MANIFEST_PATH}"
    ),
)

require(
    local_path_is_inside(
        NB06_OUTPUT_MANIFEST_PATH,
        V01_ROOT,
    ),
    "Notebook 06 output manifest is outside V0.1.",
)

require(
    not local_path_is_inside(
        NB06_OUTPUT_MANIFEST_PATH,
        V00_ROOT,
    ),
    "Notebook 06 output manifest is inside immutable V0.0.",
)

registered_manifest_file_sha256 = str(
    nb06_manifest_reference[
        "sha256"
    ]
)

observed_manifest_file_sha256 = (
    local_file_sha256(
        NB06_OUTPUT_MANIFEST_PATH
    )
)

require(
    observed_manifest_file_sha256
    == registered_manifest_file_sha256,
    "Notebook 06 output-manifest file hash does not match its handoff.",
)

with NB06_OUTPUT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as manifest_file:
    nb06_output_manifest = json.load(
        manifest_file
    )

require(
    isinstance(
        nb06_output_manifest,
        dict,
    ),
    "Notebook 06 output manifest is not a JSON object.",
)

require(
    nb06_output_manifest.get(
        "artifact_type"
    )
    == "NOTEBOOK_06_OUTPUT_MANIFEST",
    "Unexpected Notebook 06 output-manifest artifact type.",
)

require(
    nb06_output_manifest.get(
        "status"
    )
    == "PASS",
    "Notebook 06 output manifest did not pass.",
)

registered_manifest_payload_sha256 = str(
    nb06_output_manifest.get(
        "payload_sha256"
    )
)

manifest_payload_without_hash = {
    str(
        key
    ): value
    for key, value in (
        nb06_output_manifest.items()
    )
    if key
    != "payload_sha256"
}

recomputed_manifest_payload_sha256 = (
    local_canonical_json_sha256(
        manifest_payload_without_hash
    )
)

require(
    recomputed_manifest_payload_sha256
    == registered_manifest_payload_sha256,
    "Notebook 06 output-manifest semantic hash is invalid.",
)

require(
    registered_manifest_payload_sha256
    == str(
        nb06_manifest_reference[
            "payload_sha256"
        ]
    ),
    (
        "Notebook 06 output-manifest payload hash differs "
        "from the verified handoff."
    ),
)

require(
    nb06_output_manifest.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX,
    "Notebook 06 manifest source-run identity differs.",
)

require(
    nb06_output_manifest.get(
        "source_set_sha256"
    )
    == SOURCE_SET_SHA256,
    "Notebook 06 manifest source-set hash differs.",
)

require(
    nb06_output_manifest.get(
        "v0_1_run_id"
    )
    == V0_1_RUN_ID,
    "Notebook 06 manifest V0.1 run ID differs.",
)

require(
    nb06_output_manifest.get(
        "run_config_sha256"
    )
    == RUN_CONFIG_SHA256,
    "Notebook 06 manifest run-config hash differs.",
)

require(
    nb06_output_manifest.get(
        "run_identity_sha256"
    )
    == RUN_IDENTITY_SHA256,
    "Notebook 06 manifest run-identity hash differs.",
)


# ------------------------------------------------------------
# Index only actually persisted Notebook 06 result tables
# ------------------------------------------------------------

nb06_artifact_records = (
    nb06_output_manifest.get(
        "artifacts"
    )
)

require(
    isinstance(
        nb06_artifact_records,
        list,
    ),
    "Notebook 06 output manifest lacks an artifact list.",
)

manifest_table_records: dict[
    str,
    dict[str, Any],
] = {}

for artifact_record in (
    nb06_artifact_records
):
    if not isinstance(
        artifact_record,
        dict,
    ):
        continue

    if (
        artifact_record.get(
            "artifact_type"
        )
        != "NOTEBOOK_06_RESULT_TABLE"
    ):
        continue

    variable_name = str(
        artifact_record.get(
            "variable_name",
            "",
        )
    ).strip()

    require(
        variable_name,
        "A Notebook 06 table artifact lacks variable_name.",
    )

    require(
        variable_name
        not in manifest_table_records,
        (
            "Notebook 06 output manifest contains duplicate "
            f"table records for {variable_name}."
        ),
    )

    manifest_table_records[
        variable_name
    ] = artifact_record


NB06_REQUIRED_PERSISTED_TABLE_NAMES: Final[
    tuple[str, ...]
] = (
    "COUNT_BASELINE_MODEL_REGISTRY",
    "UNIFIED_COUNT_BASELINE_SCORE_TABLE",
    "CALIBRATION_COUNT_SCORE_BLOCKS",
    "BOOTSTRAP_COVERAGE_AUDIT",
)

NB06_OPTIONAL_PERSISTED_TABLE_NAMES: Final[
    tuple[str, ...]
] = (
    "COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON",
    "COUNT_SCORE_RECONCILIATION",
)

missing_required_persisted_tables = sorted(
    set(
        NB06_REQUIRED_PERSISTED_TABLE_NAMES
    )
    - set(
        manifest_table_records
    )
)

require(
    not missing_required_persisted_tables,
    (
        "Notebook 06 output manifest lacks required persisted "
        f"source tables: {missing_required_persisted_tables}"
    ),
)


# ------------------------------------------------------------
# Verified table readback
# ------------------------------------------------------------

NB06_BASELINE_TABLES: dict[
    str,
    pd.DataFrame,
] = {}

nb06_table_audit_records: list[
    dict[str, Any]
] = []

tables_to_load = (
    list(
        NB06_REQUIRED_PERSISTED_TABLE_NAMES
    )
    + [
        table_name
        for table_name in (
            NB06_OPTIONAL_PERSISTED_TABLE_NAMES
        )
        if table_name
        in manifest_table_records
    ]
)

for table_name in (
    tables_to_load
):
    artifact_record = (
        manifest_table_records[
            table_name
        ]
    )

    table_path = Path(
        str(
            artifact_record[
                "path"
            ]
        )
    )

    require(
        table_path.is_file(),
        (
            f"Persisted Notebook 06 table does not exist: "
            f"{table_name} -> {table_path}"
        ),
    )

    require(
        local_path_is_inside(
            table_path,
            V01_ROOT,
        ),
        f"{table_name} is outside V0.1.",
    )

    require(
        not local_path_is_inside(
            table_path,
            V00_ROOT,
        ),
        f"{table_name} is inside immutable V0.0.",
    )

    observed_file_sha256 = (
        local_file_sha256(
            table_path
        )
    )

    registered_file_sha256 = str(
        artifact_record[
            "sha256"
        ]
    )

    require(
        observed_file_sha256
        == registered_file_sha256,
        f"{table_name} file hash does not match its manifest.",
    )

    require(
        artifact_record.get(
            "format"
        )
        == "CSV_GZIP",
        f"{table_name} has an unexpected persisted format.",
    )

    loaded_table = pd.read_csv(
        table_path,
        compression="gzip",
        low_memory=False,
    )

    expected_columns = [
        str(
            column_name
        )
        for column_name in (
            artifact_record[
                "columns"
            ]
        )
    ]

    require(
        len(
            loaded_table
        )
        == int(
            artifact_record[
                "row_count"
            ]
        ),
        f"{table_name} row count does not match its manifest.",
    )

    require(
        loaded_table.shape[
            1
        ]
        == int(
            artifact_record[
                "column_count"
            ]
        ),
        f"{table_name} column count does not match its manifest.",
    )

    require(
        [
            str(
                column_name
            )
            for column_name in (
                loaded_table.columns
            )
        ]
        == expected_columns,
        f"{table_name} column order does not match its manifest.",
    )

    NB06_BASELINE_TABLES[
        table_name
    ] = loaded_table

    nb06_table_audit_records.append(
        {
            "table_name": table_name,
            "row_count": int(
                len(
                    loaded_table
                )
            ),
            "column_count": int(
                loaded_table.shape[
                    1
                ]
            ),
            "file_hash_matches": True,
            "row_count_matches": True,
            "column_count_matches": True,
            "column_order_matches": True,
            "status": "PASS",
        }
    )


NB06_BASELINE_TABLE_READBACK_AUDIT = (
    pd.DataFrame.from_records(
        nb06_table_audit_records
    )
    .sort_values(
        "table_name",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NB06_BASELINE_TABLE_READBACK_AUDIT[
        "status"
    ].eq(
        "PASS"
    ).all(),
    "At least one Notebook 06 baseline table failed readback.",
)


# ------------------------------------------------------------
# Normalize loaded Notebook 06 tables
# ------------------------------------------------------------

COUNT_BASELINE_MODEL_REGISTRY = (
    NB06_BASELINE_TABLES[
        "COUNT_BASELINE_MODEL_REGISTRY"
    ].copy()
)

UNIFIED_COUNT_BASELINE_SCORE_TABLE = (
    NB06_BASELINE_TABLES[
        "UNIFIED_COUNT_BASELINE_SCORE_TABLE"
    ].copy()
)

CALIBRATION_COUNT_SCORE_BLOCKS = (
    NB06_BASELINE_TABLES[
        "CALIBRATION_COUNT_SCORE_BLOCKS"
    ].copy()
)

BOOTSTRAP_COVERAGE_AUDIT = (
    NB06_BASELINE_TABLES[
        "BOOTSTRAP_COVERAGE_AUDIT"
    ].copy()
)

for boolean_column in (
    "primary_simple_count_baseline_flag",
    "formal_count_comparator_flag",
    "calibration_used_for_selection",
):
    require(
        boolean_column
        in COUNT_BASELINE_MODEL_REGISTRY.columns,
        (
            "Notebook 06 model registry lacks "
            f"{boolean_column}."
        ),
    )

    COUNT_BASELINE_MODEL_REGISTRY[
        boolean_column
    ] = normalize_loaded_boolean(
        COUNT_BASELINE_MODEL_REGISTRY[
            boolean_column
        ],
        label=(
            "COUNT_BASELINE_MODEL_REGISTRY."
            f"{boolean_column}"
        ),
    )

for boolean_column in (
    "primary_simple_count_baseline_flag",
    "formal_count_comparator_flag",
    "calibration_used_for_selection",
    "locked_before_calibration",
):
    require(
        boolean_column
        in UNIFIED_COUNT_BASELINE_SCORE_TABLE.columns,
        (
            "Notebook 06 unified score table lacks "
            f"{boolean_column}."
        ),
    )

    UNIFIED_COUNT_BASELINE_SCORE_TABLE[
        boolean_column
    ] = normalize_loaded_boolean(
        UNIFIED_COUNT_BASELINE_SCORE_TABLE[
            boolean_column
        ],
        label=(
            "UNIFIED_COUNT_BASELINE_SCORE_TABLE."
            f"{boolean_column}"
        ),
    )

CALIBRATION_COUNT_SCORE_BLOCKS[
    "complete_equal_exposure_block_flag"
] = normalize_loaded_boolean(
    CALIBRATION_COUNT_SCORE_BLOCKS[
        "complete_equal_exposure_block_flag"
    ],
    label=(
        "CALIBRATION_COUNT_SCORE_BLOCKS."
        "complete_equal_exposure_block_flag"
    ),
)

for boolean_column in (
    "equal_exposure_blocks_only",
    "partial_final_block_excluded_from_ci",
    "full_calibration_score_still_reported",
):
    BOOTSTRAP_COVERAGE_AUDIT[
        boolean_column
    ] = normalize_loaded_boolean(
        BOOTSTRAP_COVERAGE_AUDIT[
            boolean_column
        ],
        label=(
            "BOOTSTRAP_COVERAGE_AUDIT."
            f"{boolean_column}"
        ),
    )


# ------------------------------------------------------------
# Resolve formal and primary baseline identities
# ------------------------------------------------------------

formal_comparator_rows = (
    COUNT_BASELINE_MODEL_REGISTRY.loc[
        COUNT_BASELINE_MODEL_REGISTRY[
            "formal_count_comparator_flag"
        ]
    ]
)

primary_simple_rows = (
    COUNT_BASELINE_MODEL_REGISTRY.loc[
        COUNT_BASELINE_MODEL_REGISTRY[
            "primary_simple_count_baseline_flag"
        ]
    ]
)

require(
    len(
        formal_comparator_rows
    )
    == 1,
    "Notebook 06 must identify exactly one formal comparator.",
)

require(
    len(
        primary_simple_rows
    )
    == 1,
    "Notebook 06 must identify exactly one primary simple baseline.",
)

FORMAL_COUNT_COMPARATOR_MODEL_ID = str(
    formal_comparator_rows.iloc[
        0
    ][
        "model_id"
    ]
)

PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID = str(
    primary_simple_rows.iloc[
        0
    ][
        "model_id"
    ]
)

require(
    FORMAL_COUNT_COMPARATOR_MODEL_ID
    == "D0_SIDE_CONSTANT_POISSON",
    "Unexpected Notebook 06 formal comparator identity.",
)

require(
    PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
    == "E_SIDE_EWMA_250MS_POISSON",
    "Unexpected Notebook 06 primary simple baseline identity.",
)


# ------------------------------------------------------------
# Reconstruct the non-persisted locked CALIBRATION ranking
# ------------------------------------------------------------

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING = (
    UNIFIED_COUNT_BASELINE_SCORE_TABLE.loc[
        UNIFIED_COUNT_BASELINE_SCORE_TABLE[
            "scored_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .copy()
    .sort_values(
        [
            "log_score_total",
            "model_id",
        ],
        ascending=[
            False,
            True,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    not LOCKED_CALIBRATION_COUNT_BASELINE_RANKING.empty,
    "Notebook 06 contains no locked CALIBRATION baseline scores.",
)

require(
    LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
        "locked_before_calibration"
    ].all(),
    "A ranked Notebook 06 baseline was not locked before CALIBRATION.",
)

require(
    not LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
        "calibration_used_for_selection"
    ].any(),
    "A Notebook 06 baseline used CALIBRATION for selection.",
)

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
    "calibration_score_rank"
] = np.arange(
    1,
    len(
        LOCKED_CALIBRATION_COUNT_BASELINE_RANKING
    )
    + 1,
    dtype=np.int64,
)

formal_comparator_calibration_score = float(
    LOCKED_CALIBRATION_COUNT_BASELINE_RANKING.loc[
        LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
            "model_id"
        ].eq(
            FORMAL_COUNT_COMPARATOR_MODEL_ID
        ),
        "log_score_total",
    ].iloc[
        0
    ]
)

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
    "log_score_improvement_over_formal_comparator"
] = (
    LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
        "log_score_total"
    ]
    - formal_comparator_calibration_score
)

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
    "ranking_role"
] = (
    "RECONSTRUCTED_FROM_PERSISTED_UNIFIED_SCORE_TABLE"
)


# ------------------------------------------------------------
# Compute frozen H1 scores in Notebook 06's exact score space
# ------------------------------------------------------------

HAWKES_BASELINE_SCORE_SPACE: Final[str] = (
    "BIVARIATE_SIDE_COUNT_NATIVE_1MS"
)

h1_partition_score_records: list[
    dict[str, Any]
] = []

H1_COMMON_GRID_CELL_LOG_SCORES: dict[
    str,
    np.ndarray,
] = {}

for partition_name in (
    ANALYTICAL_PARTITIONS
):
    diagnostic_inputs = (
        HAWKES_COMMON_GRID_DIAGNOSTIC_INPUTS[
            partition_name
        ]
    )

    buy_log_score_terms = (
        poisson_log_score_terms(
            diagnostic_inputs.buy_counts,
            diagnostic_inputs.expected_buy_counts,
        )
    )

    sell_log_score_terms = (
        poisson_log_score_terms(
            diagnostic_inputs.sell_counts,
            diagnostic_inputs.expected_sell_counts,
        )
    )

    joint_log_score_terms = (
        buy_log_score_terms
        + sell_log_score_terms
    )

    require(
        joint_log_score_terms.shape
        == (
            diagnostic_inputs.cell_count,
        ),
        (
            f"{partition_name} H1 joint score shape differs "
            "from the normalized common grid."
        ),
    )

    require(
        np.isfinite(
            joint_log_score_terms
        ).all(),
        f"{partition_name} H1 cell scores contain non-finite values.",
    )

    frozen_joint_scores = (
        joint_log_score_terms.copy()
    )

    frozen_joint_scores.setflags(
        write=False
    )

    H1_COMMON_GRID_CELL_LOG_SCORES[
        partition_name
    ] = frozen_joint_scores

    observed_event_count = int(
        diagnostic_inputs
        .pooled_counts
        .sum(
            dtype=np.int64
        )
    )

    predicted_event_count = float(
        diagnostic_inputs
        .expected_pooled_counts
        .sum(
            dtype=np.float64
        )
    )

    total_log_score = float(
        frozen_joint_scores.sum(
            dtype=np.float64
        )
    )

    h1_partition_score_records.append(
        {
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "model_family": (
                EXPECTED_SELECTED_MODEL_FAMILY
            ),
            "declared_roles": (
                "FROZEN_RESTRICTED_HAWKES_CANDIDATE"
            ),
            "fitted_partition": (
                "DEVELOPMENT"
            ),
            "scored_partition": (
                partition_name
            ),
            "score_space": (
                HAWKES_BASELINE_SCORE_SPACE
            ),
            "native_grid_width_ms": 1,
            "cell_count": int(
                diagnostic_inputs.cell_count
            ),
            "exposure_seconds": float(
                diagnostic_inputs
                .total_exposure_seconds
            ),
            "observed_event_count": (
                observed_event_count
            ),
            "predicted_event_count": (
                predicted_event_count
            ),
            "observed_minus_predicted": (
                observed_event_count
                - predicted_event_count
            ),
            "log_score_total": (
                total_log_score
            ),
            "log_score_per_second": (
                total_log_score
                / diagnostic_inputs
                .total_exposure_seconds
            ),
            "log_score_per_observed_event": (
                total_log_score
                / observed_event_count
            ),
            "primary_simple_count_baseline_flag": False,
            "formal_count_comparator_flag": False,
            "calibration_used_for_selection": False,
            "locked_before_calibration": True,
            "status": "PASS",
        }
    )


HAWKES_COMMON_GRID_COMPARABLE_SCORE_TABLE = (
    pd.DataFrame.from_records(
        h1_partition_score_records
    )
    .sort_values(
        "scored_partition",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    set(
        HAWKES_COMMON_GRID_COMPARABLE_SCORE_TABLE[
            "scored_partition"
        ]
    )
    == set(
        ANALYTICAL_PARTITIONS
    ),
    "The comparable H1 score table lacks an analytical partition.",
)


# ------------------------------------------------------------
# Combined locked CALIBRATION ranking
# ------------------------------------------------------------

h1_calibration_score_row = (
    HAWKES_COMMON_GRID_COMPARABLE_SCORE_TABLE.loc[
        HAWKES_COMMON_GRID_COMPARABLE_SCORE_TABLE[
            "scored_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .copy()
)

require(
    len(
        h1_calibration_score_row
    )
    == 1,
    "Expected exactly one locked H1 CALIBRATION score.",
)

HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING = (
    pd.concat(
        [
            LOCKED_CALIBRATION_COUNT_BASELINE_RANKING.drop(
                columns=[
                    "calibration_score_rank",
                    "log_score_improvement_over_formal_comparator",
                    "ranking_role",
                ],
                errors="ignore",
            ),
            h1_calibration_score_row,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "log_score_total",
            "model_id",
        ],
        ascending=[
            False,
            True,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
    "calibration_score_rank"
] = np.arange(
    1,
    len(
        HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING
    )
    + 1,
    dtype=np.int64,
)

HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
    "log_score_improvement_over_formal_comparator"
] = (
    HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
        "log_score_total"
    ]
    - formal_comparator_calibration_score
)

primary_simple_calibration_score = float(
    HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING.loc[
        HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
            "model_id"
        ].eq(
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
        ),
        "log_score_total",
    ].iloc[
        0
    ]
)

HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
    "log_score_improvement_over_primary_simple_baseline"
] = (
    HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
        "log_score_total"
    ]
    - primary_simple_calibration_score
)

HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
    "candidate_role"
] = np.where(
    HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
        "model_id"
    ].eq(
        EXPECTED_SELECTED_MODEL_ID
    ),
    "FROZEN_HAWKES",
    "NOTEBOOK_06_BASELINE",
)

HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
    "score_only_superiority_claim_authorized"
] = False

HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
    "status"
] = "PASS_SCORE_COMPARISON_ONLY"


# ------------------------------------------------------------
# Exact Notebook 06 block geometry
# ------------------------------------------------------------

require(
    len(
        BOOTSTRAP_COVERAGE_AUDIT
    )
    == 1,
    "Notebook 06 bootstrap coverage audit must contain one row.",
)

bootstrap_coverage_row = (
    BOOTSTRAP_COVERAGE_AUDIT.iloc[
        0
    ]
)

require(
    str(
        bootstrap_coverage_row[
            "scored_partition"
        ]
    )
    == "CALIBRATION",
    "Notebook 06 formal bootstrap partition is not CALIBRATION.",
)

require(
    bool(
        bootstrap_coverage_row[
            "equal_exposure_blocks_only"
        ]
    ),
    "Notebook 06 bootstrap did not use equal-exposure blocks.",
)

require(
    bool(
        bootstrap_coverage_row[
            "partial_final_block_excluded_from_ci"
        ]
    ),
    "Notebook 06 did not exclude the partial final block.",
)

formal_complete_block_count = int(
    bootstrap_coverage_row[
        "complete_block_count"
    ]
)

cells_per_complete_block = int(
    bootstrap_coverage_row[
        "cells_per_complete_block"
    ]
)

formal_cell_count = int(
    bootstrap_coverage_row[
        "formal_cell_count"
    ]
)

formal_exposure_ns = int(
    bootstrap_coverage_row[
        "formal_exposure_ns"
    ]
)

excluded_tail_exposure_ns = int(
    bootstrap_coverage_row[
        "excluded_tail_exposure_ns"
    ]
)

block_width_ns = int(
    bootstrap_coverage_row[
        "block_width_ns"
    ]
)

require(
    formal_complete_block_count
    >= 2,
    "At least two complete CALIBRATION blocks are required.",
)

require(
    formal_cell_count
    == (
        formal_complete_block_count
        * cells_per_complete_block
    ),
    "Notebook 06 formal bootstrap cell geometry is inconsistent.",
)

require(
    block_width_ns
    == (
        cells_per_complete_block
        * COUNT_GRID_WIDTH_NS
    ),
    "Notebook 06 block width differs from its cell geometry.",
)

require(
    formal_exposure_ns
    == (
        formal_complete_block_count
        * block_width_ns
    ),
    "Notebook 06 formal bootstrap exposure is inconsistent.",
)


# ------------------------------------------------------------
# Verify baseline block-table geometry
# ------------------------------------------------------------

required_block_columns: Final[
    set[str]
] = {
    "model_id",
    "scored_partition",
    "block_index",
    "block_start_ns",
    "block_end_exclusive_ns",
    "block_exposure_seconds",
    "log_score_total",
    "complete_equal_exposure_block_flag",
    "status",
}

missing_block_columns = sorted(
    required_block_columns
    - set(
        CALIBRATION_COUNT_SCORE_BLOCKS.columns
    )
)

require(
    not missing_block_columns,
    (
        "Notebook 06 CALIBRATION block table lacks columns: "
        f"{missing_block_columns}"
    ),
)

require(
    CALIBRATION_COUNT_SCORE_BLOCKS[
        "scored_partition"
    ].eq(
        "CALIBRATION"
    ).all(),
    "Notebook 06 block table contains a non-CALIBRATION row.",
)

require(
    CALIBRATION_COUNT_SCORE_BLOCKS[
        "complete_equal_exposure_block_flag"
    ].all(),
    "Notebook 06 block table contains a non-complete block.",
)

require(
    CALIBRATION_COUNT_SCORE_BLOCKS[
        "status"
    ].eq(
        "PASS"
    ).all(),
    "Notebook 06 block table contains a failed row.",
)

baseline_model_ids = tuple(
    COUNT_BASELINE_MODEL_REGISTRY[
        "model_id"
    ].astype(
        str
    )
)

require(
    set(
        CALIBRATION_COUNT_SCORE_BLOCKS[
            "model_id"
        ].astype(
            str
        )
    )
    == set(
        baseline_model_ids
    ),
    "Notebook 06 block-score model set differs from its registry.",
)

block_counts_by_model = (
    CALIBRATION_COUNT_SCORE_BLOCKS.groupby(
        "model_id",
        observed=True,
        sort=True,
        dropna=False,
    )[
        "block_index"
    ]
    .nunique()
)

require(
    block_counts_by_model.eq(
        formal_complete_block_count
    ).all(),
    "A Notebook 06 baseline lacks complete formal blocks.",
)

block_geometry_consistency = (
    CALIBRATION_COUNT_SCORE_BLOCKS.groupby(
        "block_index",
        observed=True,
        sort=True,
        dropna=False,
    )
    .agg(
        block_start_ns_nunique=(
            "block_start_ns",
            "nunique",
        ),
        block_end_exclusive_ns_nunique=(
            "block_end_exclusive_ns",
            "nunique",
        ),
        block_exposure_seconds_nunique=(
            "block_exposure_seconds",
            "nunique",
        ),
        model_count=(
            "model_id",
            "nunique",
        ),
    )
    .reset_index()
)

require(
    block_geometry_consistency[
        [
            "block_start_ns_nunique",
            "block_end_exclusive_ns_nunique",
            "block_exposure_seconds_nunique",
        ]
    ].eq(
        1
    ).all().all(),
    "Notebook 06 baseline models do not share identical block geometry.",
)

require(
    block_geometry_consistency[
        "model_count"
    ].eq(
        len(
            baseline_model_ids
        )
    ).all(),
    "At least one formal block lacks a baseline model.",
)

formal_block_geometry = (
    CALIBRATION_COUNT_SCORE_BLOCKS[
        [
            "block_index",
            "block_start_ns",
            "block_end_exclusive_ns",
            "block_exposure_seconds",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "block_index",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    len(
        formal_block_geometry
    )
    == formal_complete_block_count,
    "Formal block geometry has an incorrect block count.",
)

require(
    formal_block_geometry[
        "block_index"
    ].to_numpy(
        dtype=np.int64
    ).tolist()
    == list(
        range(
            formal_complete_block_count
        )
    ),
    "Formal block indices are not contiguous from zero.",
)


# ------------------------------------------------------------
# Construct H1 scores on the exact same formal blocks
# ------------------------------------------------------------

h1_calibration_cell_scores = (
    H1_COMMON_GRID_CELL_LOG_SCORES[
        "CALIBRATION"
    ]
)

require(
    len(
        h1_calibration_cell_scores
    )
    >= formal_cell_count,
    "H1 CALIBRATION score coverage is shorter than the formal blocks.",
)

h1_formal_cell_scores = (
    h1_calibration_cell_scores[
        :formal_cell_count
    ]
)

h1_excluded_tail_cell_scores = (
    h1_calibration_cell_scores[
        formal_cell_count:
    ]
)

h1_block_scores = (
    h1_formal_cell_scores.reshape(
        formal_complete_block_count,
        cells_per_complete_block,
    )
    .sum(
        axis=1,
        dtype=np.float64,
    )
)

require(
    np.isfinite(
        h1_block_scores
    ).all(),
    "H1 formal block scores contain non-finite values.",
)

require(
    math.isclose(
        float(
            h1_calibration_cell_scores.sum(
                dtype=np.float64
            )
        ),
        (
            float(
                h1_block_scores.sum(
                    dtype=np.float64
                )
            )
            + float(
                h1_excluded_tail_cell_scores.sum(
                    dtype=np.float64
                )
            )
        ),
        rel_tol=1e-13,
        abs_tol=1e-9,
    ),
    "H1 complete-block and excluded-tail scores do not reconcile.",
)

H1_CALIBRATION_COUNT_SCORE_BLOCKS = (
    formal_block_geometry.copy()
)

H1_CALIBRATION_COUNT_SCORE_BLOCKS.insert(
    0,
    "model_id",
    EXPECTED_SELECTED_MODEL_ID,
)

H1_CALIBRATION_COUNT_SCORE_BLOCKS.insert(
    1,
    "scored_partition",
    "CALIBRATION",
)

H1_CALIBRATION_COUNT_SCORE_BLOCKS[
    "log_score_total"
] = h1_block_scores

H1_CALIBRATION_COUNT_SCORE_BLOCKS[
    "complete_equal_exposure_block_flag"
] = True

H1_CALIBRATION_COUNT_SCORE_BLOCKS[
    "status"
] = "PASS"


# ------------------------------------------------------------
# Paired complete-block bootstrap: H1 versus every baseline
# ------------------------------------------------------------

HAWKES_BASELINE_BOOTSTRAP_REPLICATES: Final[int] = int(
    globals().get(
        "N_BLOCK_BOOTSTRAP_REPLICATES",
        20_000,
    )
)

require(
    HAWKES_BASELINE_BOOTSTRAP_REPLICATES
    >= 1_000,
    "At least 1,000 paired block-bootstrap replicates are required.",
)

formal_exposure_seconds = (
    formal_exposure_ns
    / NANOSECONDS_PER_SECOND
)

baseline_block_score_lookup: dict[
    str,
    np.ndarray,
] = {}

for baseline_model_id in (
    baseline_model_ids
):
    baseline_scores = (
        CALIBRATION_COUNT_SCORE_BLOCKS.loc[
            CALIBRATION_COUNT_SCORE_BLOCKS[
                "model_id"
            ].astype(
                str
            ).eq(
                baseline_model_id
            )
        ]
        .sort_values(
            "block_index",
            kind="stable",
        )[
            "log_score_total"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    require(
        baseline_scores.shape
        == h1_block_scores.shape,
        (
            f"H1 and {baseline_model_id} formal block-score "
            "arrays differ in shape."
        ),
    )

    require(
        np.isfinite(
            baseline_scores
        ).all(),
        f"{baseline_model_id} block scores contain non-finite values.",
    )

    baseline_block_score_lookup[
        baseline_model_id
    ] = baseline_scores


def paired_block_bootstrap_h1_difference(
    baseline_scores: np.ndarray,
    *,
    replicate_count: int,
    seed_sequence: tuple[int, ...],
) -> dict[str, Any]:
    """Bootstrap H1 minus baseline total score differences."""
    baseline = np.asarray(
        baseline_scores,
        dtype=np.float64,
    )

    require(
        baseline.shape
        == h1_block_scores.shape,
        "Paired bootstrap arrays differ in shape.",
    )

    paired_difference = (
        h1_block_scores
        - baseline
    )

    rng = np.random.default_rng(
        np.random.SeedSequence(
            seed_sequence
        )
    )

    sampled_indices = rng.integers(
        low=0,
        high=paired_difference.size,
        size=(
            replicate_count,
            paired_difference.size,
        ),
        endpoint=False,
    )

    bootstrap_total_differences = (
        paired_difference[
            sampled_indices
        ]
        .sum(
            axis=1,
            dtype=np.float64,
        )
    )

    quantiles = np.quantile(
        bootstrap_total_differences,
        [
            0.025,
            0.50,
            0.975,
        ],
        method="linear",
    )

    probability_nonpositive = (
        1.0
        + float(
            np.count_nonzero(
                bootstrap_total_differences
                <= 0.0
            )
        )
    ) / (
        replicate_count
        + 1.0
    )

    probability_nonnegative = (
        1.0
        + float(
            np.count_nonzero(
                bootstrap_total_differences
                >= 0.0
            )
        )
    ) / (
        replicate_count
        + 1.0
    )

    observed_total_difference = float(
        paired_difference.sum(
            dtype=np.float64
        )
    )

    positive_supported = bool(
        quantiles[
            0
        ]
        > 0.0
    )

    negative_supported = bool(
        quantiles[
            2
        ]
        < 0.0
    )

    return {
        "observed_total_log_score_difference": (
            observed_total_difference
        ),
        "observed_log_score_difference_per_block": float(
            paired_difference.mean()
        ),
        "observed_log_score_difference_per_second": (
            observed_total_difference
            / formal_exposure_seconds
        ),
        "bootstrap_total_difference_q025": float(
            quantiles[
                0
            ]
        ),
        "bootstrap_total_difference_median": float(
            quantiles[
                1
            ]
        ),
        "bootstrap_total_difference_q975": float(
            quantiles[
                2
            ]
        ),
        "bootstrap_difference_per_second_q025": float(
            quantiles[
                0
            ]
            / formal_exposure_seconds
        ),
        "bootstrap_difference_per_second_median": float(
            quantiles[
                1
            ]
            / formal_exposure_seconds
        ),
        "bootstrap_difference_per_second_q975": float(
            quantiles[
                2
            ]
            / formal_exposure_seconds
        ),
        "bootstrap_probability_difference_nonpositive": (
            probability_nonpositive
        ),
        "bootstrap_probability_difference_nonnegative": (
            probability_nonnegative
        ),
        "positive_improvement_supported_at_95pct": (
            positive_supported
        ),
        "negative_difference_supported_at_95pct": (
            negative_supported
        ),
        "comparison_status": (
            "H1_POSITIVE_SUPPORTED"
            if positive_supported
            else (
                "H1_NEGATIVE_SUPPORTED"
                if negative_supported
                else "INCONCLUSIVE"
            )
        ),
    }


h1_baseline_bootstrap_records: list[
    dict[str, Any]
] = []

for comparison_index, baseline_model_id in enumerate(
    baseline_model_ids,
    start=1,
):
    bootstrap_result = (
        paired_block_bootstrap_h1_difference(
            baseline_block_score_lookup[
                baseline_model_id
            ],
            replicate_count=(
                HAWKES_BASELINE_BOOTSTRAP_REPLICATES
            ),
            seed_sequence=(
                int(
                    RANDOM_SEED
                ),
                8,
                6,
                comparison_index,
            ),
        )
    )

    baseline_registry_row = (
        COUNT_BASELINE_MODEL_REGISTRY.loc[
            COUNT_BASELINE_MODEL_REGISTRY[
                "model_id"
            ].astype(
                str
            ).eq(
                baseline_model_id
            )
        ]
        .iloc[
            0
        ]
    )

    h1_baseline_bootstrap_records.append(
        {
            "candidate_model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "comparator_model_id": (
                baseline_model_id
            ),
            "comparator_model_family": str(
                baseline_registry_row[
                    "model_family"
                ]
            ),
            "formal_comparator_flag": bool(
                baseline_registry_row[
                    "formal_count_comparator_flag"
                ]
            ),
            "primary_simple_baseline_flag": bool(
                baseline_registry_row[
                    "primary_simple_count_baseline_flag"
                ]
            ),
            "scored_partition": (
                "CALIBRATION"
            ),
            "score_space": (
                HAWKES_BASELINE_SCORE_SPACE
            ),
            "grid_width_ms": 1,
            "block_width_seconds": (
                block_width_ns
                / NANOSECONDS_PER_SECOND
            ),
            "complete_block_count": (
                formal_complete_block_count
            ),
            "formal_bootstrap_exposure_seconds": (
                formal_exposure_seconds
            ),
            "excluded_tail_exposure_seconds": (
                excluded_tail_exposure_ns
                / NANOSECONDS_PER_SECOND
            ),
            "bootstrap_replicates": (
                HAWKES_BASELINE_BOOTSTRAP_REPLICATES
            ),
            **bootstrap_result,
            "calibration_used_for_model_selection": False,
            "parameter_refit_performed": False,
            "calibration_parameter_updates": 0,
            "full_hawkes_superiority_claim_authorized": False,
            "status": "PASS_SCORE_COMPARISON_ONLY",
        }
    )


HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP = (
    pd.DataFrame.from_records(
        h1_baseline_bootstrap_records
    )
    .sort_values(
        [
            "formal_comparator_flag",
            "primary_simple_baseline_flag",
            "comparator_model_id",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Primary comparison extraction
# ------------------------------------------------------------

HAWKES_VS_FORMAL_COMPARATOR_RESULT = (
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP.loc[
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
            "comparator_model_id"
        ].eq(
            FORMAL_COUNT_COMPARATOR_MODEL_ID
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

HAWKES_VS_PRIMARY_SIMPLE_BASELINE_RESULT = (
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP.loc[
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
            "comparator_model_id"
        ].eq(
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

require(
    len(
        HAWKES_VS_FORMAL_COMPARATOR_RESULT
    )
    == 1,
    "Expected one H1 versus formal-comparator result.",
)

require(
    len(
        HAWKES_VS_PRIMARY_SIMPLE_BASELINE_RESULT
    )
    == 1,
    "Expected one H1 versus primary-simple-baseline result.",
)


# ------------------------------------------------------------
# Score-comparison scope ledger
# ------------------------------------------------------------

HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER = pd.DataFrame(
    [
        {
            "comparison_component": (
                "NOTEBOOK_06_BASELINE_ARTIFACT_READBACK"
            ),
            "completed": True,
            "acceptance_role": (
                "BLOCKING_LINEAGE_AND_HASH_VERIFICATION"
            ),
            "full_hawkes_superiority_claim_authorized": False,
            "status": "PASS",
        },
        {
            "comparison_component": (
                "LOCKED_CALIBRATION_SCORE_RANKING"
            ),
            "completed": True,
            "acceptance_role": (
                "COMPARABLE_SCORE_EVIDENCE"
            ),
            "full_hawkes_superiority_claim_authorized": False,
            "status": "PASS",
        },
        {
            "comparison_component": (
                "PAIRED_COMPLETE_BLOCK_BOOTSTRAP"
            ),
            "completed": True,
            "acceptance_role": (
                "SCORE_UNCERTAINTY_EVIDENCE"
            ),
            "full_hawkes_superiority_claim_authorized": False,
            "status": "PASS",
        },
        {
            "comparison_component": (
                "COARSENING_AWARE_MODEL_ADEQUACY"
            ),
            "completed": True,
            "acceptance_role": (
                "REQUIRED_SEPARATE_DIAGNOSTIC_GATE"
            ),
            "full_hawkes_superiority_claim_authorized": False,
            "status": (
                "AWAITING_TERMINAL_SYNTHESIS"
            ),
        },
        {
            "comparison_component": (
                "FINAL_HAWKES_SUPERIORITY_DECISION"
            ),
            "completed": False,
            "acceptance_role": (
                "TERMINAL_DECISION_ONLY"
            ),
            "full_hawkes_superiority_claim_authorized": False,
            "status": (
                "NOT_YET_AUTHORIZED"
            ),
        },
    ]
)


# ------------------------------------------------------------
# Verification matrix
# ------------------------------------------------------------

expected_combined_ranking_rows = (
    len(
        COUNT_BASELINE_MODEL_REGISTRY
    )
    + 1
)

require(
    len(
        HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING
    )
    == expected_combined_ranking_rows,
    "The combined H1/baseline ranking has an incorrect row count.",
)

require(
    HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
        "model_id"
    ].nunique()
    == expected_combined_ranking_rows,
    "The combined H1/baseline ranking contains duplicate models.",
)

require(
    len(
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP
    )
    == len(
        COUNT_BASELINE_MODEL_REGISTRY
    ),
    "The H1 paired-bootstrap table lacks a Notebook 06 baseline.",
)

require(
    np.isfinite(
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
            [
                "observed_total_log_score_difference",
                "bootstrap_total_difference_q025",
                "bootstrap_total_difference_median",
                "bootstrap_total_difference_q975",
                "bootstrap_probability_difference_nonpositive",
            ]
        ].to_numpy(
            dtype=np.float64
        )
    ).all(),
    "A Hawkes-versus-baseline bootstrap result is non-finite.",
)

require(
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        "bootstrap_total_difference_q025"
    ].le(
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
            "bootstrap_total_difference_median"
        ]
    ).all(),
    "A paired-bootstrap lower quantile exceeds its median.",
)

require(
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        "bootstrap_total_difference_median"
    ].le(
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
            "bootstrap_total_difference_q975"
        ]
    ).all(),
    "A paired-bootstrap median exceeds its upper quantile.",
)

require(
    not HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        "parameter_refit_performed"
    ].any(),
    "The baseline comparison refitted H1.",
)

require(
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        "calibration_parameter_updates"
    ].eq(
        0
    ).all(),
    "The baseline comparison updated CALIBRATION parameters.",
)

require(
    not HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        "full_hawkes_superiority_claim_authorized"
    ].any(),
    "A score-only comparison incorrectly authorized full superiority.",
)


HAWKES_BASELINE_COUNT_COMPARISON_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "notebook_06_manifest_file_hash_verified"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "notebook_06_manifest_semantic_hash_verified"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "required_persisted_source_tables_loaded"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "all_loaded_table_hashes_verified"
            ),
            "passed": bool(
                NB06_BASELINE_TABLE_READBACK_AUDIT[
                    "file_hash_matches"
                ].all()
            ),
        },
        {
            "check_name": (
                "locked_baseline_ranking_reconstructed_from_unified_scores"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "formal_comparator_identity_verified"
            ),
            "passed": (
                FORMAL_COUNT_COMPARATOR_MODEL_ID
                == "D0_SIDE_CONSTANT_POISSON"
            ),
        },
        {
            "check_name": (
                "primary_simple_baseline_identity_verified"
            ),
            "passed": (
                PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
                == "E_SIDE_EWMA_250MS_POISSON"
            ),
        },
        {
            "check_name": (
                "identical_native_one_ms_score_space_used"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "formal_complete_block_geometry_inherited"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "h1_complete_blocks_and_tail_reconcile"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "paired_bootstrap_covers_every_baseline"
            ),
            "passed": (
                len(
                    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP
                )
                == len(
                    COUNT_BASELINE_MODEL_REGISTRY
                )
            ),
        },
        {
            "check_name": (
                "score_only_superiority_claim_disabled"
            ),
            "passed": bool(
                not HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
                    "full_hawkes_superiority_claim_authorized"
                ].any()
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    HAWKES_BASELINE_COUNT_COMPARISON_VERIFICATION[
        "passed"
    ].all(),
    "At least one Hawkes-versus-baseline verification failed.",
)


# ------------------------------------------------------------
# Publish cell state
# ------------------------------------------------------------

HAWKES_BASELINE_COUNT_COMPARISON_BUILT = True
HAWKES_COUNT_SCORE_SUPERIORITY_DECISION = (
    "PENDING_TERMINAL_DIAGNOSTIC_SYNTHESIS"
)
HAWKES_SUPERIORITY_AUTHORIZED = False
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "This cell performed a filesystem write.",
)


display(
    NB06_BASELINE_TABLE_READBACK_AUDIT
)

display(
    LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
        [
            "calibration_score_rank",
            "model_id",
            "model_family",
            "observed_event_count",
            "predicted_event_count",
            "log_score_total",
            "log_score_per_second",
            "log_score_improvement_over_formal_comparator",
            "primary_simple_count_baseline_flag",
            "formal_count_comparator_flag",
            "ranking_role",
            "status",
        ]
    ]
)

display(
    HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING[
        [
            "calibration_score_rank",
            "model_id",
            "model_family",
            "candidate_role",
            "observed_event_count",
            "predicted_event_count",
            "log_score_total",
            "log_score_per_second",
            "log_score_per_observed_event",
            "log_score_improvement_over_formal_comparator",
            "log_score_improvement_over_primary_simple_baseline",
            "score_only_superiority_claim_authorized",
            "status",
        ]
    ]
)

display(
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        [
            "candidate_model_id",
            "comparator_model_id",
            "formal_comparator_flag",
            "primary_simple_baseline_flag",
            "complete_block_count",
            "bootstrap_replicates",
            "observed_total_log_score_difference",
            "observed_log_score_difference_per_second",
            "bootstrap_total_difference_q025",
            "bootstrap_total_difference_median",
            "bootstrap_total_difference_q975",
            "bootstrap_probability_difference_nonpositive",
            "positive_improvement_supported_at_95pct",
            "negative_difference_supported_at_95pct",
            "comparison_status",
            "full_hawkes_superiority_claim_authorized",
            "status",
        ]
    ]
)

display(
    HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER
)

display(
    HAWKES_BASELINE_COUNT_COMPARISON_VERIFICATION
)

print(
    "Verified Notebook 06 baseline artifacts were loaded from the "
    "persisted output manifest. The non-persisted locked CALIBRATION "
    "ranking was reconstructed deterministically from the persisted "
    "unified score table. Frozen H1 was scored against every authorized "
    "Notebook 06 baseline on the identical bivariate side-specific "
    "one-millisecond Poisson-count interface. Paired uncertainty used "
    "Notebook 06's exact complete CALIBRATION blocks and excluded only "
    "the registered partial terminal tail. This cell provides score "
    "evidence only; full Hawkes superiority remains unauthorized pending "
    "terminal synthesis of the model-adequacy diagnostics. No filesystem "
    "writes were performed."
)

,table_name,row_count,column_count,file_hash_matches,row_count_matches,column_count_matches,column_order_matches,status
0,BOOTSTRAP_COVERAGE_AUDIT,1,14,True,True,True,True,PASS
1,CALIBRATION_COUNT_SCORE_BLOCKS,288,11,True,True,True,True,PASS
2,COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON,3,24,True,True,True,True,PASS
3,COUNT_BASELINE_MODEL_REGISTRY,4,11,True,True,True,True,PASS
4,COUNT_SCORE_RECONCILIATION,8,8,True,True,True,True,PASS
5,UNIFIED_COUNT_BASELINE_SCORE_TABLE,8,20,True,True,True,True,PASS


,calibration_score_rank,model_id,model_family,observed_event_count,predicted_event_count,log_score_total,log_score_per_second,log_score_improvement_over_formal_comparator,primary_simple_count_baseline_flag,formal_count_comparator_flag,ranking_role,status
0,1,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,2493,"2,673.479105","-18,072.44329",-25.09019004,368.3327884,True,False,RECONSTRUCTED_FROM_PERSISTED_UNIFIED_SCORE_TABLE,PASS
1,2,R_SIDE_ROLLING_250MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,2493,"2,699.452555","-18,095.79606",-25.12261098,344.9800145,False,False,RECONSTRUCTED_FROM_PERSISTED_UNIFIED_SCORE_TABLE,PASS
2,3,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,INDEPENDENT_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,2493,"2,799.871581","-18,440.77608",-25.60155087,0,False,False,RECONSTRUCTED_FROM_PERSISTED_UNIFIED_SCORE_TABLE,PASS
3,4,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,2493,"2,799.871581","-18,440.77608",-25.60155087,0,False,True,RECONSTRUCTED_FROM_PERSISTED_UNIFIED_SCORE_TABLE,PASS


,calibration_score_rank,model_id,model_family,candidate_role,observed_event_count,predicted_event_count,log_score_total,log_score_per_second,log_score_per_observed_event,log_score_improvement_over_formal_comparator,log_score_improvement_over_primary_simple_baseline,score_only_superiority_claim_authorized,status
0,1,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,FROZEN_HAWKES,2493,"2,754.976987","-14,914.46158",-20.70592611,-5.982535733,"3,526.314495","3,157.981706",False,PASS_SCORE_COMPARISON_ONLY
1,2,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,NOTEBOOK_06_BASELINE,2493,"2,673.479105","-18,072.44329",-25.09019004,-7.249275286,368.3327884,0,False,PASS_SCORE_COMPARISON_ONLY
2,3,R_SIDE_ROLLING_250MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,NOTEBOOK_06_BASELINE,2493,"2,699.452555","-18,095.79606",-25.12261098,-7.258642624,344.9800145,-23.35277388,False,PASS_SCORE_COMPARISON_ONLY
3,4,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,INDEPENDENT_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,NOTEBOOK_06_BASELINE,2493,"2,799.871581","-18,440.77608",-25.60155087,-7.397022093,0,-368.3327884,False,PASS_SCORE_COMPARISON_ONLY
4,5,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,NOTEBOOK_06_BASELINE,2493,"2,799.871581","-18,440.77608",-25.60155087,-7.397022093,0,-368.3327884,False,PASS_SCORE_COMPARISON_ONLY


,candidate_model_id,comparator_model_id,formal_comparator_flag,primary_simple_baseline_flag,complete_block_count,bootstrap_replicates,observed_total_log_score_difference,observed_log_score_difference_per_second,bootstrap_total_difference_q025,bootstrap_total_difference_median,bootstrap_total_difference_q975,bootstrap_probability_difference_nonpositive,positive_improvement_supported_at_95pct,negative_difference_supported_at_95pct,comparison_status,full_hawkes_superiority_claim_authorized,status
0,H1_DIAGONAL_SHARED_DECAY,D0_SIDE_CONSTANT_POISSON,True,False,72,2000,"3,526.145892",4.89742485,"3,102.105784","3,518.309313","3,981.651151",0.0004997501249,True,False,H1_POSITIVE_SUPPORTED,False,PASS_SCORE_COMPARISON_ONLY
1,H1_DIAGONAL_SHARED_DECAY,E_SIDE_EWMA_250MS_POISSON,False,True,72,2000,"3,157.846627",4.385898093,"2,862.365487","3,142.108249","3,475.409094",0.0004997501249,True,False,H1_POSITIVE_SUPPORTED,False,PASS_SCORE_COMPARISON_ONLY
2,H1_DIAGONAL_SHARED_DECAY,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,False,False,72,2000,"3,526.145892",4.89742485,"3,112.409349","3,514.702898","4,017.43975",0.0004997501249,True,False,H1_POSITIVE_SUPPORTED,False,PASS_SCORE_COMPARISON_ONLY
3,H1_DIAGONAL_SHARED_DECAY,R_SIDE_ROLLING_250MS_POISSON,False,False,72,2000,"3,181.26831",4.418428208,"2,884.313871","3,175.670248","3,500.741288",0.0004997501249,True,False,H1_POSITIVE_SUPPORTED,False,PASS_SCORE_COMPARISON_ONLY


,comparison_component,completed,acceptance_role,full_hawkes_superiority_claim_authorized,status
0,NOTEBOOK_06_BASELINE_ARTIFACT_READBACK,True,BLOCKING_LINEAGE_AND_HASH_VERIFICATION,False,PASS
1,LOCKED_CALIBRATION_SCORE_RANKING,True,COMPARABLE_SCORE_EVIDENCE,False,PASS
2,PAIRED_COMPLETE_BLOCK_BOOTSTRAP,True,SCORE_UNCERTAINTY_EVIDENCE,False,PASS
3,COARSENING_AWARE_MODEL_ADEQUACY,True,REQUIRED_SEPARATE_DIAGNOSTIC_GATE,False,AWAITING_TERMINAL_SYNTHESIS
4,FINAL_HAWKES_SUPERIORITY_DECISION,False,TERMINAL_DECISION_ONLY,False,NOT_YET_AUTHORIZED


,check_name,passed
0,notebook_06_manifest_file_hash_verified,True
1,notebook_06_manifest_semantic_hash_verified,True
2,required_persisted_source_tables_loaded,True
3,all_loaded_table_hashes_verified,True
4,locked_baseline_ranking_reconstructed_from_uni...,True
5,formal_comparator_identity_verified,True
6,primary_simple_baseline_identity_verified,True
7,identical_native_one_ms_score_space_used,True
8,formal_complete_block_geometry_inherited,True
9,h1_complete_blocks_and_tail_reconcile,True


Verified Notebook 06 baseline artifacts were loaded from the persisted output manifest. The non-persisted locked CALIBRATION ranking was reconstructed deterministically from the persisted unified score table. Frozen H1 was scored against every authorized Notebook 06 baseline on the identical bivariate side-specific one-millisecond Poisson-count interface. Paired uncertainty used Notebook 06's exact complete CALIBRATION blocks and excluded only the registered partial terminal tail. This cell provides score evidence only; full Hawkes superiority remains unauthorized pending terminal synthesis of the model-adequacy diagnostics. No filesystem writes were performed.


In [26]:
# ============================================================
# Notebook 08 terminal diagnostic synthesis
#
# Terminal gate follows the Notebook 08 exit criteria in the
# frozen V0.1 plan: higher likelihood alone is insufficient.
#
# This cell:
#   - separates narrow count-score superiority from full model
#     adequacy;
#   - evaluates locked CALIBRATION envelope failures by family;
#   - creates the formal rejection and decision ledgers;
#   - determines whether Notebook 09 may open VALIDATION.
#
# No artifacts are written in this cell.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "COARSENING_AWARE_MODEL_ENVELOPES_BUILT",
            False,
        )
    ),
    "Coarsening-aware model envelopes are unavailable.",
)

require(
    bool(
        globals().get(
            "HAWKES_BASELINE_COUNT_COMPARISON_BUILT",
            False,
        )
    ),
    "The Hawkes-versus-baseline comparison is unavailable.",
)

require(
    isinstance(
        COARSENING_AWARE_DIAGNOSTIC_ENVELOPES,
        pd.DataFrame,
    )
    and not COARSENING_AWARE_DIAGNOSTIC_ENVELOPES.empty,
    "The diagnostic-envelope table is unavailable.",
)

require(
    isinstance(
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP,
        pd.DataFrame,
    )
    and not HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP.empty,
    "The paired Hawkes-versus-baseline bootstrap is unavailable.",
)

require(
    not bool(
        globals().get(
            "NOTEBOOK08_TERMINAL_DECISION_CREATED",
            False,
        )
    ),
    "The Notebook 08 terminal decision already exists.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED
    == 0,
    "CALIBRATION parameters were updated.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is False,
    "No filesystem writes are permitted in this cell.",
)


# ------------------------------------------------------------
# Frozen terminal-gate configuration
# ------------------------------------------------------------

MODEL_ADEQUACY_MAX_OUTSIDE_FRACTION = 0.20

RESIDUAL_DISTRIBUTION_METRICS = (
    "ABS_TAU_MEAN_ERROR",
    "ABS_TAU_VARIANCE_ERROR",
    "KS_D_UNIFORM",
    "CRAMER_VON_MISES",
)

RESIDUAL_SERIAL_METRICS = (
    "MAX_ABS_ACF_1_TO_20",
    "MAX_ABS_ACF_1_TO_50",
    "LJUNG_BOX_Q_10",
    "LJUNG_BOX_Q_20",
    "LJUNG_BOX_Q_50",
)

PRIMARY_CALIBRATION_FAMILIES = (
    "TIME_RESCALING_RESIDUAL",
    "PATH_COUNTS",
    "EXACT_TIME_MULTIPLICITY",
    "SIDE_MARK_CALIBRATION",
)

require(
    0.0
    <= MODEL_ADEQUACY_MAX_OUTSIDE_FRACTION
    < 1.0,
    "The model-adequacy outside-envelope tolerance is invalid.",
)


# ------------------------------------------------------------
# Validate the final envelope schema
# ------------------------------------------------------------

required_envelope_columns = {
    "event_partition",
    "partition_order",
    "diagnostic_scope",
    "diagnostic_family",
    "metric_name",
    "tail_direction",
    "simulation_replicates",
    "observed_value",
    "simulation_q025",
    "simulation_median",
    "simulation_q975",
    "outside_two_sided_95pct_envelope",
    "outside_directional_95pct_envelope",
    "envelope_interpretation",
    "parameter_refit_performed",
    "calibration_parameter_updates",
    "status",
}

missing_envelope_columns = sorted(
    required_envelope_columns
    - set(
        COARSENING_AWARE_DIAGNOSTIC_ENVELOPES.columns
    )
)

require(
    not missing_envelope_columns,
    (
        "The final diagnostic-envelope table lacks columns: "
        f"{missing_envelope_columns}"
    ),
)

require(
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "simulation_replicates"
    ].eq(
        N_MODEL_ENVELOPE_REPLICATES
    ).all(),
    "A diagnostic envelope lacks the full simulation count.",
)

require(
    not COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "parameter_refit_performed"
    ].any(),
    "A model-envelope replicate refitted parameters.",
)

require(
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
        "calibration_parameter_updates"
    ].eq(
        0
    ).all(),
    "A model-envelope replicate updated CALIBRATION parameters.",
)

require(
    set(
        PRIMARY_CALIBRATION_FAMILIES
    ).issubset(
        set(
            COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
                "diagnostic_family"
            ]
        )
    ),
    "A required diagnostic family is absent.",
)


# ------------------------------------------------------------
# Locked CALIBRATION envelope summaries
# ------------------------------------------------------------

LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES = (
    COARSENING_AWARE_DIAGNOSTIC_ENVELOPES.loc[
        COARSENING_AWARE_DIAGNOSTIC_ENVELOPES[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .copy()
    .sort_values(
        [
            "diagnostic_family",
            "diagnostic_scope",
            "metric_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    not LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES.empty,
    "No locked CALIBRATION envelopes are available.",
)

require(
    set(
        LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES[
            "diagnostic_family"
        ]
    )
    == set(
        PRIMARY_CALIBRATION_FAMILIES
    ),
    (
        "Locked CALIBRATION contains an unexpected or missing "
        "primary diagnostic family."
    ),
)


def summarize_envelope_groups(
    source_table: pd.DataFrame,
    *,
    grouping_columns: list[str],
) -> pd.DataFrame:
    """Summarize directional 95% envelope exceedances."""
    summary = (
        source_table.groupby(
            grouping_columns,
            observed=True,
            sort=True,
            dropna=False,
        )
        .agg(
            metric_count=(
                "metric_name",
                "size",
            ),
            outside_directional_envelope_count=(
                "outside_directional_95pct_envelope",
                "sum",
            ),
            outside_two_sided_envelope_count=(
                "outside_two_sided_95pct_envelope",
                "sum",
            ),
            minimum_directional_tail_probability=(
                "directional_tail_probability_report_only",
                "min",
            ),
            maximum_observed_to_upper_envelope_ratio=(
                "observed_value",
                lambda values: float(
                    np.nan
                ),
            ),
        )
        .reset_index()
    )

    summary[
        "outside_directional_envelope_fraction"
    ] = (
        summary[
            "outside_directional_envelope_count"
        ]
        / summary[
            "metric_count"
        ]
    )

    summary[
        "outside_two_sided_envelope_fraction"
    ] = (
        summary[
            "outside_two_sided_envelope_count"
        ]
        / summary[
            "metric_count"
        ]
    )

    summary = summary.drop(
        columns=[
            "maximum_observed_to_upper_envelope_ratio",
        ]
    )

    summary[
        "within_declared_adequacy_tolerance"
    ] = (
        summary[
            "outside_directional_envelope_fraction"
        ]
        <= MODEL_ADEQUACY_MAX_OUTSIDE_FRACTION
    )

    summary[
        "status"
    ] = np.where(
        summary[
            "within_declared_adequacy_tolerance"
        ],
        "PASS",
        "FAIL",
    )

    return summary


CALIBRATION_ENVELOPE_FAMILY_SUMMARY = (
    summarize_envelope_groups(
        LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES,
        grouping_columns=[
            "diagnostic_family",
        ],
    )
    .sort_values(
        "diagnostic_family",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

CALIBRATION_ENVELOPE_SCOPE_SUMMARY = (
    summarize_envelope_groups(
        LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES,
        grouping_columns=[
            "diagnostic_family",
            "diagnostic_scope",
        ],
    )
    .sort_values(
        [
            "diagnostic_family",
            "diagnostic_scope",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Residual distribution and serial-dependence summaries
# ------------------------------------------------------------

calibration_residual_envelopes = (
    LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES.loc[
        LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES[
            "diagnostic_family"
        ].eq(
            "TIME_RESCALING_RESIDUAL"
        )
    ]
    .copy()
)

calibration_distribution_envelopes = (
    calibration_residual_envelopes.loc[
        calibration_residual_envelopes[
            "metric_name"
        ].isin(
            RESIDUAL_DISTRIBUTION_METRICS
        )
    ]
    .copy()
)

calibration_serial_envelopes = (
    calibration_residual_envelopes.loc[
        calibration_residual_envelopes[
            "metric_name"
        ].isin(
            RESIDUAL_SERIAL_METRICS
        )
    ]
    .copy()
)

require(
    len(
        calibration_distribution_envelopes
    )
    == (
        3
        * len(
            RESIDUAL_DISTRIBUTION_METRICS
        )
    ),
    "The locked CALIBRATION residual-distribution grid is incomplete.",
)

require(
    len(
        calibration_serial_envelopes
    )
    == (
        3
        * len(
            RESIDUAL_SERIAL_METRICS
        )
    ),
    "The locked CALIBRATION residual-serial grid is incomplete.",
)

CALIBRATION_RESIDUAL_DISTRIBUTION_SCOPE_SUMMARY = (
    summarize_envelope_groups(
        calibration_distribution_envelopes,
        grouping_columns=[
            "diagnostic_scope",
        ],
    )
    .sort_values(
        "diagnostic_scope",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

CALIBRATION_RESIDUAL_SERIAL_SCOPE_SUMMARY = (
    summarize_envelope_groups(
        calibration_serial_envelopes,
        grouping_columns=[
            "diagnostic_scope",
        ],
    )
    .sort_values(
        "diagnostic_scope",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

CALIBRATION_RESIDUAL_DISTRIBUTION_PASS = bool(
    CALIBRATION_RESIDUAL_DISTRIBUTION_SCOPE_SUMMARY[
        "within_declared_adequacy_tolerance"
    ].all()
)

CALIBRATION_RESIDUAL_SERIAL_PASS = bool(
    CALIBRATION_RESIDUAL_SERIAL_SCOPE_SUMMARY[
        "within_declared_adequacy_tolerance"
    ].all()
)


# ------------------------------------------------------------
# Count, multiplicity, and mark calibration gates
# ------------------------------------------------------------

calibration_path_count_envelopes = (
    LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES.loc[
        LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES[
            "diagnostic_family"
        ].eq(
            "PATH_COUNTS"
        )
    ]
)

calibration_multiplicity_envelopes = (
    LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES.loc[
        LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES[
            "diagnostic_family"
        ].eq(
            "EXACT_TIME_MULTIPLICITY"
        )
    ]
)

calibration_mark_envelopes = (
    LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES.loc[
        LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES[
            "diagnostic_family"
        ].eq(
            "SIDE_MARK_CALIBRATION"
        )
    ]
)

require(
    len(
        calibration_path_count_envelopes
    )
    == 4,
    "The locked CALIBRATION path-count envelope grid is incomplete.",
)

require(
    len(
        calibration_multiplicity_envelopes
    )
    == 7,
    "The locked CALIBRATION multiplicity envelope grid is incomplete.",
)

require(
    len(
        calibration_mark_envelopes
    )
    == 3,
    "The locked CALIBRATION mark envelope grid is incomplete.",
)

CALIBRATION_PATH_COUNT_PASS = bool(
    not calibration_path_count_envelopes[
        "outside_directional_95pct_envelope"
    ].any()
)

CALIBRATION_MULTIPLICITY_OUTSIDE_FRACTION = float(
    calibration_multiplicity_envelopes[
        "outside_directional_95pct_envelope"
    ].mean()
)

CALIBRATION_MULTIPLICITY_PASS = bool(
    CALIBRATION_MULTIPLICITY_OUTSIDE_FRACTION
    <= MODEL_ADEQUACY_MAX_OUTSIDE_FRACTION
)

CALIBRATION_SIDE_MARK_PASS = bool(
    not calibration_mark_envelopes[
        "outside_directional_95pct_envelope"
    ].any()
)


# ------------------------------------------------------------
# Narrow score-superiority evidence
# ------------------------------------------------------------

formal_comparator_result = (
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP.loc[
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
            "formal_comparator_flag"
        ]
    ]
)

primary_simple_result = (
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP.loc[
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
            "primary_simple_baseline_flag"
        ]
    ]
)

require(
    len(
        formal_comparator_result
    )
    == 1,
    "Expected exactly one formal-comparator score result.",
)

require(
    len(
        primary_simple_result
    )
    == 1,
    "Expected exactly one primary-simple-baseline score result.",
)

SCORE_IMPROVEMENT_OVER_FORMAL_COMPARATOR_PASS = bool(
    formal_comparator_result.iloc[
        0
    ][
        "positive_improvement_supported_at_95pct"
    ]
    and float(
        formal_comparator_result.iloc[
            0
        ][
            "bootstrap_total_difference_q025"
        ]
    )
    > 0.0
)

SCORE_IMPROVEMENT_OVER_PRIMARY_SIMPLE_PASS = bool(
    primary_simple_result.iloc[
        0
    ][
        "positive_improvement_supported_at_95pct"
    ]
    and float(
        primary_simple_result.iloc[
            0
        ][
            "bootstrap_total_difference_q025"
        ]
    )
    > 0.0
)

SCORE_IMPROVEMENT_OVER_ALL_BASELINES_PASS = bool(
    HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        "positive_improvement_supported_at_95pct"
    ].all()
    and HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
        "bootstrap_total_difference_q025"
    ].gt(
        0.0
    ).all()
)


# ------------------------------------------------------------
# Stationarity and frozen-model integrity
# ------------------------------------------------------------

FROZEN_H1_SPECTRAL_RADIUS = float(
    max(
        FROZEN_H1_PARAMETERS.kappa_buy,
        FROZEN_H1_PARAMETERS.kappa_sell,
    )
)

FROZEN_H1_STATIONARITY_MARGIN = (
    1.0
    - FROZEN_H1_SPECTRAL_RADIUS
)

STATIONARITY_MARGIN_PASS = bool(
    FROZEN_H1_SPECTRAL_RADIUS
    < 1.0
    and FROZEN_H1_SPECTRAL_RADIUS
    <= 0.98
)

CALIBRATION_ZERO_UPDATE_PASS = bool(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED
    == 0
)

PROTECTED_PARTITIONS_UNOPENED_PASS = bool(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False
    and not any(
        PARTITION_CONTENT_LOADED[
            partition_name
        ]
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    )
)

FILESYSTEM_WRITE_GUARD_PASS = bool(
    FILESYSTEM_WRITES_PERFORMED
    is False
)


# ------------------------------------------------------------
# Cross-side diagnostic completion
# ------------------------------------------------------------

CROSS_SIDE_DESCRIPTIVE_REPORT_COMPLETE = bool(
    isinstance(
        globals().get(
            "HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY"
        ),
        pd.DataFrame,
    )
    and not HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY.empty
    and isinstance(
        globals().get(
            "HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS"
        ),
        pd.DataFrame,
    )
    and not HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS.empty
)

CROSS_SIDE_FORMAL_SIMULATION_ENVELOPE_COMPLETE = bool(
    globals().get(
        "COARSENING_AWARE_CROSS_SIDE_ENVELOPES_BUILT",
        False,
    )
)

CROSS_EXCITATION_AUTHORIZED = False


# ------------------------------------------------------------
# Model-adequacy and terminal decision
# ------------------------------------------------------------

CALIBRATION_MODEL_ADEQUACY_PASS = bool(
    CALIBRATION_RESIDUAL_DISTRIBUTION_PASS
    and CALIBRATION_RESIDUAL_SERIAL_PASS
    and CALIBRATION_PATH_COUNT_PASS
    and CALIBRATION_MULTIPLICITY_PASS
    and CALIBRATION_SIDE_MARK_PASS
)

HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED = bool(
    SCORE_IMPROVEMENT_OVER_FORMAL_COMPARATOR_PASS
    and SCORE_IMPROVEMENT_OVER_PRIMARY_SIMPLE_PASS
    and SCORE_IMPROVEMENT_OVER_ALL_BASELINES_PASS
)

HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS = bool(
    HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    and STATIONARITY_MARGIN_PASS
    and CALIBRATION_MODEL_ADEQUACY_PASS
    and CALIBRATION_ZERO_UPDATE_PASS
    and PROTECTED_PARTITIONS_UNOPENED_PASS
    and FILESYSTEM_WRITE_GUARD_PASS
)

if HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS:
    NOTEBOOK08_TERMINAL_CLASS = "PASS"

    NOTEBOOK08_TERMINAL_STATUS = (
        "PASS_HAWKES_DIAGNOSTICS"
    )

    NOTEBOOK08_TERMINAL_INTERPRETATION = (
        "Frozen H1 materially improves on simpler controls, "
        "retains its stationarity margin, and passes locked "
        "CALIBRATION model-adequacy gates."
    )

elif (
    HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    and STATIONARITY_MARGIN_PASS
    and not CALIBRATION_MODEL_ADEQUACY_PASS
):
    NOTEBOOK08_TERMINAL_CLASS = "FAIL"

    NOTEBOOK08_TERMINAL_STATUS = (
        "FAIL_HAWKES_DIAGNOSTIC_ADEQUACY_"
        "DESPITE_SCORE_SUPERIORITY"
    )

    NOTEBOOK08_TERMINAL_INTERPRETATION = (
        "Frozen H1 produces materially better locked CALIBRATION "
        "count scores than every authorized Notebook 06 baseline, "
        "but it fails the required model-adequacy gate. The score "
        "gain does not override systematic residual, event-count, "
        "or multiplicity-envelope failures."
    )

elif not HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED:
    NOTEBOOK08_TERMINAL_CLASS = "FAIL"

    NOTEBOOK08_TERMINAL_STATUS = (
        "FAIL_HAWKES_NO_MATERIAL_BASELINE_IMPROVEMENT"
    )

    NOTEBOOK08_TERMINAL_INTERPRETATION = (
        "Frozen H1 did not establish material locked CALIBRATION "
        "score improvement over the required simpler controls."
    )

else:
    NOTEBOOK08_TERMINAL_CLASS = "FAIL"

    NOTEBOOK08_TERMINAL_STATUS = (
        "FAIL_HAWKES_DIAGNOSTICS"
    )

    NOTEBOOK08_TERMINAL_INTERPRETATION = (
        "At least one blocking Hawkes diagnostic gate failed."
    )


# ------------------------------------------------------------
# Formal diagnostic gate ledger
# ------------------------------------------------------------

diagnostic_gate_records = [
    {
        "gate_order": 1,
        "gate_id": (
            "MATERIAL_SCORE_IMPROVEMENT_OVER_FORMAL_COMPARATOR"
        ),
        "gate_family": "SCORE_SUPERIORITY",
        "blocking": True,
        "passed": (
            SCORE_IMPROVEMENT_OVER_FORMAL_COMPARATOR_PASS
        ),
        "observed_value": float(
            formal_comparator_result.iloc[
                0
            ][
                "observed_total_log_score_difference"
            ]
        ),
        "reference_value": 0.0,
        "decision_rule": (
            "PAIRED_BOOTSTRAP_Q025_GREATER_THAN_ZERO"
        ),
        "status": (
            "PASS"
            if SCORE_IMPROVEMENT_OVER_FORMAL_COMPARATOR_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 2,
        "gate_id": (
            "MATERIAL_SCORE_IMPROVEMENT_OVER_PRIMARY_SIMPLE_BASELINE"
        ),
        "gate_family": "SCORE_SUPERIORITY",
        "blocking": True,
        "passed": (
            SCORE_IMPROVEMENT_OVER_PRIMARY_SIMPLE_PASS
        ),
        "observed_value": float(
            primary_simple_result.iloc[
                0
            ][
                "observed_total_log_score_difference"
            ]
        ),
        "reference_value": 0.0,
        "decision_rule": (
            "PAIRED_BOOTSTRAP_Q025_GREATER_THAN_ZERO"
        ),
        "status": (
            "PASS"
            if SCORE_IMPROVEMENT_OVER_PRIMARY_SIMPLE_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 3,
        "gate_id": (
            "MATERIAL_SCORE_IMPROVEMENT_OVER_ALL_AUTHORIZED_BASELINES"
        ),
        "gate_family": "SCORE_SUPERIORITY",
        "blocking": True,
        "passed": (
            SCORE_IMPROVEMENT_OVER_ALL_BASELINES_PASS
        ),
        "observed_value": float(
            HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP[
                "bootstrap_total_difference_q025"
            ].min()
        ),
        "reference_value": 0.0,
        "decision_rule": (
            "EVERY_PAIRED_BOOTSTRAP_Q025_GREATER_THAN_ZERO"
        ),
        "status": (
            "PASS"
            if SCORE_IMPROVEMENT_OVER_ALL_BASELINES_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 4,
        "gate_id": "DEFENSIBLE_STATIONARITY_MARGIN",
        "gate_family": "STATIONARITY",
        "blocking": True,
        "passed": STATIONARITY_MARGIN_PASS,
        "observed_value": (
            FROZEN_H1_SPECTRAL_RADIUS
        ),
        "reference_value": 0.98,
        "decision_rule": (
            "SPECTRAL_RADIUS_LESS_THAN_OR_EQUAL_TO_0_98"
        ),
        "status": (
            "PASS"
            if STATIONARITY_MARGIN_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 5,
        "gate_id": (
            "LOCKED_CALIBRATION_RESIDUAL_DISTRIBUTION"
        ),
        "gate_family": "MODEL_ADEQUACY",
        "blocking": True,
        "passed": (
            CALIBRATION_RESIDUAL_DISTRIBUTION_PASS
        ),
        "observed_value": float(
            CALIBRATION_RESIDUAL_DISTRIBUTION_SCOPE_SUMMARY[
                "outside_directional_envelope_fraction"
            ].max()
        ),
        "reference_value": (
            MODEL_ADEQUACY_MAX_OUTSIDE_FRACTION
        ),
        "decision_rule": (
            "MAX_SCOPE_OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_TO_0_20"
        ),
        "status": (
            "PASS"
            if CALIBRATION_RESIDUAL_DISTRIBUTION_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 6,
        "gate_id": (
            "LOCKED_CALIBRATION_RESIDUAL_SERIAL_DEPENDENCE"
        ),
        "gate_family": "MODEL_ADEQUACY",
        "blocking": True,
        "passed": (
            CALIBRATION_RESIDUAL_SERIAL_PASS
        ),
        "observed_value": float(
            CALIBRATION_RESIDUAL_SERIAL_SCOPE_SUMMARY[
                "outside_directional_envelope_fraction"
            ].max()
        ),
        "reference_value": (
            MODEL_ADEQUACY_MAX_OUTSIDE_FRACTION
        ),
        "decision_rule": (
            "MAX_SCOPE_OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_TO_0_20"
        ),
        "status": (
            "PASS"
            if CALIBRATION_RESIDUAL_SERIAL_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 7,
        "gate_id": (
            "LOCKED_CALIBRATION_PATH_COUNT_CALIBRATION"
        ),
        "gate_family": "MODEL_ADEQUACY",
        "blocking": True,
        "passed": (
            CALIBRATION_PATH_COUNT_PASS
        ),
        "observed_value": float(
            calibration_path_count_envelopes[
                "outside_directional_95pct_envelope"
            ].mean()
        ),
        "reference_value": 0.0,
        "decision_rule": (
            "NO_PRIMARY_PATH_COUNT_OUTSIDE_95PCT_ENVELOPE"
        ),
        "status": (
            "PASS"
            if CALIBRATION_PATH_COUNT_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 8,
        "gate_id": (
            "LOCKED_CALIBRATION_EXACT_TIME_MULTIPLICITY"
        ),
        "gate_family": "MODEL_ADEQUACY",
        "blocking": True,
        "passed": (
            CALIBRATION_MULTIPLICITY_PASS
        ),
        "observed_value": (
            CALIBRATION_MULTIPLICITY_OUTSIDE_FRACTION
        ),
        "reference_value": (
            MODEL_ADEQUACY_MAX_OUTSIDE_FRACTION
        ),
        "decision_rule": (
            "OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_TO_0_20"
        ),
        "status": (
            "PASS"
            if CALIBRATION_MULTIPLICITY_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 9,
        "gate_id": (
            "LOCKED_CALIBRATION_SIDE_MARK_CALIBRATION"
        ),
        "gate_family": "MODEL_ADEQUACY",
        "blocking": True,
        "passed": (
            CALIBRATION_SIDE_MARK_PASS
        ),
        "observed_value": float(
            calibration_mark_envelopes[
                "outside_directional_95pct_envelope"
            ].mean()
        ),
        "reference_value": 0.0,
        "decision_rule": (
            "NO_PRIMARY_MARK_METRIC_OUTSIDE_DIRECTIONAL_ENVELOPE"
        ),
        "status": (
            "PASS"
            if CALIBRATION_SIDE_MARK_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 10,
        "gate_id": (
            "CROSS_SIDE_DESCRIPTIVE_REPORT_COMPLETE"
        ),
        "gate_family": "CROSS_SIDE_DEPENDENCE",
        "blocking": False,
        "passed": (
            CROSS_SIDE_DESCRIPTIVE_REPORT_COMPLETE
        ),
        "observed_value": float(
            CROSS_SIDE_DESCRIPTIVE_REPORT_COMPLETE
        ),
        "reference_value": 1.0,
        "decision_rule": (
            "DESCRIPTIVE_CCF_PORTMANTEAU_AND_COINCIDENCE_TABLES_PRESENT"
        ),
        "status": (
            "PASS"
            if CROSS_SIDE_DESCRIPTIVE_REPORT_COMPLETE
            else "FAIL"
        ),
    },
    {
        "gate_order": 11,
        "gate_id": (
            "CROSS_SIDE_FORMAL_SIMULATION_ENVELOPE"
        ),
        "gate_family": "CROSS_SIDE_DEPENDENCE",
        "blocking": False,
        "passed": (
            CROSS_SIDE_FORMAL_SIMULATION_ENVELOPE_COMPLETE
        ),
        "observed_value": float(
            CROSS_SIDE_FORMAL_SIMULATION_ENVELOPE_COMPLETE
        ),
        "reference_value": 1.0,
        "decision_rule": (
            "FORMAL_COARSENING_AWARE_CROSS_SIDE_ENVELOPE_AVAILABLE"
        ),
        "status": (
            "PASS"
            if CROSS_SIDE_FORMAL_SIMULATION_ENVELOPE_COMPLETE
            else "NOT_COMPLETED"
        ),
    },
    {
        "gate_order": 12,
        "gate_id": (
            "CALIBRATION_PARAMETER_UPDATES_ZERO"
        ),
        "gate_family": "GOVERNANCE",
        "blocking": True,
        "passed": (
            CALIBRATION_ZERO_UPDATE_PASS
        ),
        "observed_value": float(
            CALIBRATION_PARAMETER_UPDATES_PERFORMED
        ),
        "reference_value": 0.0,
        "decision_rule": "EXACTLY_ZERO",
        "status": (
            "PASS"
            if CALIBRATION_ZERO_UPDATE_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 13,
        "gate_id": (
            "PROTECTED_PARTITIONS_UNOPENED"
        ),
        "gate_family": "GOVERNANCE",
        "blocking": True,
        "passed": (
            PROTECTED_PARTITIONS_UNOPENED_PASS
        ),
        "observed_value": float(
            PROTECTED_PARTITIONS_UNOPENED_PASS
        ),
        "reference_value": 1.0,
        "decision_rule": (
            "VALIDATION_AND_ENGINEERING_HOLDOUT_CONTENT_FALSE"
        ),
        "status": (
            "PASS"
            if PROTECTED_PARTITIONS_UNOPENED_PASS
            else "FAIL"
        ),
    },
    {
        "gate_order": 14,
        "gate_id": (
            "FILESYSTEM_WRITES_ABSENT"
        ),
        "gate_family": "GOVERNANCE",
        "blocking": True,
        "passed": (
            FILESYSTEM_WRITE_GUARD_PASS
        ),
        "observed_value": float(
            FILESYSTEM_WRITE_GUARD_PASS
        ),
        "reference_value": 1.0,
        "decision_rule": (
            "NO_FILESYSTEM_WRITES_BEFORE_PERSISTENCE"
        ),
        "status": (
            "PASS"
            if FILESYSTEM_WRITE_GUARD_PASS
            else "FAIL"
        ),
    },
]

HAWKES_DIAGNOSTIC_GATE_LEDGER = (
    pd.DataFrame.from_records(
        diagnostic_gate_records
    )
    .sort_values(
        "gate_order",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

blocking_gate_mask = (
    HAWKES_DIAGNOSTIC_GATE_LEDGER[
        "blocking"
    ]
)

BLOCKING_DIAGNOSTIC_GATE_COUNT = int(
    blocking_gate_mask.sum()
)

PASSED_BLOCKING_DIAGNOSTIC_GATE_COUNT = int(
    (
        blocking_gate_mask
        & HAWKES_DIAGNOSTIC_GATE_LEDGER[
            "passed"
        ]
    ).sum()
)

FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT = (
    BLOCKING_DIAGNOSTIC_GATE_COUNT
    - PASSED_BLOCKING_DIAGNOSTIC_GATE_COUNT
)

require(
    (
        FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT
        == 0
    )
    == HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS,
    (
        "Terminal acceptance does not reconcile with the "
        "blocking diagnostic-gate ledger."
    ),
)


# ------------------------------------------------------------
# Model rejection ledger
# ------------------------------------------------------------

rejection_reason_map = {
    (
        "MATERIAL_SCORE_IMPROVEMENT_OVER_FORMAL_COMPARATOR"
    ): (
        "H1 did not establish positive paired score improvement "
        "over the formal Notebook 06 comparator."
    ),
    (
        "MATERIAL_SCORE_IMPROVEMENT_OVER_PRIMARY_SIMPLE_BASELINE"
    ): (
        "H1 did not establish positive paired score improvement "
        "over the primary simple Notebook 06 baseline."
    ),
    (
        "MATERIAL_SCORE_IMPROVEMENT_OVER_ALL_AUTHORIZED_BASELINES"
    ): (
        "At least one authorized simpler baseline was not "
        "materially outscored."
    ),
    "DEFENSIBLE_STATIONARITY_MARGIN": (
        "The fitted branching matrix did not retain the required "
        "engineering stationarity margin."
    ),
    (
        "LOCKED_CALIBRATION_RESIDUAL_DISTRIBUTION"
    ): (
        "Locked CALIBRATION time-rescaling residual distributions "
        "show systematic departures from frozen-H1 simulation."
    ),
    (
        "LOCKED_CALIBRATION_RESIDUAL_SERIAL_DEPENDENCE"
    ): (
        "Locked CALIBRATION residuals retain serial dependence "
        "outside frozen-H1 simulation envelopes."
    ),
    (
        "LOCKED_CALIBRATION_PATH_COUNT_CALIBRATION"
    ): (
        "Locked CALIBRATION BUY, SELL, pooled, or batch counts "
        "fall outside frozen-H1 simulation envelopes."
    ),
    (
        "LOCKED_CALIBRATION_EXACT_TIME_MULTIPLICITY"
    ): (
        "Observed exact-time multiplicity is not reproduced "
        "adequately by frozen H1 after one-millisecond coarsening."
    ),
    (
        "LOCKED_CALIBRATION_SIDE_MARK_CALIBRATION"
    ): (
        "Locked CALIBRATION BUY/SELL mark calibration falls "
        "outside the directional frozen-H1 envelope."
    ),
    (
        "CALIBRATION_PARAMETER_UPDATES_ZERO"
    ): (
        "CALIBRATION parameters were changed after DEVELOPMENT."
    ),
    "PROTECTED_PARTITIONS_UNOPENED": (
        "A protected partition was opened before authorization."
    ),
    "FILESYSTEM_WRITES_ABSENT": (
        "Unexpected filesystem writes occurred before persistence."
    ),
}

failed_blocking_gates = (
    HAWKES_DIAGNOSTIC_GATE_LEDGER.loc[
        HAWKES_DIAGNOSTIC_GATE_LEDGER[
            "blocking"
        ]
        & ~HAWKES_DIAGNOSTIC_GATE_LEDGER[
            "passed"
        ]
    ]
)

model_rejection_records = []

for rejection_order, gate_row in enumerate(
    failed_blocking_gates.itertuples(
        index=False
    ),
    start=1,
):
    model_rejection_records.append(
        {
            "rejection_order": int(
                rejection_order
            ),
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "gate_id": str(
                gate_row.gate_id
            ),
            "gate_family": str(
                gate_row.gate_family
            ),
            "observed_value": float(
                gate_row.observed_value
            ),
            "reference_value": float(
                gate_row.reference_value
            ),
            "decision_rule": str(
                gate_row.decision_rule
            ),
            "rejection_reason": (
                rejection_reason_map.get(
                    str(
                        gate_row.gate_id
                    ),
                    (
                        "A blocking Notebook 08 diagnostic "
                        "gate failed."
                    ),
                )
            ),
            "repair_performed": False,
            "parameter_refit_performed": False,
            "calibration_parameter_updates": 0,
            "status": "BLOCKING_REJECTION",
        }
    )

HAWKES_MODEL_REJECTION_LEDGER = (
    pd.DataFrame.from_records(
        model_rejection_records
    )
)

if HAWKES_MODEL_REJECTION_LEDGER.empty:
    HAWKES_MODEL_REJECTION_LEDGER = pd.DataFrame(
        columns=[
            "rejection_order",
            "model_id",
            "gate_id",
            "gate_family",
            "observed_value",
            "reference_value",
            "decision_rule",
            "rejection_reason",
            "repair_performed",
            "parameter_refit_performed",
            "calibration_parameter_updates",
            "status",
        ]
    )


# ------------------------------------------------------------
# Frozen model disposition
# ------------------------------------------------------------

if HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS:
    FROZEN_HAWKES_DIAGNOSTIC_DISPOSITION = (
        "ACCEPTED_FOR_NOTEBOOK09_SIGNAL_VALIDATION"
    )

    FROZEN_HAWKES_ALLOWED_ROLE = (
        "FROZEN_INTENSITY_SIGNAL_CANDIDATE"
    )

else:
    FROZEN_HAWKES_DIAGNOSTIC_DISPOSITION = (
        "REJECTED_FOR_DOWNSTREAM_SIGNAL_AUTHORIZATION"
    )

    FROZEN_HAWKES_ALLOWED_ROLE = (
        "DIAGNOSTIC_REFERENCE_AND_FAILURE_ANALYSIS_ONLY"
    )


FROZEN_HAWKES_SPECIFICATION_DISPOSITION = pd.DataFrame(
    [
        {
            "model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "model_family": (
                EXPECTED_SELECTED_MODEL_FAMILY
            ),
            "event_representation": (
                "SAME_MS_SAME_SIDE_BURSTS"
            ),
            "timestamp_authority": (
                "event_time_ns"
            ),
            "fitted_partition": (
                "DEVELOPMENT"
            ),
            "locked_evaluation_partition": (
                "CALIBRATION"
            ),
            "mu_buy_per_second": float(
                FROZEN_H1_PARAMETERS.mu_buy
            ),
            "mu_sell_per_second": float(
                FROZEN_H1_PARAMETERS.mu_sell
            ),
            "kappa_buy_to_buy": float(
                FROZEN_H1_PARAMETERS.kappa_buy
            ),
            "kappa_sell_to_sell": float(
                FROZEN_H1_PARAMETERS.kappa_sell
            ),
            "kappa_buy_to_sell": 0.0,
            "kappa_sell_to_buy": 0.0,
            "beta_shared_per_second": float(
                FROZEN_H1_PARAMETERS.beta_buy
            ),
            "spectral_radius": (
                FROZEN_H1_SPECTRAL_RADIUS
            ),
            "stationarity_margin": (
                FROZEN_H1_STATIONARITY_MARGIN
            ),
            "count_score_superiority_authorized": (
                HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
            ),
            "full_model_adequacy_authorized": (
                CALIBRATION_MODEL_ADEQUACY_PASS
            ),
            "full_hawkes_superiority_authorized": (
                HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS
            ),
            "cross_excitation_authorized": False,
            "state_dependent_hawkes_authorized": False,
            "parameter_refit_performed": False,
            "calibration_parameter_updates": 0,
            "diagnostic_disposition": (
                FROZEN_HAWKES_DIAGNOSTIC_DISPOSITION
            ),
            "allowed_downstream_role": (
                FROZEN_HAWKES_ALLOWED_ROLE
            ),
            "status": (
                NOTEBOOK08_TERMINAL_CLASS
            ),
        }
    ]
)


# ------------------------------------------------------------
# Notebook 08 terminal decision
# ------------------------------------------------------------

NOTEBOOK09_AUTHORIZED = bool(
    HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS
)

VALIDATION_PARTITION_OPENING_AUTHORIZED = bool(
    NOTEBOOK09_AUTHORIZED
)

HAWKES_SUPERIORITY_AUTHORIZED = bool(
    HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS
)

NOTEBOOK08_TERMINAL_DECISION = pd.DataFrame(
    [
        {
            "notebook": (
                "08_HAWKES_DIAGNOSTICS.ipynb"
            ),
            "selected_model_id": (
                EXPECTED_SELECTED_MODEL_ID
            ),
            "terminal_class": (
                NOTEBOOK08_TERMINAL_CLASS
            ),
            "terminal_status": (
                NOTEBOOK08_TERMINAL_STATUS
            ),
            "blocking_gate_count": (
                BLOCKING_DIAGNOSTIC_GATE_COUNT
            ),
            "passed_blocking_gate_count": (
                PASSED_BLOCKING_DIAGNOSTIC_GATE_COUNT
            ),
            "failed_blocking_gate_count": (
                FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT
            ),
            "count_score_superiority_authorized": (
                HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
            ),
            "stationarity_margin_passed": (
                STATIONARITY_MARGIN_PASS
            ),
            "calibration_residual_distribution_passed": (
                CALIBRATION_RESIDUAL_DISTRIBUTION_PASS
            ),
            "calibration_residual_serial_passed": (
                CALIBRATION_RESIDUAL_SERIAL_PASS
            ),
            "calibration_path_count_passed": (
                CALIBRATION_PATH_COUNT_PASS
            ),
            "calibration_multiplicity_passed": (
                CALIBRATION_MULTIPLICITY_PASS
            ),
            "calibration_side_mark_passed": (
                CALIBRATION_SIDE_MARK_PASS
            ),
            "calibration_model_adequacy_passed": (
                CALIBRATION_MODEL_ADEQUACY_PASS
            ),
            "cross_excitation_authorized": False,
            "state_dependent_hawkes_authorized": False,
            "full_hawkes_superiority_authorized": (
                HAWKES_SUPERIORITY_AUTHORIZED
            ),
            "notebook09_authorized": (
                NOTEBOOK09_AUTHORIZED
            ),
            "validation_partition_opening_authorized": (
                VALIDATION_PARTITION_OPENING_AUTHORIZED
            ),
            "engineering_holdout_opened": False,
            "calibration_parameter_updates": 0,
            "interpretation": (
                NOTEBOOK08_TERMINAL_INTERPRETATION
            ),
            "status": (
                NOTEBOOK08_TERMINAL_CLASS
            ),
        }
    ]
)


# ------------------------------------------------------------
# Update comparison-scope ledger without changing evidence
# ------------------------------------------------------------

HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER = (
    HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER.copy()
)

adequacy_scope_mask = (
    HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER[
        "comparison_component"
    ].eq(
        "COARSENING_AWARE_MODEL_ADEQUACY"
    )
)

terminal_scope_mask = (
    HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER[
        "comparison_component"
    ].eq(
        "FINAL_HAWKES_SUPERIORITY_DECISION"
    )
)

HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER.loc[
    adequacy_scope_mask,
    [
        "completed",
        "full_hawkes_superiority_claim_authorized",
        "status",
    ],
] = [
    True,
    HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS,
    (
        "PASS"
        if CALIBRATION_MODEL_ADEQUACY_PASS
        else "FAIL_MODEL_ADEQUACY"
    ),
]

HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER.loc[
    terminal_scope_mask,
    [
        "completed",
        "full_hawkes_superiority_claim_authorized",
        "status",
    ],
] = [
    True,
    HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS,
    NOTEBOOK08_TERMINAL_STATUS,
]


# ------------------------------------------------------------
# Terminal verification
# ------------------------------------------------------------

require(
    len(
        NOTEBOOK08_TERMINAL_DECISION
    )
    == 1,
    "Notebook 08 terminal decision must contain one row.",
)

require(
    len(
        FROZEN_HAWKES_SPECIFICATION_DISPOSITION
    )
    == 1,
    "Frozen Hawkes disposition must contain one row.",
)

require(
    len(
        HAWKES_MODEL_REJECTION_LEDGER
    )
    == FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT,
    (
        "The rejection-ledger row count does not match the "
        "failed blocking-gate count."
    ),
)

require(
    (
        NOTEBOOK09_AUTHORIZED
        and VALIDATION_PARTITION_OPENING_AUTHORIZED
    )
    or (
        not NOTEBOOK09_AUTHORIZED
        and not VALIDATION_PARTITION_OPENING_AUTHORIZED
    ),
    "Notebook 09 and VALIDATION authorization disagree.",
)

require(
    HAWKES_SUPERIORITY_AUTHORIZED
    == HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS,
    "Full Hawkes-superiority authority is inconsistent.",
)

require(
    CROSS_EXCITATION_AUTHORIZED
    is False,
    "Cross-excitation was incorrectly authorized.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED
    == 0,
    "CALIBRATION parameter updates are nonzero.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False,
    "Protected partition content was loaded.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is False,
    "This cell performed a filesystem write.",
)


NOTEBOOK08_TERMINAL_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "locked_calibration_envelopes_complete"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "score_superiority_separated_from_model_adequacy"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "blocking_gate_ledger_reconciles"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "rejection_ledger_reconciles"
            ),
            "passed": (
                len(
                    HAWKES_MODEL_REJECTION_LEDGER
                )
                == FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT
            ),
        },
        {
            "check_name": (
                "higher_likelihood_not_used_as_sufficient_condition"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "cross_excitation_remains_unauthorized"
            ),
            "passed": (
                CROSS_EXCITATION_AUTHORIZED
                is False
            ),
        },
        {
            "check_name": (
                "state_dependent_hawkes_remains_unauthorized"
            ),
            "passed": True,
        },
        {
            "check_name": (
                "notebook09_authorization_matches_terminal_gate"
            ),
            "passed": (
                NOTEBOOK09_AUTHORIZED
                == HAWKES_FULL_DIAGNOSTIC_ACCEPTANCE_PASS
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
            ),
        },
        {
            "check_name": (
                "filesystem_writes_absent"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

require(
    NOTEBOOK08_TERMINAL_VERIFICATION[
        "passed"
    ].all(),
    "At least one Notebook 08 terminal verification failed.",
)


# ------------------------------------------------------------
# Publish terminal state
# ------------------------------------------------------------

NOTEBOOK08_TERMINAL_DECISION_CREATED = True
NOTEBOOK08_PERSISTENCE_AUTHORIZED = True
NOTEBOOK08_COMPLETED_IN_MEMORY = True

FILESYSTEM_WRITES_PERFORMED = False


display(
    CALIBRATION_ENVELOPE_FAMILY_SUMMARY
)

display(
    CALIBRATION_RESIDUAL_DISTRIBUTION_SCOPE_SUMMARY
)

display(
    CALIBRATION_RESIDUAL_SERIAL_SCOPE_SUMMARY
)

display(
    HAWKES_DIAGNOSTIC_GATE_LEDGER
)

display(
    HAWKES_MODEL_REJECTION_LEDGER
)

display(
    FROZEN_HAWKES_SPECIFICATION_DISPOSITION
)

display(
    NOTEBOOK08_TERMINAL_DECISION
)

display(
    NOTEBOOK08_TERMINAL_VERIFICATION
)

print(
    f"Notebook 08 terminal status: "
    f"{NOTEBOOK08_TERMINAL_STATUS}. "
    f"Frozen H1 count-score superiority over the authorized "
    f"Notebook 06 baselines is "
    f"{'supported' if HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED else 'not supported'}, "
    f"but full diagnostic adequacy is "
    f"{'accepted' if CALIBRATION_MODEL_ADEQUACY_PASS else 'rejected'}. "
    f"Notebook 09 and opening of VALIDATION are "
    f"{'authorized' if NOTEBOOK09_AUTHORIZED else 'not authorized'}. "
    "The fitted parameters remain unchanged, CALIBRATION updates "
    "remain zero, VALIDATION and ENGINEERING_HOLDOUT remain unopened, "
    "and no filesystem writes were performed."
)

,diagnostic_family,metric_count,outside_directional_envelope_count,outside_two_sided_envelope_count,minimum_directional_tail_probability,outside_directional_envelope_fraction,outside_two_sided_envelope_fraction,within_declared_adequacy_tolerance,status
0,EXACT_TIME_MULTIPLICITY,7,4,4,0.000999000999,0.5714285714,0.5714285714,False,FAIL
1,PATH_COUNTS,4,4,4,0.001998001998,1,1,False,FAIL
2,SIDE_MARK_CALIBRATION,3,0,2,0.1508491508,0,0.6666666667,True,PASS
3,TIME_RESCALING_RESIDUAL,27,26,26,0.000999000999,0.962962963,0.962962963,False,FAIL


,diagnostic_scope,metric_count,outside_directional_envelope_count,outside_two_sided_envelope_count,minimum_directional_tail_probability,outside_directional_envelope_fraction,outside_two_sided_envelope_fraction,within_declared_adequacy_tolerance,status
0,BUY,4,4,4,0.000999000999,1,1,False,FAIL
1,POOLED,4,4,4,0.000999000999,1,1,False,FAIL
2,SELL,4,3,3,0.000999000999,0.75,0.75,False,FAIL


,diagnostic_scope,metric_count,outside_directional_envelope_count,outside_two_sided_envelope_count,minimum_directional_tail_probability,outside_directional_envelope_fraction,outside_two_sided_envelope_fraction,within_declared_adequacy_tolerance,status
0,BUY,5,5,5,0.000999000999,1,1,False,FAIL
1,POOLED,5,5,5,0.000999000999,1,1,False,FAIL
2,SELL,5,5,5,0.000999000999,1,1,False,FAIL


,gate_order,gate_id,gate_family,blocking,passed,observed_value,reference_value,decision_rule,status
0,1,MATERIAL_SCORE_IMPROVEMENT_OVER_FORMAL_COMPARATOR,SCORE_SUPERIORITY,True,True,"3,526.145892",0,PAIRED_BOOTSTRAP_Q025_GREATER_THAN_ZERO,PASS
1,2,MATERIAL_SCORE_IMPROVEMENT_OVER_PRIMARY_SIMPLE...,SCORE_SUPERIORITY,True,True,"3,157.846627",0,PAIRED_BOOTSTRAP_Q025_GREATER_THAN_ZERO,PASS
2,3,MATERIAL_SCORE_IMPROVEMENT_OVER_ALL_AUTHORIZED...,SCORE_SUPERIORITY,True,True,"2,862.365487",0,EVERY_PAIRED_BOOTSTRAP_Q025_GREATER_THAN_ZERO,PASS
3,4,DEFENSIBLE_STATIONARITY_MARGIN,STATIONARITY,True,True,0.1892556591,0.98,SPECTRAL_RADIUS_LESS_THAN_OR_EQUAL_TO_0_98,PASS
4,5,LOCKED_CALIBRATION_RESIDUAL_DISTRIBUTION,MODEL_ADEQUACY,True,False,1,0.2,MAX_SCOPE_OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_...,FAIL
5,6,LOCKED_CALIBRATION_RESIDUAL_SERIAL_DEPENDENCE,MODEL_ADEQUACY,True,False,1,0.2,MAX_SCOPE_OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_...,FAIL
6,7,LOCKED_CALIBRATION_PATH_COUNT_CALIBRATION,MODEL_ADEQUACY,True,False,1,0,NO_PRIMARY_PATH_COUNT_OUTSIDE_95PCT_ENVELOPE,FAIL
7,8,LOCKED_CALIBRATION_EXACT_TIME_MULTIPLICITY,MODEL_ADEQUACY,True,False,0.5714285714,0.2,OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_TO_0_20,FAIL
8,9,LOCKED_CALIBRATION_SIDE_MARK_CALIBRATION,MODEL_ADEQUACY,True,True,0,0,NO_PRIMARY_MARK_METRIC_OUTSIDE_DIRECTIONAL_ENV...,PASS
9,10,CROSS_SIDE_DESCRIPTIVE_REPORT_COMPLETE,CROSS_SIDE_DEPENDENCE,False,True,1,1,DESCRIPTIVE_CCF_PORTMANTEAU_AND_COINCIDENCE_TA...,PASS


,rejection_order,model_id,gate_id,gate_family,observed_value,reference_value,decision_rule,rejection_reason,repair_performed,parameter_refit_performed,calibration_parameter_updates,status
0,1,H1_DIAGONAL_SHARED_DECAY,LOCKED_CALIBRATION_RESIDUAL_DISTRIBUTION,MODEL_ADEQUACY,1,0.2,MAX_SCOPE_OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_...,Locked CALIBRATION time-rescaling residual dis...,False,False,0,BLOCKING_REJECTION
1,2,H1_DIAGONAL_SHARED_DECAY,LOCKED_CALIBRATION_RESIDUAL_SERIAL_DEPENDENCE,MODEL_ADEQUACY,1,0.2,MAX_SCOPE_OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_...,Locked CALIBRATION residuals retain serial dep...,False,False,0,BLOCKING_REJECTION
2,3,H1_DIAGONAL_SHARED_DECAY,LOCKED_CALIBRATION_PATH_COUNT_CALIBRATION,MODEL_ADEQUACY,1,0,NO_PRIMARY_PATH_COUNT_OUTSIDE_95PCT_ENVELOPE,"Locked CALIBRATION BUY, SELL, pooled, or batch...",False,False,0,BLOCKING_REJECTION
3,4,H1_DIAGONAL_SHARED_DECAY,LOCKED_CALIBRATION_EXACT_TIME_MULTIPLICITY,MODEL_ADEQUACY,0.5714285714,0.2,OUTSIDE_FRACTION_LESS_THAN_OR_EQUAL_TO_0_20,Observed exact-time multiplicity is not reprod...,False,False,0,BLOCKING_REJECTION


,model_id,model_family,event_representation,timestamp_authority,fitted_partition,locked_evaluation_partition,mu_buy_per_second,mu_sell_per_second,kappa_buy_to_buy,kappa_sell_to_sell,kappa_buy_to_sell,kappa_sell_to_buy,beta_shared_per_second,spectral_radius,stationarity_margin,count_score_superiority_authorized,full_model_adequacy_authorized,full_hawkes_superiority_authorized,cross_excitation_authorized,state_dependent_hawkes_authorized,parameter_refit_performed,calibration_parameter_updates,diagnostic_disposition,allowed_downstream_role,status
0,H1_DIAGONAL_SHARED_DECAY,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,SAME_MS_SAME_SIDE_BURSTS,event_time_ns,DEVELOPMENT,CALIBRATION,1.536124799,1.767916532,0.1892556591,0.1126637105,0,0,70.67941658,0.1892556591,0.8107443409,True,False,False,False,False,False,0,REJECTED_FOR_DOWNSTREAM_SIGNAL_AUTHORIZATION,DIAGNOSTIC_REFERENCE_AND_FAILURE_ANALYSIS_ONLY,FAIL


,notebook,selected_model_id,terminal_class,terminal_status,blocking_gate_count,passed_blocking_gate_count,failed_blocking_gate_count,count_score_superiority_authorized,stationarity_margin_passed,calibration_residual_distribution_passed,calibration_residual_serial_passed,calibration_path_count_passed,calibration_multiplicity_passed,calibration_side_mark_passed,calibration_model_adequacy_passed,cross_excitation_authorized,state_dependent_hawkes_authorized,full_hawkes_superiority_authorized,notebook09_authorized,validation_partition_opening_authorized,engineering_holdout_opened,calibration_parameter_updates,interpretation,status
0,08_HAWKES_DIAGNOSTICS.ipynb,H1_DIAGONAL_SHARED_DECAY,FAIL,FAIL_HAWKES_DIAGNOSTIC_ADEQUACY_DESPITE_SCORE_...,12,8,4,True,True,False,False,False,False,True,False,False,False,False,False,False,False,0,Frozen H1 produces materially better locked CA...,FAIL


,check_name,passed
0,locked_calibration_envelopes_complete,True
1,score_superiority_separated_from_model_adequacy,True
2,blocking_gate_ledger_reconciles,True
3,rejection_ledger_reconciles,True
4,higher_likelihood_not_used_as_sufficient_condi...,True
5,cross_excitation_remains_unauthorized,True
6,state_dependent_hawkes_remains_unauthorized,True
7,notebook09_authorization_matches_terminal_gate,True
8,calibration_parameter_updates_zero,True
9,protected_partitions_unopened,True


Notebook 08 terminal status: FAIL_HAWKES_DIAGNOSTIC_ADEQUACY_DESPITE_SCORE_SUPERIORITY. Frozen H1 count-score superiority over the authorized Notebook 06 baselines is supported, but full diagnostic adequacy is rejected. Notebook 09 and opening of VALIDATION are not authorized. The fitted parameters remain unchanged, CALIBRATION updates remain zero, VALIDATION and ENGINEERING_HOLDOUT remain unopened, and no filesystem writes were performed.


In [27]:
# ============================================================
# Persist authoritative Notebook 08 core diagnostic artifacts
#
# The terminal result is persisted even though H1 failed full
# adequacy. Persistence records the supported narrow score result,
# the blocking diagnostic failures, the frozen rejected model
# disposition, and the prohibition on opening VALIDATION.
#
# This is the authorized core-write stage. Semantic readback,
# final acceptance, and the blocked Notebook 09 handoff follow in
# the next cell.
# ============================================================

from __future__ import annotations

import dataclasses
import gzip
import hashlib
import io
import json
import math
import os
import tempfile
from pathlib import Path
from typing import Any, Final, Mapping

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "NOTEBOOK08_TERMINAL_DECISION_CREATED",
            False,
        )
    ),
    "The Notebook 08 terminal decision is unavailable.",
)

require(
    bool(
        globals().get(
            "NOTEBOOK08_PERSISTENCE_AUTHORIZED",
            False,
        )
    ),
    "Notebook 08 persistence is not authorized.",
)

require(
    bool(
        globals().get(
            "NOTEBOOK08_COMPLETED_IN_MEMORY",
            False,
        )
    ),
    "Notebook 08 in-memory computation is incomplete.",
)

require(
    not bool(
        globals().get(
            "NOTEBOOK08_PERSISTENCE_COMPLETED",
            False,
        )
    ),
    "Notebook 08 core persistence has already completed.",
)

require(
    isinstance(
        NOTEBOOK08_TERMINAL_DECISION,
        pd.DataFrame,
    )
    and len(
        NOTEBOOK08_TERMINAL_DECISION
    )
    == 1,
    "The Notebook 08 terminal-decision table is invalid.",
)

require(
    isinstance(
        HAWKES_DIAGNOSTIC_GATE_LEDGER,
        pd.DataFrame,
    )
    and not HAWKES_DIAGNOSTIC_GATE_LEDGER.empty,
    "The Hawkes diagnostic-gate ledger is unavailable.",
)

require(
    isinstance(
        FROZEN_HAWKES_SPECIFICATION_DISPOSITION,
        pd.DataFrame,
    )
    and len(
        FROZEN_HAWKES_SPECIFICATION_DISPOSITION
    )
    == 1,
    "The frozen Hawkes disposition is unavailable.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED
    == 0,
    "CALIBRATION parameter updates must remain zero.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False,
    "Protected partition content must remain unopened.",
)

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition_name
        ]
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is False,
    (
        "Unexpected filesystem writes were recorded before the "
        "authorized Notebook 08 persistence stage."
    ),
)


# ------------------------------------------------------------
# Resolve frozen persistence identity
# ------------------------------------------------------------

NOTEBOOK08_NAME: Final[str] = (
    "08_HAWKES_DIAGNOSTICS.ipynb"
)

NOTEBOOK08_STAGE: Final[str] = (
    "HAWKES_DIAGNOSTICS"
)

NOTEBOOK08_RUN_ID: Final[str] = str(
    globals().get(
        "V0_1_RUN_ID",
        globals().get(
            "V01_RUN_ID",
            "",
        ),
    )
)

require(
    bool(
        NOTEBOOK08_RUN_ID
    ),
    "The V0.1 run ID is unavailable.",
)

NOTEBOOK08_OUTPUT_PREFIX: Final[str] = str(
    globals().get(
        "CURRENT_COMBINED_OUTPUT_PREFIX",
        globals().get(
            "COMBINED_OUTPUT_PREFIX",
            (
                str(
                    SOURCE_RUN_PREFIX
                )
                + "__"
                + NOTEBOOK08_RUN_ID
            ),
        ),
    )
)

require(
    bool(
        NOTEBOOK08_OUTPUT_PREFIX
    ),
    "The Notebook 08 output prefix is unavailable.",
)

NOTEBOOK08_V01_ROOT: Final[Path] = Path(
    globals().get(
        "V01_ROOT",
        r"D:\Clown Project\V0.1",
    )
)

NOTEBOOK08_V00_ROOT: Final[Path] = Path(
    globals().get(
        "V00_ROOT",
        r"D:\Clown Project\V0.0",
    )
)

require(
    NOTEBOOK08_V01_ROOT.is_dir(),
    (
        "The V0.1 persistence root does not exist: "
        f"{NOTEBOOK08_V01_ROOT}"
    ),
)


# ------------------------------------------------------------
# Authorized output directories
# ------------------------------------------------------------

NOTEBOOK08_ARTIFACT_ROOT: Final[Path] = (
    NOTEBOOK08_V01_ROOT
    / "artifacts"
)

NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_ARTIFACT_ROOT
    / "diagnostics"
)

NOTEBOOK08_RECONCILIATION_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_ARTIFACT_ROOT
    / "reconciliation"
)

NOTEBOOK08_COMPARISON_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_ARTIFACT_ROOT
    / "comparisons"
)

NOTEBOOK08_SIMULATION_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_ARTIFACT_ROOT
    / "simulations"
)

NOTEBOOK08_AUDIT_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_ARTIFACT_ROOT
    / "audit_tables"
)

NOTEBOOK08_MANIFEST_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_ARTIFACT_ROOT
    / "manifests"
)

NOTEBOOK08_HANDOFF_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_ARTIFACT_ROOT
    / "handoff"
)

NOTEBOOK08_LOG_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK08_V01_ROOT
    / "logs"
)


def persistence_normalized_path(
    source_path: Path,
) -> str:
    """Return one normalized absolute path string."""
    return os.path.normcase(
        os.path.abspath(
            os.fspath(
                source_path
            )
        )
    )


def persistence_path_is_inside(
    candidate_path: Path,
    parent_path: Path,
) -> bool:
    """Return whether candidate_path is contained by parent_path."""
    candidate_normalized = (
        persistence_normalized_path(
            candidate_path
        )
    )

    parent_normalized = (
        persistence_normalized_path(
            parent_path
        )
    )

    try:
        common_path = os.path.commonpath(
            [
                candidate_normalized,
                parent_normalized,
            ]
        )

    except ValueError:
        return False

    return (
        common_path
        == parent_normalized
    )


def require_authorized_output_path(
    output_path: Path,
) -> None:
    """Require one path to lie inside writable V0.1 and outside V0.0."""
    require(
        persistence_path_is_inside(
            output_path,
            NOTEBOOK08_V01_ROOT,
        ),
        (
            "Unauthorized output path outside V0.1: "
            f"{output_path}"
        ),
    )

    require(
        not persistence_path_is_inside(
            output_path,
            NOTEBOOK08_V00_ROOT,
        ),
        (
            "Unauthorized output path inside immutable V0.0: "
            f"{output_path}"
        ),
    )


for authorized_directory in (
    NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    NOTEBOOK08_RECONCILIATION_OUTPUT_DIR,
    NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    NOTEBOOK08_AUDIT_OUTPUT_DIR,
    NOTEBOOK08_MANIFEST_OUTPUT_DIR,
    NOTEBOOK08_HANDOFF_OUTPUT_DIR,
    NOTEBOOK08_LOG_OUTPUT_DIR,
):
    require_authorized_output_path(
        authorized_directory
    )


# ------------------------------------------------------------
# Deterministic serialization helpers
# ------------------------------------------------------------

def persistence_file_sha256(
    source_path: Path,
    chunk_size: int = 1 << 20,
) -> str:
    """Return the SHA-256 digest of one physical file."""
    require(
        source_path.is_file(),
        f"Persisted file does not exist: {source_path}",
    )

    digest = hashlib.sha256()

    with source_path.open(
        "rb"
    ) as source_file:
        for chunk in iter(
            lambda: source_file.read(
                chunk_size
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def persistence_json_safe(
    value: Any,
) -> Any:
    """Convert nested Python, pandas, and NumPy values to JSON-safe values."""
    if value is None or value is pd.NA:
        return None

    if dataclasses.is_dataclass(
        value
    ):
        return persistence_json_safe(
            dataclasses.asdict(
                value
            )
        )

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        Mapping,
    ):
        return {
            str(
                key
            ): persistence_json_safe(
                nested_value
            )
            for key, nested_value in (
                value.items()
            )
        }

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            persistence_json_safe(
                nested_value
            )
            for nested_value in value
        ]

    if isinstance(
        value,
        np.ndarray,
    ):
        return persistence_json_safe(
            value.tolist()
        )

    if isinstance(
        value,
        (
            np.integer,
            pd.Int64Dtype,
        ),
    ):
        return int(
            value
        )

    if isinstance(
        value,
        np.floating,
    ):
        numeric_value = float(
            value
        )

        return (
            numeric_value
            if math.isfinite(
                numeric_value
            )
            else None
        )

    if isinstance(
        value,
        float,
    ):
        return (
            value
            if math.isfinite(
                value
            )
            else None
        )

    if isinstance(
        value,
        (
            np.bool_,
            bool,
        ),
    ):
        return bool(
            value
        )

    if isinstance(
        value,
        (
            pd.Timestamp,
            pd.Timedelta,
        ),
    ):
        return str(
            value
        )

    if isinstance(
        value,
        (
            str,
            int,
        ),
    ):
        return value

    return str(
        value
    )


def canonical_json_bytes(
    payload: Any,
) -> bytes:
    """Return canonical compact JSON bytes."""
    safe_payload = persistence_json_safe(
        payload
    )

    return json.dumps(
        safe_payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        ensure_ascii=False,
        allow_nan=False,
    ).encode(
        "utf-8"
    )


def canonical_json_sha256(
    payload: Any,
) -> str:
    """Return a canonical semantic JSON hash."""
    return hashlib.sha256(
        canonical_json_bytes(
            payload
        )
    ).hexdigest()


def formatted_json_bytes(
    payload: Any,
) -> bytes:
    """Return deterministic human-readable JSON bytes."""
    safe_payload = persistence_json_safe(
        payload
    )

    serialized = json.dumps(
        safe_payload,
        sort_keys=True,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

    return (
        serialized
        + "\n"
    ).encode(
        "utf-8"
    )


def atomic_install_bytes(
    output_path: Path,
    payload_bytes: bytes,
) -> str:
    """
    Atomically install deterministic bytes.

    An existing identical file is accepted. An existing file with
    different bytes is a blocking collision.
    """
    require_authorized_output_path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    payload_sha256 = hashlib.sha256(
        payload_bytes
    ).hexdigest()

    if output_path.exists():
        require(
            output_path.is_file(),
            (
                "An output path exists but is not a file: "
                f"{output_path}"
            ),
        )

        existing_sha256 = (
            persistence_file_sha256(
                output_path
            )
        )

        require(
            existing_sha256
            == payload_sha256,
            (
                "Persistence collision with different content: "
                f"{output_path}"
            ),
        )

        return existing_sha256

    temporary_file = tempfile.NamedTemporaryFile(
        mode="wb",
        dir=output_path.parent,
        prefix=(
            output_path.name
            + "."
        ),
        suffix=".tmp",
        delete=False,
    )

    temporary_path = Path(
        temporary_file.name
    )

    try:
        with temporary_file:
            temporary_file.write(
                payload_bytes
            )

            temporary_file.flush()

            os.fsync(
                temporary_file.fileno()
            )

        os.replace(
            temporary_path,
            output_path,
        )

    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    observed_sha256 = (
        persistence_file_sha256(
            output_path
        )
    )

    require(
        observed_sha256
        == payload_sha256,
        (
            "Atomic byte persistence hash mismatch: "
            f"{output_path}"
        ),
    )

    return observed_sha256


def dataframe_gzip_bytes(
    dataframe: pd.DataFrame,
) -> bytes:
    """Serialize one DataFrame as deterministic UTF-8 CSV.GZIP."""
    raw_buffer = io.BytesIO()

    with gzip.GzipFile(
        filename="",
        mode="wb",
        fileobj=raw_buffer,
        compresslevel=9,
        mtime=0,
    ) as gzip_file:
        with io.TextIOWrapper(
            gzip_file,
            encoding="utf-8",
            newline="",
        ) as text_file:
            dataframe.to_csv(
                text_file,
                index=False,
                lineterminator="\n",
                float_format="%.17g",
            )

    return raw_buffer.getvalue()


def table_schema_sha256(
    dataframe: pd.DataFrame,
) -> str:
    """Hash table column order and pandas dtype declarations."""
    schema_payload = {
        "columns": [
            str(
                column_name
            )
            for column_name in (
                dataframe.columns
            )
        ],
        "dtypes": {
            str(
                column_name
            ): str(
                dataframe[
                    column_name
                ].dtype
            )
            for column_name in (
                dataframe.columns
            )
        },
    }

    return canonical_json_sha256(
        schema_payload
    )


def persistence_slug(
    variable_name: str,
) -> str:
    """Convert a Python variable name to one stable file slug."""
    return (
        str(
            variable_name
        )
        .strip()
        .lower()
        .replace(
            "__",
            "_",
        )
    )


def write_dataframe_artifact(
    *,
    variable_name: str,
    dataframe: pd.DataFrame,
    artifact_role: str,
    output_directory: Path,
) -> dict[str, Any]:
    """Persist and register one DataFrame artifact."""
    require(
        isinstance(
            dataframe,
            pd.DataFrame,
        ),
        (
            f"{variable_name} is not a pandas DataFrame."
        ),
    )

    output_path = (
        output_directory
        / (
            NOTEBOOK08_OUTPUT_PREFIX
            + "__08_"
            + persistence_slug(
                variable_name
            )
            + ".csv.gz"
        )
    )

    payload_bytes = dataframe_gzip_bytes(
        dataframe
    )

    file_sha256 = atomic_install_bytes(
        output_path,
        payload_bytes,
    )

    return {
        "artifact_type": (
            "NOTEBOOK_08_RESULT_TABLE"
        ),
        "artifact_role": (
            artifact_role
        ),
        "variable_name": (
            variable_name
        ),
        "format": "CSV_GZIP",
        "path": str(
            output_path
        ),
        "relative_path": str(
            output_path.relative_to(
                NOTEBOOK08_V01_ROOT
            )
        ),
        "row_count": int(
            len(
                dataframe
            )
        ),
        "column_count": int(
            dataframe.shape[
                1
            ]
        ),
        "columns": [
            str(
                column_name
            )
            for column_name in (
                dataframe.columns
            )
        ],
        "dtypes": {
            str(
                column_name
            ): str(
                dataframe[
                    column_name
                ].dtype
            )
            for column_name in (
                dataframe.columns
            )
        },
        "schema_sha256": (
            table_schema_sha256(
                dataframe
            )
        ),
        "file_size_bytes": int(
            output_path.stat().st_size
        ),
        "sha256": (
            file_sha256
        ),
        "status": "PASS",
    }


def write_self_hashed_json_artifact(
    *,
    payload: dict[str, Any],
    output_path: Path,
    artifact_role: str,
    variable_name: str,
) -> dict[str, Any]:
    """Persist and register one self-hashed JSON artifact."""
    safe_payload = persistence_json_safe(
        payload
    )

    require(
        isinstance(
            safe_payload,
            dict,
        ),
        "A persisted JSON artifact must have a top-level object.",
    )

    require(
        "payload_sha256"
        not in safe_payload,
        (
            f"{variable_name} payload already contains "
            "payload_sha256."
        ),
    )

    payload_sha256 = canonical_json_sha256(
        safe_payload
    )

    document = {
        **safe_payload,
        "payload_sha256": (
            payload_sha256
        ),
    }

    file_sha256 = atomic_install_bytes(
        output_path,
        formatted_json_bytes(
            document
        ),
    )

    return {
        "artifact_type": str(
            safe_payload.get(
                "artifact_type",
                "NOTEBOOK_08_JSON_ARTIFACT",
            )
        ),
        "artifact_role": (
            artifact_role
        ),
        "variable_name": (
            variable_name
        ),
        "format": "JSON",
        "path": str(
            output_path
        ),
        "relative_path": str(
            output_path.relative_to(
                NOTEBOOK08_V01_ROOT
            )
        ),
        "row_count": None,
        "column_count": None,
        "columns": None,
        "dtypes": None,
        "schema_sha256": None,
        "file_size_bytes": int(
            output_path.stat().st_size
        ),
        "sha256": (
            file_sha256
        ),
        "payload_sha256": (
            payload_sha256
        ),
        "status": "PASS",
    }


def dataframe_records(
    dataframe: pd.DataFrame,
) -> list[dict[str, Any]]:
    """Return JSON-safe records preserving displayed table fields."""
    return persistence_json_safe(
        dataframe.to_dict(
            orient="records"
        )
    )


# ------------------------------------------------------------
# Core table registry
# ------------------------------------------------------------

TABLE_PERSISTENCE_SPECS: Final[
    tuple[
        tuple[
            str,
            str,
            Path,
        ],
        ...,
    ]
] = (
    # Exact replay and compensator reconciliation
    (
        "EXACT_COMPENSATOR_PATH",
        "exact_compensator_path",
        NOTEBOOK08_RECONCILIATION_OUTPUT_DIR,
    ),
    (
        "EXACT_COMPENSATOR_PARTITION_SUMMARY",
        "exact_compensator_partition_summary",
        NOTEBOOK08_RECONCILIATION_OUTPUT_DIR,
    ),
    (
        "BOUNDARY_COMPENSATOR_RECONCILIATION",
        "boundary_compensator_reconciliation",
        NOTEBOOK08_RECONCILIATION_OUTPUT_DIR,
    ),
    (
        "EXACT_COMPENSATOR_VERIFICATION_SUMMARY",
        "exact_compensator_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    # Time-rescaling residuals
    (
        "TIME_RESCALING_RESIDUALS",
        "time_rescaling_residuals",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "TIME_RESCALING_RESIDUAL_SUMMARY",
        "time_rescaling_residual_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "PRIMARY_ELIGIBLE_RESIDUAL_SUMMARY",
        "primary_eligible_residual_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "TIME_RESCALING_RIGHT_CENSORING_LEDGER",
        "time_rescaling_right_censoring",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "TIME_RESCALING_VERIFICATION_SUMMARY",
        "time_rescaling_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    (
        "EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS",
        "empirical_residual_distribution",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "EMPIRICAL_RESIDUAL_ACF",
        "empirical_residual_acf",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "EMPIRICAL_RESIDUAL_LJUNG_BOX",
        "empirical_residual_ljung_box",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "EMPIRICAL_RESIDUAL_QQ_COORDINATES",
        "empirical_residual_qq_coordinates",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "EMPIRICAL_RESIDUAL_SERIAL_SUMMARY",
        "empirical_residual_serial_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "EMPIRICAL_RESIDUAL_DEGRADATION_SUMMARY",
        "empirical_residual_degradation",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "EMPIRICAL_RESIDUAL_DIAGNOSTIC_VERIFICATION",
        "empirical_residual_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    # Marks and multiplicity
    (
        "SIDE_MARK_BATCH_DIAGNOSTICS",
        "side_mark_batch_diagnostics",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "SIDE_MARK_PARTITION_SUMMARY",
        "side_mark_partition_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "SIDE_MARK_BATCH_CLASS_SUMMARY",
        "side_mark_batch_class_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "SIDE_MARK_RELIABILITY_TABLE",
        "side_mark_reliability",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "BATCH_MULTIPLICITY_DISTRIBUTION",
        "batch_multiplicity_distribution",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "SIDE_MULTIPLICITY_DISTRIBUTION",
        "side_multiplicity_distribution",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "MULTIPLICITY_BY_FROZEN_INTENSITY_BIN",
        "multiplicity_by_frozen_intensity",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "MULTIPLICITY_DIAGNOSTIC_SCOPE_LEDGER",
        "multiplicity_scope_ledger",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "SIDE_MARK_MULTIPLICITY_VERIFICATION",
        "side_mark_multiplicity_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    # Common 1 ms grid
    (
        "COMMON_GRID_DIAGNOSTIC_INPUT_SUMMARY",
        "common_grid_input_summary",
        NOTEBOOK08_RECONCILIATION_OUTPUT_DIR,
    ),
    (
        "COMMON_GRID_DIAGNOSTIC_INPUT_VERIFICATION",
        "common_grid_input_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    (
        "HAWKES_COMMON_GRID_CONTRACT_SUMMARY",
        "common_grid_contract_summary",
        NOTEBOOK08_RECONCILIATION_OUTPUT_DIR,
    ),
    (
        "HAWKES_COMMON_GRID_SCORE_TABLE",
        "common_grid_score_table",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_COMMON_GRID_RESIDUAL_SUMMARY",
        "common_grid_residual_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_COUNT_CATEGORY_CALIBRATION",
        "common_grid_count_category_calibration",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_COMMON_GRID_VERIFICATION",
        "common_grid_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    # Cross-side diagnostics
    (
        "HAWKES_CROSS_SIDE_RESIDUAL_CCF",
        "cross_side_residual_ccf",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_CROSS_SIDE_PORTMANTEAU",
        "cross_side_portmanteau",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS",
        "cross_side_coincidence",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_CROSS_SIDE_RESIDUAL_SUMMARY",
        "cross_side_residual_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_CROSS_SIDE_DEGRADATION_SUMMARY",
        "cross_side_degradation",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_CROSS_SIDE_VERIFICATION",
        "cross_side_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    # Simulation engine and envelopes
    (
        "COARSENING_AWARE_SIMULATION_CONTRACT",
        "simulation_contract",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "FROZEN_H1_STATIONARY_RATE_REFERENCE",
        "stationary_rate_reference",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "COARSENING_AWARE_SIMULATION_PILOT_SUMMARY",
        "simulation_pilot_summary",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "COARSENING_AWARE_SIMULATION_ENGINE_VERIFICATION",
        "simulation_engine_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    (
        "OBSERVED_COARSENING_AWARE_DIAGNOSTIC_METRICS",
        "observed_envelope_metric_registry",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS",
        "model_envelope_replicate_metrics",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_SUMMARY",
        "model_envelope_replicate_summary",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "COARSENING_AWARE_REPLICATE_COUNT_SUMMARY",
        "model_envelope_count_summary",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "COARSENING_AWARE_DIAGNOSTIC_ENVELOPES",
        "coarsening_aware_diagnostic_envelopes",
        NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    ),
    (
        "COARSENING_AWARE_MODEL_ENVELOPE_VERIFICATION",
        "model_envelope_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    # Notebook 06 comparison
    (
        "NB06_BASELINE_TABLE_READBACK_AUDIT",
        "notebook06_baseline_readback_audit",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    (
        "LOCKED_CALIBRATION_COUNT_BASELINE_RANKING",
        "locked_calibration_baseline_ranking",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "HAWKES_COMMON_GRID_COMPARABLE_SCORE_TABLE",
        "hawkes_comparable_score_table",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING",
        "hawkes_and_baseline_ranking",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "H1_CALIBRATION_COUNT_SCORE_BLOCKS",
        "hawkes_calibration_score_blocks",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP",
        "hawkes_vs_baseline_block_bootstrap",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "HAWKES_VS_FORMAL_COMPARATOR_RESULT",
        "hawkes_vs_formal_comparator",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "HAWKES_VS_PRIMARY_SIMPLE_BASELINE_RESULT",
        "hawkes_vs_primary_simple_baseline",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "HAWKES_BASELINE_COMPARISON_SCOPE_LEDGER",
        "hawkes_baseline_comparison_scope",
        NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    ),
    (
        "HAWKES_BASELINE_COUNT_COMPARISON_VERIFICATION",
        "hawkes_baseline_comparison_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
    # Terminal synthesis
    (
        "LOCKED_CALIBRATION_DIAGNOSTIC_ENVELOPES",
        "locked_calibration_envelopes",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "CALIBRATION_ENVELOPE_FAMILY_SUMMARY",
        "calibration_envelope_family_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "CALIBRATION_ENVELOPE_SCOPE_SUMMARY",
        "calibration_envelope_scope_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "CALIBRATION_RESIDUAL_DISTRIBUTION_SCOPE_SUMMARY",
        "calibration_residual_distribution_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "CALIBRATION_RESIDUAL_SERIAL_SCOPE_SUMMARY",
        "calibration_residual_serial_summary",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_DIAGNOSTIC_GATE_LEDGER",
        "diagnostic_gate_ledger",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "HAWKES_MODEL_REJECTION_LEDGER",
        "model_rejection_ledger",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "FROZEN_HAWKES_SPECIFICATION_DISPOSITION",
        "frozen_hawkes_disposition",
        NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    ),
    (
        "NOTEBOOK08_TERMINAL_DECISION",
        "terminal_decision_table",
        NOTEBOOK08_MANIFEST_OUTPUT_DIR,
    ),
    (
        "NOTEBOOK08_TERMINAL_VERIFICATION",
        "terminal_verification",
        NOTEBOOK08_AUDIT_OUTPUT_DIR,
    ),
)


NOTEBOOK08_REQUIRED_TABLE_VARIABLES: Final[
    tuple[str, ...]
] = (
    "EXACT_COMPENSATOR_PATH",
    "TIME_RESCALING_RESIDUALS",
    "EMPIRICAL_RESIDUAL_DISTRIBUTION_DIAGNOSTICS",
    "EMPIRICAL_RESIDUAL_ACF",
    "EMPIRICAL_RESIDUAL_LJUNG_BOX",
    "SIDE_MARK_BATCH_DIAGNOSTICS",
    "HAWKES_CROSS_SIDE_RESIDUAL_CCF",
    "HAWKES_CROSS_SIDE_PORTMANTEAU",
    "HAWKES_CROSS_SIDE_COINCIDENCE_DIAGNOSTICS",
    "COARSENING_AWARE_MODEL_ENVELOPE_REPLICATE_METRICS",
    "COARSENING_AWARE_DIAGNOSTIC_ENVELOPES",
    "HAWKES_AND_BASELINE_LOCKED_CALIBRATION_RANKING",
    "HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP",
    "HAWKES_DIAGNOSTIC_GATE_LEDGER",
    "HAWKES_MODEL_REJECTION_LEDGER",
    "FROZEN_HAWKES_SPECIFICATION_DISPOSITION",
    "NOTEBOOK08_TERMINAL_DECISION",
    "NOTEBOOK08_TERMINAL_VERIFICATION",
)

missing_required_table_variables = [
    variable_name
    for variable_name in (
        NOTEBOOK08_REQUIRED_TABLE_VARIABLES
    )
    if not isinstance(
        globals().get(
            variable_name
        ),
        pd.DataFrame,
    )
]

require(
    not missing_required_table_variables,
    (
        "Required Notebook 08 tables are unavailable: "
        f"{missing_required_table_variables}"
    ),
)


# ------------------------------------------------------------
# Begin authorized writes
# ------------------------------------------------------------

FILESYSTEM_WRITES_PERFORMED = True

for authorized_directory in (
    NOTEBOOK08_DIAGNOSTIC_OUTPUT_DIR,
    NOTEBOOK08_RECONCILIATION_OUTPUT_DIR,
    NOTEBOOK08_COMPARISON_OUTPUT_DIR,
    NOTEBOOK08_SIMULATION_OUTPUT_DIR,
    NOTEBOOK08_AUDIT_OUTPUT_DIR,
    NOTEBOOK08_MANIFEST_OUTPUT_DIR,
    NOTEBOOK08_HANDOFF_OUTPUT_DIR,
    NOTEBOOK08_LOG_OUTPUT_DIR,
):
    authorized_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Persist all available registered result tables
# ------------------------------------------------------------

persisted_artifact_records: list[
    dict[str, Any]
] = []

persisted_table_names: list[
    str
] = []

for (
    variable_name,
    artifact_role,
    output_directory,
) in TABLE_PERSISTENCE_SPECS:
    candidate_table = globals().get(
        variable_name
    )

    if candidate_table is None:
        continue

    require(
        isinstance(
            candidate_table,
            pd.DataFrame,
        ),
        (
            f"Registered Notebook 08 artifact "
            f"{variable_name} is not a DataFrame."
        ),
    )

    artifact_record = write_dataframe_artifact(
        variable_name=(
            variable_name
        ),
        dataframe=(
            candidate_table
        ),
        artifact_role=(
            artifact_role
        ),
        output_directory=(
            output_directory
        ),
    )

    persisted_artifact_records.append(
        artifact_record
    )

    persisted_table_names.append(
        variable_name
    )


require(
    set(
        NOTEBOOK08_REQUIRED_TABLE_VARIABLES
    ).issubset(
        set(
            persisted_table_names
        )
    ),
    "At least one required Notebook 08 table was not persisted.",
)

require(
    len(
        persisted_table_names
    )
    == len(
        set(
            persisted_table_names
        )
    ),
    "The Notebook 08 table registry contains duplicate variables.",
)


# ------------------------------------------------------------
# Build authoritative JSON packages
# ------------------------------------------------------------

terminal_decision_record = persistence_json_safe(
    NOTEBOOK08_TERMINAL_DECISION.iloc[
        0
    ].to_dict()
)

frozen_disposition_record = persistence_json_safe(
    FROZEN_HAWKES_SPECIFICATION_DISPOSITION.iloc[
        0
    ].to_dict()
)

formal_comparator_record = persistence_json_safe(
    HAWKES_VS_FORMAL_COMPARATOR_RESULT.iloc[
        0
    ].to_dict()
)

primary_simple_record = persistence_json_safe(
    HAWKES_VS_PRIMARY_SIMPLE_BASELINE_RESULT.iloc[
        0
    ].to_dict()
)


NOTEBOOK08_TERMINAL_PACKAGE = {
    "artifact_type": (
        "NOTEBOOK_08_TERMINAL_DECISION"
    ),
    "schema_version": (
        "NOTEBOOK_08_TERMINAL_DECISION_V1"
    ),
    "producer": NOTEBOOK08_NAME,
    "notebook_stage": (
        NOTEBOOK08_STAGE
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "selected_model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "selected_model_family": (
        EXPECTED_SELECTED_MODEL_FAMILY
    ),
    "terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "terminal_interpretation": (
        NOTEBOOK08_TERMINAL_INTERPRETATION
    ),
    "terminal_decision": (
        terminal_decision_record
    ),
    "blocking_gate_count": (
        BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "passed_blocking_gate_count": (
        PASSED_BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "failed_blocking_gate_count": (
        FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "count_score_superiority_authorized": (
        HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    ),
    "calibration_model_adequacy_passed": (
        CALIBRATION_MODEL_ADEQUACY_PASS
    ),
    "full_hawkes_superiority_authorized": (
        HAWKES_SUPERIORITY_AUTHORIZED
    ),
    "notebook09_authorized": (
        NOTEBOOK09_AUTHORIZED
    ),
    "validation_partition_opening_authorized": (
        VALIDATION_PARTITION_OPENING_AUTHORIZED
    ),
    "engineering_holdout_opened": False,
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition_name: bool(
            PARTITION_CONTENT_LOADED[
                partition_name
            ]
        )
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    },
    "status": "PERSISTED_TERMINAL_DECISION",
}


NOTEBOOK08_FROZEN_MODEL_PACKAGE = {
    "artifact_type": (
        "NOTEBOOK_08_FROZEN_HAWKES_DISPOSITION"
    ),
    "schema_version": (
        "NOTEBOOK_08_FROZEN_HAWKES_DISPOSITION_V1"
    ),
    "producer": NOTEBOOK08_NAME,
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "selected_model_package_sha256": globals().get(
        "SELECTED_HAWKES_MODEL_PACKAGE_SHA256"
    ),
    "model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "model_family": (
        EXPECTED_SELECTED_MODEL_FAMILY
    ),
    "event_representation": (
        PRIMARY_EVENT_REPRESENTATION
    ),
    "timestamp_authority": (
        PRIMARY_EVENT_TIME_COLUMN
    ),
    "timestamp_interface": (
        PRIMARY_TIMESTAMP_INTERFACE
    ),
    "fitted_partition": (
        "DEVELOPMENT"
    ),
    "locked_evaluation_partition": (
        "CALIBRATION"
    ),
    "parameters": {
        "mu_buy_per_second": float(
            FROZEN_H1_PARAMETERS.mu_buy
        ),
        "mu_sell_per_second": float(
            FROZEN_H1_PARAMETERS.mu_sell
        ),
        "kappa_buy_to_buy": float(
            FROZEN_H1_PARAMETERS.kappa_buy
        ),
        "kappa_sell_to_sell": float(
            FROZEN_H1_PARAMETERS.kappa_sell
        ),
        "kappa_buy_to_sell": 0.0,
        "kappa_sell_to_buy": 0.0,
        "beta_buy_per_second": float(
            FROZEN_H1_PARAMETERS.beta_buy
        ),
        "beta_sell_per_second": float(
            FROZEN_H1_PARAMETERS.beta_sell
        ),
        "spectral_radius": (
            FROZEN_H1_SPECTRAL_RADIUS
        ),
        "stationarity_margin": (
            FROZEN_H1_STATIONARITY_MARGIN
        ),
    },
    "disposition": (
        frozen_disposition_record
    ),
    "count_score_superiority_authorized": (
        HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    ),
    "full_model_adequacy_authorized": (
        CALIBRATION_MODEL_ADEQUACY_PASS
    ),
    "full_hawkes_superiority_authorized": (
        HAWKES_SUPERIORITY_AUTHORIZED
    ),
    "cross_excitation_authorized": False,
    "state_dependent_hawkes_authorized": False,
    "allowed_downstream_role": (
        FROZEN_HAWKES_ALLOWED_ROLE
    ),
    "diagnostic_disposition": (
        FROZEN_HAWKES_DIAGNOSTIC_DISPOSITION
    ),
    "parameter_refit_performed": False,
    "calibration_parameter_updates": 0,
    "status": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
}


NOTEBOOK08_DIAGNOSTIC_GATE_PACKAGE = {
    "artifact_type": (
        "NOTEBOOK_08_DIAGNOSTIC_GATE_PACKAGE"
    ),
    "schema_version": (
        "NOTEBOOK_08_DIAGNOSTIC_GATE_PACKAGE_V1"
    ),
    "producer": NOTEBOOK08_NAME,
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "gate_ledger": dataframe_records(
        HAWKES_DIAGNOSTIC_GATE_LEDGER
    ),
    "rejection_ledger": dataframe_records(
        HAWKES_MODEL_REJECTION_LEDGER
    ),
    "calibration_envelope_family_summary": (
        dataframe_records(
            CALIBRATION_ENVELOPE_FAMILY_SUMMARY
        )
    ),
    "calibration_residual_distribution_summary": (
        dataframe_records(
            CALIBRATION_RESIDUAL_DISTRIBUTION_SCOPE_SUMMARY
        )
    ),
    "calibration_residual_serial_summary": (
        dataframe_records(
            CALIBRATION_RESIDUAL_SERIAL_SCOPE_SUMMARY
        )
    ),
    "blocking_gate_count": (
        BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "passed_blocking_gate_count": (
        PASSED_BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "failed_blocking_gate_count": (
        FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "higher_likelihood_sufficient": False,
    "status": "PERSISTED_DIAGNOSTIC_GATES",
}


NOTEBOOK08_SIMULATION_ENVELOPE_PACKAGE = {
    "artifact_type": (
        "NOTEBOOK_08_COARSENING_AWARE_ENVELOPE_PACKAGE"
    ),
    "schema_version": (
        "NOTEBOOK_08_COARSENING_AWARE_ENVELOPE_PACKAGE_V1"
    ),
    "producer": NOTEBOOK08_NAME,
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "simulation_replicates": int(
        N_MODEL_ENVELOPE_REPLICATES
    ),
    "simulation_runtime_seconds": float(
        MODEL_ENVELOPE_RUNTIME_SECONDS
    ),
    "simulation_base_seed": int(
        MODEL_ENVELOPE_BASE_SEED
    ),
    "first_simulation_seed": int(
        MODEL_ENVELOPE_SEEDS[
            0
        ]
    ),
    "last_simulation_seed": int(
        MODEL_ENVELOPE_SEEDS[
            -1
        ]
    ),
    "simulation_contract": dataframe_records(
        COARSENING_AWARE_SIMULATION_CONTRACT
    ),
    "replicate_count_summary": dataframe_records(
        COARSENING_AWARE_REPLICATE_COUNT_SUMMARY
    ),
    "verification": dataframe_records(
        COARSENING_AWARE_MODEL_ENVELOPE_VERIFICATION
    ),
    "parameter_refit_performed": False,
    "calibration_parameter_updates": 0,
    "protected_partitions_loaded": False,
    "status": "PERSISTED_SIMULATION_ENVELOPES",
}


NOTEBOOK08_BASELINE_COMPARISON_PACKAGE = {
    "artifact_type": (
        "NOTEBOOK_08_BASELINE_COMPARISON_PACKAGE"
    ),
    "schema_version": (
        "NOTEBOOK_08_BASELINE_COMPARISON_PACKAGE_V1"
    ),
    "producer": NOTEBOOK08_NAME,
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "candidate_model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "formal_comparator_model_id": (
        FORMAL_COUNT_COMPARATOR_MODEL_ID
    ),
    "primary_simple_baseline_model_id": (
        PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
    ),
    "score_space": (
        HAWKES_BASELINE_SCORE_SPACE
    ),
    "formal_comparator_result": (
        formal_comparator_record
    ),
    "primary_simple_baseline_result": (
        primary_simple_record
    ),
    "all_baseline_comparisons": dataframe_records(
        HAWKES_VS_COUNT_BASELINE_BLOCK_BOOTSTRAP
    ),
    "count_score_superiority_authorized": (
        HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    ),
    "full_hawkes_superiority_authorized": (
        HAWKES_SUPERIORITY_AUTHORIZED
    ),
    "score_only_superiority_claim": True,
    "full_model_adequacy_required_separately": True,
    "status": "PERSISTED_SCORE_COMPARISON",
}


json_package_specs = (
    (
        "NOTEBOOK08_TERMINAL_PACKAGE",
        NOTEBOOK08_TERMINAL_PACKAGE,
        (
            NOTEBOOK08_MANIFEST_OUTPUT_DIR
            / (
                NOTEBOOK08_OUTPUT_PREFIX
                + "__08_hawkes_diagnostics_terminal_decision.json"
            )
        ),
        "terminal_decision",
    ),
    (
        "NOTEBOOK08_FROZEN_MODEL_PACKAGE",
        NOTEBOOK08_FROZEN_MODEL_PACKAGE,
        (
            NOTEBOOK08_MANIFEST_OUTPUT_DIR
            / (
                NOTEBOOK08_OUTPUT_PREFIX
                + "__08_frozen_hawkes_disposition.json"
            )
        ),
        "frozen_hawkes_disposition",
    ),
    (
        "NOTEBOOK08_DIAGNOSTIC_GATE_PACKAGE",
        NOTEBOOK08_DIAGNOSTIC_GATE_PACKAGE,
        (
            NOTEBOOK08_MANIFEST_OUTPUT_DIR
            / (
                NOTEBOOK08_OUTPUT_PREFIX
                + "__08_diagnostic_gate_package.json"
            )
        ),
        "diagnostic_gate_package",
    ),
    (
        "NOTEBOOK08_SIMULATION_ENVELOPE_PACKAGE",
        NOTEBOOK08_SIMULATION_ENVELOPE_PACKAGE,
        (
            NOTEBOOK08_MANIFEST_OUTPUT_DIR
            / (
                NOTEBOOK08_OUTPUT_PREFIX
                + "__08_simulation_envelope_package.json"
            )
        ),
        "simulation_envelope_package",
    ),
    (
        "NOTEBOOK08_BASELINE_COMPARISON_PACKAGE",
        NOTEBOOK08_BASELINE_COMPARISON_PACKAGE,
        (
            NOTEBOOK08_MANIFEST_OUTPUT_DIR
            / (
                NOTEBOOK08_OUTPUT_PREFIX
                + "__08_baseline_comparison_package.json"
            )
        ),
        "baseline_comparison_package",
    ),
)

for (
    variable_name,
    package_payload,
    output_path,
    artifact_role,
) in json_package_specs:
    artifact_record = (
        write_self_hashed_json_artifact(
            payload=(
                package_payload
            ),
            output_path=(
                output_path
            ),
            artifact_role=(
                artifact_role
            ),
            variable_name=(
                variable_name
            ),
        )
    )

    persisted_artifact_records.append(
        artifact_record
    )


# ------------------------------------------------------------
# Immediate physical persistence verification
# ------------------------------------------------------------

immediate_verification_records: list[
    dict[str, Any]
] = []

for artifact_record in (
    persisted_artifact_records
):
    artifact_path = Path(
        str(
            artifact_record[
                "path"
            ]
        )
    )

    observed_sha256 = (
        persistence_file_sha256(
            artifact_path
        )
    )

    observed_size_bytes = int(
        artifact_path.stat().st_size
    )

    hash_matches = (
        observed_sha256
        == str(
            artifact_record[
                "sha256"
            ]
        )
    )

    size_matches = (
        observed_size_bytes
        == int(
            artifact_record[
                "file_size_bytes"
            ]
        )
    )

    immediate_verification_records.append(
        {
            "variable_name": str(
                artifact_record[
                    "variable_name"
                ]
            ),
            "artifact_role": str(
                artifact_record[
                    "artifact_role"
                ]
            ),
            "format": str(
                artifact_record[
                    "format"
                ]
            ),
            "file_exists": (
                artifact_path.is_file()
            ),
            "file_size_bytes": (
                observed_size_bytes
            ),
            "file_size_matches": (
                size_matches
            ),
            "sha256_matches": (
                hash_matches
            ),
            "path_inside_v0_1": (
                persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V01_ROOT,
                )
            ),
            "path_outside_v0_0": (
                not persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V00_ROOT,
                )
            ),
            "passed": bool(
                artifact_path.is_file()
                and size_matches
                and hash_matches
                and persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V01_ROOT,
                )
                and not persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V00_ROOT,
                )
            ),
            "status": (
                "PASS"
                if (
                    artifact_path.is_file()
                    and size_matches
                    and hash_matches
                )
                else "FAIL"
            ),
        }
    )


NOTEBOOK08_IMMEDIATE_PERSISTENCE_VERIFICATION = (
    pd.DataFrame.from_records(
        immediate_verification_records
    )
    .sort_values(
        [
            "artifact_role",
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NOTEBOOK08_IMMEDIATE_PERSISTENCE_VERIFICATION[
        "passed"
    ].all(),
    "At least one core persisted artifact failed physical verification.",
)


immediate_audit_record = write_dataframe_artifact(
    variable_name=(
        "NOTEBOOK08_IMMEDIATE_PERSISTENCE_VERIFICATION"
    ),
    dataframe=(
        NOTEBOOK08_IMMEDIATE_PERSISTENCE_VERIFICATION
    ),
    artifact_role=(
        "immediate_persistence_verification"
    ),
    output_directory=(
        NOTEBOOK08_AUDIT_OUTPUT_DIR
    ),
)

persisted_artifact_records.append(
    immediate_audit_record
)


# ------------------------------------------------------------
# Persistence log
# ------------------------------------------------------------

NOTEBOOK08_PERSISTENCE_LOG = {
    "artifact_type": (
        "NOTEBOOK_08_PERSISTENCE_LOG"
    ),
    "schema_version": (
        "NOTEBOOK_08_PERSISTENCE_LOG_V1"
    ),
    "producer": NOTEBOOK08_NAME,
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "persisted_table_count": int(
        len(
            persisted_table_names
        )
    ),
    "persisted_json_package_count": int(
        len(
            json_package_specs
        )
    ),
    "core_artifact_count_before_manifest": int(
        len(
            persisted_artifact_records
        )
    ),
    "notebook09_authorized": (
        NOTEBOOK09_AUTHORIZED
    ),
    "validation_partition_opening_authorized": (
        VALIDATION_PARTITION_OPENING_AUTHORIZED
    ),
    "calibration_parameter_updates": 0,
    "protected_partitions_loaded": False,
    "filesystem_writes_performed": True,
    "status": "PASS_CORE_PERSISTENCE",
}

persistence_log_record = (
    write_self_hashed_json_artifact(
        payload=(
            NOTEBOOK08_PERSISTENCE_LOG
        ),
        output_path=(
            NOTEBOOK08_LOG_OUTPUT_DIR
            / (
                NOTEBOOK08_OUTPUT_PREFIX
                + "__08_hawkes_diagnostics_persistence_log.json"
            )
        ),
        artifact_role=(
            "persistence_log"
        ),
        variable_name=(
            "NOTEBOOK08_PERSISTENCE_LOG"
        ),
    )
)

persisted_artifact_records.append(
    persistence_log_record
)


# ------------------------------------------------------------
# Core manifest CSV
# ------------------------------------------------------------

NOTEBOOK08_CORE_MANIFEST_CSV_PATH: Final[Path] = (
    NOTEBOOK08_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK08_OUTPUT_PREFIX
        + "__08_hawkes_diagnostics_core_manifest.csv.gz"
    )
)

core_manifest_table = (
    pd.DataFrame.from_records(
        persisted_artifact_records
    )
    .sort_values(
        [
            "artifact_role",
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

core_manifest_csv_sha256 = (
    atomic_install_bytes(
        NOTEBOOK08_CORE_MANIFEST_CSV_PATH,
        dataframe_gzip_bytes(
            core_manifest_table
        ),
    )
)

core_manifest_csv_record = {
    "artifact_type": (
        "NOTEBOOK_08_CORE_MANIFEST_TABLE"
    ),
    "artifact_role": (
        "core_manifest_table"
    ),
    "variable_name": (
        "NOTEBOOK08_CORE_MANIFEST_TABLE"
    ),
    "format": "CSV_GZIP",
    "path": str(
        NOTEBOOK08_CORE_MANIFEST_CSV_PATH
    ),
    "relative_path": str(
        NOTEBOOK08_CORE_MANIFEST_CSV_PATH.relative_to(
            NOTEBOOK08_V01_ROOT
        )
    ),
    "row_count": int(
        len(
            core_manifest_table
        )
    ),
    "column_count": int(
        core_manifest_table.shape[
            1
        ]
    ),
    "columns": [
        str(
            column_name
        )
        for column_name in (
            core_manifest_table.columns
        )
    ],
    "dtypes": {
        str(
            column_name
        ): str(
            core_manifest_table[
                column_name
            ].dtype
        )
        for column_name in (
            core_manifest_table.columns
        )
    },
    "schema_sha256": (
        table_schema_sha256(
            core_manifest_table
        )
    ),
    "file_size_bytes": int(
        NOTEBOOK08_CORE_MANIFEST_CSV_PATH.stat().st_size
    ),
    "sha256": (
        core_manifest_csv_sha256
    ),
    "status": "PASS",
}

persisted_artifact_records.append(
    core_manifest_csv_record
)


# ------------------------------------------------------------
# Self-hashed core output manifest JSON
# ------------------------------------------------------------

NOTEBOOK08_CORE_OUTPUT_MANIFEST_PATH: Final[Path] = (
    NOTEBOOK08_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK08_OUTPUT_PREFIX
        + "__08_hawkes_diagnostics_core_output_manifest.json"
    )
)

NOTEBOOK08_CORE_OUTPUT_MANIFEST = {
    "artifact_type": (
        "NOTEBOOK_08_CORE_OUTPUT_MANIFEST"
    ),
    "schema_version": (
        "NOTEBOOK_08_CORE_OUTPUT_MANIFEST_V1"
    ),
    "producer": NOTEBOOK08_NAME,
    "notebook_stage": (
        NOTEBOOK08_STAGE
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "combined_output_prefix": (
        NOTEBOOK08_OUTPUT_PREFIX
    ),
    "selected_model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "count_score_superiority_authorized": (
        HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    ),
    "calibration_model_adequacy_passed": (
        CALIBRATION_MODEL_ADEQUACY_PASS
    ),
    "full_hawkes_superiority_authorized": (
        HAWKES_SUPERIORITY_AUTHORIZED
    ),
    "notebook09_authorized": (
        NOTEBOOK09_AUTHORIZED
    ),
    "validation_partition_opening_authorized": (
        VALIDATION_PARTITION_OPENING_AUTHORIZED
    ),
    "engineering_holdout_opened": False,
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition_name: bool(
            PARTITION_CONTENT_LOADED[
                partition_name
            ]
        )
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    },
    "core_artifact_count": int(
        len(
            persisted_artifact_records
        )
    ),
    "persisted_table_count": int(
        len(
            persisted_table_names
        )
        + 2
    ),
    "artifacts": persistence_json_safe(
        persisted_artifact_records
    ),
    "persistence_stage": (
        "CORE_ARTIFACTS_WRITTEN_"
        "SEMANTIC_READBACK_PENDING"
    ),
    "status": "PASS_CORE_PERSISTENCE",
}

core_manifest_json_record = (
    write_self_hashed_json_artifact(
        payload=(
            NOTEBOOK08_CORE_OUTPUT_MANIFEST
        ),
        output_path=(
            NOTEBOOK08_CORE_OUTPUT_MANIFEST_PATH
        ),
        artifact_role=(
            "core_output_manifest"
        ),
        variable_name=(
            "NOTEBOOK08_CORE_OUTPUT_MANIFEST"
        ),
    )
)

persisted_artifact_records.append(
    core_manifest_json_record
)


# ------------------------------------------------------------
# Final persisted-file registry
# ------------------------------------------------------------

NOTEBOOK08_PERSISTED_FILE_REGISTRY = (
    pd.DataFrame.from_records(
        persisted_artifact_records
    )
    .sort_values(
        [
            "artifact_role",
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NOTEBOOK08_PERSISTED_FILE_REGISTRY[
        "path"
    ].is_unique,
    "Notebook 08 persisted-file paths are not unique.",
)

require(
    NOTEBOOK08_PERSISTED_FILE_REGISTRY[
        "variable_name"
    ].is_unique,
    "Notebook 08 persisted variable names are not unique.",
)

require(
    NOTEBOOK08_PERSISTED_FILE_REGISTRY[
        "status"
    ].eq(
        "PASS"
    ).all(),
    "A Notebook 08 persisted-file registry row did not pass.",
)

for registry_row in (
    NOTEBOOK08_PERSISTED_FILE_REGISTRY.itertuples(
        index=False
    )
):
    registry_path = Path(
        str(
            registry_row.path
        )
    )

    require(
        registry_path.is_file(),
        (
            "A registered Notebook 08 artifact is missing: "
            f"{registry_path}"
        ),
    )

    require(
        persistence_file_sha256(
            registry_path
        )
        == str(
            registry_row.sha256
        ),
        (
            "A registered Notebook 08 artifact hash changed "
            f"after persistence: {registry_path}"
        ),
    )


# ------------------------------------------------------------
# Core persistence verification matrix
# ------------------------------------------------------------

NOTEBOOK08_CORE_PERSISTENCE_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "terminal_decision_created"
            ),
            "passed": (
                NOTEBOOK08_TERMINAL_DECISION_CREATED
            ),
        },
        {
            "check_name": (
                "failure_result_persisted_without_repair"
            ),
            "passed": (
                NOTEBOOK08_TERMINAL_CLASS
                == "FAIL"
                and not FROZEN_HAWKES_SPECIFICATION_DISPOSITION[
                    "parameter_refit_performed"
                ].any()
            ),
        },
        {
            "check_name": (
                "all_required_tables_persisted"
            ),
            "passed": (
                set(
                    NOTEBOOK08_REQUIRED_TABLE_VARIABLES
                ).issubset(
                    set(
                        persisted_table_names
                    )
                )
            ),
        },
        {
            "check_name": (
                "all_core_files_physically_verified"
            ),
            "passed": bool(
                NOTEBOOK08_IMMEDIATE_PERSISTENCE_VERIFICATION[
                    "passed"
                ].all()
            ),
        },
        {
            "check_name": (
                "all_persisted_paths_unique"
            ),
            "passed": bool(
                NOTEBOOK08_PERSISTED_FILE_REGISTRY[
                    "path"
                ].is_unique
            ),
        },
        {
            "check_name": (
                "all_persisted_variable_names_unique"
            ),
            "passed": bool(
                NOTEBOOK08_PERSISTED_FILE_REGISTRY[
                    "variable_name"
                ].is_unique
            ),
        },
        {
            "check_name": (
                "core_manifest_written"
            ),
            "passed": (
                NOTEBOOK08_CORE_OUTPUT_MANIFEST_PATH.is_file()
            ),
        },
        {
            "check_name": (
                "notebook09_remains_unauthorized"
            ),
            "passed": (
                NOTEBOOK09_AUTHORIZED
                is False
            ),
        },
        {
            "check_name": (
                "validation_opening_remains_unauthorized"
            ),
            "passed": (
                VALIDATION_PARTITION_OPENING_AUTHORIZED
                is False
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "protected_partitions_unopened"
            ),
            "passed": (
                PROTECTED_PARTITION_CONTENT_LOADED
                is False
                and not any(
                    PARTITION_CONTENT_LOADED[
                        partition_name
                    ]
                    for partition_name in (
                        PROTECTED_PARTITIONS
                    )
                )
            ),
        },
        {
            "check_name": (
                "authorized_filesystem_writes_recorded"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is True
            ),
        },
    ]
)

require(
    NOTEBOOK08_CORE_PERSISTENCE_VERIFICATION[
        "passed"
    ].all(),
    "At least one Notebook 08 core-persistence verification failed.",
)


# ------------------------------------------------------------
# Publish persistence state
# ------------------------------------------------------------

NOTEBOOK08_CORE_PERSISTENCE_FILE_COUNT = int(
    len(
        NOTEBOOK08_PERSISTED_FILE_REGISTRY
    )
)

NOTEBOOK08_CORE_PERSISTENCE_TABLE_COUNT = int(
    len(
        persisted_table_names
    )
)

NOTEBOOK08_PERSISTENCE_COMPLETED = True
NOTEBOOK08_READBACK_AUTHORIZED = True
NOTEBOOK08_FINAL_ACCEPTANCE_CREATED = False
NOTEBOOK08_HANDOFF_CREATED = False

require(
    FILESYSTEM_WRITES_PERFORMED
    is True,
    "Authorized Notebook 08 writes were not recorded.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False,
    "Protected partition content was loaded during persistence.",
)


display(
    NOTEBOOK08_PERSISTED_FILE_REGISTRY[
        [
            "artifact_role",
            "variable_name",
            "format",
            "row_count",
            "column_count",
            "file_size_bytes",
            "sha256",
            "status",
        ]
    ]
)

display(
    NOTEBOOK08_IMMEDIATE_PERSISTENCE_VERIFICATION
)

display(
    NOTEBOOK08_CORE_PERSISTENCE_VERIFICATION
)

print(
    f"Notebook 08 core persistence completed with "
    f"{NOTEBOOK08_CORE_PERSISTENCE_FILE_COUNT:,} registered files "
    f"and {NOTEBOOK08_CORE_PERSISTENCE_TABLE_COUNT:,} diagnostic "
    "tables. The failed H1 adequacy decision, supported count-score "
    "superiority result, blocking rejection ledger, frozen model "
    "disposition, simulation envelopes, and Notebook 06 comparisons "
    "were persisted without refitting or repair. Notebook 09 and "
    "VALIDATION remain unauthorized. Semantic readback and the final "
    "blocked handoff are now authorized."
)

,artifact_role,variable_name,format,row_count,column_count,file_size_bytes,sha256,status
0,baseline_comparison_package,NOTEBOOK08_BASELINE_COMPARISON_PACKAGE,JSON,NaN,NaN,11515,75ef8d54ff149678e956c7b75265779240498c344995ba...,PASS
1,batch_multiplicity_distribution,BATCH_MULTIPLICITY_DISTRIBUTION,CSV_GZIP,10,12,530,00f3237303e7c68819ef77beea2f6a5fa398b24d305eb8...,PASS
2,boundary_compensator_reconciliation,BOUNDARY_COMPENSATOR_RECONCILIATION,CSV_GZIP,2,8,244,ba2fbf5f0556853ad52afe32d7468cd31ac77b5722f693...,PASS
3,calibration_envelope_family_summary,CALIBRATION_ENVELOPE_FAMILY_SUMMARY,CSV_GZIP,4,9,315,574ad108b411d26d654897140cefe35eafa40a315accd5...,PASS
4,calibration_envelope_scope_summary,CALIBRATION_ENVELOPE_SCOPE_SUMMARY,CSV_GZIP,8,10,366,bd6f019cbbdfd4e15ca1935aa802f0b5312f52dd7c8a29...,PASS
...,...,...,...,...,...,...,...,...
68,terminal_verification,NOTEBOOK08_TERMINAL_VERIFICATION,CSV_GZIP,11,2,287,93694705e11575bda7c31778279c2b91a4927387baaeac...,PASS
69,time_rescaling_residual_summary,TIME_RESCALING_RESIDUAL_SUMMARY,CSV_GZIP,6,16,762,4b55b4875fad332d8ae0185c87bff4c04b4ad70b709414...,PASS
70,time_rescaling_residuals,TIME_RESCALING_RESIDUALS,CSV_GZIP,"18,549",31,1309891,c95089408af3f082c0cd16611098bfdaa80c5124fdb794...,PASS
71,time_rescaling_right_censoring,TIME_RESCALING_RIGHT_CENSORING_LEDGER,CSV_GZIP,6,10,559,cebc39cca29f11581d8da55eb8a03a473f9965343b7801...,PASS


,variable_name,artifact_role,format,file_exists,file_size_bytes,file_size_matches,sha256_matches,path_inside_v0_1,path_outside_v0_0,passed,status
0,NOTEBOOK08_BASELINE_COMPARISON_PACKAGE,baseline_comparison_package,JSON,True,11515,True,True,True,True,True,PASS
1,BATCH_MULTIPLICITY_DISTRIBUTION,batch_multiplicity_distribution,CSV_GZIP,True,530,True,True,True,True,True,PASS
2,BOUNDARY_COMPENSATOR_RECONCILIATION,boundary_compensator_reconciliation,CSV_GZIP,True,244,True,True,True,True,True,PASS
3,CALIBRATION_ENVELOPE_FAMILY_SUMMARY,calibration_envelope_family_summary,CSV_GZIP,True,315,True,True,True,True,True,PASS
4,CALIBRATION_ENVELOPE_SCOPE_SUMMARY,calibration_envelope_scope_summary,CSV_GZIP,True,366,True,True,True,True,True,PASS
...,...,...,...,...,...,...,...,...,...,...,...
64,NOTEBOOK08_TERMINAL_VERIFICATION,terminal_verification,CSV_GZIP,True,287,True,True,True,True,True,PASS
65,TIME_RESCALING_RESIDUAL_SUMMARY,time_rescaling_residual_summary,CSV_GZIP,True,762,True,True,True,True,True,PASS
66,TIME_RESCALING_RESIDUALS,time_rescaling_residuals,CSV_GZIP,True,1309891,True,True,True,True,True,PASS
67,TIME_RESCALING_RIGHT_CENSORING_LEDGER,time_rescaling_right_censoring,CSV_GZIP,True,559,True,True,True,True,True,PASS


,check_name,passed
0,terminal_decision_created,True
1,failure_result_persisted_without_repair,True
2,all_required_tables_persisted,True
3,all_core_files_physically_verified,True
4,all_persisted_paths_unique,True
5,all_persisted_variable_names_unique,True
6,core_manifest_written,True
7,notebook09_remains_unauthorized,True
8,validation_opening_remains_unauthorized,True
9,calibration_parameter_updates_zero,True


Notebook 08 core persistence completed with 73 registered files and 64 diagnostic tables. The failed H1 adequacy decision, supported count-score superiority result, blocking rejection ledger, frozen model disposition, simulation envelopes, and Notebook 06 comparisons were persisted without refitting or repair. Notebook 09 and VALIDATION remain unauthorized. Semantic readback and the final blocked handoff are now authorized.


In [29]:
# ============================================================
# Complete Notebook 08 semantic readback and finalization
#
# The semantic-readback CSV path is taken directly from the
# persistence record returned by write_dataframe_artifact.
# No independently constructed CSV-path assertion is used.
#
# H1 remains rejected for downstream signal validation.
# Notebook 09 is explicitly blocked. Notebook 10 is authorized
# only to audit and close the failed V0.1 diagnostic branch.
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "NOTEBOOK08_PERSISTENCE_COMPLETED",
            False,
        )
    ),
    "Notebook 08 core persistence is incomplete.",
)

require(
    bool(
        globals().get(
            "NOTEBOOK08_READBACK_AUTHORIZED",
            False,
        )
    ),
    "Notebook 08 semantic readback is not authorized.",
)

require(
    not bool(
        globals().get(
            "NOTEBOOK08_READBACK_COMPLETED",
            False,
        )
    ),
    "Notebook 08 semantic readback has already completed.",
)

require(
    isinstance(
        globals().get(
            "NOTEBOOK08_SEMANTIC_READBACK_AUDIT"
        ),
        pd.DataFrame,
    ),
    (
        "The previously completed Notebook 08 semantic-readback "
        "audit table is unavailable."
    ),
)

require(
    not NOTEBOOK08_SEMANTIC_READBACK_AUDIT.empty,
    "The semantic-readback audit is empty.",
)

require(
    "passed"
    in NOTEBOOK08_SEMANTIC_READBACK_AUDIT.columns,
    "The semantic-readback audit lacks its passed column.",
)

require(
    NOTEBOOK08_SEMANTIC_READBACK_AUDIT[
        "passed"
    ].astype(
        bool
    ).all(),
    "At least one core artifact failed semantic readback.",
)

require(
    isinstance(
        NOTEBOOK08_PERSISTED_FILE_REGISTRY,
        pd.DataFrame,
    )
    and not NOTEBOOK08_PERSISTED_FILE_REGISTRY.empty,
    "The Notebook 08 persisted-file registry is unavailable.",
)

require(
    NOTEBOOK08_TERMINAL_CLASS
    == "FAIL",
    "This finalization cell expects the frozen Notebook 08 failure result.",
)

require(
    NOTEBOOK08_TERMINAL_STATUS
    == (
        "FAIL_HAWKES_DIAGNOSTIC_ADEQUACY_"
        "DESPITE_SCORE_SUPERIORITY"
    ),
    "Unexpected Notebook 08 terminal status.",
)

require(
    HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    is True,
    "The narrow count-score result is not the expected supported result.",
)

require(
    CALIBRATION_MODEL_ADEQUACY_PASS
    is False,
    "Full model adequacy unexpectedly passed.",
)

require(
    NOTEBOOK09_AUTHORIZED
    is False,
    "Notebook 09 must remain unauthorized.",
)

require(
    VALIDATION_PARTITION_OPENING_AUTHORIZED
    is False,
    "VALIDATION opening must remain unauthorized.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED
    == 0,
    "CALIBRATION parameter updates must remain zero.",
)

require(
    PROTECTED_PARTITION_CONTENT_LOADED
    is False,
    "Protected partition content was loaded.",
)

require(
    not any(
        PARTITION_CONTENT_LOADED[
            partition_name
        ]
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    ),
    "A protected partition was materialized.",
)

require(
    FILESYSTEM_WRITES_PERFORMED
    is True,
    "The authorized persistence stage was not recorded.",
)


# ------------------------------------------------------------
# Final output paths
# ------------------------------------------------------------

NOTEBOOK08_SEMANTIC_READBACK_AUDIT_JSON_PATH = (
    NOTEBOOK08_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK08_OUTPUT_PREFIX
        + "__08_hawkes_diagnostics_semantic_readback_audit.json"
    )
)

NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF_PATH = (
    NOTEBOOK08_HANDOFF_OUTPUT_DIR
    / (
        NOTEBOOK08_OUTPUT_PREFIX
        + "__08_to_09_intensity_signal_validation_blocked.json"
    )
)

NOTEBOOK08_FINAL_DISPOSITION_PATH = (
    NOTEBOOK08_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK08_OUTPUT_PREFIX
        + "__08_hawkes_diagnostics_final_disposition.json"
    )
)

NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF_PATH = (
    NOTEBOOK08_HANDOFF_OUTPUT_DIR
    / (
        NOTEBOOK08_OUTPUT_PREFIX
        + "__08_to_10_final_audit_handoff.json"
    )
)

NOTEBOOK08_FINAL_OUTPUT_MANIFEST_JSON_PATH = (
    NOTEBOOK08_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK08_OUTPUT_PREFIX
        + "__08_hawkes_diagnostics_output_manifest.json"
    )
)

for final_output_path in (
    NOTEBOOK08_SEMANTIC_READBACK_AUDIT_JSON_PATH,
    NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF_PATH,
    NOTEBOOK08_FINAL_DISPOSITION_PATH,
    NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF_PATH,
    NOTEBOOK08_FINAL_OUTPUT_MANIFEST_JSON_PATH,
):
    require_authorized_output_path(
        final_output_path
    )


# ------------------------------------------------------------
# Self-hashed JSON verification helper
# ------------------------------------------------------------

def verify_persisted_self_hashed_json(
    artifact_record: Mapping[str, Any],
) -> dict[str, Any]:
    """Verify one JSON artifact's physical and semantic hashes."""
    artifact_path = Path(
        str(
            artifact_record[
                "path"
            ]
        )
    )

    require(
        artifact_path.is_file(),
        f"Persisted JSON artifact is missing: {artifact_path}",
    )

    require(
        persistence_file_sha256(
            artifact_path
        )
        == str(
            artifact_record[
                "sha256"
            ]
        ),
        f"Persisted JSON file hash differs: {artifact_path}",
    )

    with artifact_path.open(
        "r",
        encoding="utf-8",
    ) as artifact_file:
        loaded_document = json.load(
            artifact_file
        )

    require(
        isinstance(
            loaded_document,
            dict,
        ),
        f"Persisted JSON is not an object: {artifact_path}",
    )

    registered_payload_sha256 = str(
        loaded_document.get(
            "payload_sha256",
            "",
        )
    )

    require(
        bool(
            registered_payload_sha256
        ),
        f"Persisted JSON lacks payload_sha256: {artifact_path}",
    )

    semantic_payload = {
        str(
            key
        ): value
        for key, value in (
            loaded_document.items()
        )
        if key
        != "payload_sha256"
    }

    require(
        canonical_json_sha256(
            semantic_payload
        )
        == registered_payload_sha256,
        f"Persisted JSON semantic hash differs: {artifact_path}",
    )

    require(
        registered_payload_sha256
        == str(
            artifact_record[
                "payload_sha256"
            ]
        ),
        (
            "Persisted JSON semantic hash differs from its "
            f"artifact record: {artifact_path}"
        ),
    )

    return loaded_document


# ------------------------------------------------------------
# Reverify all core persisted files
# ------------------------------------------------------------

core_reverification_records: list[
    dict[str, Any]
] = []

for registry_row in (
    NOTEBOOK08_PERSISTED_FILE_REGISTRY.itertuples(
        index=False
    )
):
    artifact_path = Path(
        str(
            registry_row.path
        )
    )

    file_exists = artifact_path.is_file()

    observed_sha256 = (
        persistence_file_sha256(
            artifact_path
        )
        if file_exists
        else None
    )

    sha256_matches = bool(
        file_exists
        and observed_sha256
        == str(
            registry_row.sha256
        )
    )

    path_inside_v0_1 = (
        persistence_path_is_inside(
            artifact_path,
            NOTEBOOK08_V01_ROOT,
        )
    )

    path_outside_v0_0 = (
        not persistence_path_is_inside(
            artifact_path,
            NOTEBOOK08_V00_ROOT,
        )
    )

    core_reverification_records.append(
        {
            "variable_name": str(
                registry_row.variable_name
            ),
            "artifact_role": str(
                registry_row.artifact_role
            ),
            "format": str(
                registry_row.format
            ),
            "path": str(
                artifact_path
            ),
            "file_exists": (
                file_exists
            ),
            "expected_sha256": str(
                registry_row.sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "sha256_matches": (
                sha256_matches
            ),
            "path_inside_v0_1": (
                path_inside_v0_1
            ),
            "path_outside_v0_0": (
                path_outside_v0_0
            ),
            "passed": bool(
                file_exists
                and sha256_matches
                and path_inside_v0_1
                and path_outside_v0_0
            ),
            "status": (
                "PASS"
                if (
                    file_exists
                    and sha256_matches
                    and path_inside_v0_1
                    and path_outside_v0_0
                )
                else "FAIL"
            ),
        }
    )


NOTEBOOK08_CORE_FILE_REVERIFICATION = (
    pd.DataFrame.from_records(
        core_reverification_records
    )
    .sort_values(
        [
            "artifact_role",
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NOTEBOOK08_CORE_FILE_REVERIFICATION[
        "passed"
    ].all(),
    "At least one core artifact failed final physical reverification.",
)


# ------------------------------------------------------------
# Freeze semantic-readback summary
# ------------------------------------------------------------

semantic_readback_path_column = next(
    (
        candidate_column
        for candidate_column in (
            "relative_path",
            "path",
            "absolute_path",
        )
        if candidate_column
        in NOTEBOOK08_SEMANTIC_READBACK_AUDIT.columns
    ),
    None,
)

if semantic_readback_path_column is not None:
    require(
        NOTEBOOK08_SEMANTIC_READBACK_AUDIT[
            semantic_readback_path_column
        ].astype(
            str
        ).is_unique,
        "The semantic-readback audit contains duplicate artifact paths.",
    )


NOTEBOOK08_SEMANTIC_READBACK_SUMMARY = pd.DataFrame(
    [
        {
            "field": (
                "core_registered_file_count"
            ),
            "value": int(
                len(
                    NOTEBOOK08_PERSISTED_FILE_REGISTRY
                )
            ),
        },
        {
            "field": (
                "semantic_readback_record_count"
            ),
            "value": int(
                len(
                    NOTEBOOK08_SEMANTIC_READBACK_AUDIT
                )
            ),
        },
        {
            "field": (
                "semantic_readback_pass_count"
            ),
            "value": int(
                NOTEBOOK08_SEMANTIC_READBACK_AUDIT[
                    "passed"
                ].astype(
                    bool
                ).sum()
            ),
        },
        {
            "field": (
                "semantic_readback_failure_count"
            ),
            "value": int(
                (
                    ~NOTEBOOK08_SEMANTIC_READBACK_AUDIT[
                        "passed"
                    ].astype(
                        bool
                    )
                ).sum()
            ),
        },
        {
            "field": (
                "core_file_reverification_passed"
            ),
            "value": bool(
                NOTEBOOK08_CORE_FILE_REVERIFICATION[
                    "passed"
                ].all()
            ),
        },
        {
            "field": (
                "terminal_status"
            ),
            "value": (
                NOTEBOOK08_TERMINAL_STATUS
            ),
        },
        {
            "field": (
                "count_score_superiority_authorized"
            ),
            "value": (
                HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
            ),
        },
        {
            "field": (
                "full_model_adequacy_authorized"
            ),
            "value": (
                CALIBRATION_MODEL_ADEQUACY_PASS
            ),
        },
        {
            "field": (
                "notebook09_authorized"
            ),
            "value": (
                NOTEBOOK09_AUTHORIZED
            ),
        },
        {
            "field": (
                "validation_partition_opening_authorized"
            ),
            "value": (
                VALIDATION_PARTITION_OPENING_AUTHORIZED
            ),
        },
        {
            "field": (
                "calibration_parameter_updates"
            ),
            "value": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
            ),
        },
        {
            "field": (
                "protected_partitions_loaded"
            ),
            "value": False,
        },
        {
            "field": (
                "status"
            ),
            "value": (
                "PASS_SEMANTIC_READBACK"
            ),
        },
    ]
)


# ------------------------------------------------------------
# Persist semantic-readback CSV using its returned path
# ------------------------------------------------------------

semantic_readback_csv_record = (
    write_dataframe_artifact(
        variable_name=(
            "NOTEBOOK08_SEMANTIC_READBACK_AUDIT"
        ),
        dataframe=(
            NOTEBOOK08_SEMANTIC_READBACK_AUDIT
        ),
        artifact_role=(
            "semantic_readback_audit_table"
        ),
        output_directory=(
            NOTEBOOK08_MANIFEST_OUTPUT_DIR
        ),
    )
)

NOTEBOOK08_SEMANTIC_READBACK_AUDIT_CSV_PATH = Path(
    str(
        semantic_readback_csv_record[
            "path"
        ]
    )
)

require(
    NOTEBOOK08_SEMANTIC_READBACK_AUDIT_CSV_PATH.is_file(),
    "The persisted semantic-readback CSV is missing.",
)

require(
    persistence_file_sha256(
        NOTEBOOK08_SEMANTIC_READBACK_AUDIT_CSV_PATH
    )
    == str(
        semantic_readback_csv_record[
            "sha256"
        ]
    ),
    "The semantic-readback CSV failed file-hash verification.",
)


# ------------------------------------------------------------
# Persist semantic-readback JSON package
# ------------------------------------------------------------

NOTEBOOK08_SEMANTIC_READBACK_PACKAGE = {
    "artifact_type": (
        "NOTEBOOK_08_HAWKES_DIAGNOSTICS_SEMANTIC_READBACK_AUDIT"
    ),
    "schema_version": (
        "NOTEBOOK_08_HAWKES_DIAGNOSTICS_"
        "SEMANTIC_READBACK_AUDIT_V1"
    ),
    "producer": (
        NOTEBOOK08_NAME
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "selected_model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "core_registered_file_count": int(
        len(
            NOTEBOOK08_PERSISTED_FILE_REGISTRY
        )
    ),
    "semantic_readback_record_count": int(
        len(
            NOTEBOOK08_SEMANTIC_READBACK_AUDIT
        )
    ),
    "semantic_readback_failure_count": 0,
    "semantic_readback_csv_path": str(
        NOTEBOOK08_SEMANTIC_READBACK_AUDIT_CSV_PATH.relative_to(
            NOTEBOOK08_V01_ROOT
        )
    ),
    "semantic_readback_csv_sha256": str(
        semantic_readback_csv_record[
            "sha256"
        ]
    ),
    "all_core_files_physically_reverified": True,
    "all_semantic_checks_passed": True,
    "records": dataframe_records(
        NOTEBOOK08_SEMANTIC_READBACK_AUDIT
    ),
    "summary": dataframe_records(
        NOTEBOOK08_SEMANTIC_READBACK_SUMMARY
    ),
    "count_score_superiority_authorized": (
        HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
    ),
    "full_model_adequacy_authorized": (
        CALIBRATION_MODEL_ADEQUACY_PASS
    ),
    "full_hawkes_superiority_authorized": False,
    "notebook09_authorized": False,
    "validation_partition_opening_authorized": False,
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition_name: False
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    },
    "status": (
        "PASS_SEMANTIC_READBACK_OF_FAILED_DIAGNOSTIC_RESULT"
    ),
}

semantic_readback_json_record = (
    write_self_hashed_json_artifact(
        payload=(
            NOTEBOOK08_SEMANTIC_READBACK_PACKAGE
        ),
        output_path=(
            NOTEBOOK08_SEMANTIC_READBACK_AUDIT_JSON_PATH
        ),
        artifact_role=(
            "semantic_readback_audit_package"
        ),
        variable_name=(
            "NOTEBOOK08_SEMANTIC_READBACK_PACKAGE"
        ),
    )
)

verify_persisted_self_hashed_json(
    semantic_readback_json_record
)


# ------------------------------------------------------------
# Explicit blocked Notebook 09 control artifact
# ------------------------------------------------------------

NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF = {
    "artifact_type": (
        "NOTEBOOK_08_TO_NOTEBOOK_09_BLOCKED_HANDOFF"
    ),
    "schema_version": (
        "NOTEBOOK_08_TO_NOTEBOOK_09_BLOCKED_HANDOFF_V1"
    ),
    "producer": (
        NOTEBOOK08_NAME
    ),
    "consumer": (
        "09_INTENSITY_SIGNAL_VALIDATION.ipynb"
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "selected_model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "notebook08_terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "notebook08_terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "authorization_granted": False,
    "authorization_scope": (
        "NONE"
    ),
    "blocking_reason": (
        "FROZEN_H1_FAILED_LOCKED_CALIBRATION_MODEL_ADEQUACY"
    ),
    "failed_blocking_gates": dataframe_records(
        HAWKES_MODEL_REJECTION_LEDGER
    ),
    "count_score_superiority_authorized": True,
    "full_model_adequacy_authorized": False,
    "full_hawkes_superiority_authorized": False,
    "intensity_signal_validation_authorized": False,
    "validation_partition_opening_authorized": False,
    "engineering_holdout_opening_authorized": False,
    "cross_excitation_authorized": False,
    "state_dependent_hawkes_authorized": False,
    "parameter_refit_performed": False,
    "calibration_parameter_updates": 0,
    "permitted_use_of_frozen_h1": (
        "DIAGNOSTIC_REFERENCE_AND_FAILURE_ANALYSIS_ONLY"
    ),
    "prohibited_next_actions": [
        "OPEN_VALIDATION_EVENT_CONTENT",
        "OPEN_ENGINEERING_HOLDOUT_EVENT_CONTENT",
        "RUN_INCREMENTAL_INTENSITY_SIGNAL_VALIDATION",
        "USE_H1_AS_AN_AUTHORIZED_SIGNAL",
        "CLAIM_FULL_HAWKES_SUPERIORITY",
        "AUTHORIZE_QUOTING_OR_STRATEGY_LOGIC",
    ],
    "status": (
        "BLOCKED_BY_NOTEBOOK08_DIAGNOSTIC_FAILURE"
    ),
}

notebook09_blocked_handoff_record = (
    write_self_hashed_json_artifact(
        payload=(
            NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF
        ),
        output_path=(
            NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF_PATH
        ),
        artifact_role=(
            "notebook08_to_notebook09_blocked_handoff"
        ),
        variable_name=(
            "NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF"
        ),
    )
)

verify_persisted_self_hashed_json(
    notebook09_blocked_handoff_record
)


# ------------------------------------------------------------
# Final Notebook 08 disposition
# ------------------------------------------------------------

NOTEBOOK08_FINAL_DISPOSITION = {
    "artifact_type": (
        "NOTEBOOK_08_HAWKES_DIAGNOSTICS_FINAL_DISPOSITION"
    ),
    "schema_version": (
        "NOTEBOOK_08_HAWKES_DIAGNOSTICS_FINAL_DISPOSITION_V1"
    ),
    "producer": (
        NOTEBOOK08_NAME
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "selected_model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "selected_model_family": (
        EXPECTED_SELECTED_MODEL_FAMILY
    ),
    "terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "terminal_interpretation": (
        NOTEBOOK08_TERMINAL_INTERPRETATION
    ),
    "terminal_decision": persistence_json_safe(
        NOTEBOOK08_TERMINAL_DECISION.iloc[
            0
        ].to_dict()
    ),
    "blocking_gate_count": (
        BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "passed_blocking_gate_count": (
        PASSED_BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "failed_blocking_gate_count": (
        FAILED_BLOCKING_DIAGNOSTIC_GATE_COUNT
    ),
    "failed_blocking_gates": dataframe_records(
        HAWKES_MODEL_REJECTION_LEDGER
    ),
    "narrow_findings": {
        "count_score_superiority_over_authorized_baselines": True,
        "stationarity_margin_passed": True,
        "side_mark_directional_calibration_passed": True,
    },
    "blocking_findings": {
        "time_rescaling_distribution_adequacy_passed": False,
        "time_rescaling_serial_independence_passed": False,
        "path_count_calibration_passed": False,
        "exact_time_multiplicity_adequacy_passed": False,
    },
    "full_model_adequacy_authorized": False,
    "full_hawkes_superiority_authorized": False,
    "cross_excitation_authorized": False,
    "state_dependent_hawkes_authorized": False,
    "notebook09_authorized": False,
    "validation_partition_opening_authorized": False,
    "engineering_holdout_opening_authorized": False,
    "parameter_refit_performed": False,
    "repair_performed": False,
    "calibration_parameter_updates": 0,
    "frozen_model_disposition": (
        FROZEN_HAWKES_DIAGNOSTIC_DISPOSITION
    ),
    "allowed_downstream_role": (
        FROZEN_HAWKES_ALLOWED_ROLE
    ),
    "semantic_readback_payload_sha256": str(
        semantic_readback_json_record[
            "payload_sha256"
        ]
    ),
    "semantic_readback_file_sha256": str(
        semantic_readback_json_record[
            "sha256"
        ]
    ),
    "notebook09_blocked_handoff_payload_sha256": str(
        notebook09_blocked_handoff_record[
            "payload_sha256"
        ]
    ),
    "notebook09_blocked_handoff_file_sha256": str(
        notebook09_blocked_handoff_record[
            "sha256"
        ]
    ),
    "next_authorized_operation": (
        "BEGIN_NOTEBOOK10_FINAL_AUDIT_WITH_NOTEBOOK09_SKIPPED"
    ),
    "status": (
        "PASS_FINALIZATION_OF_FAILED_DIAGNOSTIC_RESULT"
    ),
}

final_disposition_record = (
    write_self_hashed_json_artifact(
        payload=(
            NOTEBOOK08_FINAL_DISPOSITION
        ),
        output_path=(
            NOTEBOOK08_FINAL_DISPOSITION_PATH
        ),
        artifact_role=(
            "final_disposition"
        ),
        variable_name=(
            "NOTEBOOK08_FINAL_DISPOSITION"
        ),
    )
)

verify_persisted_self_hashed_json(
    final_disposition_record
)


# ------------------------------------------------------------
# Audit-only handoff to Notebook 10
# ------------------------------------------------------------

NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF = {
    "artifact_type": (
        "NOTEBOOK_08_TO_NOTEBOOK_10_FINAL_AUDIT_HANDOFF"
    ),
    "schema_version": (
        "NOTEBOOK_08_TO_NOTEBOOK_10_FINAL_AUDIT_HANDOFF_V1"
    ),
    "producer": (
        NOTEBOOK08_NAME
    ),
    "consumer": (
        "10_V01_FINAL_AUDIT_AND_HANDOFF.ipynb"
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "notebook08_terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "notebook08_terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "authorization_granted": True,
    "authorization_scope": (
        "AUDIT_AND_CLOSE_FAILED_V0_1_DIAGNOSTIC_BRANCH_ONLY"
    ),
    "notebook09_executed": False,
    "notebook09_skip_reason": (
        "NOTEBOOK08_DID_NOT_PRODUCE_AN_ACCEPTED_DIAGNOSTIC_HANDOFF"
    ),
    "notebook09_authorized": False,
    "validation_partition_opening_authorized": False,
    "engineering_holdout_opening_authorized": False,
    "model_signal_handoff_authorized": False,
    "v0_2_model_handoff_authorized": False,
    "required_audit_findings": [
        "PRESERVE_SUPPORTED_COUNT_SCORE_SUPERIORITY_AS_NARROW_EVIDENCE",
        "PRESERVE_FAILED_MODEL_ADEQUACY_AS_BLOCKING_EVIDENCE",
        "RECORD_NOTEBOOK09_AS_SKIPPED_NOT_PASSED",
        "RECORD_VALIDATION_AS_UNOPENED",
        "RECORD_ENGINEERING_HOLDOUT_AS_UNOPENED",
        "PRESERVE_FROZEN_H1_WITHOUT_REFIT_OR_REPAIR",
        "CLOSE_V0_1_WITHOUT_STRATEGY_OR_SIGNAL_AUTHORIZATION",
    ],
    "authoritative_artifacts": {
        "core_output_manifest": str(
            NOTEBOOK08_CORE_OUTPUT_MANIFEST_PATH.relative_to(
                NOTEBOOK08_V01_ROOT
            )
        ),
        "semantic_readback_audit": str(
            NOTEBOOK08_SEMANTIC_READBACK_AUDIT_JSON_PATH.relative_to(
                NOTEBOOK08_V01_ROOT
            )
        ),
        "final_disposition": str(
            NOTEBOOK08_FINAL_DISPOSITION_PATH.relative_to(
                NOTEBOOK08_V01_ROOT
            )
        ),
        "notebook09_blocked_handoff": str(
            NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF_PATH.relative_to(
                NOTEBOOK08_V01_ROOT
            )
        ),
    },
    "final_disposition_payload_sha256": str(
        final_disposition_record[
            "payload_sha256"
        ]
    ),
    "semantic_readback_payload_sha256": str(
        semantic_readback_json_record[
            "payload_sha256"
        ]
    ),
    "notebook09_blocked_handoff_payload_sha256": str(
        notebook09_blocked_handoff_record[
            "payload_sha256"
        ]
    ),
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition_name: False
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    },
    "status": (
        "PASS_HANDOFF_TO_FINAL_AUDIT_AFTER_DIAGNOSTIC_FAILURE"
    ),
}

notebook10_audit_handoff_record = (
    write_self_hashed_json_artifact(
        payload=(
            NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF
        ),
        output_path=(
            NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF_PATH
        ),
        artifact_role=(
            "notebook08_to_notebook10_audit_handoff"
        ),
        variable_name=(
            "NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF"
        ),
    )
)

verify_persisted_self_hashed_json(
    notebook10_audit_handoff_record
)


# ------------------------------------------------------------
# Finalization verification artifact
# ------------------------------------------------------------

NOTEBOOK08_FINALIZATION_VERIFICATION = pd.DataFrame(
    [
        {
            "check_name": (
                "core_persistence_completed"
            ),
            "passed": (
                NOTEBOOK08_PERSISTENCE_COMPLETED
            ),
        },
        {
            "check_name": (
                "all_core_files_physically_reverified"
            ),
            "passed": bool(
                NOTEBOOK08_CORE_FILE_REVERIFICATION[
                    "passed"
                ].all()
            ),
        },
        {
            "check_name": (
                "all_semantic_readback_records_passed"
            ),
            "passed": bool(
                NOTEBOOK08_SEMANTIC_READBACK_AUDIT[
                    "passed"
                ].astype(
                    bool
                ).all()
            ),
        },
        {
            "check_name": (
                "semantic_readback_csv_path_taken_from_artifact_record"
            ),
            "passed": (
                NOTEBOOK08_SEMANTIC_READBACK_AUDIT_CSV_PATH
                == Path(
                    str(
                        semantic_readback_csv_record[
                            "path"
                        ]
                    )
                )
            ),
        },
        {
            "check_name": (
                "semantic_readback_json_verified"
            ),
            "passed": (
                NOTEBOOK08_SEMANTIC_READBACK_AUDIT_JSON_PATH.is_file()
            ),
        },
        {
            "check_name": (
                "failed_terminal_result_preserved"
            ),
            "passed": (
                NOTEBOOK08_TERMINAL_CLASS
                == "FAIL"
            ),
        },
        {
            "check_name": (
                "narrow_score_superiority_preserved"
            ),
            "passed": (
                HAWKES_COUNT_SCORE_SUPERIORITY_AUTHORIZED
                is True
            ),
        },
        {
            "check_name": (
                "full_model_adequacy_remains_rejected"
            ),
            "passed": (
                CALIBRATION_MODEL_ADEQUACY_PASS
                is False
            ),
        },
        {
            "check_name": (
                "notebook09_blocked_handoff_verified"
            ),
            "passed": (
                NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF_PATH.is_file()
            ),
        },
        {
            "check_name": (
                "notebook10_audit_only_handoff_verified"
            ),
            "passed": (
                NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF_PATH.is_file()
            ),
        },
        {
            "check_name": (
                "notebook09_remains_unauthorized"
            ),
            "passed": (
                NOTEBOOK09_AUTHORIZED
                is False
            ),
        },
        {
            "check_name": (
                "validation_remains_unopened"
            ),
            "passed": (
                VALIDATION_PARTITION_OPENING_AUTHORIZED
                is False
                and PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                is False
            ),
        },
        {
            "check_name": (
                "engineering_holdout_remains_unopened"
            ),
            "passed": (
                PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
                is False
            ),
        },
        {
            "check_name": (
                "calibration_parameter_updates_zero"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "check_name": (
                "no_refit_or_repair_authorized"
            ),
            "passed": (
                not FROZEN_HAWKES_SPECIFICATION_DISPOSITION[
                    "parameter_refit_performed"
                ].astype(
                    bool
                ).any()
            ),
        },
        {
            "check_name": (
                "authorized_filesystem_writes_recorded"
            ),
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is True
            ),
        },
    ]
)

require(
    NOTEBOOK08_FINALIZATION_VERIFICATION[
        "passed"
    ].all(),
    "At least one Notebook 08 finalization verification failed.",
)

finalization_verification_record = (
    write_dataframe_artifact(
        variable_name=(
            "NOTEBOOK08_FINALIZATION_VERIFICATION"
        ),
        dataframe=(
            NOTEBOOK08_FINALIZATION_VERIFICATION
        ),
        artifact_role=(
            "finalization_verification"
        ),
        output_directory=(
            NOTEBOOK08_AUDIT_OUTPUT_DIR
        ),
    )
)


# ------------------------------------------------------------
# Build pre-manifest final file registry
# ------------------------------------------------------------

final_additional_records = [
    semantic_readback_csv_record,
    semantic_readback_json_record,
    notebook09_blocked_handoff_record,
    final_disposition_record,
    notebook10_audit_handoff_record,
    finalization_verification_record,
]

NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY = (
    pd.concat(
        [
            NOTEBOOK08_PERSISTED_FILE_REGISTRY.copy(),
            pd.DataFrame.from_records(
                final_additional_records
            ),
        ],
        ignore_index=True,
        sort=False,
    )
    .sort_values(
        [
            "artifact_role",
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY[
        "path"
    ].is_unique,
    "The pre-manifest registry contains duplicate file paths.",
)

require(
    NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY[
        "variable_name"
    ].is_unique,
    "The pre-manifest registry contains duplicate variable names.",
)

require(
    NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY[
        "status"
    ].eq(
        "PASS"
    ).all(),
    "A pre-manifest artifact record did not pass.",
)


# ------------------------------------------------------------
# Persist final manifest table
# ------------------------------------------------------------

final_manifest_table_record = (
    write_dataframe_artifact(
        variable_name=(
            "NOTEBOOK08_FINAL_OUTPUT_MANIFEST_TABLE"
        ),
        dataframe=(
            NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY
        ),
        artifact_role=(
            "final_output_manifest_table"
        ),
        output_directory=(
            NOTEBOOK08_MANIFEST_OUTPUT_DIR
        ),
    )
)

NOTEBOOK08_FINAL_OUTPUT_MANIFEST_CSV_PATH = Path(
    str(
        final_manifest_table_record[
            "path"
        ]
    )
)


# ------------------------------------------------------------
# Persist final self-hashed output manifest
# ------------------------------------------------------------

NOTEBOOK08_FINAL_OUTPUT_MANIFEST = {
    "artifact_type": (
        "NOTEBOOK_08_HAWKES_DIAGNOSTICS_OUTPUT_MANIFEST"
    ),
    "schema_version": (
        "NOTEBOOK_08_HAWKES_DIAGNOSTICS_OUTPUT_MANIFEST_V1"
    ),
    "producer": (
        NOTEBOOK08_NAME
    ),
    "notebook_stage": (
        NOTEBOOK08_STAGE
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        NOTEBOOK08_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "combined_output_prefix": (
        NOTEBOOK08_OUTPUT_PREFIX
    ),
    "selected_model_id": (
        EXPECTED_SELECTED_MODEL_ID
    ),
    "selected_model_family": (
        EXPECTED_SELECTED_MODEL_FAMILY
    ),
    "terminal_class": (
        NOTEBOOK08_TERMINAL_CLASS
    ),
    "terminal_status": (
        NOTEBOOK08_TERMINAL_STATUS
    ),
    "terminal_interpretation": (
        NOTEBOOK08_TERMINAL_INTERPRETATION
    ),
    "count_score_superiority_authorized": True,
    "full_model_adequacy_authorized": False,
    "full_hawkes_superiority_authorized": False,
    "notebook09_authorized": False,
    "notebook09_status": (
        "SKIPPED_BLOCKED_BY_NOTEBOOK08"
    ),
    "validation_partition_opening_authorized": False,
    "engineering_holdout_opening_authorized": False,
    "notebook10_audit_authorized": True,
    "notebook10_authorization_scope": (
        "AUDIT_AND_CLOSE_FAILED_V0_1_DIAGNOSTIC_BRANCH_ONLY"
    ),
    "semantic_readback_payload_sha256": str(
        semantic_readback_json_record[
            "payload_sha256"
        ]
    ),
    "final_disposition_payload_sha256": str(
        final_disposition_record[
            "payload_sha256"
        ]
    ),
    "notebook09_blocked_handoff_payload_sha256": str(
        notebook09_blocked_handoff_record[
            "payload_sha256"
        ]
    ),
    "notebook10_audit_handoff_payload_sha256": str(
        notebook10_audit_handoff_record[
            "payload_sha256"
        ]
    ),
    "non_manifest_file_count": int(
        len(
            NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY
        )
    ),
    "non_manifest_total_byte_count": int(
        NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY[
            "file_size_bytes"
        ].fillna(
            0
        ).astype(
            np.int64
        ).sum()
    ),
    "files": persistence_json_safe(
        NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY.to_dict(
            orient="records"
        )
    ),
    "manifest_table": {
        "path": str(
            NOTEBOOK08_FINAL_OUTPUT_MANIFEST_CSV_PATH.relative_to(
                NOTEBOOK08_V01_ROOT
            )
        ),
        "sha256": str(
            final_manifest_table_record[
                "sha256"
            ]
        ),
        "row_count": int(
            final_manifest_table_record[
                "row_count"
            ]
        ),
        "column_count": int(
            final_manifest_table_record[
                "column_count"
            ]
        ),
    },
    "self_reference_policy": {
        "manifest_json_listed_inside_itself": False,
        "reason": (
            "AVOID_RECURSIVE_SELF_HASH_DEPENDENCY"
        ),
        "manifest_json_verified_after_write": True,
    },
    "parameter_refit_performed": False,
    "repair_performed": False,
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition_name: False
        for partition_name in (
            PROTECTED_PARTITIONS
        )
    },
    "next_operation": (
        "BEGIN_NOTEBOOK10_FINAL_AUDIT_WITH_NOTEBOOK09_SKIPPED"
    ),
    "status": (
        "PASS_COMPLETE_OUTPUT_MANIFEST_FOR_FAILED_DIAGNOSTIC_RESULT"
    ),
}

final_manifest_json_record = (
    write_self_hashed_json_artifact(
        payload=(
            NOTEBOOK08_FINAL_OUTPUT_MANIFEST
        ),
        output_path=(
            NOTEBOOK08_FINAL_OUTPUT_MANIFEST_JSON_PATH
        ),
        artifact_role=(
            "final_output_manifest"
        ),
        variable_name=(
            "NOTEBOOK08_FINAL_OUTPUT_MANIFEST"
        ),
    )
)

final_manifest_loaded = (
    verify_persisted_self_hashed_json(
        final_manifest_json_record
    )
)

require(
    final_manifest_loaded.get(
        "status"
    )
    == (
        "PASS_COMPLETE_OUTPUT_MANIFEST_"
        "FOR_FAILED_DIAGNOSTIC_RESULT"
    ),
    "The final output manifest has an unexpected status.",
)


# ------------------------------------------------------------
# Complete final file registry
# ------------------------------------------------------------

NOTEBOOK08_FINAL_FILE_REGISTRY = (
    pd.concat(
        [
            NOTEBOOK08_PRE_MANIFEST_FILE_REGISTRY,
            pd.DataFrame.from_records(
                [
                    final_manifest_table_record,
                    final_manifest_json_record,
                ]
            ),
        ],
        ignore_index=True,
        sort=False,
    )
    .sort_values(
        [
            "artifact_role",
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NOTEBOOK08_FINAL_FILE_REGISTRY[
        "path"
    ].is_unique,
    "The final Notebook 08 registry contains duplicate paths.",
)

require(
    NOTEBOOK08_FINAL_FILE_REGISTRY[
        "variable_name"
    ].is_unique,
    "The final Notebook 08 registry contains duplicate variable names.",
)

require(
    NOTEBOOK08_FINAL_FILE_REGISTRY[
        "status"
    ].eq(
        "PASS"
    ).all(),
    "A final Notebook 08 artifact record did not pass.",
)


# ------------------------------------------------------------
# Final physical verification of every registered file
# ------------------------------------------------------------

final_registry_verification_records: list[
    dict[str, Any]
] = []

for registry_row in (
    NOTEBOOK08_FINAL_FILE_REGISTRY.itertuples(
        index=False
    )
):
    artifact_path = Path(
        str(
            registry_row.path
        )
    )

    file_exists = artifact_path.is_file()

    observed_sha256 = (
        persistence_file_sha256(
            artifact_path
        )
        if file_exists
        else None
    )

    expected_sha256 = str(
        registry_row.sha256
    )

    sha256_matches = bool(
        file_exists
        and observed_sha256
        == expected_sha256
    )

    final_registry_verification_records.append(
        {
            "artifact_role": str(
                registry_row.artifact_role
            ),
            "variable_name": str(
                registry_row.variable_name
            ),
            "format": str(
                registry_row.format
            ),
            "file_exists": (
                file_exists
            ),
            "expected_sha256": (
                expected_sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "sha256_matches": (
                sha256_matches
            ),
            "path_inside_v0_1": (
                persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V01_ROOT,
                )
            ),
            "path_outside_v0_0": (
                not persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V00_ROOT,
                )
            ),
            "passed": bool(
                file_exists
                and sha256_matches
                and persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V01_ROOT,
                )
                and not persistence_path_is_inside(
                    artifact_path,
                    NOTEBOOK08_V00_ROOT,
                )
            ),
            "status": (
                "PASS"
                if (
                    file_exists
                    and sha256_matches
                )
                else "FAIL"
            ),
        }
    )


NOTEBOOK08_FINAL_FILE_VERIFICATION = (
    pd.DataFrame.from_records(
        final_registry_verification_records
    )
    .sort_values(
        [
            "artifact_role",
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NOTEBOOK08_FINAL_FILE_VERIFICATION[
        "passed"
    ].all(),
    "At least one final Notebook 08 artifact failed verification.",
)


# ------------------------------------------------------------
# Final completion gate ledger
# ------------------------------------------------------------

NOTEBOOK08_FINAL_COMPLETION_GATE_LEDGER = pd.DataFrame(
    [
        {
            "gate_order": 1,
            "gate_id": (
                "CORE_PERSISTENCE_COMPLETE"
            ),
            "passed": (
                NOTEBOOK08_PERSISTENCE_COMPLETED
            ),
            "status": "PASS",
        },
        {
            "gate_order": 2,
            "gate_id": (
                "SEMANTIC_READBACK_COMPLETE"
            ),
            "passed": True,
            "status": "PASS",
        },
        {
            "gate_order": 3,
            "gate_id": (
                "FINAL_DISPOSITION_PERSISTED"
            ),
            "passed": (
                NOTEBOOK08_FINAL_DISPOSITION_PATH.is_file()
            ),
            "status": "PASS",
        },
        {
            "gate_order": 4,
            "gate_id": (
                "NOTEBOOK09_BLOCKED_HANDOFF_PERSISTED"
            ),
            "passed": (
                NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF_PATH.is_file()
            ),
            "status": "PASS",
        },
        {
            "gate_order": 5,
            "gate_id": (
                "NOTEBOOK10_AUDIT_HANDOFF_PERSISTED"
            ),
            "passed": (
                NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF_PATH.is_file()
            ),
            "status": "PASS",
        },
        {
            "gate_order": 6,
            "gate_id": (
                "FINAL_OUTPUT_MANIFEST_PERSISTED"
            ),
            "passed": (
                NOTEBOOK08_FINAL_OUTPUT_MANIFEST_JSON_PATH.is_file()
                and NOTEBOOK08_FINAL_OUTPUT_MANIFEST_CSV_PATH.is_file()
            ),
            "status": "PASS",
        },
        {
            "gate_order": 7,
            "gate_id": (
                "ALL_FINAL_FILES_HASH_VERIFIED"
            ),
            "passed": bool(
                NOTEBOOK08_FINAL_FILE_VERIFICATION[
                    "passed"
                ].all()
            ),
            "status": "PASS",
        },
        {
            "gate_order": 8,
            "gate_id": (
                "NOTEBOOK09_REMAINS_UNAUTHORIZED"
            ),
            "passed": (
                NOTEBOOK09_AUTHORIZED
                is False
            ),
            "status": "PASS",
        },
        {
            "gate_order": 9,
            "gate_id": (
                "VALIDATION_REMAINS_UNOPENED"
            ),
            "passed": (
                PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                is False
            ),
            "status": "PASS",
        },
        {
            "gate_order": 10,
            "gate_id": (
                "ENGINEERING_HOLDOUT_REMAINS_UNOPENED"
            ),
            "passed": (
                PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
                is False
            ),
            "status": "PASS",
        },
        {
            "gate_order": 11,
            "gate_id": (
                "CALIBRATION_PARAMETER_UPDATES_ZERO"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
            "status": "PASS",
        },
        {
            "gate_order": 12,
            "gate_id": (
                "FAILED_DIAGNOSTIC_RESULT_NOT_REPAIRED"
            ),
            "passed": (
                NOTEBOOK08_TERMINAL_CLASS
                == "FAIL"
                and CALIBRATION_MODEL_ADEQUACY_PASS
                is False
            ),
            "status": "PASS",
        },
    ]
)

require(
    NOTEBOOK08_FINAL_COMPLETION_GATE_LEDGER[
        "passed"
    ].all(),
    "At least one Notebook 08 final completion gate failed.",
)


# ------------------------------------------------------------
# Publish final Notebook 08 state
# ------------------------------------------------------------

NOTEBOOK08_READBACK_COMPLETED = True
NOTEBOOK08_FINAL_ACCEPTANCE_CREATED = True
NOTEBOOK08_FINAL_DISPOSITION_CREATED = True
NOTEBOOK08_HANDOFF_CREATED = True
NOTEBOOK08_NOTEBOOK09_BLOCKED_HANDOFF_CREATED = True
NOTEBOOK08_NOTEBOOK10_AUDIT_HANDOFF_CREATED = True
NOTEBOOK08_FINAL_OUTPUT_MANIFEST_COMPLETED = True
NOTEBOOK08_COMPLETED = True

NOTEBOOK08_FINAL_REGISTERED_FILE_COUNT = int(
    len(
        NOTEBOOK08_FINAL_FILE_REGISTRY
    )
)

NOTEBOOK08_NEXT_OPERATION = (
    "BEGIN_NOTEBOOK10_FINAL_AUDIT_WITH_NOTEBOOK09_SKIPPED"
)

NOTEBOOK08_FINAL_AUTHORIZATION_STATE = (
    "NOTEBOOK09_BLOCKED_NOTEBOOK10_AUDIT_ONLY"
)

FILESYSTEM_WRITES_PERFORMED = True


# ------------------------------------------------------------
# Final summaries
# ------------------------------------------------------------

NOTEBOOK08_FINAL_SUMMARY = pd.DataFrame(
    [
        {
            "field": "notebook",
            "value": NOTEBOOK08_NAME,
        },
        {
            "field": "terminal_class",
            "value": NOTEBOOK08_TERMINAL_CLASS,
        },
        {
            "field": "terminal_status",
            "value": NOTEBOOK08_TERMINAL_STATUS,
        },
        {
            "field": (
                "count_score_superiority_authorized"
            ),
            "value": True,
        },
        {
            "field": (
                "full_model_adequacy_authorized"
            ),
            "value": False,
        },
        {
            "field": (
                "full_hawkes_superiority_authorized"
            ),
            "value": False,
        },
        {
            "field": (
                "notebook09_authorized"
            ),
            "value": False,
        },
        {
            "field": (
                "validation_partition_opening_authorized"
            ),
            "value": False,
        },
        {
            "field": (
                "notebook10_audit_authorized"
            ),
            "value": True,
        },
        {
            "field": (
                "final_registered_file_count"
            ),
            "value": (
                NOTEBOOK08_FINAL_REGISTERED_FILE_COUNT
            ),
        },
        {
            "field": (
                "semantic_readback_audit_path"
            ),
            "value": str(
                NOTEBOOK08_SEMANTIC_READBACK_AUDIT_JSON_PATH
            ),
        },
        {
            "field": (
                "final_disposition_path"
            ),
            "value": str(
                NOTEBOOK08_FINAL_DISPOSITION_PATH
            ),
        },
        {
            "field": (
                "final_output_manifest_path"
            ),
            "value": str(
                NOTEBOOK08_FINAL_OUTPUT_MANIFEST_JSON_PATH
            ),
        },
        {
            "field": (
                "notebook09_blocked_handoff_path"
            ),
            "value": str(
                NOTEBOOK08_TO_NOTEBOOK09_BLOCKED_HANDOFF_PATH
            ),
        },
        {
            "field": (
                "notebook10_audit_handoff_path"
            ),
            "value": str(
                NOTEBOOK08_TO_NOTEBOOK10_AUDIT_HANDOFF_PATH
            ),
        },
        {
            "field": (
                "calibration_parameter_updates"
            ),
            "value": 0,
        },
        {
            "field": (
                "protected_partitions_loaded"
            ),
            "value": False,
        },
        {
            "field": (
                "next_operation"
            ),
            "value": (
                NOTEBOOK08_NEXT_OPERATION
            ),
        },
    ]
)


display(
    NOTEBOOK08_SEMANTIC_READBACK_SUMMARY
)

display(
    NOTEBOOK08_FINALIZATION_VERIFICATION
)

display(
    NOTEBOOK08_FINAL_FILE_REGISTRY[
        [
            "artifact_role",
            "variable_name",
            "format",
            "row_count",
            "column_count",
            "file_size_bytes",
            "sha256",
            "status",
        ]
    ]
)

display(
    NOTEBOOK08_FINAL_FILE_VERIFICATION[
        [
            "artifact_role",
            "variable_name",
            "format",
            "file_exists",
            "sha256_matches",
            "path_inside_v0_1",
            "path_outside_v0_0",
            "passed",
            "status",
        ]
    ]
)

display(
    NOTEBOOK08_FINAL_COMPLETION_GATE_LEDGER
)

display(
    NOTEBOOK08_FINAL_SUMMARY
)

print(
    "Notebook 08 is complete. All core artifacts passed semantic "
    "readback and final file-hash verification. The frozen H1 model "
    "retains supported count-score superiority over every authorized "
    "Notebook 06 baseline, but fails the required locked CALIBRATION "
    "model-adequacy gates. Notebook 09 and VALIDATION remain blocked. "
    "Notebook 10 is authorized only to audit and close the failed "
    "V0.1 diagnostic branch. No parameters were refitted or repaired, "
    "CALIBRATION updates remain zero, and VALIDATION and "
    "ENGINEERING_HOLDOUT remain unopened."
)

,field,value
0,core_registered_file_count,73
1,semantic_readback_record_count,73
2,semantic_readback_pass_count,73
3,semantic_readback_failure_count,0
4,core_file_reverification_passed,True
5,terminal_status,FAIL_HAWKES_DIAGNOSTIC_ADEQUACY_DESPITE_SCORE_...
6,count_score_superiority_authorized,True
7,full_model_adequacy_authorized,False
8,notebook09_authorized,False
9,validation_partition_opening_authorized,False


,check_name,passed
0,core_persistence_completed,True
1,all_core_files_physically_reverified,True
2,all_semantic_readback_records_passed,True
3,semantic_readback_csv_path_taken_from_artifact...,True
4,semantic_readback_json_verified,True
5,failed_terminal_result_preserved,True
6,narrow_score_superiority_preserved,True
7,full_model_adequacy_remains_rejected,True
8,notebook09_blocked_handoff_verified,True
9,notebook10_audit_only_handoff_verified,True


,artifact_role,variable_name,format,row_count,column_count,file_size_bytes,sha256,status
0,baseline_comparison_package,NOTEBOOK08_BASELINE_COMPARISON_PACKAGE,JSON,NaN,NaN,11515,75ef8d54ff149678e956c7b75265779240498c344995ba...,PASS
1,batch_multiplicity_distribution,BATCH_MULTIPLICITY_DISTRIBUTION,CSV_GZIP,10,12,530,00f3237303e7c68819ef77beea2f6a5fa398b24d305eb8...,PASS
2,boundary_compensator_reconciliation,BOUNDARY_COMPENSATOR_RECONCILIATION,CSV_GZIP,2,8,244,ba2fbf5f0556853ad52afe32d7468cd31ac77b5722f693...,PASS
3,calibration_envelope_family_summary,CALIBRATION_ENVELOPE_FAMILY_SUMMARY,CSV_GZIP,4,9,315,574ad108b411d26d654897140cefe35eafa40a315accd5...,PASS
4,calibration_envelope_scope_summary,CALIBRATION_ENVELOPE_SCOPE_SUMMARY,CSV_GZIP,8,10,366,bd6f019cbbdfd4e15ca1935aa802f0b5312f52dd7c8a29...,PASS
...,...,...,...,...,...,...,...,...
76,terminal_verification,NOTEBOOK08_TERMINAL_VERIFICATION,CSV_GZIP,11,2,287,93694705e11575bda7c31778279c2b91a4927387baaeac...,PASS
77,time_rescaling_residual_summary,TIME_RESCALING_RESIDUAL_SUMMARY,CSV_GZIP,6,16,762,4b55b4875fad332d8ae0185c87bff4c04b4ad70b709414...,PASS
78,time_rescaling_residuals,TIME_RESCALING_RESIDUALS,CSV_GZIP,"18,549",31,1309891,c95089408af3f082c0cd16611098bfdaa80c5124fdb794...,PASS
79,time_rescaling_right_censoring,TIME_RESCALING_RIGHT_CENSORING_LEDGER,CSV_GZIP,6,10,559,cebc39cca29f11581d8da55eb8a03a473f9965343b7801...,PASS


,artifact_role,variable_name,format,file_exists,sha256_matches,path_inside_v0_1,path_outside_v0_0,passed,status
0,baseline_comparison_package,NOTEBOOK08_BASELINE_COMPARISON_PACKAGE,JSON,True,True,True,True,True,PASS
1,batch_multiplicity_distribution,BATCH_MULTIPLICITY_DISTRIBUTION,CSV_GZIP,True,True,True,True,True,PASS
2,boundary_compensator_reconciliation,BOUNDARY_COMPENSATOR_RECONCILIATION,CSV_GZIP,True,True,True,True,True,PASS
3,calibration_envelope_family_summary,CALIBRATION_ENVELOPE_FAMILY_SUMMARY,CSV_GZIP,True,True,True,True,True,PASS
4,calibration_envelope_scope_summary,CALIBRATION_ENVELOPE_SCOPE_SUMMARY,CSV_GZIP,True,True,True,True,True,PASS
...,...,...,...,...,...,...,...,...,...
76,terminal_verification,NOTEBOOK08_TERMINAL_VERIFICATION,CSV_GZIP,True,True,True,True,True,PASS
77,time_rescaling_residual_summary,TIME_RESCALING_RESIDUAL_SUMMARY,CSV_GZIP,True,True,True,True,True,PASS
78,time_rescaling_residuals,TIME_RESCALING_RESIDUALS,CSV_GZIP,True,True,True,True,True,PASS
79,time_rescaling_right_censoring,TIME_RESCALING_RIGHT_CENSORING_LEDGER,CSV_GZIP,True,True,True,True,True,PASS


,gate_order,gate_id,passed,status
0,1,CORE_PERSISTENCE_COMPLETE,True,PASS
1,2,SEMANTIC_READBACK_COMPLETE,True,PASS
2,3,FINAL_DISPOSITION_PERSISTED,True,PASS
3,4,NOTEBOOK09_BLOCKED_HANDOFF_PERSISTED,True,PASS
4,5,NOTEBOOK10_AUDIT_HANDOFF_PERSISTED,True,PASS
5,6,FINAL_OUTPUT_MANIFEST_PERSISTED,True,PASS
6,7,ALL_FINAL_FILES_HASH_VERIFIED,True,PASS
7,8,NOTEBOOK09_REMAINS_UNAUTHORIZED,True,PASS
8,9,VALIDATION_REMAINS_UNOPENED,True,PASS
9,10,ENGINEERING_HOLDOUT_REMAINS_UNOPENED,True,PASS


,field,value
0,notebook,08_HAWKES_DIAGNOSTICS.ipynb
1,terminal_class,FAIL
2,terminal_status,FAIL_HAWKES_DIAGNOSTIC_ADEQUACY_DESPITE_SCORE_...
3,count_score_superiority_authorized,True
4,full_model_adequacy_authorized,False
5,full_hawkes_superiority_authorized,False
6,notebook09_authorized,False
7,validation_partition_opening_authorized,False
8,notebook10_audit_authorized,True
9,final_registered_file_count,81


Notebook 08 is complete. All core artifacts passed semantic readback and final file-hash verification. The frozen H1 model retains supported count-score superiority over every authorized Notebook 06 baseline, but fails the required locked CALIBRATION model-adequacy gates. Notebook 09 and VALIDATION remain blocked. Notebook 10 is authorized only to audit and close the failed V0.1 diagnostic branch. No parameters were refitted or repaired, CALIBRATION updates remain zero, and VALIDATION and ENGINEERING_HOLDOUT remain unopened.
